# PyGGUF: 100% Python + PyTorch + Triton GGUF OpenAI Server on Kaggle (`0% llama.cpp`)

**Key Capabilities (Built for Kaggle `GPU T4 x2` — 30 GB VRAM + 640 GB/s Combined Bandwidth + 1.1 TB `/kaggle/tmp`):**
- **Zero C++ / Zero `llama.cpp`**: Pure Python `mmap` GGUF v2/v3 parser + **25 `@triton.jit` GPU kernels** (warp-per-row `int32`-coalesced `W4A16`/`W8A16` GEMV, weight-reusing multi-row `M=2..8` verify GEMV, Turing `sm_75` cuBLAS Tensor-Core prefill, `INT4` & `INT8` Split-KV Flash-Decoding + Fused Combine, Fused Residual-Add + RMSNorm, Fused SwiGLU+RowScale, and 2-launch Gated-DeltaNet `S=1` Decode & Multi-Row `2 <= S <= 16` Verify).
- **256K+ Long-Context `INT4` KV Cache + Sequential Peak-VRAM-Safe Growth**:
  - **`kv_dtype="int4"` + Triton `INT4` Split-KV Flash-Decoding**: Cuts KV cache VRAM & GDDR6 decode bandwidth by **~75% vs FP16** and **~50% vs INT8** (`256,000` tokens on `Qwen3.8-27B` takes only **~2.1 GiB total** across Dual T4!) while sequential single-buffer growth eliminates 2x reallocation VRAM spikes.
  - **GQA-native short/medium attention** (`SDPA enable_gqa`) and **GQA-folded chunked online-softmax** for big batches/softcap — K/V are never expanded, with FP32 softmax bookkeeping over tensor-core matmuls.
  - **YaRN / Linear / NTK-aware RoPE context extension**: Either the checkpoint's GGUF `{arch}.rope.scaling.*` metadata (llama.cpp-compatible keys, used automatically) or launch kwargs (`rope_scaling_type="yarn", rope_scaling_factor=4.0, rope_scaling_original_context_length=32768`) extend a checkpoint's trained window up to `max_context_len` (YaRN mscale attention factor applied automatically, incl. the draft-mtp head).
- **Turbo Decode + Adaptive Native GGUF `draft-mtp` Speculative Engine**:
  - **Native Embedded GGUF `draft-mtp` (`blk.{L}.nextn.*`) + Ring KV (`mtp_kv_window=4096`) + Confidence-Gated Adaptive Depth (`k = 1..3`)**: Caps private MTP KV memory to a `4096`-token ring buffer (`~4 MiB` even at 256K context, slicing long prompt prefills to the trailing `4096` tokens in `< 3 ms`) and dynamically chains `k = 1..3` draft tokens when GPU softmax confidence `>= 0.55` (with 3-level enable/disable controls: load-time `speculative_mtp`, CLI `--no-mtp`, and live per-request `extra_body={"speculative_mtp": False}`).
  - **Fused Multi-Row (`2 <= S <= 16`) Gated-DeltaNet Verify + Zero-Recompute Rollback**: Collapses ~864 eager PyTorch ops per verify step across 48 DeltaNet layers into 96 register-resident Triton launches and settles rejected states in `~0.2 ms` without re-forwarding trunk layers.
  - **2-GPU Tensor Parallelism (`TP=2`) & `CUDAGraph`**: Streams weights from **both** Tesla T4 GPUs simultaneously (`2 × 320 = 640 GB/s`) for Dense & Hybrid `Qwen3.8-27B` / `Qwen3` models, plus per-stage **`torch.cuda.CUDAGraph`** replay for `PP` / single-GPU decode.
- **Full Agent-Harness Support (OpenCode, Claude Code, Cline, Roo Code, Continue, Cursor, Aider, OpenAI SDK)**:
  - **Zero `<think></think>` Leakage**: Separates `<think>...</think>` into `reasoning_content` so `content` stays 100% clean.
  - **Native Tool Calling (`tools` / `tool_choice` / `role="tool"` / `role="developer"`)**: Hardened Jinja2 templates + dual XML (`<function=...><parameter=...>`) & JSON (`{"name": ..., "arguments": ...}`) tool-call parser with JSON-Schema type coercion.


In [ ]:
# 1. Install lightweight Python web/hub packages (PyTorch + OpenAI Triton are pre-installed on Kaggle!)
!pip install -q fastapi uvicorn openai gguf "huggingface_hub[hf_transfer]" tokenizers jinja2
!mkdir -p pygguf_engine


In [ ]:
%%writefile pygguf_engine/__init__.py
"""
pygguf_engine
=============
100% Python + PyTorch + OpenAI Triton (`@triton.jit`) GGUF Inference Engine & OpenAI Server.
Zero `llama.cpp`, zero C++ compilation — supports both Dense and Sparse MoE GGUF models,
INT8 Quantized KV Cache for 32K-128K+ context, Dual-GPU sharding on Kaggle T4 x2,
Prefix KV Cache reuse, N-gram Self-Speculative Decoding, and Cloudflare HTTPS Tunnels.
"""

from .agentic_protocol import (
    QWEN3_AGENTIC_JINJA_TEMPLATE,
    QWEN35_AGENTIC_JINJA_TEMPLATE,
    ParsedToolCall,
    ReasoningAndToolStreamParser,
)
from .engine import GenerationResult, PyGGUFEngine, SamplingConfig, StreamChunk
from .gguf_parser import GGUFFile, GGUFModelConfig, GGUFTokenizer
from .openai_server import CloudflareTunnel, create_openai_app

__all__ = [
    "CloudflareTunnel",
    "GGUFFile",
    "GGUFModelConfig",
    "GGUFTokenizer",
    "GenerationResult",
    "ParsedToolCall",
    "PyGGUFEngine",
    "QWEN35_AGENTIC_JINJA_TEMPLATE",
    "QWEN3_AGENTIC_JINJA_TEMPLATE",
    "ReasoningAndToolStreamParser",
    "SamplingConfig",
    "StreamChunk",
    "create_openai_app",
]


In [ ]:
%%writefile pygguf_engine/agentic_protocol.py
"""
Agentic Chat Template, Reasoning (`<think>...</think>`), and Tool-Call Protocol Engine
for `PyGGUFEngine` OpenAI-Compatible Serving (`0% llama.cpp`).

Solves the two classic agent-harness pain points on Qwen3 / Qwen3.5 / Qwen3.6 / Qwen3.8 / Llama / DeepSeek GGUFs:
1. `<think>...</think>` leakage:
   - Incremental state-machine parser strips `<think>...</think>` (including pre-opened `<think>` Prompts
     on `Qwen3.5` / `Qwen3.8`, empty `<think></think>` blocks, and orphan `</think>` tags) out of `content`
     and routes non-empty thoughts cleanly to `reasoning_content` (`delta.reasoning_content` in SSE).
2. Multi-Turn Tool Calling across Coding & Agent Harnesses (OpenCode, Claude Code, Cline, Roo Code,
   Continue, Cursor, Aider, OpenAI Agents SDK, LangGraph, Qwen-Agent):
   - Built-in hardened Jinja2 templates supporting `developer` role, mid-conversation `system` messages,
     `tool` responses, and string-or-dict `tool_calls.function.arguments` without Jinja crashes.
   - Dual-protocol `<tool_call>` output parser supporting BOTH:
     * Qwen3.5 / Qwen3.8 / Qwen3-Coder XML (`<function=name><parameter=k>v</parameter></function>`)
       with JSON-Schema-aware type coercion (`int`, `float`, `bool`, `dict`, `list`, `str`) and safe
       handling of `<`, `>`, `&` inside code parameters!
     * Qwen3 / Qwen2.5 / Hermes / Llama JSON (`{"name": ..., "arguments": ...}`).
"""

from __future__ import annotations

import json
import re
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Sequence, Tuple, Union


# ==============================================================================
# 1. Hardened Agentic Jinja2 Templates (Qwen3.5/3.8 XML & Qwen3 JSON)
# ==============================================================================

# Battle-tested Qwen3.5 / Qwen3.6 / Qwen3.8 (`qwen35`, `qwen35moe`, `qwen3next`) Jinja template:
# - Supports `developer` role and mid-conversation `system` messages
# - Supports system-only or tool-only turns without raising exceptions
# - Normalizes `tool_call.arguments` whether mapping or JSON string
# - Never injects empty `<think></think>` into past history (preserves 100% KV cache prefix hits!)
# - Enforces `</think>` closure BEFORE `<tool_call>`
QWEN35_AGENTIC_JINJA_TEMPLATE = r"""{%- macro render_content(content) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if item is string %}
                {{- item }}
            {%- elif item is mapping and 'text' in item %}
                {{- item.text }}
            {%- endif %}
        {%- endfor %}
    {%- elif content is none or content is undefined %}
        {{- '' }}
    {%- else %}
        {{- content | string }}
    {%- endif %}
{%- endmacro %}
{%- set ns_flags = namespace(enable_thinking=true) %}
{%- if enable_thinking is defined %}
    {%- set ns_flags.enable_thinking = enable_thinking %}
{%- endif %}
{%- set system_content = '' %}
{%- set has_first_system = false %}
{%- if messages and (messages[0].role == 'system' or messages[0].role == 'developer') %}
    {%- set has_first_system = true %}
    {%- set system_content = render_content(messages[0].content) | trim %}
    {%- if '<|think_off|>' in system_content or '/no_think' in system_content %}
        {%- if enable_thinking is not defined %}
            {%- set ns_flags.enable_thinking = false %}
        {%- endif %}
        {%- set system_content = system_content | replace('<|think_off|>', '') | trim %}
    {%- endif %}
    {%- if '<|think_on|>' in system_content %}
        {%- set ns_flags.enable_thinking = true %}
        {%- set system_content = system_content | replace('<|think_on|>', '') | trim %}
    {%- endif %}
{%- endif %}
{%- if tools and tools is iterable and tools is not mapping %}
    {{- '<|im_start|>system\n' }}
    {{- '# Tools\n\nYou have access to the following functions:\n\n<tools>' }}
    {%- for tool in tools %}
        {{- '\n' }}
        {%- set fn = tool.function if (tool is mapping and tool.function is defined) else tool %}
        {{- fn | tojson }}
    {%- endfor %}
    {{- '\n</tools>\n\nIf you choose to call a function ONLY reply in the following format with NO suffix:\n\n' }}
    {%- if ns_flags.enable_thinking %}
        {{- '<think>\nBrief reasoning and plan\n</think>\n' }}
    {%- endif %}
    {{- '<tool_call>\n<function=example_function_name>\n<parameter=example_parameter_1>\nvalue_1\n</parameter>\n<parameter=example_parameter_2>\nThis is the value for the second parameter\nthat can span\nmultiple lines\n</parameter>\n</function>\n</tool_call>\n\n<IMPORTANT>\nReminder:\n- Function calls MUST follow the specified format: an inner <function=...></function> block nested within <tool_call></tool_call> XML tags.\n- Required parameters MUST be specified.\n- Place internal reasoning inside <think></think> and close </think> BEFORE outputting <tool_call>.\n- The <tool_call> and <function=...> tags MUST start at the beginning of a new line with NO indentation.\n- To call multiple functions, emit a separate closed <tool_call></tool_call> block for each function.\n- If no tool call is needed, answer the user normally right after </think>.\n</IMPORTANT>' }}
    {%- if has_first_system and system_content %}
        {{- '\n\n' + system_content }}
    {%- endif %}
    {{- '<|im_end|>\n' }}
{%- elif has_first_system and system_content %}
    {{- '<|im_start|>system\n' + system_content + '<|im_end|>\n' }}
{%- endif %}
{%- for message in messages %}
    {%- set role = message.role %}
    {%- set is_sys = (role == 'system' or role == 'developer') %}
    {%- set content = render_content(message.content) | trim %}
    {%- if '<|think_off|>' in content %}
        {%- set ns_flags.enable_thinking = false %}
        {%- set content = content | replace('<|think_off|>', '') | trim %}
    {%- elif '<|think_on|>' in content %}
        {%- set ns_flags.enable_thinking = true %}
        {%- set content = content | replace('<|think_on|>', '') | trim %}
    {%- endif %}
    {%- if is_sys %}
        {%- if not loop.first and content %}
            {{- '<|im_start|>system\n' + content + '<|im_end|>\n' }}
        {%- endif %}
    {%- elif role == 'user' %}
        {{- '<|im_start|>user\n' + content + '<|im_end|>\n' }}
    {%- elif role == 'assistant' %}
        {%- set reasoning_content = '' %}
        {%- if message.reasoning_content is defined and message.reasoning_content is string %}
            {%- set reasoning_content = message.reasoning_content | trim %}
        {%- elif '</think>' in content %}
            {%- set reasoning_content = content.split('</think>')[0].split('<think>')[-1] | trim %}
            {%- set content = content.split('</think>')[-1] | trim %}
        {%- endif %}
        {%- if message.tool_calls and '<tool_call>' in content %}
            {%- set content = content.split('<tool_call>')[0] | trim %}
        {%- endif %}
        {{- '<|im_start|>assistant' }}
        {%- if reasoning_content and ns_flags.enable_thinking %}
            {{- '\n<think>\n' + reasoning_content + '\n</think>' }}
        {%- endif %}
        {%- if content %}
            {{- '\n' + content }}
        {%- elif not message.tool_calls %}
            {{- '\n' }}
        {%- endif %}
        {%- if message.tool_calls and message.tool_calls is iterable and message.tool_calls is not mapping %}
            {%- for tc in message.tool_calls %}
                {%- set fn = tc.function if (tc is mapping and tc.function is defined) else tc %}
                {{- '\n<tool_call>\n<function=' ~ fn.name ~ '>\n' }}
                {%- if fn.arguments is defined and fn.arguments is mapping %}
                    {%- for k in fn.arguments %}
                        {%- set v = fn.arguments[k] %}
                        {{- '<parameter=' ~ k ~ '>\n' }}
                        {%- if v is mapping or (v is iterable and v is not string) %}
                            {{- v | tojson }}
                        {%- else %}
                            {{- v | string }}
                        {%- endif %}
                        {{- '\n</parameter>\n' }}
                    {%- endfor %}
                {%- elif fn.arguments is defined and fn.arguments is string and (fn.arguments | trim) %}
                    {{- (fn.arguments | trim) ~ '\n' }}
                {%- endif %}
                {{- '</function>\n</tool_call>' }}
            {%- endfor %}
        {%- endif %}
        {{- '<|im_end|>\n' }}
    {%- elif role == 'tool' %}
        {%- if loop.first or messages[loop.index0 - 1].role != 'tool' %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' + content + '\n</tool_response>' }}
        {%- if loop.last or messages[loop.index0 + 1].role != 'tool' %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- else %}
        {{- '<|im_start|>' + role + '\n' + content + '<|im_end|>\n' }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
    {%- if ns_flags.enable_thinking is false %}
        {{- '<think>\n\n</think>\n\n' }}
    {%- else %}
        {{- '<think>\n' }}
    {%- endif %}
{%- endif %}"""


# Hardened Qwen3 / Qwen3-MoE / Qwen2.5 JSON tool-calling Jinja template
QWEN3_AGENTIC_JINJA_TEMPLATE = r"""{%- macro render_content(content) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if item is string %}
                {{- item }}
            {%- elif item is mapping and 'text' in item %}
                {{- item.text }}
            {%- endif %}
        {%- endfor %}
    {%- elif content is none or content is undefined %}
        {{- '' }}
    {%- else %}
        {{- content | string }}
    {%- endif %}
{%- endmacro %}
{%- set has_first_system = messages and (messages[0].role == 'system' or messages[0].role == 'developer') %}
{%- set first_sys = render_content(messages[0].content) | trim if has_first_system else '' %}
{%- if tools and tools is iterable and tools is not mapping %}
    {{- '<|im_start|>system\n' }}
    {%- if first_sys %}
        {{- first_sys + '\n\n' }}
    {%- endif %}
    {{- '# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>' }}
    {%- for tool in tools %}
        {{- '\n' + (tool | tojson) }}
    {%- endfor %}
    {{- '\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n' }}
{%- elif has_first_system and first_sys %}
    {{- '<|im_start|>system\n' + first_sys + '<|im_end|>\n' }}
{%- endif %}
{%- for message in messages %}
    {%- set role = message.role %}
    {%- set is_sys = (role == 'system' or role == 'developer') %}
    {%- set content = render_content(message.content) | trim %}
    {%- if is_sys %}
        {%- if not loop.first and content %}
            {{- '<|im_start|>system\n' + content + '<|im_end|>\n' }}
        {%- endif %}
    {%- elif role == 'user' %}
        {{- '<|im_start|>user\n' + content + '<|im_end|>\n' }}
    {%- elif role == 'assistant' %}
        {%- set reasoning_content = '' %}
        {%- if message.reasoning_content is defined and message.reasoning_content is string %}
            {%- set reasoning_content = message.reasoning_content | trim %}
        {%- elif '</think>' in content %}
            {%- set reasoning_content = content.split('</think>')[0].split('<think>')[-1] | trim %}
            {%- set content = content.split('</think>')[-1] | trim %}
        {%- endif %}
        {{- '<|im_start|>assistant\n' }}
        {%- if reasoning_content %}
            {{- '<think>\n' + reasoning_content + '\n</think>\n\n' }}
        {%- endif %}
        {{- content }}
        {%- if message.tool_calls and message.tool_calls is iterable and message.tool_calls is not mapping %}
            {%- for tc in message.tool_calls %}
                {%- if (loop.first and content) or (not loop.first) %}
                    {{- '\n' }}
                {%- endif %}
                {%- set fn = tc.function if (tc is mapping and tc.function is defined) else tc %}
                {%- set args_json = fn.arguments if fn.arguments is string else (fn.arguments | tojson) %}
                {{- '<tool_call>\n{"name": "' ~ fn.name ~ '", "arguments": ' ~ args_json ~ '}\n</tool_call>' }}
            {%- endfor %}
        {%- endif %}
        {{- '<|im_end|>\n' }}
    {%- elif role == 'tool' %}
        {%- if loop.first or messages[loop.index0 - 1].role != 'tool' %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' + content + '\n</tool_response>' }}
        {%- if loop.last or messages[loop.index0 + 1].role != 'tool' %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- else %}
        {{- '<|im_start|>' + role + '\n' + content + '<|im_end|>\n' }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
    {%- if enable_thinking is defined and enable_thinking is false %}
        {{- '<think>\n\n</think>\n\n' }}
    {%- endif %}
{%- endif %}"""


def normalize_messages_for_template(messages: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Normalize OpenAI-format messages before Jinja rendering so NO template crashes on:
    - `role="developer"` (used by OpenCode, Codex CLI, OpenAI o-series)
    - `content=None` (sent on assistant tool_calls messages)
    - `tool_calls[*].function.arguments` passed as a JSON string when Jinja uses `|items`!
    """
    normalized: List[Dict[str, Any]] = []
    for raw_msg in messages:
        msg = dict(raw_msg)
        role = str(msg.get("role", "user"))
        if role == "developer":
            msg["role"] = "system"

        # Normalize content (handle None and OpenAI content-part arrays)
        content = msg.get("content")
        if content is None:
            msg["content"] = ""
        elif isinstance(content, list):
            parts: List[str] = []
            for item in content:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, dict) and "text" in item:
                    parts.append(str(item.get("text", "")))
            msg["content"] = "\n".join(parts)

        # Normalize assistant tool_calls so `function.arguments` is a parsed dict when valid JSON
        raw_tcs = msg.get("tool_calls")
        if isinstance(raw_tcs, list) and raw_tcs:
            clean_tcs: List[Dict[str, Any]] = []
            for tc in raw_tcs:
                if not isinstance(tc, dict):
                    continue
                tc_copy = dict(tc)
                fn = tc_copy.get("function")
                if isinstance(fn, dict):
                    fn_copy = dict(fn)
                    args = fn_copy.get("arguments", {})
                    if isinstance(args, str):
                        args_str = args.strip()
                        if args_str:
                            try:
                                parsed = json.loads(args_str)
                                if isinstance(parsed, dict):
                                    fn_copy["arguments"] = parsed
                            except Exception:
                                pass
                        else:
                            fn_copy["arguments"] = {}
                    elif args is None:
                        fn_copy["arguments"] = {}
                    tc_copy["function"] = fn_copy
                clean_tcs.append(tc_copy)
            msg["tool_calls"] = clean_tcs

        normalized.append(msg)
    return normalized


def apply_tool_choice_directive(
    messages: List[Dict[str, Any]],
    tools: Optional[Sequence[Dict[str, Any]]],
    tool_choice: Any,
) -> Tuple[List[Dict[str, Any]], Optional[List[Dict[str, Any]]]]:
    """Handle OpenAI `tool_choice` (`"none"`, `"auto"`, `"required"`, or specific function dict)."""
    if not tools or tool_choice == "none":
        return messages, None

    tool_list = list(tools)
    directive: Optional[str] = None
    if tool_choice == "required":
        directive = "IMPORTANT: You MUST call at least one tool from <tools> in this response using <tool_call>."
    elif isinstance(tool_choice, dict):
        fn_name = (
            tool_choice.get("function", {}).get("name")
            if isinstance(tool_choice.get("function"), dict)
            else tool_choice.get("name")
        )
        if fn_name:
            tool_list = [
                t
                for t in tool_list
                if (t.get("function", {}).get("name") if isinstance(t.get("function"), dict) else t.get("name"))
                == fn_name
            ] or tool_list
            directive = f'IMPORTANT: You MUST call the function "{fn_name}" in this response using <tool_call>.'

    if directive:
        msgs = [dict(m) for m in messages]
        if msgs and msgs[0].get("role") == "system":
            msgs[0]["content"] = f"{msgs[0].get('content', '')}\n\n{directive}".strip()
        else:
            msgs.insert(0, {"role": "system", "content": directive})
        return msgs, tool_list

    return messages, tool_list


def resolve_enable_thinking(
    messages: Sequence[Dict[str, Any]],
    explicit_enable_thinking: Optional[bool] = None,
    chat_template_kwargs: Optional[Dict[str, Any]] = None,
    reasoning_effort: Optional[str] = None,
    thinking_param: Optional[Dict[str, Any]] = None,
    default_enable_thinking: bool = True,
) -> bool:
    """
    Unified reasoning-mode resolver supporting OpenAI, vLLM, Anthropic/OpenCode, and Qwen conventions:
    1. Explicit `enable_thinking: bool`
    2. `chat_template_kwargs: {"enable_thinking": bool}`
    3. `thinking: {"type": "disabled" | "enabled"}` (Claude Code / OpenCode / LiteLLM)
    4. `reasoning_effort: "none"` -> False; `"low" | "medium" | "high"` -> True (OpenAI o-series)
    5. Inline `/no_think`, `/think`, `<|think_off|>`, `<|think_on|>` tags in system/user messages
    6. Server-wide `default_enable_thinking`
    """
    if explicit_enable_thinking is not None:
        return bool(explicit_enable_thinking)
    if chat_template_kwargs and "enable_thinking" in chat_template_kwargs:
        return bool(chat_template_kwargs["enable_thinking"])
    if isinstance(thinking_param, dict) and "type" in thinking_param:
        t_type = str(thinking_param["type"]).lower()
        if t_type in ("disabled", "off", "none", "false"):
            return False
        if t_type in ("enabled", "on", "true", "adaptive"):
            return True
    if reasoning_effort is not None:
        eff = str(reasoning_effort).strip().lower()
        if eff in ("none", "off", "disabled", "false", "0"):
            return False
        if eff in ("low", "medium", "high", "minimal", "max"):
            return True

    # Check inline directives in messages (latest directive wins)
    state: Optional[bool] = None
    for m in messages:
        if m.get("role") in ("system", "developer", "user"):
            txt = str(m.get("content", ""))
            if "/no_think" in txt or "<|think_off|>" in txt:
                state = False
            if "/think" in txt or "<|think_on|>" in txt:
                # Ensure `/no_think` didn't falsely match `/think`
                cleaned = txt.replace("/no_think", "")
                if "/think" in cleaned or "<|think_on|>" in cleaned:
                    state = True
    if state is not None:
        return state

    return default_enable_thinking


# ==============================================================================
# 2. Schema-Aware Dual XML (`qwen3_xml`) + JSON (`qwen3_json`) Tool Call Parser
# ==============================================================================

@dataclass
class ParsedToolCall:
    id: str
    name: str
    arguments_json: str

    def to_openai_dict(self) -> Dict[str, Any]:
        return {
            "id": self.id,
            "type": "function",
            "function": {
                "name": self.name,
                "arguments": self.arguments_json,
            },
        }

    def to_openai_stream_delta(self, index: int) -> Dict[str, Any]:
        return {
            "index": index,
            "id": self.id,
            "type": "function",
            "function": {
                "name": self.name,
                "arguments": self.arguments_json,
            },
        }


def _build_tool_param_schema_map(
    tools: Optional[Sequence[Dict[str, Any]]],
) -> Dict[str, Dict[str, Any]]:
    """Map `func_name -> {param_name -> json_schema_dict}` for accurate XML value type coercion."""
    schema_map: Dict[str, Dict[str, Any]] = {}
    if not tools:
        return schema_map
    for t in tools:
        if not isinstance(t, dict):
            continue
        fn = t.get("function") if isinstance(t.get("function"), dict) else t
        if not isinstance(fn, dict):
            continue
        name = fn.get("name")
        params = fn.get("parameters")
        if isinstance(name, str) and isinstance(params, dict):
            props = params.get("properties")
            if isinstance(props, dict):
                schema_map[name] = {str(k): (v if isinstance(v, dict) else {}) for k, v in props.items()}
            else:
                schema_map[name] = {}
    return schema_map


def _extract_schema_types(prop_schema: Dict[str, Any]) -> List[str]:
    """Extract allowed JSON schema types (handling `type: "integer"`, `type: ["integer", "null"]`, `anyOf`, `oneOf`)."""
    types: List[str] = []
    t = prop_schema.get("type")
    if isinstance(t, str):
        types.append(t)
    elif isinstance(t, list):
        types.extend(str(x) for x in t)
    for key in ("anyOf", "oneOf"):
        sub = prop_schema.get(key)
        if isinstance(sub, list):
            for item in sub:
                if isinstance(item, dict):
                    types.extend(_extract_schema_types(item))
    return types


def _coerce_xml_param_value(raw_val: str, prop_schema: Optional[Dict[str, Any]]) -> Any:
    """
    Coerce a raw XML parameter string to the expected JSON Schema type WITHOUT corrupting
    multi-line source code or strings containing `<`, `>`, `&`, or JSON snippets!
    """
    allowed = _extract_schema_types(prop_schema) if prop_schema else []
    stripped = raw_val.strip()

    # If schema explicitly says ONLY string, preserve exact string (do NOT turn "123" or "{}" into int/dict!)
    if allowed and all(t in ("string", "null") for t in allowed):
        if "null" in allowed and stripped.lower() == "null":
            return None
        return raw_val

    if stripped.lower() == "null" and (not allowed or "null" in allowed):
        return None

    if not allowed or "boolean" in allowed:
        if stripped.lower() == "true" and ("boolean" in allowed or not allowed):
            if "string" not in allowed:
                return True
        if stripped.lower() == "false" and ("boolean" in allowed or not allowed):
            if "string" not in allowed:
                return False

    if "integer" in allowed:
        try:
            return int(stripped)
        except ValueError:
            pass

    if "number" in allowed:
        try:
            return float(stripped)
        except ValueError:
            pass

    if "object" in allowed or "array" in allowed:
        try:
            return json.loads(stripped)
        except Exception:
            pass

    # Schema-less fallback: parse obvious JSON objects/arrays, booleans, or numbers if unambiguous
    if not allowed:
        if (stripped.startswith("{") and stripped.endswith("}")) or (
            stripped.startswith("[") and stripped.endswith("]")
        ):
            try:
                return json.loads(stripped)
            except Exception:
                pass
        if stripped.lower() == "true":
            return True
        if stripped.lower() == "false":
            return False
        if re.fullmatch(r"-?(?:0|[1-9]\d*)", stripped):
            try:
                return int(stripped)
            except ValueError:
                pass
        if re.fullmatch(r"-?(?:0|[1-9]\d*)\.\d+", stripped):
            try:
                return float(stripped)
            except ValueError:
                pass

    return raw_val


_FUNC_HEADER_RE = re.compile(r"<function\s*=\s*([^>\s]+)\s*>", re.DOTALL)
_PARAM_OPEN_RE = re.compile(r"<parameter\s*=\s*([^>\s]+)\s*>", re.DOTALL)


def parse_single_tool_call_block(
    block_body: str,
    schema_map: Optional[Dict[str, Dict[str, Any]]] = None,
) -> List[ParsedToolCall]:
    """
    Parse the interior of a `<tool_call> ... </tool_call>` block.
    Supports:
      1. Qwen3.5 / Qwen3.8 / Qwen3-Coder XML: `<function=name><parameter=k>v</parameter></function>`
         (Uses exact `<parameter=...>` ... `</parameter>` boundary slicing so `<` and `>` inside code NEVER break!)
      2. Qwen3 / Qwen2.5 / Hermes JSON: `{"name": "fn", "arguments": {...}}` (or JSON list of calls)
    """
    text = block_body.strip()
    if not text:
        return []

    results: List[ParsedToolCall] = []

    # --- Format 1: XML `<function=NAME>...</function>` ---
    fn_matches = list(_FUNC_HEADER_RE.finditer(text))
    if fn_matches:
        for idx, fm in enumerate(fn_matches):
            fn_name = fm.group(1).strip().strip("\"'")
            body_start = fm.end()
            next_fn_start = fn_matches[idx + 1].start() if idx + 1 < len(fn_matches) else len(text)
            fn_chunk = text[body_start:next_fn_start]
            end_fn_pos = fn_chunk.rfind("</function>")
            if end_fn_pos != -1:
                fn_chunk = fn_chunk[:end_fn_pos]

            fn_props = (schema_map or {}).get(fn_name, {})
            args_dict: Dict[str, Any] = {}

            # Scan `<parameter=KEY>` ... `</parameter>` sequentially (safe for `<` / `>` inside values!)
            p_opens = list(_PARAM_OPEN_RE.finditer(fn_chunk))
            if p_opens:
                for p_idx, pm in enumerate(p_opens):
                    p_name = pm.group(1).strip().strip("\"'")
                    val_start = pm.end()
                    val_limit = p_opens[p_idx + 1].start() if p_idx + 1 < len(p_opens) else len(fn_chunk)
                    raw_segment = fn_chunk[val_start:val_limit]
                    close_pos = raw_segment.rfind("</parameter>")
                    if close_pos != -1:
                        raw_val = raw_segment[:close_pos]
                    else:
                        raw_val = raw_segment
                    # Strip the single formatting newline right after `<parameter=k>\n` and right before `\n</parameter>`
                    if raw_val.startswith("\r\n"):
                        raw_val = raw_val[2:]
                    elif raw_val.startswith("\n"):
                        raw_val = raw_val[1:]
                    if raw_val.endswith("\r\n"):
                        raw_val = raw_val[:-2]
                    elif raw_val.endswith("\n"):
                        raw_val = raw_val[:-1]

                    args_dict[p_name] = _coerce_xml_param_value(raw_val, fn_props.get(p_name))
            else:
                # Fallback if model put raw JSON inside `<function=name>{...}</function>`
                inner = fn_chunk.strip()
                if inner.startswith("{"):
                    try:
                        parsed_inner = json.loads(inner)
                        if isinstance(parsed_inner, dict):
                            args_dict = parsed_inner
                    except Exception:
                        pass

            results.append(
                ParsedToolCall(
                    id=f"call_{uuid.uuid4().hex[:24]}",
                    name=fn_name,
                    arguments_json=json.dumps(args_dict, ensure_ascii=False),
                )
            )
        return results

    # --- Format 2: JSON `{"name": ..., "arguments": ...}` (or `[...]`) ---
    # Strip optional markdown fences if a smaller model wrapped JSON in ```json ... ```
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text).strip()

    candidates: List[Any] = []
    try:
        loaded = json.loads(text)
        candidates = loaded if isinstance(loaded, list) else [loaded]
    except Exception:
        # Try line-by-line or brace-balanced extraction
        decoder = json.JSONDecoder()
        pos = 0
        while pos < len(text):
            brace = text.find("{", pos)
            if brace == -1:
                break
            try:
                obj, end_idx = decoder.raw_decode(text, brace)
                candidates.append(obj)
                pos = end_idx
            except Exception:
                pos = brace + 1

    for item in candidates:
        if not isinstance(item, dict):
            continue
        fn_obj = item.get("function") if isinstance(item.get("function"), dict) else item
        name = fn_obj.get("name") or fn_obj.get("tool")
        if not isinstance(name, str) or not name.strip():
            continue
        args = fn_obj.get("arguments", fn_obj.get("parameters", {}))
        if isinstance(args, str):
            try:
                json.loads(args)
                args_json = args
            except Exception:
                args_json = json.dumps({"raw": args}, ensure_ascii=False)
        else:
            args_json = json.dumps(args if args is not None else {}, ensure_ascii=False)
        results.append(
            ParsedToolCall(
                id=f"call_{uuid.uuid4().hex[:24]}",
                name=name.strip(),
                arguments_json=args_json,
            )
        )
    return results


# ==============================================================================
# 3. Incremental Streaming & Non-Streaming Reasoning + Tool State Machine
# ==============================================================================

_THINK_OPEN_TAGS = ("<think>", "<thinking>", "<redacted_thinking>")
_THINK_CLOSE_TAGS = ("</think>", "</thinking>", "</redacted_thinking>")
_TOOL_OPEN_TAGS = ("<tool_call>", "<|python_tag|>")
_TOOL_CLOSE_TAG = "</tool_call>"


def _longest_partial_tag_suffix(text: str, tags: Sequence[str]) -> int:
    """
    Return the length of the longest suffix of `text` that is a proper non-empty prefix
    of any tag in `tags`. Returns 0 immediately when no tag prefix matches at the end!
    """
    if not text:
        return 0
    max_len = max(len(t) for t in tags)
    tail = text[-max_len:]
    for k in range(min(len(tail), max_len - 1), 0, -1):
        suffix = tail[-k:]
        for tag in tags:
            if tag.startswith(suffix):
                return k
    return 0


def _find_earliest_tag(text: str, tags: Sequence[str]) -> Tuple[int, str]:
    best_pos = -1
    best_tag = ""
    for tag in tags:
        p = text.find(tag)
        if p != -1 and (best_pos == -1 or p < best_pos):
            best_pos = p
            best_tag = tag
    return best_pos, best_tag


@dataclass
class StreamParseEmission:
    reasoning_delta: str = ""
    content_delta: str = ""
    tool_calls: List[ParsedToolCall] = field(default_factory=list)


class ReasoningAndToolStreamParser:
    """
    Real-time incremental parser for both SSE streaming and non-streaming `/v1/chat/completions`:
    - Separates `<think>...</think>` into `reasoning_delta` (NEVER emitting `<think></think>` into `content_delta`!).
    - Suppresses pure-whitespace empty think blocks (`<think>\\n\\n</think>`) completely!
    - Handles prompts that pre-open `<think>\\n` (`Qwen3.5` / `Qwen3.8`) as well as models that emit `<think>` themselves (`Qwen3`).
    - Intercepts `<tool_call>...</tool_call>` blocks (including auto-closing unclosed `<think>` before `<tool_call>`)
      and converts them into structured OpenAI `ParsedToolCall` objects!
    """

    def __init__(
        self,
        prompt_str: str,
        tools: Optional[Sequence[Dict[str, Any]]] = None,
        parse_tool_calls: bool = True,
        emit_reasoning: bool = True,
    ) -> None:
        self.schema_map = _build_tool_param_schema_map(tools)
        self.parse_tool_calls = parse_tool_calls
        self.emit_reasoning = emit_reasoning

        rstripped = prompt_str.rstrip(" \t\r\n")
        self.prompt_opened_think = any(rstripped.endswith(t) for t in _THINK_OPEN_TAGS)

        self._state: Literal["START", "IN_THINK", "AFTER_THINK", "IN_CONTENT", "IN_TOOL"] = (
            "IN_THINK" if self.prompt_opened_think else "START"
        )
        self._buf: str = ""
        self._think_started_non_ws: bool = False
        self._active_tool_open_tag: str = "<tool_call>"

        # Full accumulated outputs for non-streaming summary & finish_reason inspection
        self.reasoning_parts: List[str] = []
        self.content_parts: List[str] = []
        self.parsed_tool_calls: List[ParsedToolCall] = []

    def feed(self, delta_text: str) -> StreamParseEmission:
        if not delta_text:
            return StreamParseEmission()
        self._buf += delta_text
        return self._drain(final=False)

    def finalize(self) -> StreamParseEmission:
        return self._drain(final=True)

    @property
    def has_tool_calls(self) -> bool:
        return len(self.parsed_tool_calls) > 0

    @property
    def full_reasoning(self) -> str:
        return "".join(self.reasoning_parts).strip()

    @property
    def full_content(self) -> str:
        return "".join(self.content_parts)

    def _emit_reasoning_piece(self, text: str, out: StreamParseEmission) -> None:
        if not text:
            return
        if not self._think_started_non_ws:
            stripped_lead = text.lstrip(" \t\r\n")
            if not stripped_lead:
                return
            text = stripped_lead
            self._think_started_non_ws = True
        self.reasoning_parts.append(text)
        if self.emit_reasoning:
            out.reasoning_delta += text

    def _emit_content_piece(self, text: str, out: StreamParseEmission) -> None:
        if not text:
            return
        self.content_parts.append(text)
        out.content_delta += text

    def _drain(self, final: bool) -> StreamParseEmission:
        out = StreamParseEmission()

        while self._buf:
            if self._state == "START":
                stripped = self._buf.lstrip(" \t\r\n")
                if not stripped:
                    if final:
                        self._buf = ""
                    break

                # 1. Model opened `<think>` at start of reply
                matched_open = next((t for t in _THINK_OPEN_TAGS if stripped.startswith(t)), None)
                if matched_open is not None:
                    self._buf = stripped[len(matched_open) :]
                    self._think_started_non_ws = False
                    self._state = "IN_THINK"
                    continue

                # 2. Orphan `</think>` at start when thinking was bypassed in prompt
                matched_orphan_close = next((t for t in _THINK_CLOSE_TAGS if stripped.startswith(t)), None)
                if matched_orphan_close is not None:
                    self._buf = stripped[len(matched_orphan_close) :]
                    self._state = "AFTER_THINK"
                    continue

                # 3. Direct `<tool_call>` at start of reply
                if self.parse_tool_calls:
                    matched_tool = next((t for t in _TOOL_OPEN_TAGS if stripped.startswith(t)), None)
                    if matched_tool is not None:
                        self._active_tool_open_tag = matched_tool
                        self._buf = stripped[len(matched_tool) :]
                        self._state = "IN_TOOL"
                        continue

                # 4. Wait if `stripped` is still a partial prefix of `<think>` / `</think>` / `<tool_call>`
                start_watch = list(_THINK_OPEN_TAGS) + list(_THINK_CLOSE_TAGS)
                if self.parse_tool_calls:
                    start_watch.extend(_TOOL_OPEN_TAGS)
                if not final and any(tag.startswith(stripped) for tag in start_watch):
                    break

                # Otherwise normal assistant content begins immediately!
                self._state = "IN_CONTENT"
                continue

            if self._state == "IN_THINK":
                close_pos, close_tag = _find_earliest_tag(self._buf, _THINK_CLOSE_TAGS)
                # Also guard against unclosed `<think>` going straight into `<tool_call>`!
                tool_pos = self._buf.find("<tool_call>") if self.parse_tool_calls else -1

                if close_pos != -1 and (tool_pos == -1 or close_pos < tool_pos):
                    before = self._buf[:close_pos]
                    if self._think_started_non_ws:
                        before = before.rstrip(" \t\r\n")
                    self._emit_reasoning_piece(before, out)
                    self._buf = self._buf[close_pos + len(close_tag) :]
                    self._state = "AFTER_THINK"
                    continue

                if tool_pos != -1:
                    before = self._buf[:tool_pos].rstrip(" \t\r\n")
                    self._emit_reasoning_piece(before, out)
                    self._active_tool_open_tag = "<tool_call>"
                    self._buf = self._buf[tool_pos + len("<tool_call>") :]
                    self._state = "IN_TOOL"
                    continue

                if final:
                    # Stream ended before `</think>` (e.g. hit `max_tokens` during reasoning)
                    self._emit_reasoning_piece(self._buf, out)
                    self._buf = ""
                    break

                # If we haven't seen non-whitespace yet, hold pure whitespace in buffer
                # so empty `<think>\n\n</think>` emits 0 reasoning chunks!
                if not self._think_started_non_ws and not self._buf.strip(" \t\r\n"):
                    break

                watch_tags = list(_THINK_CLOSE_TAGS) + (["<tool_call>"] if self.parse_tool_calls else [])
                hold = _longest_partial_tag_suffix(self._buf, watch_tags)
                safe_len = len(self._buf) - hold
                if safe_len > 0:
                    self._emit_reasoning_piece(self._buf[:safe_len], out)
                    self._buf = self._buf[safe_len:]
                break

            if self._state == "AFTER_THINK":
                # Strip formatting newlines right after `</think>` before content or `<tool_call>`
                stripped = self._buf.lstrip(" \t\r\n")
                if not stripped:
                    if not final:
                        self._buf = ""
                    else:
                        self._buf = ""
                    break
                self._buf = stripped
                # Check if another `<think>` or orphan `</think>` or immediate `<tool_call>` follows
                if any(self._buf.startswith(t) for t in _THINK_OPEN_TAGS) or any(
                    self._buf.startswith(t) for t in _THINK_CLOSE_TAGS
                ):
                    self._state = "START"
                    continue
                if self.parse_tool_calls:
                    matched_tool = next((t for t in _TOOL_OPEN_TAGS if self._buf.startswith(t)), None)
                    if matched_tool is not None:
                        self._active_tool_open_tag = matched_tool
                        self._buf = self._buf[len(matched_tool) :]
                        self._state = "IN_TOOL"
                        continue
                    if not final and any(t.startswith(self._buf) for t in _TOOL_OPEN_TAGS):
                        break
                self._state = "IN_CONTENT"
                continue

            if self._state == "IN_CONTENT":
                watch_content: List[str] = list(_THINK_CLOSE_TAGS)
                if self.parse_tool_calls:
                    watch_content.extend(_TOOL_OPEN_TAGS)

                pos, tag = _find_earliest_tag(self._buf, watch_content)
                if pos != -1:
                    if tag in _THINK_CLOSE_TAGS:
                        # Strip stray orphan `</think>` in mid-stream
                        self._emit_content_piece(self._buf[:pos], out)
                        self._buf = self._buf[pos + len(tag) :].lstrip("\r\n")
                        continue
                    # Tool call opened! Trim trailing whitespace right before `<tool_call>`
                    before_tool = self._buf[:pos].rstrip(" \t\r\n")
                    if before_tool:
                        self._emit_content_piece(before_tool, out)
                    self._active_tool_open_tag = tag
                    self._buf = self._buf[pos + len(tag) :]
                    self._state = "IN_TOOL"
                    continue

                if final:
                    self._emit_content_piece(self._buf, out)
                    self._buf = ""
                    break

                hold = _longest_partial_tag_suffix(self._buf, watch_content)
                safe_len = len(self._buf) - hold
                if safe_len > 0:
                    self._emit_content_piece(self._buf[:safe_len], out)
                    self._buf = self._buf[safe_len:]
                break

            if self._state == "IN_TOOL":
                if self._active_tool_open_tag == "<|python_tag|>":
                    # Llama-3 `<|python_tag|>` runs until EOS (`final=True`)
                    if final:
                        calls = parse_single_tool_call_block(self._buf, self.schema_map)
                        self.parsed_tool_calls.extend(calls)
                        out.tool_calls.extend(calls)
                        self._buf = ""
                    break

                close_idx = self._buf.find(_TOOL_CLOSE_TAG)
                if close_idx != -1:
                    raw_block = self._buf[:close_idx]
                    self._buf = self._buf[close_idx + len(_TOOL_CLOSE_TAG) :]
                    calls = parse_single_tool_call_block(raw_block, self.schema_map)
                    self.parsed_tool_calls.extend(calls)
                    out.tool_calls.extend(calls)
                    # After `</tool_call>`, transition to `AFTER_THINK` so consecutive `<tool_call>` blocks
                    # or trailing text are handled seamlessly!
                    self._state = "AFTER_THINK"
                    continue

                if final:
                    # Auto-heal if generation stopped right after `</function>` or `}` before `</tool_call>`
                    calls = parse_single_tool_call_block(self._buf, self.schema_map)
                    if calls:
                        self.parsed_tool_calls.extend(calls)
                        out.tool_calls.extend(calls)
                    self._buf = ""
                break

        return out


In [ ]:
%%writefile pygguf_engine/gguf_parser.py
"""
Pure-Python zero-copy `mmap` GGUF v2/v3 parser, architecture config extractor,
and embedded GGUF tokenizer / chat-template renderer (0% llama.cpp).
"""

from __future__ import annotations

import json
import math
import mmap
import struct
from dataclasses import dataclass, field
from enum import IntEnum
from pathlib import Path
from typing import Any, BinaryIO, Dict, List, Literal, Optional, Sequence, Tuple, Union

import numpy as np


class GGMLQuantType(IntEnum):
    F32 = 0
    F16 = 1
    Q4_0 = 2
    Q4_1 = 3
    Q5_0 = 6
    Q5_1 = 7
    Q8_0 = 8
    Q8_1 = 9
    Q2_K = 10
    Q3_K = 11
    Q4_K = 12
    Q5_K = 13
    Q6_K = 14
    Q8_K = 15
    IQ2_XXS = 16
    IQ2_XS = 17
    IQ3_XXS = 18
    IQ1_S = 19
    IQ4_NL = 20
    IQ3_S = 21
    IQ2_S = 22
    IQ4_XS = 23
    I8 = 24
    I16 = 25
    I32 = 26
    I64 = 27
    F64 = 28
    IQ1_M = 29
    BF16 = 30
    TQ1_0 = 34
    TQ2_0 = 35
    MXFP4 = 39
    NVFP4 = 40


# (elements_per_block, bytes_per_block)
GGML_BLOCK_INFO: Dict[int, Tuple[int, int]] = {
    GGMLQuantType.F32: (1, 4),
    GGMLQuantType.F16: (1, 2),
    GGMLQuantType.BF16: (1, 2),
    GGMLQuantType.Q4_0: (32, 18),
    GGMLQuantType.Q4_1: (32, 20),
    GGMLQuantType.Q5_0: (32, 22),
    GGMLQuantType.Q5_1: (32, 24),
    GGMLQuantType.Q8_0: (32, 34),
    GGMLQuantType.Q8_1: (32, 40),
    GGMLQuantType.Q2_K: (256, 84),
    GGMLQuantType.Q3_K: (256, 110),
    GGMLQuantType.Q4_K: (256, 144),
    GGMLQuantType.Q5_K: (256, 176),
    GGMLQuantType.Q6_K: (256, 210),
    GGMLQuantType.Q8_K: (256, 292),
    GGMLQuantType.IQ2_XXS: (256, 66),
    GGMLQuantType.IQ2_XS: (256, 74),
    GGMLQuantType.IQ3_XXS: (256, 98),
    GGMLQuantType.IQ1_S: (256, 50),
    GGMLQuantType.IQ4_NL: (32, 18),
    GGMLQuantType.IQ3_S: (256, 110),
    GGMLQuantType.IQ2_S: (256, 82),
    GGMLQuantType.IQ4_XS: (256, 136),
    GGMLQuantType.I8: (1, 1),
    GGMLQuantType.I16: (1, 2),
    GGMLQuantType.I32: (1, 4),
    GGMLQuantType.I64: (1, 8),
    GGMLQuantType.F64: (1, 8),
    GGMLQuantType.IQ1_M: (256, 56),
    GGMLQuantType.TQ1_0: (256, 54),
    GGMLQuantType.TQ2_0: (256, 66),
    GGMLQuantType.MXFP4: (32, 17),
    GGMLQuantType.NVFP4: (64, 36),
}


class _GGUFValueType(IntEnum):
    UINT8 = 0
    INT8 = 1
    UINT16 = 2
    INT16 = 3
    UINT32 = 4
    INT32 = 5
    FLOAT32 = 6
    BOOL = 7
    STRING = 8
    ARRAY = 9
    UINT64 = 10
    INT64 = 11
    FLOAT64 = 12


_SCALAR_STRUCTS: Dict[int, Tuple[str, int, Any]] = {
    _GGUFValueType.UINT8: ("<B", 1, np.uint8),
    _GGUFValueType.INT8: ("<b", 1, np.int8),
    _GGUFValueType.UINT16: ("<H", 2, np.uint16),
    _GGUFValueType.INT16: ("<h", 2, np.int16),
    _GGUFValueType.UINT32: ("<I", 4, np.uint32),
    _GGUFValueType.INT32: ("<i", 4, np.int32),
    _GGUFValueType.FLOAT32: ("<f", 4, np.float32),
    _GGUFValueType.BOOL: ("<?", 1, np.bool_),
    _GGUFValueType.UINT64: ("<Q", 8, np.uint64),
    _GGUFValueType.INT64: ("<q", 8, np.int64),
    _GGUFValueType.FLOAT64: ("<d", 8, np.float64),
}


@dataclass
class GGUFTensorInfo:
    name: str
    ggml_dims: Tuple[int, ...]  # GGML order [ne0, ne1, ...] where ne0 is innermost contiguous
    shape: Tuple[int, ...]      # PyTorch row-major shape (reversed ggml_dims)
    qtype: int
    offset: int                 # Byte offset relative to tensor_data_offset
    n_elements: int
    n_bytes: int

    @property
    def qtype_name(self) -> str:
        try:
            return GGMLQuantType(self.qtype).name
        except ValueError:
            return f"UNKNOWN_{self.qtype}"


@dataclass
class GGUFModelConfig:
    arch: str
    name: str
    vocab_size: int
    n_layers: int
    hidden_size: int
    intermediate_size: int
    n_heads: int
    n_kv_heads: int
    head_dim: int
    rms_norm_eps: float = 1e-6
    rope_theta: float = 10000.0
    rope_dim: int = 0
    rope_neox: bool = True
    max_context_len: int = 32768
    # RoPE context-extension scaling (linear / YaRN / NTK-aware) for running beyond the trained context
    rope_scaling_type: str = "none"
    rope_scaling_factor: float = 1.0
    rope_original_context_len: int = 0
    rope_yarn_beta_fast: float = 32.0
    rope_yarn_beta_slow: float = 1.0
    rope_yarn_ext_factor: float = 1.0
    rope_yarn_attn_factor: float = 0.0
    rope_yarn_log_mul: float = 0.1
    # MoE parameters (0 for dense models)
    n_experts: int = 0
    n_experts_used: int = 0
    moe_intermediate_size: int = 0
    shared_expert_intermediate_size: int = 0
    norm_topk_prob: bool = True
    # Hybrid Gated-DeltaNet / SSM & Multi-Token Prediction (Qwen3.5 / Qwen3.6 / Qwen3.8 / Qwen3Next)
    nextn_predict_layers: int = 0
    full_attention_interval: int = 0
    ssm_conv_kernel: int = 0
    ssm_inner_size: int = 0
    ssm_state_size: int = 0
    ssm_dt_rank: int = 0
    ssm_group_count: int = 0
    # Gemma-specific scaling / softcapping
    embed_scale: float = 1.0
    attn_logit_softcapping: float = 0.0
    final_logit_softcapping: float = 0.0
    query_pre_attn_scalar: float = 0.0
    # Special tokens & chat template
    bos_token_id: Optional[int] = None
    eos_token_ids: List[int] = field(default_factory=list)
    chat_template: Optional[str] = None

    @property
    def is_moe(self) -> bool:
        return self.n_experts > 1

    @property
    def is_hybrid_ssm(self) -> bool:
        return self.ssm_inner_size > 0 and self.ssm_state_size > 0


class GGUFFile:
    """
    Zero-copy memory-mapped GGUF v2/v3 file reader in pure Python.
    Streams tensor slices directly from OS page cache without duplicating model memory.
    """

    def __init__(self, path: Union[str, Path]) -> None:
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"GGUF file not found: {self.path}")

        self._fh: BinaryIO = open(self.path, "rb")
        self._mm = mmap.mmap(self._fh.fileno(), 0, access=mmap.ACCESS_READ)
        self._pos = 0

        magic = self._read_bytes(4)
        if magic != b"GGUF":
            self.close()
            raise ValueError(f"Invalid GGUF magic {magic!r} in {self.path}")

        self.version = self._read_scalar("<I", 4)
        if self.version not in (2, 3):
            self.close()
            raise ValueError(f"Unsupported GGUF version {self.version} (expected 2 or 3)")

        self.tensor_count = self._read_scalar("<Q", 8)
        self.kv_count = self._read_scalar("<Q", 8)

        self.metadata: Dict[str, Any] = {}
        for _ in range(self.kv_count):
            key = self._read_string()
            val_type = self._read_scalar("<I", 4)
            self.metadata[key] = self._read_value(val_type)

        self.alignment = int(self.metadata.get("general.alignment", 32))
        self.tensors: Dict[str, GGUFTensorInfo] = {}

        for _ in range(self.tensor_count):
            name = self._read_string()
            n_dims = self._read_scalar("<I", 4)
            ggml_dims = tuple(self._read_scalar("<Q", 8) for _ in range(n_dims))
            qtype = self._read_scalar("<I", 4)
            rel_offset = self._read_scalar("<Q", 8)

            shape = tuple(reversed(ggml_dims)) if n_dims > 0 else (1,)
            n_elements = 1
            for d in ggml_dims:
                n_elements *= d

            if qtype not in GGML_BLOCK_INFO:
                raise ValueError(f"Unsupported GGML tensor quantization type id={qtype} on {name}")
            block_elems, block_bytes = GGML_BLOCK_INFO[qtype]
            if n_elements % block_elems != 0:
                raise ValueError(
                    f"Tensor {name} has {n_elements} elements, not divisible by block size {block_elems}"
                )
            n_bytes = (n_elements // block_elems) * block_bytes

            self.tensors[name] = GGUFTensorInfo(
                name=name,
                ggml_dims=ggml_dims,
                shape=shape,
                qtype=qtype,
                offset=rel_offset,
                n_elements=n_elements,
                n_bytes=n_bytes,
            )

        # Align end of header to `alignment` bytes
        rem = self._pos % self.alignment
        self.tensor_data_offset = self._pos + ((self.alignment - rem) if rem else 0)

    def _read_bytes(self, n: int) -> bytes:
        b = self._mm[self._pos : self._pos + n]
        self._pos += n
        return b

    def _read_scalar(self, fmt: str, size: int) -> Any:
        val = struct.unpack_from(fmt, self._mm, self._pos)[0]
        self._pos += size
        return val

    def _read_string(self) -> str:
        length = struct.unpack_from("<Q", self._mm, self._pos)[0]
        self._pos += 8
        raw = self._mm[self._pos : self._pos + length]
        self._pos += length
        return raw.decode("utf-8", errors="replace")

    def _read_value(self, vtype: int) -> Any:
        if vtype == _GGUFValueType.STRING:
            return self._read_string()
        if vtype == _GGUFValueType.ARRAY:
            elem_type = self._read_scalar("<I", 4)
            length = self._read_scalar("<Q", 8)
            if elem_type == _GGUFValueType.STRING:
                # Fast tight loop using local unpack_from for 150k+ vocab arrays
                mm = self._mm
                pos = self._pos
                unpack_q = struct.unpack_from
                out_strs: List[str] = []
                append = out_strs.append
                for _ in range(length):
                    slen = unpack_q("<Q", mm, pos)[0]
                    pos += 8
                    append(mm[pos : pos + slen].decode("utf-8", errors="replace"))
                    pos += slen
                self._pos = pos
                return out_strs
            if elem_type in _SCALAR_STRUCTS:
                _, elem_size, np_dtype = _SCALAR_STRUCTS[elem_type]
                nbytes = length * elem_size
                arr = np.frombuffer(self._mm, dtype=np_dtype, count=length, offset=self._pos).copy()
                self._pos += nbytes
                return arr.tolist()
            return [self._read_value(elem_type) for _ in range(length)]

        if vtype in _SCALAR_STRUCTS:
            fmt, size, _ = _SCALAR_STRUCTS[vtype]
            return self._read_scalar(fmt, size)
        raise ValueError(f"Unknown GGUF metadata value type: {vtype}")

    def get_tensor_bytes(self, name: str) -> np.ndarray:
        """Return a zero-copy uint8 NumPy view into the memory-mapped tensor bytes."""
        info = self.tensors[name]
        abs_offset = self.tensor_data_offset + info.offset
        return np.frombuffer(self._mm, dtype=np.uint8, count=info.n_bytes, offset=abs_offset)

    def build_model_config(self) -> GGUFModelConfig:
        meta = self.metadata
        arch = str(meta.get("general.architecture", "llama")).lower()
        name = str(meta.get("general.name", self.path.stem))

        def _get(suffix: str, default: Any = None) -> Any:
            if f"{arch}.{suffix}" in meta:
                return meta[f"{arch}.{suffix}"]
            return default

        def _max_nonzero_int(val: Any, fallback: int) -> int:
            if isinstance(val, list):
                nz = [int(x) for x in val if int(x) > 0]
                return max(nz) if nz else fallback
            return int(val) if val is not None else fallback

        hidden_size = int(_get("embedding_length", 4096))
        total_blocks = int(_get("block_count", 32))
        nextn_layers = int(_get("nextn_predict_layers", 0) or 0)

        # Exclude trailing Multi-Token Prediction (`nextn` / MTP) draft blocks from trunk layers
        # (e.g., Qwen3.8-27B has block_count=65 with blk.64.nextn.* and nextn_predict_layers=1 -> 64 trunk layers)
        if nextn_layers > 0 and total_blocks > nextn_layers:
            n_layers = total_blocks - nextn_layers
        else:
            n_layers = total_blocks
            while n_layers > 1 and (
                f"blk.{n_layers - 1}.nextn.eh_proj.weight" in self.tensors
                or (
                    f"blk.{n_layers - 1}.attn_norm.weight" not in self.tensors
                    and f"blk.{n_layers - 1}.attn_q.weight" not in self.tensors
                    and f"blk.{n_layers - 1}.attn_qkv.weight" not in self.tensors
                )
            ):
                n_layers -= 1
                nextn_layers += 1

        n_heads = _max_nonzero_int(_get("attention.head_count", 32), 32)
        n_kv_heads = _max_nonzero_int(_get("attention.head_count_kv", n_heads), n_heads)

        head_dim = _max_nonzero_int(_get("attention.key_length", hidden_size // n_heads), hidden_size // n_heads)
        ffn_raw = _get("feed_forward_length", hidden_size * 4)
        intermediate_size = _max_nonzero_int(ffn_raw, hidden_size * 4)

        # Infer vocab_size from token_embd tensor shape [vocab_size, hidden_size] or tokenizer tokens
        if "token_embd.weight" in self.tensors:
            vocab_size = int(self.tensors["token_embd.weight"].shape[0])
        else:
            tokens = meta.get("tokenizer.ggml.tokens", [])
            vocab_size = len(tokens)

        rms_eps = float(_get("attention.layer_norm_rms_epsilon", 1e-6))
        rope_theta = float(_get("rope.freq_base", 10000.0))
        rope_dim = int(_get("rope.dimension_count", head_dim))
        max_ctx = int(_get("context_length", 32768))

        # Optional RoPE scaling metadata (llama.cpp keys: {arch}.rope.scaling.*)
        rope_scaling_type = str(_get("rope.scaling.type", "none") or "none").strip().lower().replace("_", "-")
        rope_scaling_factor = float(_get("rope.scaling.factor", 1.0) or 1.0)
        rope_original_context_len = int(_get("rope.scaling.original_context_length", 0) or 0)
        rope_yarn_beta_fast = float(_get("rope.scaling.yarn_beta_fast", 32.0) or 32.0)
        rope_yarn_beta_slow = float(_get("rope.scaling.yarn_beta_slow", 1.0) or 1.0)
        rope_yarn_ext_factor = float(_get("rope.scaling.yarn_ext_factor", 1.0) or 1.0)
        rope_yarn_attn_factor = float(_get("rope.scaling.yarn_attn_factor", 0.0) or 0.0)
        rope_yarn_log_mul = float(_get("rope.scaling.yarn_log_mul", 0.1) or 0.1)

        # GGUF `llama` uses interleaved/adjacent-pair RoPE because convert_hf_to_gguf permutes Q/K.
        # Qwen2/Qwen3/Qwen3.5/Qwen3.8/Gemma/Phi3 keep HF's NeoX split-half RoPE.
        rope_neox = arch not in ("llama",)

        # MoE fields
        n_experts = int(_get("expert_count", 0) or 0)
        n_experts_used = int(_get("expert_used_count", 0) or 0)
        moe_ffn = int(_get("expert_feed_forward_length", intermediate_size) or intermediate_size)
        shared_moe_ffn = int(_get("expert_shared_feed_forward_length", 0) or 0)
        norm_topk = bool(
            _get("expert_weights_norm", arch in ("qwen3moe", "qwen35moe", "qwen3next", "mixtral", "llama"))
        )

        # Hybrid Gated-DeltaNet / SSM fields (`qwen35`, `qwen35moe`, `qwen3next`)
        full_attn_interval = int(_get("full_attention_interval", 0) or 0)
        ssm_conv_kernel = int(_get("ssm.conv_kernel", 0) or 0)
        ssm_inner_size = int(_get("ssm.inner_size", 0) or 0)
        ssm_state_size = int(_get("ssm.state_size", 0) or 0)
        ssm_dt_rank = int(_get("ssm.time_step_rank", 0) or 0)
        ssm_group_count = int(_get("ssm.group_count", 0) or 0)

        # Gemma family specifics
        embed_scale = math.sqrt(hidden_size) if arch.startswith("gemma") else 1.0
        attn_softcap = float(_get("attn_logit_softcapping", 0.0) or 0.0)
        final_softcap = float(_get("final_logit_softcapping", 0.0) or 0.0)
        query_scalar = float(_get("attention.query_pre_attn_scalar", 0.0) or 0.0)

        bos_id = meta.get("tokenizer.ggml.bos_token_id")
        eos_ids: List[int] = []
        if "tokenizer.ggml.eos_token_id" in meta:
            eos_ids.append(int(meta["tokenizer.ggml.eos_token_id"]))
        extra_eos = meta.get("tokenizer.ggml.eos_token_ids")
        if isinstance(extra_eos, list):
            for eid in extra_eos:
                if int(eid) not in eos_ids:
                    eos_ids.append(int(eid))

        chat_template = meta.get("tokenizer.chat_template")

        return GGUFModelConfig(
            arch=arch,
            name=name,
            vocab_size=vocab_size,
            n_layers=n_layers,
            hidden_size=hidden_size,
            intermediate_size=intermediate_size,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            head_dim=head_dim,
            rms_norm_eps=rms_eps,
            rope_theta=rope_theta,
            rope_dim=rope_dim,
            rope_neox=rope_neox,
            max_context_len=max_ctx,
            rope_scaling_type=rope_scaling_type,
            rope_scaling_factor=rope_scaling_factor,
            rope_original_context_len=rope_original_context_len,
            rope_yarn_beta_fast=rope_yarn_beta_fast,
            rope_yarn_beta_slow=rope_yarn_beta_slow,
            rope_yarn_ext_factor=rope_yarn_ext_factor,
            rope_yarn_attn_factor=rope_yarn_attn_factor,
            rope_yarn_log_mul=rope_yarn_log_mul,
            n_experts=n_experts,
            n_experts_used=n_experts_used,
            moe_intermediate_size=moe_ffn,
            shared_expert_intermediate_size=shared_moe_ffn,
            norm_topk_prob=norm_topk,
            nextn_predict_layers=nextn_layers,
            full_attention_interval=full_attn_interval,
            ssm_conv_kernel=ssm_conv_kernel,
            ssm_inner_size=ssm_inner_size,
            ssm_state_size=ssm_state_size,
            ssm_dt_rank=ssm_dt_rank,
            ssm_group_count=ssm_group_count,
            embed_scale=embed_scale,
            attn_logit_softcapping=attn_softcap,
            final_logit_softcapping=final_softcap,
            query_pre_attn_scalar=query_scalar,
            bos_token_id=int(bos_id) if bos_id is not None else None,
            eos_token_ids=eos_ids,
            chat_template=str(chat_template) if chat_template else None,
        )

    def close(self) -> None:
        if getattr(self, "_mm", None) is not None:
            self._mm.close()
            self._mm = None  # type: ignore[assignment]
        if getattr(self, "_fh", None) is not None:
            self._fh.close()
            self._fh = None  # type: ignore[assignment]

    def __enter__(self) -> "GGUFFile":
        return self

    def __exit__(self, exc_type: Any, exc_val: Any, exc_tb: Any) -> None:
        self.close()


class GGUFTokenizer:
    """
    Tokenizes text and renders OpenAI chat messages either via:
    1. An optional HuggingFace tokenizer (`hf_tokenizer="Qwen/Qwen3-8B"` or local dir), OR
    2. Directly from the embedded `tokenizer.ggml.*` tables inside the `.gguf` file!
    """

    def __init__(
        self,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        hf_tokenizer: Optional[str] = None,
        chat_template: Optional[str] = None,
        chat_template_mode: Literal["auto", "gguf", "qwen35_xml", "qwen3_json"] = "auto",
    ) -> None:
        self.config = config
        self.chat_template_mode = chat_template_mode
        self.custom_chat_template: Optional[str] = None
        if chat_template:
            cand_path = Path(chat_template)
            if cand_path.is_file():
                self.custom_chat_template = cand_path.read_text(encoding="utf-8")
            else:
                self.custom_chat_template = chat_template
        self._hf_tok: Any = None
        self._rust_tok: Any = None
        self._vocab_list: List[str] = gguf_file.metadata.get("tokenizer.ggml.tokens", [])
        self._token_to_id: Dict[str, int] = {t: i for i, t in enumerate(self._vocab_list)}
        self.add_bos_token: bool = bool(
            gguf_file.metadata.get("tokenizer.ggml.add_bos_token", config.arch in ("llama", "gemma", "gemma2", "gemma3"))
        )

        # Register Common Chat Stop Tokens (<|im_end|>, <|eot_id|>, <end_of_turn>, <|endoftext|>)
        for special_stop in ("<|im_end|>", "<|eot_id|>", "<end_of_turn>", "<|endoftext|>", "</s>"):
            sid = self._token_to_id.get(special_stop)
            if sid is not None and sid not in self.config.eos_token_ids:
                self.config.eos_token_ids.append(sid)

        if hf_tokenizer:
            try:
                from transformers import AutoTokenizer

                self._hf_tok = AutoTokenizer.from_pretrained(hf_tokenizer, trust_remote_code=True)
                if self._hf_tok.eos_token_id is not None and self._hf_tok.eos_token_id not in self.config.eos_token_ids:
                    self.config.eos_token_ids.append(int(self._hf_tok.eos_token_id))
                return
            except Exception as exc:
                print(f"[PyGGUF] Note: Could not load hf_tokenizer='{hf_tokenizer}' ({exc}); using embedded GGUF tokenizer.")

        # Build fast Rust `tokenizers.Tokenizer` directly from GGUF metadata when available
        self._rust_tok = self._build_embedded_rust_tokenizer(gguf_file.metadata)

    def _build_embedded_rust_tokenizer(self, meta: Dict[str, Any]) -> Any:
        tokens: List[str] = meta.get("tokenizer.ggml.tokens", [])
        if not tokens:
            return None
        try:
            from tokenizers import AddedToken, Tokenizer
        except ImportError:
            return None

        tok_model_type = str(meta.get("tokenizer.ggml.model", "gpt2")).lower()
        merges_raw: List[str] = meta.get("tokenizer.ggml.merges", [])
        token_types: List[int] = meta.get("tokenizer.ggml.token_type", [])

        vocab_dict = {tok: idx for idx, tok in enumerate(tokens)}

        def _keep_visible_in_decode(tok: str) -> bool:
            # Keep reasoning & tool-call delimiters visible so `ReasoningAndToolStreamParser`
            # can cleanly separate `<think>...</think>` into `reasoning_content` and parse `<tool_call>`!
            if tok in (
                "<think>",
                "</think>",
                "<thinking>",
                "</thinking>",
                "<redacted_thinking>",
                "</redacted_thinking>",
                "<tool_call>",
                "</tool_call>",
                "<tool_response>",
                "</tool_response>",
                "</function>",
                "</parameter>",
                "<|python_tag|>",
            ):
                return True
            if tok.startswith("<function=") or tok.startswith("<parameter="):
                return True
            return False

        try:
            if tok_model_type in ("gpt2", "bpe") and merges_raw:
                merges = []
                for m in merges_raw:
                    parts = m.split(" ", 1)
                    if len(parts) == 2:
                        merges.append((parts[0], parts[1]))
                # Construct JSON representation for tokenizers.Tokenizer.from_str (works across versions)
                tok_json = {
                    "version": "1.0",
                    "truncation": None,
                    "padding": None,
                    "added_tokens": [
                        {
                            "id": idx,
                            "content": tok,
                            "single_word": False,
                            "lstrip": False,
                            "rstrip": False,
                            "normalized": False,
                            "special": not _keep_visible_in_decode(tok),
                        }
                        for idx, tok in enumerate(tokens)
                        if (idx < len(token_types) and token_types[idx] in (3, 4))
                        or (tok.startswith("<|") and tok.endswith("|>"))
                    ],
                    "normalizer": None,
                    "pre_tokenizer": {"type": "ByteLevel", "add_prefix_space": False, "trim_offsets": True, "use_regex": True},
                    "post_processor": None,
                    "decoder": {"type": "ByteLevel", "add_prefix_space": False, "trim_offsets": False, "use_regex": False},
                    "model": {
                        "type": "BPE",
                        "dropout": None,
                        "unk_token": None,
                        "continuing_subword_prefix": "",
                        "end_of_word_suffix": "",
                        "fuse_unk": False,
                        "byte_fallback": False,
                        "vocab": vocab_dict,
                        "merges": merges,
                    },
                }
                return Tokenizer.from_str(json.dumps(tok_json))

            if tok_model_type == "llama":
                scores = meta.get("tokenizer.ggml.scores", [0.0] * len(tokens))
                unk_id = int(meta.get("tokenizer.ggml.unknown_token_id", 0))
                tok_json = {
                    "version": "1.0",
                    "truncation": None,
                    "padding": None,
                    "added_tokens": [
                        {
                            "id": idx,
                            "content": tok,
                            "single_word": False,
                            "lstrip": False,
                            "rstrip": False,
                            "normalized": False,
                            "special": not _keep_visible_in_decode(tok),
                        }
                        for idx, tok in enumerate(tokens)
                        if idx < len(token_types) and token_types[idx] in (3, 4)
                    ],
                    "normalizer": {"type": "Prepend", "prepend": "▁"},
                    "pre_tokenizer": {"type": "Metaspace", "replacement": "▁", "prepend_scheme": "never"},
                    "post_processor": None,
                    "decoder": {"type": "Metaspace", "replacement": "▁", "prepend_scheme": "always"},
                    "model": {
                        "type": "Unigram",
                        "unk_id": unk_id,
                        "vocab": [[tok, float(scores[idx]) if idx < len(scores) else 0.0] for idx, tok in enumerate(tokens)],
                        "byte_fallback": True,
                    },
                }
                return Tokenizer.from_str(json.dumps(tok_json))
        except Exception:
            return None
        return None

    def encode(self, text: str, add_special_tokens: bool = True) -> List[int]:
        if self._hf_tok is not None:
            return list(self._hf_tok.encode(text, add_special_tokens=add_special_tokens))

        if self._rust_tok is not None:
            ids = list(self._rust_tok.encode(text, add_special_tokens=False).ids)
            if add_special_tokens and self.add_bos_token and self.config.bos_token_id is not None:
                if not ids or ids[0] != self.config.bos_token_id:
                    ids = [self.config.bos_token_id] + ids
            return ids

        # Pure-Python greedy longest-match fallback (useful for synthetic/tiny unit test vocabs)
        ids = self._greedy_encode(text)
        if add_special_tokens and self.add_bos_token and self.config.bos_token_id is not None:
            if not ids or ids[0] != self.config.bos_token_id:
                ids = [self.config.bos_token_id] + ids
        return ids

    def _greedy_encode(self, text: str) -> List[int]:
        if not self._vocab_list:
            return [b % max(self.config.vocab_size, 1) for b in text.encode("utf-8")]
        out: List[int] = []
        i = 0
        n = len(text)
        max_tok_len = min(32, n)
        while i < n:
            matched = False
            for end in range(min(n, i + max_tok_len), i, -1):
                sub = text[i:end]
                if sub in self._token_to_id:
                    out.append(self._token_to_id[sub])
                    i = end
                    matched = True
                    break
            if not matched:
                # Byte fallback <0xXX> or raw id 0
                byte_val = text[i].encode("utf-8")[0]
                hex_tok = f"<0x{byte_val:02X}>"
                out.append(self._token_to_id.get(hex_tok, byte_val % len(self._vocab_list)))
                i += 1
        return out

    def decode(self, token_ids: Sequence[int], skip_special_tokens: bool = True) -> str:
        if not token_ids:
            return ""
        if self._hf_tok is not None:
            return str(self._hf_tok.decode(list(token_ids), skip_special_tokens=skip_special_tokens))
        if self._rust_tok is not None:
            return str(self._rust_tok.decode(list(token_ids), skip_special_tokens=skip_special_tokens))

        pieces: List[str] = []
        eos_set = set(self.config.eos_token_ids)
        for tid in token_ids:
            if skip_special_tokens and (tid in eos_set or tid == self.config.bos_token_id):
                continue
            if 0 <= tid < len(self._vocab_list):
                tok = self._vocab_list[tid]
                pieces.append(tok.replace("Ġ", " ").replace("▁", " ").replace("Ċ", "\n"))
        return "".join(pieces)

    def _select_jinja_templates(self) -> List[str]:
        from .agentic_protocol import QWEN3_AGENTIC_JINJA_TEMPLATE, QWEN35_AGENTIC_JINJA_TEMPLATE

        candidates: List[str] = []
        if self.custom_chat_template:
            candidates.append(self.custom_chat_template)

        arch = self.config.arch
        if self.chat_template_mode == "qwen35_xml":
            candidates.append(QWEN35_AGENTIC_JINJA_TEMPLATE)
        elif self.chat_template_mode == "qwen3_json":
            candidates.append(QWEN3_AGENTIC_JINJA_TEMPLATE)
        elif self.chat_template_mode == "gguf":
            if self.config.chat_template:
                candidates.append(self.config.chat_template)
        else:
            # `"auto"` mode: prefer hardened agentic templates for Qwen3.5/3.6/3.8 & Qwen3
            # (fixes `developer` role crashes, mid-conversation `system` crashes, and `|items` string crashes!)
            if arch in ("qwen35", "qwen35moe", "qwen3next"):
                candidates.append(QWEN35_AGENTIC_JINJA_TEMPLATE)
            elif arch in ("qwen3", "qwen3moe"):
                candidates.append(QWEN3_AGENTIC_JINJA_TEMPLATE)

        if self.config.chat_template and self.config.chat_template not in candidates:
            candidates.append(self.config.chat_template)

        if arch.startswith("qwen") and QWEN3_AGENTIC_JINJA_TEMPLATE not in candidates:
            candidates.append(QWEN3_AGENTIC_JINJA_TEMPLATE)

        return candidates

    def apply_chat_template(
        self,
        messages: Sequence[Dict[str, Any]],
        add_generation_prompt: bool = True,
        enable_thinking: Optional[bool] = None,
        tools: Optional[Sequence[Dict[str, Any]]] = None,
        tool_choice: Any = None,
        default_enable_thinking: bool = True,
    ) -> str:
        from .agentic_protocol import (
            apply_tool_choice_directive,
            normalize_messages_for_template,
            resolve_enable_thinking,
        )

        eff_thinking = resolve_enable_thinking(
            messages=messages,
            explicit_enable_thinking=enable_thinking,
            default_enable_thinking=default_enable_thinking,
        )
        norm_msgs = normalize_messages_for_template(messages)
        norm_msgs, eff_tools = apply_tool_choice_directive(norm_msgs, tools, tool_choice)

        if (
            self.custom_chat_template is None
            and self.chat_template_mode == "gguf"
            and self._hf_tok is not None
            and getattr(self._hf_tok, "chat_template", None)
        ):
            try:
                return str(
                    self._hf_tok.apply_chat_template(
                        norm_msgs,
                        tools=eff_tools,
                        tokenize=False,
                        add_generation_prompt=add_generation_prompt,
                        enable_thinking=eff_thinking,
                    )
                )
            except Exception:
                pass

        # Render via Jinja2 sandbox across prioritized candidate templates
        candidates = self._select_jinja_templates()
        if candidates:
            try:
                from jinja2 import BaseLoader
                from jinja2.sandbox import ImmutableSandboxedEnvironment

                def _raise_exception(msg: str) -> str:
                    raise ValueError(msg)

                env = ImmutableSandboxedEnvironment(
                    loader=BaseLoader(),
                    trim_blocks=True,
                    lstrip_blocks=True,
                    extensions=["jinja2.ext.loopcontrols"],
                )
                env.globals["raise_exception"] = _raise_exception
                bos_str = (
                    self._vocab_list[self.config.bos_token_id]
                    if (self.config.bos_token_id is not None and self.config.bos_token_id < len(self._vocab_list))
                    else ""
                )
                eos_str = (
                    self._vocab_list[self.config.eos_token_ids[0]]
                    if (self.config.eos_token_ids and self.config.eos_token_ids[0] < len(self._vocab_list))
                    else ""
                )

                for tmpl_src in candidates:
                    try:
                        tmpl = env.from_string(tmpl_src)
                        return tmpl.render(
                            messages=norm_msgs,
                            add_generation_prompt=add_generation_prompt,
                            bos_token=bos_str,
                            eos_token=eos_str,
                            tools=eff_tools,
                            enable_thinking=eff_thinking,
                            add_vision_id=False,
                        )
                    except Exception:
                        continue
            except ImportError:
                pass

        # Architecture-aware fallback formatting (ChatML for Qwen, Gemma turn tags, Llama-3 headers)
        arch = self.config.arch
        if arch.startswith("gemma"):
            parts: List[str] = []
            for m in messages:
                role = "model" if m["role"] == "assistant" else "user"
                parts.append(f"<start_of_turn>{role}\n{m['content']}<end_of_turn>\n")
            if add_generation_prompt:
                parts.append("<start_of_turn>model\n")
            return "".join(parts)

        if arch == "llama" and "<|start_header_id|>" in self._token_to_id:
            parts = []
            for m in messages:
                parts.append(f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n{m['content']}<|eot_id|>")
            if add_generation_prompt:
                parts.append("<|start_header_id|>assistant<|end_header_id|>\n\n")
            return "".join(parts)

        # Default ChatML (Qwen2 / Qwen3 / Mistral / OpenHermes)
        parts = [f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages]
        if add_generation_prompt:
            parts.append("<|im_start|>assistant\n")
        return "".join(parts)


In [ ]:
%%writefile pygguf_engine/quant_kernels.py
"""
GPU-coalesced GGUF weight unpacking & fused OpenAI Triton (`@triton.jit`) W4A16 / W8A16
linear, embedding, and sparse Mixture-of-Experts (MoE) kernels (0% llama.cpp).

Optimized specifically for NVIDIA Tesla T4 (`sm_75` FP16 Tensor Cores + GDDR6 bandwidth)
with automatic vectorized PyTorch fallback on CPU or non-Triton environments.
"""

from __future__ import annotations

from contextlib import contextmanager
import math
import os
from typing import Dict, Iterator, List, Optional, Sequence, Tuple, Union

os.environ.setdefault("TRITON_DEFAULT_FP_FUSION", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .gguf_parser import GGML_BLOCK_INFO, GGMLQuantType, GGUFFile, GGUFTensorInfo

# Sub-5-bit GGUF formats (including Unsloth Dynamic `UD` mixes: IQ4_XS, IQ4_NL, Q3_K, Q2_K, IQ3, IQ2, MXFP4)
# are packed into 4-bit W4A16 (5.0 bits/weight) so 27B-32B models fit comfortably on Dual 15GB T4 GPUs!
SUB_5BIT_W4_QTYPES = frozenset(
    {
        GGMLQuantType.Q4_0,
        GGMLQuantType.Q4_1,
        GGMLQuantType.Q4_K,
        GGMLQuantType.Q2_K,
        GGMLQuantType.Q3_K,
        GGMLQuantType.IQ4_NL,
        GGMLQuantType.IQ4_XS,
        GGMLQuantType.IQ3_XXS,
        GGMLQuantType.IQ3_S,
        GGMLQuantType.IQ2_XXS,
        GGMLQuantType.IQ2_XS,
        GGMLQuantType.IQ2_S,
        GGMLQuantType.IQ1_S,
        GGMLQuantType.IQ1_M,
        GGMLQuantType.TQ1_0,
        GGMLQuantType.TQ2_0,
        GGMLQuantType.MXFP4,
        GGMLQuantType.NVFP4,
    }
)

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    triton = None  # type: ignore[assignment]
    tl = None  # type: ignore[assignment]
    HAS_TRITON = False


# ==============================================================================
# 1. Fused @triton.jit GPU Kernels (W4A16 & W8A16 GEMV + Tensor-Core GEMM)
# ==============================================================================

if HAS_TRITON:

    @triton.jit
    def _w4a16_gemv_kernel(
        x_ptr,
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        bias_ptr,
        out_ptr,
        M,
        N,
        K,
        HAS_BIAS: tl.constexpr,
        BLOCK_N: tl.constexpr,
        row_scale_ptr=None,
        HAS_ROW_SCALE: tl.constexpr = False,
        OUT_FP32: tl.constexpr = False,
        X_PACKED_I32: tl.constexpr = False,
    ):
        """
        Zero-barrier register-accumulated 128-byte `int32` (`ld.global.b32`) W4A16 GEMV kernel (`M == 1`).
        - Accumulates per-lane FP32 registers across the entire K-loop (`0` in-loop `bar.sync` / shared-mem stalls!)
          and reduces once at the epilogue via warp-butterfly shuffles (`shfl.sync.bfly.b32`).
        - Supports `int32`-pair vectorized `x` loads (`X_PACKED_I32=True`, 4x `ld.global.b32` instead of 8x `b16`)
          and fused `row_scale` + `OUT_FP32` epilogue for `down_proj`!
        """
        pid_n = tl.program_id(0)
        pid_m = tl.program_id(1)

        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_n = offs_n < N
        offs_n64 = offs_n.to(tl.int64)

        n_groups = K // 32
        k_i32 = K // 8
        lanes32 = tl.arange(0, 32)
        g_local = lanes32 // 4
        j_local = lanes32 % 4

        acc_lane = tl.zeros((BLOCK_N, 32), dtype=tl.float32)
        if X_PACKED_I32:
            x_row_ptr = x_ptr + pid_m.to(tl.int64) * (K // 2)
        else:
            x_row_ptr = x_ptr + pid_m.to(tl.int64) * K

        for g_base in range(0, n_groups, 8):
            offs_i32 = g_base * 4 + lanes32
            mask_k = offs_i32 < k_i32

            w32 = tl.load(
                qw_i32_ptr + offs_n64[:, None] * k_i32 + offs_i32[None, :],
                mask=mask_n[:, None] & mask_k[None, :],
                other=0,
            )

            if X_PACKED_I32:
                x_pair_lo = (g_base + g_local) * 16 + j_local * 2
                xp_lo0 = tl.load(x_row_ptr + x_pair_lo + 0, mask=mask_k, other=0)
                xp_lo1 = tl.load(x_row_ptr + x_pair_lo + 1, mask=mask_k, other=0)
                xp_hi0 = tl.load(x_row_ptr + x_pair_lo + 8, mask=mask_k, other=0)
                xp_hi1 = tl.load(x_row_ptr + x_pair_lo + 9, mask=mask_k, other=0)

                x_lo_0 = (xp_lo0 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_lo_1 = ((xp_lo0 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_lo_2 = (xp_lo1 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_lo_3 = ((xp_lo1 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)

                x_hi_0 = (xp_hi0 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_hi_1 = ((xp_hi0 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_hi_2 = (xp_hi1 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_hi_3 = ((xp_hi1 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
            else:
                x_idx_lo = (g_base + g_local) * 32 + j_local * 4
                x_lo_0 = tl.load(x_row_ptr + x_idx_lo + 0, mask=mask_k, other=0.0).to(tl.float32)
                x_lo_1 = tl.load(x_row_ptr + x_idx_lo + 1, mask=mask_k, other=0.0).to(tl.float32)
                x_lo_2 = tl.load(x_row_ptr + x_idx_lo + 2, mask=mask_k, other=0.0).to(tl.float32)
                x_lo_3 = tl.load(x_row_ptr + x_idx_lo + 3, mask=mask_k, other=0.0).to(tl.float32)

                x_hi_0 = tl.load(x_row_ptr + x_idx_lo + 16, mask=mask_k, other=0.0).to(tl.float32)
                x_hi_1 = tl.load(x_row_ptr + x_idx_lo + 17, mask=mask_k, other=0.0).to(tl.float32)
                x_hi_2 = tl.load(x_row_ptr + x_idx_lo + 18, mask=mask_k, other=0.0).to(tl.float32)
                x_hi_3 = tl.load(x_row_ptr + x_idx_lo + 19, mask=mask_k, other=0.0).to(tl.float32)

            dot32 = (
                ((w32 >> 0) & 0x0F).to(tl.float32) * x_lo_0[None, :]
                + ((w32 >> 8) & 0x0F).to(tl.float32) * x_lo_1[None, :]
                + ((w32 >> 16) & 0x0F).to(tl.float32) * x_lo_2[None, :]
                + ((w32 >> 24) & 0x0F).to(tl.float32) * x_lo_3[None, :]
                + ((w32 >> 4) & 0x0F).to(tl.float32) * x_hi_0[None, :]
                + ((w32 >> 12) & 0x0F).to(tl.float32) * x_hi_1[None, :]
                + ((w32 >> 20) & 0x0F).to(tl.float32) * x_hi_2[None, :]
                + ((w32 >> 28) & 0x0F).to(tl.float32) * x_hi_3[None, :]
            )
            x_lane_sum = x_lo_0 + x_lo_1 + x_lo_2 + x_lo_3 + x_hi_0 + x_hi_1 + x_hi_2 + x_hi_3

            offs_g_lane = g_base + g_local
            mask_gl = offs_g_lane < n_groups
            sg_ptrs = offs_n64[:, None] * n_groups + offs_g_lane[None, :]
            s = tl.load(scales_ptr + sg_ptrs, mask=mask_n[:, None] & mask_gl[None, :], other=0.0).to(tl.float32)
            z = tl.load(zeros_ptr + sg_ptrs, mask=mask_n[:, None] & mask_gl[None, :], other=0.0).to(tl.float32)

            acc_lane += s * dot32 - z * x_lane_sum[None, :]

        acc = tl.sum(acc_lane, axis=1)

        if HAS_BIAS:
            bias = tl.load(bias_ptr + offs_n64, mask=mask_n, other=0.0).to(tl.float32)
            acc += bias
        if HAS_ROW_SCALE:
            r_sc = tl.load(row_scale_ptr + pid_m).to(tl.float32)
            acc = acc * r_sc

        out_ptrs = out_ptr + pid_m.to(tl.int64) * N + offs_n64
        if OUT_FP32:
            tl.store(out_ptrs, acc, mask=mask_n)
        else:
            acc_clamped = tl.minimum(tl.maximum(acc, -65504.0), 65504.0)
            tl.store(out_ptrs, acc_clamped.to(tl.float16), mask=mask_n)

    @triton.jit
    def _w4a16_multirow_gemv_kernel(
        x_ptr,
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        bias_ptr,
        out_ptr,
        M,
        N,
        K,
        HAS_BIAS: tl.constexpr,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
        row_scale_ptr=None,
        HAS_ROW_SCALE: tl.constexpr = False,
        OUT_FP32: tl.constexpr = False,
        X_PACKED_I32: tl.constexpr = False,
    ):
        """
        Zero-barrier weight-reusing multi-row (`2 <= M <= 8`) W4A16 GEMV kernel for MTP & N-gram verify!
        Eliminates all 4 per-tile `bar.sync` shared-memory stalls by accumulating `(BLOCK_M, BLOCK_N, 32)`
        directly in registers and loading `x` via `int32` pairs (`8` `ld.global.b32` vs `24` `ld.global.b16`).
        """
        pid_n = tl.program_id(0)

        offs_m = tl.arange(0, BLOCK_M)
        mask_m = offs_m < M
        offs_m64 = offs_m.to(tl.int64)

        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_n = offs_n < N
        offs_n64 = offs_n.to(tl.int64)

        n_groups = K // 32
        k_i32 = K // 8
        lanes32 = tl.arange(0, 32)
        g_local = lanes32 // 4
        j_local = lanes32 % 4

        acc_lane = tl.zeros((BLOCK_M, BLOCK_N, 32), dtype=tl.float32)

        for g_base in range(0, n_groups, 8):
            offs_i32 = g_base * 4 + lanes32
            mask_k = offs_i32 < k_i32
            mask_mk = mask_m[:, None] & mask_k[None, :]

            w32 = tl.load(
                qw_i32_ptr + offs_n64[:, None] * k_i32 + offs_i32[None, :],
                mask=mask_n[:, None] & mask_k[None, :],
                other=0,
            )
            q_lo_0 = ((w32 >> 0) & 0x0F).to(tl.float32)
            q_lo_1 = ((w32 >> 8) & 0x0F).to(tl.float32)
            q_lo_2 = ((w32 >> 16) & 0x0F).to(tl.float32)
            q_lo_3 = ((w32 >> 24) & 0x0F).to(tl.float32)
            q_hi_0 = ((w32 >> 4) & 0x0F).to(tl.float32)
            q_hi_1 = ((w32 >> 12) & 0x0F).to(tl.float32)
            q_hi_2 = ((w32 >> 20) & 0x0F).to(tl.float32)
            q_hi_3 = ((w32 >> 28) & 0x0F).to(tl.float32)

            if X_PACKED_I32:
                x_pair_lo = (g_base + g_local) * 16 + j_local * 2
                x_base_p = x_ptr + offs_m64[:, None] * (K // 2) + x_pair_lo[None, :]
                xp_lo0 = tl.load(x_base_p + 0, mask=mask_mk, other=0)
                xp_lo1 = tl.load(x_base_p + 1, mask=mask_mk, other=0)
                xp_hi0 = tl.load(x_base_p + 8, mask=mask_mk, other=0)
                xp_hi1 = tl.load(x_base_p + 9, mask=mask_mk, other=0)

                x_lo_0 = (xp_lo0 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_lo_1 = ((xp_lo0 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_lo_2 = (xp_lo1 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_lo_3 = ((xp_lo1 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)

                x_hi_0 = (xp_hi0 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_hi_1 = ((xp_hi0 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_hi_2 = (xp_hi1 & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
                x_hi_3 = ((xp_hi1 >> 16) & 0xFFFF).to(tl.uint16).to(tl.float16, bitcast=True).to(tl.float32)
            else:
                x_idx_lo = (g_base + g_local) * 32 + j_local * 4
                x_base_lo = x_ptr + offs_m64[:, None] * K + x_idx_lo[None, :]
                x_lo_0 = tl.load(x_base_lo + 0, mask=mask_mk, other=0.0).to(tl.float32)
                x_lo_1 = tl.load(x_base_lo + 1, mask=mask_mk, other=0.0).to(tl.float32)
                x_lo_2 = tl.load(x_base_lo + 2, mask=mask_mk, other=0.0).to(tl.float32)
                x_lo_3 = tl.load(x_base_lo + 3, mask=mask_mk, other=0.0).to(tl.float32)

                x_hi_0 = tl.load(x_base_lo + 16, mask=mask_mk, other=0.0).to(tl.float32)
                x_hi_1 = tl.load(x_base_lo + 17, mask=mask_mk, other=0.0).to(tl.float32)
                x_hi_2 = tl.load(x_base_lo + 18, mask=mask_mk, other=0.0).to(tl.float32)
                x_hi_3 = tl.load(x_base_lo + 19, mask=mask_mk, other=0.0).to(tl.float32)

            dot32 = (
                q_lo_0[None, :, :] * x_lo_0[:, None, :]
                + q_lo_1[None, :, :] * x_lo_1[:, None, :]
                + q_lo_2[None, :, :] * x_lo_2[:, None, :]
                + q_lo_3[None, :, :] * x_lo_3[:, None, :]
                + q_hi_0[None, :, :] * x_hi_0[:, None, :]
                + q_hi_1[None, :, :] * x_hi_1[:, None, :]
                + q_hi_2[None, :, :] * x_hi_2[:, None, :]
                + q_hi_3[None, :, :] * x_hi_3[:, None, :]
            )
            x_lane_sum = x_lo_0 + x_lo_1 + x_lo_2 + x_lo_3 + x_hi_0 + x_hi_1 + x_hi_2 + x_hi_3

            offs_g_lane = g_base + g_local
            mask_gl = offs_g_lane < n_groups
            sg_ptrs = offs_n64[:, None] * n_groups + offs_g_lane[None, :]
            s = tl.load(scales_ptr + sg_ptrs, mask=mask_n[:, None] & mask_gl[None, :], other=0.0).to(tl.float32)
            z = tl.load(zeros_ptr + sg_ptrs, mask=mask_n[:, None] & mask_gl[None, :], other=0.0).to(tl.float32)

            acc_lane += s[None, :, :] * dot32 - z[None, :, :] * x_lane_sum[:, None, :]

        acc = tl.sum(acc_lane, axis=2)

        if HAS_BIAS:
            bias = tl.load(bias_ptr + offs_n64, mask=mask_n, other=0.0).to(tl.float32)
            acc += bias[None, :]
        if HAS_ROW_SCALE:
            r_sc = tl.load(row_scale_ptr + offs_m64, mask=mask_m, other=1.0).to(tl.float32)
            acc = acc * r_sc[:, None]

        out_ptrs = out_ptr + offs_m64[:, None] * N + offs_n64[None, :]
        mask_mn = mask_m[:, None] & mask_n[None, :]
        if OUT_FP32:
            tl.store(out_ptrs, acc, mask=mask_mn)
        else:
            acc_clamped = tl.minimum(tl.maximum(acc, -65504.0), 65504.0)
            tl.store(out_ptrs, acc_clamped.to(tl.float16), mask=mask_mn)

    @triton.jit
    def _w4a16_gemm_tc_kernel(
        x_ptr,
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        bias_ptr,
        out_ptr,
        M,
        N,
        K,
        HAS_BIAS: tl.constexpr,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
    ):
        """Coalesced `int32` W4A16 Tensor-Core GEMM kernel (`BLOCK_K=64`) for Ampere+ (`sm_80+`)."""
        pid_m = tl.program_id(0)
        pid_n = tl.program_id(1)

        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_m = offs_m < M
        mask_n = offs_n < N
        offs_m64 = offs_m.to(tl.int64)
        offs_n64 = offs_n.to(tl.int64)

        n_groups = K // 32
        k_i32 = K // 8
        lanes8 = tl.arange(0, 8)
        lanes64 = tl.arange(0, 64)
        offs_2 = tl.arange(0, 2)

        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

        for g_base in range(0, n_groups, 2):
            offs_k = g_base * 32 + lanes64
            mask_k64 = offs_k < K
            a_tile = tl.load(
                x_ptr + offs_m64[:, None] * K + offs_k[None, :],
                mask=mask_m[:, None] & mask_k64[None, :],
                other=0.0,
            ).to(tl.float16)

            offs_i32 = g_base * 4 + lanes8
            mask_i32 = offs_i32 < k_i32
            w32 = tl.load(
                qw_i32_ptr + offs_n64[:, None] * k_i32 + offs_i32[None, :],
                mask=mask_n[:, None] & mask_i32[None, :],
                other=0,
            )

            lo_ev = tl.join((w32 >> 0) & 0x0F, (w32 >> 16) & 0x0F)
            lo_od = tl.join((w32 >> 8) & 0x0F, (w32 >> 24) & 0x0F)
            q_lo = tl.reshape(tl.join(lo_ev, lo_od), (BLOCK_N, 2, 16)).to(tl.float32)

            hi_ev = tl.join((w32 >> 4) & 0x0F, (w32 >> 20) & 0x0F)
            hi_od = tl.join((w32 >> 12) & 0x0F, (w32 >> 28) & 0x0F)
            q_hi = tl.reshape(tl.join(hi_ev, hi_od), (BLOCK_N, 2, 16)).to(tl.float32)

            q_grp = tl.reshape(tl.permute(tl.join(q_lo, q_hi), (0, 1, 3, 2)), (BLOCK_N, 2, 32))

            offs_g = g_base + offs_2
            mask_g = offs_g < n_groups
            sg_ptrs = offs_n64[:, None] * n_groups + offs_g[None, :]
            s = tl.load(scales_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)[:, :, None]
            z = tl.load(zeros_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)[:, :, None]

            w_tile = tl.reshape((q_grp * s - z).to(tl.float16), (BLOCK_N, 64))
            w_tile = tl.where(mask_k64[None, :], w_tile, tl.zeros((1, 1), dtype=tl.float16))
            acc += tl.dot(a_tile, tl.trans(w_tile))

        if HAS_BIAS:
            bias = tl.load(bias_ptr + offs_n64, mask=mask_n, other=0.0).to(tl.float32)
            acc += bias[None, :]

        acc_clamped = tl.minimum(tl.maximum(acc, -65504.0), 65504.0)
        out_ptrs = out_ptr + offs_m64[:, None] * N + offs_n64[None, :]
        tl.store(out_ptrs, acc_clamped.to(tl.float16), mask=mask_m[:, None] & mask_n[None, :])

    @triton.jit
    def _w8a16_gemv_kernel(
        x_ptr,
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        bias_ptr,
        out_ptr,
        M,
        N,
        K,
        HAS_ZEROS: tl.constexpr,
        HAS_BIAS: tl.constexpr,
        BLOCK_N: tl.constexpr,
    ):
        """Coalesced 128-byte `int32` (`ld.global.b32`) W8A16 GEMV kernel for decode (M <= 8)."""
        pid_n = tl.program_id(0)
        pid_m = tl.program_id(1)

        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_n = offs_n < N
        offs_n64 = offs_n.to(tl.int64)

        n_groups = K // 16
        k_i32 = K // 4
        lanes32 = tl.arange(0, 32)
        offs_8 = tl.arange(0, 8)

        acc = tl.zeros((BLOCK_N,), dtype=tl.float32)
        x_row_ptr = x_ptr + pid_m.to(tl.int64) * K

        for g_base in range(0, n_groups, 8):
            offs_i32 = g_base * 4 + lanes32
            mask_k = offs_i32 < k_i32

            w32 = tl.load(
                qw_i32_ptr + offs_n64[:, None] * k_i32 + offs_i32[None, :],
                mask=mask_n[:, None] & mask_k[None, :],
                other=0,
            )
            q0 = ((w32 << 24) >> 24).to(tl.float32)
            q1 = ((w32 << 16) >> 24).to(tl.float32)
            q2 = ((w32 << 8) >> 24).to(tl.float32)
            q3 = (w32 >> 24).to(tl.float32)

            x_idx = offs_i32 * 4
            x0 = tl.load(x_row_ptr + x_idx + 0, mask=mask_k, other=0.0).to(tl.float32)
            x1 = tl.load(x_row_ptr + x_idx + 1, mask=mask_k, other=0.0).to(tl.float32)
            x2 = tl.load(x_row_ptr + x_idx + 2, mask=mask_k, other=0.0).to(tl.float32)
            x3 = tl.load(x_row_ptr + x_idx + 3, mask=mask_k, other=0.0).to(tl.float32)

            dot32 = q0 * x0[None, :] + q1 * x1[None, :] + q2 * x2[None, :] + q3 * x3[None, :]
            dot_g = tl.sum(tl.reshape(dot32, (BLOCK_N, 8, 4)), axis=2)

            offs_g = g_base + offs_8
            mask_g = offs_g < n_groups
            sg_ptrs = offs_n64[:, None] * n_groups + offs_g[None, :]
            s = tl.load(scales_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)

            if HAS_ZEROS:
                x_g_sum = tl.sum(tl.reshape(x0 + x1 + x2 + x3, (8, 4)), axis=1)
                z = tl.load(zeros_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)
                acc += tl.sum(s * dot_g - z * x_g_sum[None, :], axis=1)
            else:
                acc += tl.sum(s * dot_g, axis=1)

        if HAS_BIAS:
            bias = tl.load(bias_ptr + offs_n64, mask=mask_n, other=0.0).to(tl.float32)
            acc += bias

        acc_clamped = tl.minimum(tl.maximum(acc, -65504.0), 65504.0)
        tl.store(out_ptr + pid_m.to(tl.int64) * N + offs_n64, acc_clamped.to(tl.float16), mask=mask_n)

    @triton.jit
    def _w8a16_multirow_gemv_kernel(
        x_ptr,
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        bias_ptr,
        out_ptr,
        M,
        N,
        K,
        HAS_ZEROS: tl.constexpr,
        HAS_BIAS: tl.constexpr,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
    ):
        """Weight-reusing multi-row (`2 <= M <= 8`) W8A16 GEMV kernel for MTP & N-gram speculative verify!"""
        pid_n = tl.program_id(0)

        offs_m = tl.arange(0, BLOCK_M)
        mask_m = offs_m < M
        offs_m64 = offs_m.to(tl.int64)

        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_n = offs_n < N
        offs_n64 = offs_n.to(tl.int64)

        n_groups = K // 16
        k_i32 = K // 4
        lanes32 = tl.arange(0, 32)
        offs_8 = tl.arange(0, 8)

        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

        for g_base in range(0, n_groups, 8):
            offs_i32 = g_base * 4 + lanes32
            mask_k = offs_i32 < k_i32
            mask_mk = mask_m[:, None] & mask_k[None, :]

            w32 = tl.load(
                qw_i32_ptr + offs_n64[:, None] * k_i32 + offs_i32[None, :],
                mask=mask_n[:, None] & mask_k[None, :],
                other=0,
            )
            q0 = ((w32 << 24) >> 24).to(tl.float32)
            q1 = ((w32 << 16) >> 24).to(tl.float32)
            q2 = ((w32 << 8) >> 24).to(tl.float32)
            q3 = (w32 >> 24).to(tl.float32)

            x_idx = offs_i32 * 4
            x_base = x_ptr + offs_m64[:, None] * K + x_idx[None, :]
            x0 = tl.load(x_base + 0, mask=mask_mk, other=0.0).to(tl.float32)
            x1 = tl.load(x_base + 1, mask=mask_mk, other=0.0).to(tl.float32)
            x2 = tl.load(x_base + 2, mask=mask_mk, other=0.0).to(tl.float32)
            x3 = tl.load(x_base + 3, mask=mask_mk, other=0.0).to(tl.float32)

            dot32 = (
                q0[None, :, :] * x0[:, None, :]
                + q1[None, :, :] * x1[:, None, :]
                + q2[None, :, :] * x2[:, None, :]
                + q3[None, :, :] * x3[:, None, :]
            )
            dot_g = tl.sum(tl.reshape(dot32, (BLOCK_M, BLOCK_N, 8, 4)), axis=3)

            offs_g = g_base + offs_8
            mask_g = offs_g < n_groups
            sg_ptrs = offs_n64[:, None] * n_groups + offs_g[None, :]
            s = tl.load(scales_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)

            if HAS_ZEROS:
                x_g_sum = tl.sum(tl.reshape(x0 + x1 + x2 + x3, (BLOCK_M, 8, 4)), axis=2)
                z = tl.load(zeros_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)
                acc += tl.sum(s[None, :, :] * dot_g - z[None, :, :] * x_g_sum[:, None, :], axis=2)
            else:
                acc += tl.sum(s[None, :, :] * dot_g, axis=2)

        if HAS_BIAS:
            bias = tl.load(bias_ptr + offs_n64, mask=mask_n, other=0.0).to(tl.float32)
            acc += bias[None, :]

        acc_clamped = tl.minimum(tl.maximum(acc, -65504.0), 65504.0)
        tl.store(
            out_ptr + offs_m64[:, None] * N + offs_n64[None, :],
            acc_clamped.to(tl.float16),
            mask=mask_m[:, None] & mask_n[None, :],
        )

    @triton.jit
    def _w8a16_gemm_tc_kernel(
        x_ptr,
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        bias_ptr,
        out_ptr,
        M,
        N,
        K,
        HAS_ZEROS: tl.constexpr,
        HAS_BIAS: tl.constexpr,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
    ):
        """Coalesced `int32` W8A16 Tensor-Core GEMM kernel (`BLOCK_K=64`) for Ampere+ (`sm_80+`)."""
        pid_m = tl.program_id(0)
        pid_n = tl.program_id(1)

        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_m = offs_m < M
        mask_n = offs_n < N
        offs_m64 = offs_m.to(tl.int64)
        offs_n64 = offs_n.to(tl.int64)

        n_groups = K // 16
        k_i32 = K // 4
        lanes16 = tl.arange(0, 16)
        lanes64 = tl.arange(0, 64)
        offs_4 = tl.arange(0, 4)

        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

        for g_base in range(0, n_groups, 4):
            offs_k = g_base * 16 + lanes64
            mask_k64 = offs_k < K
            a_tile = tl.load(
                x_ptr + offs_m64[:, None] * K + offs_k[None, :],
                mask=mask_m[:, None] & mask_k64[None, :],
                other=0.0,
            ).to(tl.float16)

            offs_i32 = g_base * 4 + lanes16
            mask_i32 = offs_i32 < k_i32
            w32 = tl.load(
                qw_i32_ptr + offs_n64[:, None] * k_i32 + offs_i32[None, :],
                mask=mask_n[:, None] & mask_i32[None, :],
                other=0,
            )

            q0 = ((w32 << 24) >> 24).to(tl.float32)
            q1 = ((w32 << 16) >> 24).to(tl.float32)
            q2 = ((w32 << 8) >> 24).to(tl.float32)
            q3 = (w32 >> 24).to(tl.float32)
            q_grp = tl.reshape(tl.join(tl.join(q0, q2), tl.join(q1, q3)), (BLOCK_N, 4, 16))

            offs_g = g_base + offs_4
            mask_g = offs_g < n_groups
            sg_ptrs = offs_n64[:, None] * n_groups + offs_g[None, :]
            s = tl.load(scales_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)[:, :, None]
            if HAS_ZEROS:
                z = tl.load(zeros_ptr + sg_ptrs, mask=mask_n[:, None] & mask_g[None, :], other=0.0).to(tl.float32)[:, :, None]
                w_grp = (q_grp * s - z).to(tl.float16)
            else:
                w_grp = (q_grp * s).to(tl.float16)

            w_tile = tl.reshape(w_grp, (BLOCK_N, 64))
            w_tile = tl.where(mask_k64[None, :], w_tile, tl.zeros((1, 1), dtype=tl.float16))
            acc += tl.dot(a_tile, tl.trans(w_tile))

        if HAS_BIAS:
            bias = tl.load(bias_ptr + offs_n64, mask=mask_n, other=0.0).to(tl.float32)
            acc += bias[None, :]

        acc_clamped = tl.minimum(tl.maximum(acc, -65504.0), 65504.0)
        out_ptrs = out_ptr + offs_m64[:, None] * N + offs_n64[None, :]
        tl.store(out_ptrs, acc_clamped.to(tl.float16), mask=mask_m[:, None] & mask_n[None, :])

    @triton.jit
    def _dequant_w4_chunk_kernel(
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        out_fp16_ptr,
        rc,
        K,
        BLOCK_R: tl.constexpr,
    ):
        """Fast coalesced W4 -> FP16 chunk unpacker feeding Turing (`sm_75`) cuBLAS Tensor Cores during prefill."""
        pid_r = tl.program_id(0)
        pid_g = tl.program_id(1)

        offs_r = pid_r * BLOCK_R + tl.arange(0, BLOCK_R)
        mask_r = offs_r < rc
        offs_r64 = offs_r.to(tl.int64)

        n_groups = K // 32
        k_i32 = K // 8
        lanes4 = tl.arange(0, 4)
        lanes16 = tl.arange(0, 16)

        w32 = tl.load(
            qw_i32_ptr + offs_r64[:, None] * k_i32 + (pid_g * 4 + lanes4)[None, :],
            mask=mask_r[:, None],
            other=0,
        )
        s = tl.load(scales_ptr + offs_r64 * n_groups + pid_g, mask=mask_r, other=0.0).to(tl.float32)[:, None]
        z = tl.load(zeros_ptr + offs_r64 * n_groups + pid_g, mask=mask_r, other=0.0).to(tl.float32)[:, None]

        lo_ev = tl.join((w32 >> 0) & 0x0F, (w32 >> 16) & 0x0F)
        lo_od = tl.join((w32 >> 8) & 0x0F, (w32 >> 24) & 0x0F)
        q_lo = (tl.reshape(tl.join(lo_ev, lo_od), (BLOCK_R, 16)).to(tl.float32) * s - z).to(tl.float16)

        hi_ev = tl.join((w32 >> 4) & 0x0F, (w32 >> 20) & 0x0F)
        hi_od = tl.join((w32 >> 12) & 0x0F, (w32 >> 28) & 0x0F)
        q_hi = (tl.reshape(tl.join(hi_ev, hi_od), (BLOCK_R, 16)).to(tl.float32) * s - z).to(tl.float16)

        base_out = out_fp16_ptr + offs_r64[:, None] * K + pid_g * 32
        tl.store(base_out + lanes16[None, :], q_lo, mask=mask_r[:, None])
        tl.store(base_out + 16 + lanes16[None, :], q_hi, mask=mask_r[:, None])

    @triton.jit
    def _dequant_w8_chunk_kernel(
        qw_i32_ptr,
        scales_ptr,
        zeros_ptr,
        out_fp16_ptr,
        rc,
        K,
        HAS_ZEROS: tl.constexpr,
        BLOCK_R: tl.constexpr,
    ):
        """Fast coalesced W8 -> FP16 chunk unpacker feeding Turing (`sm_75`) cuBLAS Tensor Cores during prefill."""
        pid_r = tl.program_id(0)
        pid_g = tl.program_id(1)

        offs_r = pid_r * BLOCK_R + tl.arange(0, BLOCK_R)
        mask_r = offs_r < rc
        offs_r64 = offs_r.to(tl.int64)

        n_groups = K // 16
        k_i32 = K // 4
        lanes4 = tl.arange(0, 4)
        lanes16 = tl.arange(0, 16)

        w32 = tl.load(
            qw_i32_ptr + offs_r64[:, None] * k_i32 + (pid_g * 4 + lanes4)[None, :],
            mask=mask_r[:, None],
            other=0,
        )
        q0 = ((w32 << 24) >> 24).to(tl.float32)
        q1 = ((w32 << 16) >> 24).to(tl.float32)
        q2 = ((w32 << 8) >> 24).to(tl.float32)
        q3 = (w32 >> 24).to(tl.float32)
        q_16 = tl.reshape(tl.join(tl.join(q0, q2), tl.join(q1, q3)), (BLOCK_R, 16))

        s = tl.load(scales_ptr + offs_r64 * n_groups + pid_g, mask=mask_r, other=0.0).to(tl.float32)[:, None]
        if HAS_ZEROS:
            z = tl.load(zeros_ptr + offs_r64 * n_groups + pid_g, mask=mask_r, other=0.0).to(tl.float32)[:, None]
            w_16 = (q_16 * s - z).to(tl.float16)
        else:
            w_16 = (q_16 * s).to(tl.float16)

        base_out = out_fp16_ptr + offs_r64[:, None] * K + pid_g * 16 + lanes16[None, :]
        tl.store(base_out, w_16, mask=mask_r[:, None])

    @triton.jit
    def _fused_act_mul_rowscale_kernel(
        gate_ptr,
        up_ptr,
        out_fp16_ptr,
        row_scale_ptr,
        M,
        I,
        IS_CONCAT: tl.constexpr,
        USE_GELU: tl.constexpr,
        BLOCK_I: tl.constexpr,
    ):
        """
        Fused SwiGLU / GeGLU + dynamic FP16 overflow row-scaling (`max(|inter|)/16384 >= 1.0`) in 1 launch.
        Replaces 9 eager PyTorch launches per MLP layer with a single L1-cached pass.
        """
        pid_m = tl.program_id(0).to(tl.int64)
        g_row_stride = (2 * I) if IS_CONCAT else I
        g_base = gate_ptr + pid_m * g_row_stride
        u_base = (gate_ptr + pid_m * g_row_stride + I) if IS_CONCAT else (up_ptr + pid_m * I)
        out_base = out_fp16_ptr + pid_m * I

        row_max = tl.zeros((), dtype=tl.float32)
        for col_off in range(0, I, BLOCK_I):
            offs = col_off + tl.arange(0, BLOCK_I)
            mask = offs < I
            g = tl.load(g_base + offs, mask=mask, other=0.0).to(tl.float32)
            u = tl.load(u_base + offs, mask=mask, other=0.0).to(tl.float32)
            if USE_GELU:
                inner = 0.7978845608028654 * (g + 0.044715 * g * g * g)
                inner_c = tl.maximum(tl.minimum(inner, 20.0), -20.0)
                th = (2.0 / (1.0 + tl.exp(-2.0 * inner_c))) - 1.0
                act = 0.5 * g * (1.0 + th)
            else:
                act = g / (1.0 + tl.exp(-g))
            inter = act * u
            row_max = tl.maximum(row_max, tl.max(tl.abs(inter), axis=0))

        scale = tl.maximum(row_max * (1.0 / 16384.0), 1.0)
        inv_scale = 1.0 / scale
        tl.store(row_scale_ptr + pid_m, scale)

        for col_off in range(0, I, BLOCK_I):
            offs = col_off + tl.arange(0, BLOCK_I)
            mask = offs < I
            g = tl.load(g_base + offs, mask=mask, other=0.0).to(tl.float32)
            u = tl.load(u_base + offs, mask=mask, other=0.0).to(tl.float32)
            if USE_GELU:
                inner = 0.7978845608028654 * (g + 0.044715 * g * g * g)
                inner_c = tl.maximum(tl.minimum(inner, 20.0), -20.0)
                th = (2.0 / (1.0 + tl.exp(-2.0 * inner_c))) - 1.0
                act = 0.5 * g * (1.0 + th)
            else:
                act = g / (1.0 + tl.exp(-g))
            scaled = (act * u * inv_scale).to(tl.float16)
            tl.store(out_base + offs, scaled, mask=mask)

    @triton.jit
    def _fused_rmsnorm_silu_gate_kernel(
        y_ptr,
        z_ptr,
        w_ptr,
        out_fp16_ptr,
        D,
        eps,
        BLOCK_D: tl.constexpr,
    ):
        """Fused `RMSNorm(y) * SiLU(z) -> fp16` across `S * H_v` head rows in 1 launch."""
        pid_r = tl.program_id(0).to(tl.int64)
        offs = tl.arange(0, BLOCK_D)
        mask = offs < D
        base = pid_r * D + offs

        y = tl.load(y_ptr + base, mask=mask, other=0.0).to(tl.float32)
        z = tl.load(z_ptr + base, mask=mask, other=0.0).to(tl.float32)
        w = tl.load(w_ptr + offs, mask=mask, other=0.0).to(tl.float32)

        rms = tl.rsqrt(tl.sum(y * y, axis=0) * (1.0 / D) + eps)
        silu_z = z / (1.0 + tl.exp(-z))
        out = (y * rms * w * silu_z).to(tl.float16)
        tl.store(out_fp16_ptr + base, out, mask=mask)

    @triton.jit
    def _fused_rmsnorm_kernel(
        x_ptr,
        delta_ptr,
        w_ptr,
        res_out_ptr,
        out_ptr,
        D,
        eps,
        HAS_RESIDUAL: tl.constexpr,
        OUT_FP16: tl.constexpr,
        BLOCK_D: tl.constexpr,
    ):
        """
        Single-launch FP32-accurate RMSNorm + optional Residual-Add (`h = x + delta -> fp32`,
        `out = RMSNorm(h, w) -> fp16/fp32`). Collapses 3-5 PyTorch elementwise/cast launches
        per norm (~400+ launches per forward pass on 64-layer models) into 1 register-resident kernel.
        """
        pid_r = tl.program_id(0).to(tl.int64)
        offs = tl.arange(0, BLOCK_D)
        mask = offs < D
        base = pid_r * D + offs

        x = tl.load(x_ptr + base, mask=mask, other=0.0).to(tl.float32)
        if HAS_RESIDUAL:
            d = tl.load(delta_ptr + base, mask=mask, other=0.0).to(tl.float32)
            x = x + d
            tl.store(res_out_ptr + base, x, mask=mask)

        w = tl.load(w_ptr + offs, mask=mask, other=0.0).to(tl.float32)
        rms = tl.rsqrt(tl.sum(x * x, axis=0) * (1.0 / D) + eps)
        normed = x * rms * w
        if OUT_FP16:
            tl.store(out_ptr + base, normed.to(tl.float16), mask=mask)
        else:
            tl.store(out_ptr + base, normed, mask=mask)


# ==============================================================================
# 2. GGUF Block Decoders -> GPU-Coalesced SoA Buffers (Streaming Tensor-by-Tensor)
# ==============================================================================

def _unpack_k_scales_12b(scales_12b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Unpack 12-byte K-quant scale/min headers (used by Q4_K and Q5_K) into [n_blocks, 8] uint8 pairs."""
    n_blocks = scales_12b.shape[0]
    s = scales_12b.reshape(n_blocks, 3, 4)
    d_b, m_b, md_b = s[:, 0, :], s[:, 1, :], s[:, 2, :]
    sc = np.concatenate([d_b & 0x3F, (md_b & 0x0F) | ((d_b >> 2) & 0x30)], axis=-1)
    mn = np.concatenate([m_b & 0x3F, (md_b >> 4) | ((m_b >> 2) & 0x30)], axis=-1)
    return sc, mn


def repack_gguf_to_w4a16(
    raw_bytes: np.ndarray,
    qtype: int,
    rows: int,
    cols: int,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Repack Q4_0, Q4_1, or Q4_K raw GGUF bytes into coalesced GPU SoA tensors:
      qweight: uint8[rows, cols // 2]
      scales : float16[rows, cols // 32]
      zeros  : float16[rows, cols // 32]
    Exact formula: w = scale * q - zero.
    """
    if cols % 32 != 0:
        raise ValueError(f"W4A16 requires cols ({cols}) divisible by 32")

    if qtype == GGMLQuantType.Q4_0:
        blocks = raw_bytes.reshape(-1, 18)
        d = np.frombuffer(np.ascontiguousarray(blocks[:, 0:2]), dtype=np.float16).reshape(rows, cols // 32)
        z = (d.astype(np.float32) * 8.0).astype(np.float16)
        qw = np.ascontiguousarray(blocks[:, 2:18]).reshape(rows, cols // 2)

    elif qtype == GGMLQuantType.Q4_1:
        blocks = raw_bytes.reshape(-1, 20)
        d = np.frombuffer(np.ascontiguousarray(blocks[:, 0:2]), dtype=np.float16).reshape(rows, cols // 32)
        m = np.frombuffer(np.ascontiguousarray(blocks[:, 2:4]), dtype=np.float16).reshape(rows, cols // 32)
        z = (-m.astype(np.float32)).astype(np.float16)
        qw = np.ascontiguousarray(blocks[:, 4:20]).reshape(rows, cols // 2)

    elif qtype == GGMLQuantType.Q4_K:
        if cols % 256 != 0:
            raise ValueError(f"Q4_K requires cols ({cols}) divisible by 256")
        blocks = raw_bytes.reshape(-1, 144)
        n_blocks = blocks.shape[0]
        d = np.frombuffer(np.ascontiguousarray(blocks[:, 0:2]), dtype=np.float16).astype(np.float32).reshape(n_blocks, 1)
        dmin = (
            np.frombuffer(np.ascontiguousarray(blocks[:, 2:4]), dtype=np.float16)
            .astype(np.float32)
            .reshape(n_blocks, 1)
        )
        sc, mn = _unpack_k_scales_12b(blocks[:, 4:16])
        d_sub = (d * sc.astype(np.float32)).astype(np.float16).reshape(rows, cols // 32)
        z_sub = (dmin * mn.astype(np.float32)).astype(np.float16).reshape(rows, cols // 32)

        qs = blocks[:, 16:144].reshape(n_blocks, 4, 32)
        q_even = qs & np.uint8(0x0F)
        q_odd = (qs >> np.uint8(4)) & np.uint8(0x0F)
        q_unpacked = np.stack([q_even, q_odd], axis=2).reshape(n_blocks, 8, 32)
        qw = (q_unpacked[:, :, :16] | (q_unpacked[:, :, 16:] << np.uint8(4))).reshape(rows, cols // 2)
        d, z = d_sub, z_sub

    else:
        # Streaming row-chunked W4A16 repacker for sub-5-bit dynamic quants (IQ4_XS, IQ4_NL, Q3_K, Q2_K, IQ3_*, etc.)
        # Streams 4,096 rows at a time (<85 MB CPU RAM peak even on 248K vocab embeddings) directly onto GPU!
        import gguf.quants as gq

        block_elems, block_bytes = GGML_BLOCK_INFO[int(qtype)]
        gq_type = gq.GGMLQuantizationType(int(qtype))
        qw_gpu = torch.empty((rows, cols // 2), device=device, dtype=torch.uint8)
        sc_gpu = torch.empty((rows, cols // 32), device=device, dtype=torch.float16)
        zr_gpu = torch.empty((rows, cols // 32), device=device, dtype=torch.float16)

        chunk_rows = 4096  # Multiple of 256 ensures exact GGML block alignment for every row chunk
        for r0 in range(0, rows, chunk_rows):
            r1 = min(rows, r0 + chunk_rows)
            rc = r1 - r0
            b0 = ((r0 * cols) // block_elems) * block_bytes
            b1 = ((r1 * cols) // block_elems) * block_bytes
            fp32 = gq.dequantize(raw_bytes[b0:b1], gq_type).reshape(rc, cols // 32, 32)

            w_min = np.min(fp32, axis=-1, keepdims=True)
            w_max = np.max(fp32, axis=-1, keepdims=True)
            scale = (w_max - w_min) * (1.0 / 15.0)
            scale_safe = np.where(scale == 0.0, 1.0, scale)
            q_u8 = np.clip(np.round((fp32 - w_min) / scale_safe), 0, 15).astype(np.uint8)
            qw_chunk = (q_u8[:, :, :16] | (q_u8[:, :, 16:] << np.uint8(4))).reshape(rc, cols // 2)

            qw_gpu[r0:r1].copy_(torch.from_numpy(np.ascontiguousarray(qw_chunk)))
            sc_gpu[r0:r1].copy_(torch.from_numpy(np.ascontiguousarray(scale.squeeze(-1).astype(np.float16))))
            zr_gpu[r0:r1].copy_(torch.from_numpy(np.ascontiguousarray((-w_min.squeeze(-1)).astype(np.float16))))

        return qw_gpu, sc_gpu, zr_gpu

    return (
        torch.from_numpy(np.ascontiguousarray(qw)).to(device=device, dtype=torch.uint8),
        torch.from_numpy(np.ascontiguousarray(d)).to(device=device, dtype=torch.float16),
        torch.from_numpy(np.ascontiguousarray(z)).to(device=device, dtype=torch.float16),
    )


def repack_gguf_to_w8a16(
    raw_bytes: np.ndarray,
    qtype: int,
    rows: int,
    cols: int,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    """
    Repack Q8_0, Q6_K, Q5_K (or any fallback GGUF quant) into coalesced W8A16 GPU tensors:
      qweight: int8[rows, cols]
      scales : float16[rows, cols // 16]
      zeros  : Optional[float16[rows, cols // 16]]
    """
    if cols % 16 != 0:
        raise ValueError(f"W8A16 requires cols ({cols}) divisible by 16")

    if qtype == GGMLQuantType.Q8_0:
        blocks = raw_bytes.reshape(-1, 34)
        d32 = np.frombuffer(np.ascontiguousarray(blocks[:, 0:2]), dtype=np.float16).reshape(-1, 1)
        scales = np.repeat(d32, 2, axis=1).reshape(rows, cols // 16)
        qw = np.ascontiguousarray(blocks[:, 2:34]).view(np.int8).reshape(rows, cols)
        zeros = None

    elif qtype == GGMLQuantType.Q6_K:
        blocks = raw_bytes.reshape(-1, 210)
        n_blocks = blocks.shape[0]
        ql = blocks[:, 0:128].reshape(n_blocks, 2, 1, 64)
        qh = blocks[:, 128:192].reshape(n_blocks, 2, 1, 32)
        sc = np.ascontiguousarray(blocks[:, 192:208]).view(np.int8).astype(np.float32)
        d = np.frombuffer(np.ascontiguousarray(blocks[:, 208:210]), dtype=np.float16).astype(np.float32).reshape(n_blocks, 1)

        ql_u = ((ql >> np.array([0, 4], dtype=np.uint8).reshape(1, 1, 2, 1)) & np.uint8(0x0F)).reshape(n_blocks, 8, 32)
        qh_u = ((qh >> np.array([0, 2, 4, 6], dtype=np.uint8).reshape(1, 1, 4, 1)) & np.uint8(0x03)).reshape(
            n_blocks, 8, 32
        )
        qw = ((ql_u | (qh_u << np.uint8(4))).astype(np.int8) - np.int8(32)).reshape(rows, cols)
        scales = (d * sc).astype(np.float16).reshape(rows, cols // 16)
        zeros = None

    elif qtype == GGMLQuantType.Q5_K:
        blocks = raw_bytes.reshape(-1, 176)
        n_blocks = blocks.shape[0]
        d = np.frombuffer(np.ascontiguousarray(blocks[:, 0:2]), dtype=np.float16).astype(np.float32).reshape(n_blocks, 1)
        dmin = (
            np.frombuffer(np.ascontiguousarray(blocks[:, 2:4]), dtype=np.float16)
            .astype(np.float32)
            .reshape(n_blocks, 1)
        )
        sc, mn = _unpack_k_scales_12b(blocks[:, 4:16])
        qh_raw = blocks[:, 16:48].reshape(n_blocks, 1, 1, 32)
        qs_raw = blocks[:, 48:176].reshape(n_blocks, 4, 1, 32)

        ql_u = ((qs_raw >> np.array([0, 4], dtype=np.uint8).reshape(1, 1, 2, 1)) & np.uint8(0x0F)).reshape(
            n_blocks, 8, 32
        )
        qh_u = ((qh_raw >> np.arange(8, dtype=np.uint8).reshape(1, 1, 8, 1)) & np.uint8(0x01)).reshape(n_blocks, 8, 32)
        qw = (ql_u | (qh_u << np.uint8(4))).astype(np.int8).reshape(rows, cols)

        scales = np.repeat((d * sc.astype(np.float32)).astype(np.float16), 2, axis=1).reshape(rows, cols // 16)
        zeros = np.repeat((dmin * mn.astype(np.float32)).astype(np.float16), 2, axis=1).reshape(rows, cols // 16)

    else:
        # Streaming row-chunked fallback via gguf.quants.dequantize for 5/8-bit formats (Q5_0, Q5_1, Q8_1, Q8_K)
        import gguf.quants as gq

        block_elems, block_bytes = GGML_BLOCK_INFO[int(qtype)]
        gq_type = gq.GGMLQuantizationType(int(qtype))
        qw_gpu = torch.empty((rows, cols), device=device, dtype=torch.int8)
        sc_gpu = torch.empty((rows, cols // 16), device=device, dtype=torch.float16)

        chunk_rows = 4096
        for r0 in range(0, rows, chunk_rows):
            r1 = min(rows, r0 + chunk_rows)
            rc = r1 - r0
            b0 = ((r0 * cols) // block_elems) * block_bytes
            b1 = ((r1 * cols) // block_elems) * block_bytes
            grouped = gq.dequantize(raw_bytes[b0:b1], gq_type).reshape(rc, cols // 16, 16)
            amax = np.max(np.abs(grouped), axis=-1, keepdims=True)
            sc_f32 = np.where(amax == 0, 1.0, amax / 127.0)
            qw_chunk = np.clip(np.round(grouped / sc_f32), -127, 127).astype(np.int8).reshape(rc, cols)
            qw_gpu[r0:r1].copy_(torch.from_numpy(np.ascontiguousarray(qw_chunk)))
            sc_gpu[r0:r1].copy_(torch.from_numpy(np.ascontiguousarray(sc_f32.squeeze(-1).astype(np.float16))))

        return qw_gpu, sc_gpu, None

    return (
        torch.from_numpy(np.ascontiguousarray(qw)).to(device=device, dtype=torch.int8),
        torch.from_numpy(np.ascontiguousarray(scales)).to(device=device, dtype=torch.float16),
        torch.from_numpy(np.ascontiguousarray(zeros)).to(device=device, dtype=torch.float16)
        if zeros is not None
        else None,
    )


def load_float_tensor(gguf_file: GGUFFile, name: str, device: torch.device, dtype: torch.dtype = torch.float32) -> torch.Tensor:
    """Load a 1D/2D unquantized (or small dequantized) tensor such as RMSNorm weights, biases, conv1d, or router gates."""
    info = gguf_file.tensors[name]
    raw = gguf_file.get_tensor_bytes(name)
    if info.qtype == GGMLQuantType.F32:
        arr = np.frombuffer(raw, dtype=np.float32).reshape(info.shape).copy()
        return torch.from_numpy(arr).to(device=device, dtype=dtype)
    if info.qtype == GGMLQuantType.F16:
        arr = np.frombuffer(raw, dtype=np.float16).reshape(info.shape).copy()
        return torch.from_numpy(arr).to(device=device, dtype=dtype)
    if info.qtype == GGMLQuantType.BF16:
        u16 = np.frombuffer(raw, dtype=np.uint16).reshape(info.shape).copy()
        t_bf16 = torch.from_numpy(u16).view(torch.bfloat16)
        return t_bf16.to(device=device, dtype=dtype)

    # Small quantized tensor (including narrow shapes like conv1d [C, 4]) -> direct dequantize
    import gguf.quants as gq

    fp32 = gq.dequantize(raw, gq.GGMLQuantizationType(int(info.qtype))).reshape(info.shape).copy()
    return torch.from_numpy(fp32).to(device=device, dtype=dtype)


# ==============================================================================
# 3. PyTorch / Triton Execution Helpers (Multi-GPU Safe + sm_75 cuBLAS Prefill)
# ==============================================================================

_CUDA_CC_MAJOR: Dict[torch.device, int] = {}
_DEQUANT_SCRATCH: Dict[torch.device, torch.Tensor] = {}


@contextmanager
def cuda_device_scope(dev: torch.device) -> Iterator[None]:
    """
    Eagerly bind the NVIDIA CUDA driver `CUcontext` (`torch.cuda.set_device`) AND PyTorch's
    device context so OpenAI Triton's `cuPointerGetAttribute` (`driver.c`) always validates
    and launches kernels on `dev`'s primary context — even across Dual T4 GPUs without PCIe P2P
    and inside background FastAPI worker threads!
    """
    if dev.type != "cuda":
        yield
        return
    idx = dev.index if dev.index is not None else torch.cuda.current_device()
    prev = torch.cuda.current_device()
    torch.cuda.set_device(idx)
    try:
        with torch.cuda.device(idx):
            yield
    finally:
        if prev != idx:
            torch.cuda.set_device(prev)


def _get_cuda_major(dev: torch.device) -> int:
    if dev not in _CUDA_CC_MAJOR:
        major, _ = torch.cuda.get_device_capability(dev)
        _CUDA_CC_MAJOR[dev] = int(major)
    return _CUDA_CC_MAJOR[dev]


def _get_scratch_fp16(dev: torch.device, rows: int, cols: int) -> torch.Tensor:
    needed = rows * cols
    buf = _DEQUANT_SCRATCH.get(dev)
    if buf is None or buf.numel() < needed:
        buf = torch.empty((needed,), device=dev, dtype=torch.float16)
        _DEQUANT_SCRATCH[dev] = buf
    return buf[:needed].view(rows, cols)


def exec_w4a16_matmul(
    x_2d: torch.Tensor,
    qweight: torch.Tensor,
    scales: torch.Tensor,
    zeros: torch.Tensor,
    bias: Optional[torch.Tensor] = None,
    qw_i32_cached: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Execute x_2d [M, K] @ W^T [K, N] -> [M, N] using Triton on `qweight.device` or PyTorch on CPU."""
    M, K = x_2d.shape
    N = qweight.shape[0]
    dev = qweight.device
    if x_2d.device != dev:
        x_2d = x_2d.to(dev, non_blocking=True)

    if HAS_TRITON and dev.type == "cuda":
        with cuda_device_scope(dev):
            x_fp16 = x_2d if x_2d.dtype == torch.float16 else x_2d.to(torch.float16)
            x_contig = x_fp16 if x_fp16.is_contiguous() else x_fp16.contiguous()
            qw_i32 = qw_i32_cached if qw_i32_cached is not None else qweight.contiguous().view(torch.int32)
            has_bias = bias is not None
            b_ptr = bias if bias is not None else scales

            if M == 1:
                out = torch.empty((1, N), device=dev, dtype=torch.float16)
                BLOCK_N = 4
                _w4a16_gemv_kernel[(triton.cdiv(N, BLOCK_N), 1)](
                    x_contig,
                    qw_i32,
                    scales,
                    zeros,
                    b_ptr,
                    out,
                    1,
                    N,
                    K,
                    HAS_BIAS=has_bias,
                    BLOCK_N=BLOCK_N,
                    num_warps=4,
                )
                return out if x_2d.dtype == torch.float16 else out.to(x_2d.dtype)

            if M <= 8:
                out = torch.empty((M, N), device=dev, dtype=torch.float16)
                BLOCK_M = 2 if M <= 2 else (4 if M <= 4 else 8)
                BLOCK_N = 4
                _w4a16_multirow_gemv_kernel[(triton.cdiv(N, BLOCK_N),)](
                    x_contig,
                    qw_i32,
                    scales,
                    zeros,
                    b_ptr,
                    out,
                    M,
                    N,
                    K,
                    HAS_BIAS=has_bias,
                    BLOCK_M=BLOCK_M,
                    BLOCK_N=BLOCK_N,
                    num_warps=4,
                )
                return out if x_2d.dtype == torch.float16 else out.to(x_2d.dtype)

            # Prompt Prefill (M > 8):
            # On Turing (`sm_75` Tesla T4, major < 8), Triton lowers `tl.dot` to scalar FP32 FMAs;
            # unpacking 4096-row chunks via Triton into a tiny reusable scratch buffer and calling
            # `F.linear` hits real Turing FP16 Tensor Cores (~58 TFLOPS, >10x faster prefill!).
            if _get_cuda_major(dev) < 8:
                out = torch.empty((M, N), device=dev, dtype=torch.float16)
                step = 4096
                n_groups = K // 32
                for r0 in range(0, N, step):
                    r1 = min(N, r0 + step)
                    rc = r1 - r0
                    w_chunk = _get_scratch_fp16(dev, rc, K)
                    _dequant_w4_chunk_kernel[(triton.cdiv(rc, 16), n_groups)](
                        qw_i32[r0:r1],
                        scales[r0:r1],
                        zeros[r0:r1],
                        w_chunk,
                        rc,
                        K,
                        BLOCK_R=16,
                        num_warps=2,
                    )
                    b_sub = bias[r0:r1] if bias is not None else None
                    out[:, r0:r1] = F.linear(x_contig, w_chunk, b_sub)
                return out.to(x_2d.dtype)

            out = torch.empty((M, N), device=dev, dtype=torch.float16)
            BLOCK_M = 32 if M < 64 else 64
            BLOCK_N = 64
            grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
            _w4a16_gemm_tc_kernel[grid](
                x_contig,
                qw_i32,
                scales,
                zeros,
                b_ptr,
                out,
                M,
                N,
                K,
                HAS_BIAS=has_bias,
                BLOCK_M=BLOCK_M,
                BLOCK_N=BLOCK_N,
                num_warps=4,
                num_stages=2,
            )
            return out.to(x_2d.dtype)

    # Vectorized row-chunked CPU / non-Triton fallback (<16 MB peak scratch RAM even on 250K vocabs)
    n_groups = K // 32
    x_f32 = x_2d.float()
    out_cpu = torch.empty((M, N), device=x_2d.device, dtype=x_2d.dtype)
    step = 4096
    for r0 in range(0, N, step):
        r1 = min(N, r0 + step)
        rc = r1 - r0
        packed = qweight[r0:r1].view(rc, n_groups, 16)
        q_lo = (packed & 0x0F).to(torch.float32)
        q_hi = ((packed >> 4) & 0x0F).to(torch.float32)
        q_full = torch.cat([q_lo, q_hi], dim=-1)
        w = (q_full * scales[r0:r1].float().unsqueeze(-1) - zeros[r0:r1].float().unsqueeze(-1)).reshape(rc, K)
        b = bias[r0:r1].float() if bias is not None else None
        out_cpu[:, r0:r1] = F.linear(x_f32, w, b).to(x_2d.dtype)
    return out_cpu


def exec_w8a16_matmul(
    x_2d: torch.Tensor,
    qweight: torch.Tensor,
    scales: torch.Tensor,
    zeros: Optional[torch.Tensor] = None,
    bias: Optional[torch.Tensor] = None,
    qw_i32_cached: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Execute x_2d [M, K] @ W^T [K, N] -> [M, N] for W8A16 weights."""
    M, K = x_2d.shape
    N = qweight.shape[0]
    dev = qweight.device
    if x_2d.device != dev:
        x_2d = x_2d.to(dev, non_blocking=True)

    if HAS_TRITON and dev.type == "cuda":
        with cuda_device_scope(dev):
            x_fp16 = x_2d if x_2d.dtype == torch.float16 else x_2d.to(torch.float16)
            x_contig = x_fp16 if x_fp16.is_contiguous() else x_fp16.contiguous()
            qw_i32 = qw_i32_cached if qw_i32_cached is not None else qweight.contiguous().view(torch.int32)
            has_zeros = zeros is not None
            has_bias = bias is not None
            z_ptr = zeros if zeros is not None else scales
            b_ptr = bias if bias is not None else scales

            if M == 1:
                out = torch.empty((1, N), device=dev, dtype=torch.float16)
                BLOCK_N = 4
                _w8a16_gemv_kernel[(triton.cdiv(N, BLOCK_N), 1)](
                    x_contig,
                    qw_i32,
                    scales,
                    z_ptr,
                    b_ptr,
                    out,
                    1,
                    N,
                    K,
                    HAS_ZEROS=has_zeros,
                    HAS_BIAS=has_bias,
                    BLOCK_N=BLOCK_N,
                    num_warps=4,
                )
                return out if x_2d.dtype == torch.float16 else out.to(x_2d.dtype)

            if M <= 8:
                out = torch.empty((M, N), device=dev, dtype=torch.float16)
                BLOCK_M = 2 if M <= 2 else (4 if M <= 4 else 8)
                BLOCK_N = 4
                _w8a16_multirow_gemv_kernel[(triton.cdiv(N, BLOCK_N),)](
                    x_contig,
                    qw_i32,
                    scales,
                    z_ptr,
                    b_ptr,
                    out,
                    M,
                    N,
                    K,
                    HAS_ZEROS=has_zeros,
                    HAS_BIAS=has_bias,
                    BLOCK_M=BLOCK_M,
                    BLOCK_N=BLOCK_N,
                    num_warps=4,
                )
                return out if x_2d.dtype == torch.float16 else out.to(x_2d.dtype)

            if _get_cuda_major(dev) < 8:
                out = torch.empty((M, N), device=dev, dtype=torch.float16)
                step = 4096
                n_groups = K // 16
                for r0 in range(0, N, step):
                    r1 = min(N, r0 + step)
                    rc = r1 - r0
                    w_chunk = _get_scratch_fp16(dev, rc, K)
                    z_sub = zeros[r0:r1] if zeros is not None else scales[r0:r1]
                    _dequant_w8_chunk_kernel[(triton.cdiv(rc, 16), n_groups)](
                        qw_i32[r0:r1],
                        scales[r0:r1],
                        z_sub,
                        w_chunk,
                        rc,
                        K,
                        HAS_ZEROS=has_zeros,
                        BLOCK_R=16,
                        num_warps=2,
                    )
                    b_sub = bias[r0:r1] if bias is not None else None
                    out[:, r0:r1] = F.linear(x_contig, w_chunk, b_sub)
                return out.to(x_2d.dtype)

            out = torch.empty((M, N), device=dev, dtype=torch.float16)
            BLOCK_M = 32 if M < 64 else 64
            BLOCK_N = 64
            grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
            _w8a16_gemm_tc_kernel[grid](
                x_contig,
                qw_i32,
                scales,
                z_ptr,
                b_ptr,
                out,
                M,
                N,
                K,
                HAS_ZEROS=has_zeros,
                HAS_BIAS=has_bias,
                BLOCK_M=BLOCK_M,
                BLOCK_N=BLOCK_N,
                num_warps=4,
                num_stages=2,
            )
            return out.to(x_2d.dtype)

    n_groups = K // 16
    x_f32 = x_2d.float()
    out_cpu = torch.empty((M, N), device=x_2d.device, dtype=x_2d.dtype)
    step = 4096
    for r0 in range(0, N, step):
        r1 = min(N, r0 + step)
        rc = r1 - r0
        w = qweight[r0:r1].float().view(rc, n_groups, 16) * scales[r0:r1].float().unsqueeze(-1)
        if zeros is not None:
            w = w - zeros[r0:r1].float().unsqueeze(-1)
        b = bias[r0:r1].float() if bias is not None else None
        out_cpu[:, r0:r1] = F.linear(x_f32, w.reshape(rc, K), b).to(x_2d.dtype)
    return out_cpu


def exec_fused_act_mul_rowscale(
    gate_or_concat: torch.Tensor,
    up: Optional[torch.Tensor] = None,
    use_gelu: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute `inter = act(gate) * up` + dynamic FP16 overflow row-scaling `(inter / row_scale).to(fp16)`
    in a single Triton kernel when on CUDA (or vectorized PyTorch fallback on CPU).
    - If `up is None`, `gate_or_concat` has shape `[..., 2 * I]`.
    - Returns `(down_in_fp16 [..., I], row_scale_fp32 [..., 1])`.
    """
    orig_prefix = gate_or_concat.shape[:-1]
    if up is None:
        I = gate_or_concat.shape[-1] // 2
        g_2d = gate_or_concat.reshape(-1, 2 * I)
        u_2d = None
    else:
        I = gate_or_concat.shape[-1]
        g_2d = gate_or_concat.reshape(-1, I)
        u_2d = up.reshape(-1, I)
    M = g_2d.shape[0]
    dev = g_2d.device

    if HAS_TRITON and dev.type == "cuda":
        with cuda_device_scope(dev):
            g_c = g_2d if g_2d.is_contiguous() else g_2d.contiguous()
            u_c = g_c if u_2d is None else (u_2d if u_2d.is_contiguous() else u_2d.contiguous())
            out_fp16 = torch.empty((M, I), device=dev, dtype=torch.float16)
            row_scale = torch.empty((M, 1), device=dev, dtype=torch.float32)
            BLOCK_I = 1024 if I >= 1024 else triton.next_power_of_2(max(32, I))
            _fused_act_mul_rowscale_kernel[(M,)](
                g_c,
                u_c,
                out_fp16,
                row_scale,
                M,
                I,
                IS_CONCAT=(u_2d is None),
                USE_GELU=use_gelu,
                BLOCK_I=BLOCK_I,
                num_warps=8 if BLOCK_I >= 1024 else 4,
            )
            return out_fp16.reshape(*orig_prefix, I), row_scale.reshape(*orig_prefix, 1)

    if u_2d is None:
        g_f, u_f = g_2d.float().chunk(2, dim=-1)
    else:
        g_f, u_f = g_2d.float(), u_2d.float()
    act = F.gelu(g_f, approximate="tanh") if use_gelu else F.silu(g_f)
    inter = act * u_f
    r_scale = (inter.abs().amax(dim=-1, keepdim=True) / 16384.0).clamp(min=1.0)
    down_in = (inter / r_scale).to(torch.float16)
    return down_in.reshape(*orig_prefix, I), r_scale.reshape(*orig_prefix, 1)


def exec_fused_rmsnorm_silu_gate(
    y: torch.Tensor,
    z: torch.Tensor,
    norm_weight: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    """Compute `(RMSNorm(y, norm_weight, eps) * SiLU(z)).to(float16)` in 1 Triton launch."""
    orig_shape = y.shape
    D = orig_shape[-1]
    R = y.numel() // D
    dev = y.device

    if HAS_TRITON and dev.type == "cuda" and D <= 2048:
        with cuda_device_scope(dev):
            y_c = y.reshape(R, D).contiguous()
            z_c = z.reshape(R, D).contiguous()
            out = torch.empty((R, D), device=dev, dtype=torch.float16)
            BLOCK_D = triton.next_power_of_2(max(32, D))
            _fused_rmsnorm_silu_gate_kernel[(R,)](
                y_c,
                z_c,
                norm_weight,
                out,
                D,
                eps,
                BLOCK_D=BLOCK_D,
                num_warps=4 if BLOCK_D >= 256 else 2,
            )
            return out.reshape(orig_shape)

    normed = F.rms_norm(y.float(), (D,), norm_weight, eps)
    return (normed * F.silu(z.float())).to(torch.float16).reshape(orig_shape)


def exec_fused_rmsnorm(
    x: torch.Tensor,
    norm_weight: torch.Tensor,
    eps: float = 1e-6,
    out_dtype: Optional[torch.dtype] = None,
) -> torch.Tensor:
    """Compute FP32-accurate `RMSNorm(x, norm_weight, eps).to(out_dtype)` in 1 Triton launch."""
    target_dtype = out_dtype if out_dtype is not None else x.dtype
    orig_shape = x.shape
    D = orig_shape[-1]
    R = x.numel() // D
    dev = x.device

    if HAS_TRITON and dev.type == "cuda" and D <= 8192 and target_dtype in (torch.float16, torch.float32):
        with cuda_device_scope(dev):
            x_c = x.reshape(R, D)
            if not x_c.is_contiguous():
                x_c = x_c.contiguous()
            out = torch.empty((R, D), device=dev, dtype=target_dtype)
            BLOCK_D = triton.next_power_of_2(max(32, D))
            num_warps = 2 if BLOCK_D <= 128 else (4 if BLOCK_D <= 2048 else 8)
            _fused_rmsnorm_kernel[(R,)](
                x_c,
                x_c,
                norm_weight,
                x_c,
                out,
                D,
                eps,
                HAS_RESIDUAL=False,
                OUT_FP16=(target_dtype == torch.float16),
                BLOCK_D=BLOCK_D,
                num_warps=num_warps,
            )
            return out.reshape(orig_shape)

    normed = F.rms_norm(x.float(), (D,), norm_weight, eps)
    return normed.to(target_dtype)


def exec_fused_add_rmsnorm(
    x: torch.Tensor,
    delta: torch.Tensor,
    norm_weight: torch.Tensor,
    eps: float = 1e-6,
    out_dtype: torch.dtype = torch.float16,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute `h = x.float() + delta.float()` AND `normed = RMSNorm(h, norm_weight, eps).to(out_dtype)`
    in a single Triton launch. Returns `(h_fp32, normed)`.
    """
    orig_shape = x.shape
    D = orig_shape[-1]
    R = x.numel() // D
    dev = x.device

    if HAS_TRITON and dev.type == "cuda" and D <= 8192 and out_dtype in (torch.float16, torch.float32):
        with cuda_device_scope(dev):
            x_c = x.reshape(R, D)
            if not x_c.is_contiguous():
                x_c = x_c.contiguous()
            d_c = delta.reshape(R, D)
            if not d_c.is_contiguous():
                d_c = d_c.contiguous()
            res_out = torch.empty((R, D), device=dev, dtype=torch.float32)
            out = torch.empty((R, D), device=dev, dtype=out_dtype)
            BLOCK_D = triton.next_power_of_2(max(32, D))
            num_warps = 2 if BLOCK_D <= 128 else (4 if BLOCK_D <= 2048 else 8)
            _fused_rmsnorm_kernel[(R,)](
                x_c,
                d_c,
                norm_weight,
                res_out,
                out,
                D,
                eps,
                HAS_RESIDUAL=True,
                OUT_FP16=(out_dtype == torch.float16),
                BLOCK_D=BLOCK_D,
                num_warps=num_warps,
            )
            return res_out.reshape(orig_shape), out.reshape(orig_shape)

    h = x.float() + delta.float()
    normed = F.rms_norm(h, (D,), norm_weight, eps).to(out_dtype)
    return h, normed


# ==============================================================================
# 4. Quantized Linear, Horizontal Weight Fusion, TP Slicing, & 3D Sparse MoE
# ==============================================================================

class GGUFQuantLinear(nn.Module):
    """
    Drop-in quantized linear layer directly loaded from any GGUF tensor type
    (W4A16 for 4-bit quants, W8A16 for 5/6/8-bit quants, FP16 for unquantized).
    Supports zero-VRAM horizontal fusion (`fuse`) and group-exact Tensor Parallel slicing!
    """

    def __init__(
        self,
        mode: str,
        in_features: int,
        out_features: int,
        qweight: torch.Tensor,
        scales: Optional[torch.Tensor] = None,
        zeros: Optional[torch.Tensor] = None,
        bias: Optional[torch.Tensor] = None,
    ) -> None:
        super().__init__()
        self.mode = mode  # "w4", "w8", or "fp16"
        self.in_features = in_features
        self.out_features = out_features
        qw_c = qweight.contiguous()
        self.register_buffer("qweight", qw_c, persistent=False)
        self.register_buffer("scales", scales.contiguous() if scales is not None else None, persistent=False)
        self.register_buffer("zeros", zeros.contiguous() if zeros is not None else None, persistent=False)
        self.register_buffer("bias", bias.contiguous() if bias is not None else None, persistent=False)
        self.qweight_i32: Optional[torch.Tensor] = (
            qw_c.view(torch.int32) if mode in ("w4", "w8") and qw_c.shape[-1] % 4 == 0 else None
        )

    @classmethod
    def fuse(cls, linears: Sequence["GGUFQuantLinear"]) -> "GGUFQuantLinear":
        """Concatenate 2+ compatible `GGUFQuantLinear` modules along `out_features` (`dim=0`)."""
        if len(linears) == 1:
            return linears[0]
        first = linears[0]
        mode = first.mode
        K = first.in_features
        dev = first.qweight.device
        for lin in linears[1:]:
            if lin.mode != mode or lin.in_features != K or lin.qweight.device != dev:
                raise ValueError("Cannot fuse GGUFQuantLinear modules with mismatched mode/in_features/device")

        total_out = sum(lin.out_features for lin in linears)
        qw_cat = torch.cat([lin.qweight for lin in linears], dim=0)

        sc_cat: Optional[torch.Tensor] = None
        zr_cat: Optional[torch.Tensor] = None
        if mode in ("w4", "w8"):
            sc_cat = torch.cat([lin.scales for lin in linears], dim=0)  # type: ignore[arg-type]
            if any(lin.zeros is not None for lin in linears):
                grp = K // (32 if mode == "w4" else 16)
                zr_parts = [
                    lin.zeros
                    if lin.zeros is not None
                    else torch.zeros((lin.out_features, grp), device=dev, dtype=torch.float16)
                    for lin in linears
                ]
                zr_cat = torch.cat(zr_parts, dim=0)

        bias_cat: Optional[torch.Tensor] = None
        if any(lin.bias is not None for lin in linears):
            b_parts = [
                lin.bias
                if lin.bias is not None
                else torch.zeros((lin.out_features,), device=dev, dtype=torch.float16)
                for lin in linears
            ]
            bias_cat = torch.cat(b_parts, dim=0)

        return cls(mode, K, total_out, qw_cat, sc_cat, zr_cat, bias=bias_cat)

    def slice_output_rows(
        self,
        row_sel: Union[slice, Sequence[int], torch.Tensor],
        target_device: torch.device,
    ) -> "GGUFQuantLinear":
        """Extract a subset of output rows (`dim=0`) onto `target_device` with zero re-quantization."""
        if isinstance(row_sel, slice):
            qw = self.qweight[row_sel].to(target_device)
            sc = self.scales[row_sel].to(target_device) if self.scales is not None else None
            zr = self.zeros[row_sel].to(target_device) if self.zeros is not None else None
            bi = self.bias[row_sel].to(target_device) if self.bias is not None else None
        else:
            idx = (
                row_sel.to(self.qweight.device, dtype=torch.long)
                if isinstance(row_sel, torch.Tensor)
                else torch.tensor(row_sel, device=self.qweight.device, dtype=torch.long)
            )
            qw = self.qweight.index_select(0, idx).to(target_device)
            sc = self.scales.index_select(0, idx).to(target_device) if self.scales is not None else None
            zr = self.zeros.index_select(0, idx).to(target_device) if self.zeros is not None else None
            bi = self.bias.index_select(0, idx).to(target_device) if self.bias is not None else None
        return GGUFQuantLinear(self.mode, self.in_features, int(qw.shape[0]), qw, sc, zr, bias=bi)

    def slice_input_blocks(
        self,
        block_indices: Sequence[int],
        block_cols: int,
        target_device: torch.device,
        keep_bias: bool = True,
    ) -> "GGUFQuantLinear":
        """
        Extract column blocks (`dim=1`, where `block_cols % 32 == 0`) onto `target_device`
        for exact Row-Parallel Tensor Parallelism (`o_proj`, `down_proj`, and tiled `ssm_out`).
        """
        if block_cols % 32 != 0 or self.in_features % block_cols != 0:
            raise ValueError(f"block_cols={block_cols} must be divisible by 32 and divide K={self.in_features}")
        n_total_blk = self.in_features // block_cols
        n_sel = len(block_indices)
        new_K = n_sel * block_cols
        N = self.out_features
        # Common TP row-parallel case: one consecutive range. Transfer a strided
        # view directly rather than first materializing an index_select on the host.
        # Keep the original path for same-device slicing (which must own its storage).
        blocks = list(block_indices)
        if (self.qweight.device != target_device and blocks
                and 0 <= blocks[0] and blocks[-1] < n_total_blk
                and blocks == list(range(blocks[0], blocks[0] + n_sel))):
            start = blocks[0] * block_cols
            end = start + new_K
            qdiv = 2 if self.mode == "w4" else 1
            qw = self.qweight[:, start // qdiv:end // qdiv].to(target_device).contiguous()
            group = 32 if self.mode == "w4" else 16
            sc = (self.scales[:, start // group:end // group].to(target_device).contiguous()
                  if self.scales is not None else None)
            zr = (self.zeros[:, start // group:end // group].to(target_device).contiguous()
                  if self.zeros is not None else None)
            bi = self.bias.to(target_device) if keep_bias and self.bias is not None else None
            return GGUFQuantLinear(self.mode, new_K, N, qw, sc, zr, bias=bi)
        idx = torch.tensor(blocks, device=self.qweight.device, dtype=torch.long)

        if self.mode == "w4":
            assert self.scales is not None and self.zeros is not None
            qw = (
                self.qweight.view(N, n_total_blk, block_cols // 2)
                .index_select(1, idx)
                .reshape(N, new_K // 2)
                .to(target_device)
            )
            sc = (
                self.scales.view(N, n_total_blk, block_cols // 32)
                .index_select(1, idx)
                .reshape(N, new_K // 32)
                .to(target_device)
            )
            zr = (
                self.zeros.view(N, n_total_blk, block_cols // 32)
                .index_select(1, idx)
                .reshape(N, new_K // 32)
                .to(target_device)
            )
        elif self.mode == "w8":
            assert self.scales is not None
            qw = (
                self.qweight.view(N, n_total_blk, block_cols)
                .index_select(1, idx)
                .reshape(N, new_K)
                .to(target_device)
            )
            sc = (
                self.scales.view(N, n_total_blk, block_cols // 16)
                .index_select(1, idx)
                .reshape(N, new_K // 16)
                .to(target_device)
            )
            zr = (
                self.zeros.view(N, n_total_blk, block_cols // 16)
                .index_select(1, idx)
                .reshape(N, new_K // 16)
                .to(target_device)
                if self.zeros is not None
                else None
            )
        else:
            qw = (
                self.qweight.view(N, n_total_blk, block_cols)
                .index_select(1, idx)
                .reshape(N, new_K)
                .to(target_device)
            )
            sc, zr = None, None

        bi = self.bias.to(target_device) if (keep_bias and self.bias is not None) else None
        return GGUFQuantLinear(self.mode, new_K, N, qw, sc, zr, bias=bi)

    @classmethod
    def from_gguf(
        cls,
        gguf_file: GGUFFile,
        weight_name: str,
        bias_name: Optional[str] = None,
        device: torch.device = torch.device("cpu"),
    ) -> "GGUFQuantLinear":
        info: GGUFTensorInfo = gguf_file.tensors[weight_name]
        out_features, in_features = int(info.shape[0]), int(info.shape[1])
        raw = gguf_file.get_tensor_bytes(weight_name)

        bias_t: Optional[torch.Tensor] = None
        if bias_name and bias_name in gguf_file.tensors:
            bias_t = load_float_tensor(gguf_file, bias_name, device=device, dtype=torch.float16)

        if info.qtype in (GGMLQuantType.F16, GGMLQuantType.F32, GGMLQuantType.BF16):
            w_fp16 = load_float_tensor(gguf_file, weight_name, device=device, dtype=torch.float16)
            return cls("fp16", in_features, out_features, w_fp16, bias=bias_t)

        if info.qtype in SUB_5BIT_W4_QTYPES and in_features % 32 == 0:
            if info.qtype != GGMLQuantType.Q4_K or in_features % 256 == 0:
                qw, sc, zr = repack_gguf_to_w4a16(raw, info.qtype, out_features, in_features, device)
                return cls("w4", in_features, out_features, qw, sc, zr, bias=bias_t)

        qw, sc, zr = repack_gguf_to_w8a16(raw, info.qtype, out_features, in_features, device)
        return cls("w8", in_features, out_features, qw, sc, zr, bias=bias_t)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_shape = x.shape
        x_2d = x.reshape(-1, self.in_features)
        if self.mode == "w4":
            assert self.scales is not None and self.zeros is not None
            out = exec_w4a16_matmul(x_2d, self.qweight, self.scales, self.zeros, self.bias, self.qweight_i32)
        elif self.mode == "w8":
            assert self.scales is not None
            out = exec_w8a16_matmul(x_2d, self.qweight, self.scales, self.zeros, self.bias, self.qweight_i32)
        else:
            out = F.linear(x_2d.to(device=self.qweight.device, dtype=torch.float16), self.qweight, self.bias).to(x.dtype)
        return out.reshape(*orig_shape[:-1], self.out_features)

    def embed_lookup(self, token_ids: torch.Tensor, dtype: torch.dtype = torch.float16) -> torch.Tensor:
        """
        Dequantizes ONLY the active token rows for embedding lookup (~4x VRAM savings on 152K-256K vocabs),
        allowing the exact same quantized buffer to serve as both `token_embd` AND tied `lm_head`!
        """
        flat_ids = token_ids.reshape(-1)
        K = self.in_features
        if self.mode == "fp16":
            rows = self.qweight.index_select(0, flat_ids).to(dtype)
        elif self.mode == "w4":
            assert self.scales is not None and self.zeros is not None
            packed = self.qweight.index_select(0, flat_ids).view(-1, K // 32, 16)
            sc = self.scales.index_select(0, flat_ids).to(dtype).unsqueeze(-1)
            zr = self.zeros.index_select(0, flat_ids).to(dtype).unsqueeze(-1)
            q_lo = (packed & 0x0F).to(dtype)
            q_hi = ((packed >> 4) & 0x0F).to(dtype)
            rows = (torch.cat([q_lo, q_hi], dim=-1) * sc - zr).reshape(-1, K)
        else:
            assert self.scales is not None
            qw = self.qweight.index_select(0, flat_ids).to(dtype).view(-1, K // 16, 16)
            sc = self.scales.index_select(0, flat_ids).to(dtype).unsqueeze(-1)
            val = qw * sc
            if self.zeros is not None:
                val = val - self.zeros.index_select(0, flat_ids).to(dtype).unsqueeze(-1)
            rows = val.reshape(-1, K)
        return rows.reshape(*token_ids.shape, K)


class FusedLinearPack(nn.Module):
    """
    Automatically fuses sibling `GGUFQuantLinear` projections sharing the same input `x`
    by grouping compatible quant modes (`"w4"`, `"w8"`, `"fp16"`) into minimal GEMV/GEMM launches.
    Consumes 0 bytes of extra VRAM because un-fused tensors are immediately released.
    """

    def __init__(self, linears: Sequence[GGUFQuantLinear]) -> None:
        super().__init__()
        self.num_outputs = len(linears)
        self.out_sizes = [lin.out_features for lin in linears]

        # Group indices by `mode` while preserving deterministic reconstruction order
        mode_to_indices: Dict[str, List[int]] = {}
        for idx, lin in enumerate(linears):
            mode_to_indices.setdefault(lin.mode, []).append(idx)

        self.groups = nn.ModuleList()
        self.group_slots: List[Tuple[List[int], List[int]]] = []
        for _mode, idxs in mode_to_indices.items():
            fused = GGUFQuantLinear.fuse([linears[i] for i in idxs])
            self.groups.append(fused)
            self.group_slots.append((idxs, [linears[i].out_features for i in idxs]))

        self.is_single_contiguous = (
            len(self.groups) == 1 and self.group_slots[0][0] == list(range(self.num_outputs))
        )

    def forward_single_concat(self, x: torch.Tensor) -> torch.Tensor:
        """Fast path when all outputs fused into 1 contiguous matrix (e.g. `[gate | up]`)."""
        assert self.is_single_contiguous
        return self.groups[0](x)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        if self.is_single_contiguous:
            cat_out = self.groups[0](x)
            if self.num_outputs == 1:
                return [cat_out]
            return list(torch.split(cat_out, self.out_sizes, dim=-1))

        results: List[Optional[torch.Tensor]] = [None] * self.num_outputs
        for grp_mod, (idxs, sizes) in zip(self.groups, self.group_slots):
            grp_out = grp_mod(x)
            if len(idxs) == 1:
                results[idxs[0]] = grp_out
            else:
                for i_slot, part in zip(idxs, torch.split(grp_out, sizes, dim=-1)):
                    results[i_slot] = part
        return [r for r in results if r is not None]


class GGUFQuantMoEExperts(nn.Module):
    """
    3D Quantized Routed Experts `[n_experts, out_features, in_features]` for MoE GGUF models
    (e.g., Qwen3-30B-A3B, Qwen2-MoE, Mixtral).
    Reads ONLY the routed `top_k` expert slices from GPU VRAM per token!
    """

    def __init__(
        self,
        mode: str,
        n_experts: int,
        in_features: int,
        out_features: int,
        qweight: torch.Tensor,
        scales: Optional[torch.Tensor] = None,
        zeros: Optional[torch.Tensor] = None,
    ) -> None:
        super().__init__()
        self.mode = mode
        self.n_experts = n_experts
        self.in_features = in_features
        self.out_features = out_features
        qw_c = qweight.contiguous()
        self.register_buffer("qweight", qw_c, persistent=False)
        self.register_buffer("scales", scales.contiguous() if scales is not None else None, persistent=False)
        self.register_buffer("zeros", zeros.contiguous() if zeros is not None else None, persistent=False)
        self.qweight_i32: Optional[torch.Tensor] = (
            qw_c.view(torch.int32) if mode in ("w4", "w8") and qw_c.shape[-1] % 4 == 0 else None
        )

    @classmethod
    def from_gguf(
        cls,
        gguf_file: GGUFFile,
        weight_name: str,
        device: torch.device = torch.device("cpu"),
    ) -> "GGUFQuantMoEExperts":
        info = gguf_file.tensors[weight_name]
        if len(info.shape) != 3:
            raise ValueError(f"Expected 3D MoE tensor for {weight_name}, got shape={info.shape}")
        n_experts, out_features, in_features = int(info.shape[0]), int(info.shape[1]), int(info.shape[2])
        total_rows = n_experts * out_features
        raw = gguf_file.get_tensor_bytes(weight_name)

        if info.qtype in (GGMLQuantType.F16, GGMLQuantType.F32, GGMLQuantType.BF16):
            w_fp16 = load_float_tensor(gguf_file, weight_name, device=device, dtype=torch.float16)
            return cls("fp16", n_experts, in_features, out_features, w_fp16)

        if info.qtype in SUB_5BIT_W4_QTYPES and in_features % 32 == 0:
            if info.qtype != GGMLQuantType.Q4_K or in_features % 256 == 0:
                qw, sc, zr = repack_gguf_to_w4a16(raw, info.qtype, total_rows, in_features, device)
                return cls(
                    "w4",
                    n_experts,
                    in_features,
                    out_features,
                    qw.view(n_experts, out_features, in_features // 2),
                    sc.view(n_experts, out_features, in_features // 32),
                    zr.view(n_experts, out_features, in_features // 32),
                )

        qw, sc, zr = repack_gguf_to_w8a16(raw, info.qtype, total_rows, in_features, device)
        return cls(
            "w8",
            n_experts,
            in_features,
            out_features,
            qw.view(n_experts, out_features, in_features),
            sc.view(n_experts, out_features, in_features // 16),
            zr.view(n_experts, out_features, in_features // 16) if zr is not None else None,
        )

    def forward_expert(self, expert_idx: int, x_tokens: torch.Tensor) -> torch.Tensor:
        """Run `x_tokens` `[M_e, in_features]` through a single routed expert `expert_idx`."""
        qw_i32_e = self.qweight_i32[expert_idx] if self.qweight_i32 is not None else None
        if self.mode == "w4":
            assert self.scales is not None and self.zeros is not None
            return exec_w4a16_matmul(
                x_tokens,
                self.qweight[expert_idx],
                self.scales[expert_idx],
                self.zeros[expert_idx],
                qw_i32_cached=qw_i32_e,
            )
        if self.mode == "w8":
            assert self.scales is not None
            return exec_w8a16_matmul(
                x_tokens,
                self.qweight[expert_idx],
                self.scales[expert_idx],
                self.zeros[expert_idx] if self.zeros is not None else None,
                qw_i32_cached=qw_i32_e,
            )
        return F.linear(x_tokens.to(device=self.qweight.device, dtype=torch.float16), self.qweight[expert_idx]).to(x_tokens.dtype)


In [ ]:
%%writefile pygguf_engine/kv_cache.py
"""
Big-Context Quantized KV Cache (`int4`, `int8`, & `fp16`) with:
1. Sequential peak-VRAM-minimizing paged capacity growth + Sliding-Window Ring KV (`mtp_kv_window`)
2. `@triton.jit` Fused INT4 & INT8 GQA Split-KV Flash-Decoding kernels (reads 4-bit/8-bit K/V directly in SRAM)
3. `@triton.jit` Fused Multi-Row (`1 <= S <= 16`) Gated-DeltaNet Verify + Causal Conv1D kernels
4. Exact Online-Softmax Chunked Attention for long-context prefill (O(1) peak scratch VRAM at 64K-256K+!)
5. Automatic multi-turn Prefix KV Cache reuse for OpenAI `/v1/chat/completions`
"""

from __future__ import annotations

import math
from typing import Any, Dict, List, Literal, Optional, Sequence, Tuple

import torch
import torch.nn.functional as F

from .quant_kernels import cuda_device_scope

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    triton = None  # type: ignore[assignment]
    tl = None  # type: ignore[assignment]
    HAS_TRITON = False


if HAS_TRITON:

    @triton.jit
    def _int8_gqa_flash_decode_split_kernel(
        q_ptr,          # float16 [n_heads, head_dim]
        k_ptr,          # int8    [n_kv_heads, capacity, head_dim]
        ks_ptr,         # float16 [n_kv_heads, capacity]
        v_ptr,          # int8    [n_kv_heads, capacity, head_dim]
        vs_ptr,         # float16 [n_kv_heads, capacity]
        part_out_ptr,   # float32 [n_heads, num_splits, head_dim]
        part_lse_ptr,   # float32 [n_heads, num_splits]
        total_len,
        capacity,
        win_start,
        sm_scale,
        WIN_RING: tl.constexpr,
        GQA_GROUP: tl.constexpr,
        NUM_SPLITS: tl.constexpr,
        BLOCK_S: tl.constexpr,
        HEAD_DIM: tl.constexpr,
        Q_LEN: tl.constexpr = 1,
        WINDOW_SIZE: tl.constexpr = 0,
    ):
        """
        Fused Split-KV Flash-Decoding directly over INT8-quantized K and V caches
        (with zero-copy modulo ring addressing when `WIN_RING=True`).
        """
        head_idx = tl.program_id(0)
        split_idx = tl.program_id(1)
        kv_head = (head_idx // Q_LEN) // GQA_GROUP
        # `total_len` covers the union of all query windows, beginning at
        # win_start. Each query sees only its causal prefix and own window.
        query_end = win_start + total_len - (Q_LEN - 1 - head_idx % Q_LEN)
        if WIN_RING:
            query_start = tl.maximum(win_start, query_end - WINDOW_SIZE) if WINDOW_SIZE > 0 else win_start
        else:
            query_start = 0
        visible_len = query_end - query_start
        tokens_per_split = (visible_len + NUM_SPLITS - 1) // NUM_SPLITS
        split_start = split_idx * tokens_per_split
        split_end = tl.minimum(visible_len, split_start + tokens_per_split)

        offs_d = tl.arange(0, HEAD_DIM)
        lse_offset = head_idx * NUM_SPLITS + split_idx
        out_offset = lse_offset * HEAD_DIM

        neg_inf = tl.zeros((), dtype=tl.float32) - float("inf")
        if split_start >= split_end:
            tl.store(part_lse_ptr + lse_offset, neg_inf)
            tl.store(part_out_ptr + out_offset + offs_d, tl.zeros((HEAD_DIM,), dtype=tl.float32))
            return

        q = tl.load(q_ptr + head_idx * HEAD_DIM + offs_d).to(tl.float32) * sm_scale

        m_i = tl.zeros((), dtype=tl.float32) - float("inf")
        l_i = tl.zeros((), dtype=tl.float32)
        acc = tl.zeros((HEAD_DIM,), dtype=tl.float32)

        kv_h64 = kv_head.to(tl.int64)
        cap64 = capacity.to(tl.int64)
        k_base = k_ptr + kv_h64 * cap64 * HEAD_DIM
        v_base = v_ptr + kv_h64 * cap64 * HEAD_DIM
        ks_base = ks_ptr + kv_h64 * cap64
        vs_base = vs_ptr + kv_h64 * cap64

        for s_off in range(split_start, split_end, BLOCK_S):
            offs_s = s_off + tl.arange(0, BLOCK_S)
            mask_s = offs_s < split_end
            if WIN_RING:
                offs_s64 = ((query_start + offs_s) % capacity).to(tl.int64)
            else:
                offs_s64 = offs_s.to(tl.int64)

            k_q = tl.load(
                k_base + offs_s64[:, None] * HEAD_DIM + offs_d[None, :],
                mask=mask_s[:, None],
                other=0,
            ).to(tl.float32)
            ks = tl.load(ks_base + offs_s64, mask=mask_s, other=0.0).to(tl.float32)

            scores = tl.sum(k_q * q[None, :], axis=1) * ks
            scores = tl.where(mask_s, scores, float("-inf"))

            m_ij = tl.max(scores, axis=0)
            m_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_new)
            p = tl.exp(scores - m_new)

            v_q = tl.load(
                v_base + offs_s64[:, None] * HEAD_DIM + offs_d[None, :],
                mask=mask_s[:, None],
                other=0,
            ).to(tl.float32)
            vs = tl.load(vs_base + offs_s64, mask=mask_s, other=0.0).to(tl.float32)

            acc = acc * alpha + tl.sum((p * vs)[:, None] * v_q, axis=0)
            l_i = l_i * alpha + tl.sum(p, axis=0)
            m_i = m_new

        inv_l = 1.0 / tl.maximum(l_i, 1e-20)
        tl.store(part_out_ptr + out_offset + offs_d, acc * inv_l)
        tl.store(part_lse_ptr + lse_offset, m_i + tl.log(tl.maximum(l_i, 1e-20)))

    @triton.jit
    def _int8_gqa_flash_decode_posptr_kernel(
        q_ptr,
        k_ptr,
        ks_ptr,
        v_ptr,
        vs_ptr,
        part_out_ptr,
        part_lse_ptr,
        pos_ptr,        # int32[1] GPU scalar (`cur_pos`, the chunk base position)
        capacity,
        sm_scale,
        GQA_GROUP: tl.constexpr,
        NUM_SPLITS: tl.constexpr,
        BLOCK_S: tl.constexpr,
        HEAD_DIM: tl.constexpr,
        Q_LEN: tl.constexpr = 1,
    ):
        """CUDA-Graph-capturable Split-KV INT8 Flash-Decoding reading positions from GPU `pos_ptr`.

        `Q_LEN` queries share one captured graph (`q` rows are laid out `[head, q_row]`); query row
        `r` attends the causal prefix `[0, cur_pos + 1 + r)` (`Q_LEN == 1` reduces to the old path).
        """
        head_idx = tl.program_id(0)
        split_idx = tl.program_id(1)
        q_row = head_idx % Q_LEN
        kv_head = (head_idx // Q_LEN) // GQA_GROUP

        total_len = tl.load(pos_ptr).to(tl.int32) + 1 + q_row
        tokens_per_split = (total_len + NUM_SPLITS - 1) // NUM_SPLITS
        split_start = split_idx * tokens_per_split
        split_end = tl.minimum(total_len, split_start + tokens_per_split)

        offs_d = tl.arange(0, HEAD_DIM)
        lse_offset = head_idx * NUM_SPLITS + split_idx
        out_offset = lse_offset * HEAD_DIM

        neg_inf = tl.zeros((), dtype=tl.float32) - float("inf")
        if split_start >= split_end:
            tl.store(part_lse_ptr + lse_offset, neg_inf)
            tl.store(part_out_ptr + out_offset + offs_d, tl.zeros((HEAD_DIM,), dtype=tl.float32))
            return

        q = tl.load(q_ptr + head_idx * HEAD_DIM + offs_d).to(tl.float32) * sm_scale
        m_i = tl.zeros((), dtype=tl.float32) - float("inf")
        l_i = tl.zeros((), dtype=tl.float32)
        acc = tl.zeros((HEAD_DIM,), dtype=tl.float32)

        kv_h64 = kv_head.to(tl.int64)
        cap64 = capacity.to(tl.int64)
        k_base = k_ptr + kv_h64 * cap64 * HEAD_DIM
        v_base = v_ptr + kv_h64 * cap64 * HEAD_DIM
        ks_base = ks_ptr + kv_h64 * cap64
        vs_base = vs_ptr + kv_h64 * cap64

        for s_off in range(split_start, split_end, BLOCK_S):
            offs_s = s_off + tl.arange(0, BLOCK_S)
            mask_s = offs_s < split_end
            offs_s64 = offs_s.to(tl.int64)

            k_q = tl.load(
                k_base + offs_s64[:, None] * HEAD_DIM + offs_d[None, :],
                mask=mask_s[:, None],
                other=0,
            ).to(tl.float32)
            ks = tl.load(ks_base + offs_s64, mask=mask_s, other=0.0).to(tl.float32)

            scores = tl.sum(k_q * q[None, :], axis=1) * ks
            scores = tl.where(mask_s, scores, float("-inf"))

            m_ij = tl.max(scores, axis=0)
            m_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_new)
            p = tl.exp(scores - m_new)

            v_q = tl.load(
                v_base + offs_s64[:, None] * HEAD_DIM + offs_d[None, :],
                mask=mask_s[:, None],
                other=0,
            ).to(tl.float32)
            vs = tl.load(vs_base + offs_s64, mask=mask_s, other=0.0).to(tl.float32)

            acc = acc * alpha + tl.sum((p * vs)[:, None] * v_q, axis=0)
            l_i = l_i * alpha + tl.sum(p, axis=0)
            m_i = m_new

        inv_l = 1.0 / tl.maximum(l_i, 1e-20)
        tl.store(part_out_ptr + out_offset + offs_d, acc * inv_l)
        tl.store(part_lse_ptr + lse_offset, m_i + tl.log(tl.maximum(l_i, 1e-20)))

    @triton.jit
    def _flash_decode_combine_kernel(
        part_out_ptr,   # float32 [n_heads, NUM_SPLITS, HEAD_DIM]
        part_lse_ptr,   # float32 [n_heads, NUM_SPLITS]
        out_ptr,        # float16 [n_heads, HEAD_DIM]
        NUM_SPLITS: tl.constexpr,
        HEAD_DIM: tl.constexpr,
    ):
        """Fused Split-KV LSE reduction into FP16 output (replaces 7 eager PyTorch kernels per layer!)."""
        pid_h = tl.program_id(0)
        offs_s = tl.arange(0, NUM_SPLITS)
        offs_d = tl.arange(0, HEAD_DIM)

        lse = tl.load(part_lse_ptr + pid_h * NUM_SPLITS + offs_s)
        max_lse = tl.max(lse, axis=0)
        max_safe = tl.where(max_lse == float("-inf"), 0.0, max_lse)
        w = tl.exp(lse - max_safe)
        w = w / tl.maximum(tl.sum(w, axis=0), 1e-20)

        p_ptrs = part_out_ptr + (pid_h * NUM_SPLITS + offs_s[:, None]) * HEAD_DIM + offs_d[None, :]
        part = tl.load(p_ptrs)
        out = tl.sum(part * w[:, None], axis=0)
        tl.store(out_ptr + pid_h * HEAD_DIM + offs_d, out.to(tl.float16))

    @triton.jit
    def _kv_append_int8_kernel(
        k_in_ptr,    # float16 [n_kv_heads, s_new, head_dim]
        v_in_ptr,    # float16 [n_kv_heads, s_new, head_dim]
        k_buf_ptr,   # int8    [n_kv_heads, capacity, head_dim]
        v_buf_ptr,   # int8    [n_kv_heads, capacity, head_dim]
        ks_buf_ptr,  # float16 [n_kv_heads, capacity]
        vs_buf_ptr,  # float16 [n_kv_heads, capacity]
        start_pos,
        s_new,
        capacity,
        HEAD_DIM: tl.constexpr,
        BLOCK_D: tl.constexpr,
    ):
        """Fused single-launch K & V dynamic per-token INT8 quantizer + paged cache writer."""
        pid_h = tl.program_id(0)
        pid_s = tl.program_id(1)

        offs_d = tl.arange(0, BLOCK_D)
        mask_d = offs_d < HEAD_DIM

        in_off = (pid_h.to(tl.int64) * s_new + pid_s) * HEAD_DIM + offs_d
        k_val = tl.load(k_in_ptr + in_off, mask=mask_d, other=0.0).to(tl.float32)
        v_val = tl.load(v_in_ptr + in_off, mask=mask_d, other=0.0).to(tl.float32)

        k_amax = tl.maximum(tl.max(tl.abs(k_val), axis=0), 1e-6)
        v_amax = tl.maximum(tl.max(tl.abs(v_val), axis=0), 1e-6)

        ks = (k_amax / 127.0).to(tl.float16)
        vs = (v_amax / 127.0).to(tl.float16)

        k_scaled = k_val / ks.to(tl.float32)
        v_scaled = v_val / vs.to(tl.float32)
        kq = tl.floor(tl.abs(k_scaled) + 0.5) * tl.where(k_scaled < 0.0, -1.0, 1.0)
        vq = tl.floor(tl.abs(v_scaled) + 0.5) * tl.where(v_scaled < 0.0, -1.0, 1.0)

        kq_i8 = tl.minimum(tl.maximum(kq, -127.0), 127.0).to(tl.int8)
        vq_i8 = tl.minimum(tl.maximum(vq, -127.0), 127.0).to(tl.int8)

        dst_pos = start_pos + pid_s
        out_off = (pid_h.to(tl.int64) * capacity + dst_pos) * HEAD_DIM + offs_d
        scale_off = pid_h.to(tl.int64) * capacity + dst_pos

        tl.store(k_buf_ptr + out_off, kq_i8, mask=mask_d)
        tl.store(v_buf_ptr + out_off, vq_i8, mask=mask_d)
        tl.store(ks_buf_ptr + scale_off, ks)
        tl.store(vs_buf_ptr + scale_off, vs)

    @triton.jit
    def _kv_append_int8_posptr_kernel(
        k_in_ptr,
        v_in_ptr,
        k_buf_ptr,
        v_buf_ptr,
        ks_buf_ptr,
        vs_buf_ptr,
        pos_ptr,        # int32[1] GPU scalar (`start_pos`, chunk base position)
        capacity,
        HEAD_DIM: tl.constexpr,
        BLOCK_D: tl.constexpr,
        S_NEW: tl.constexpr = 1,
    ):
        """CUDA-Graph-capturable INT8 KV cache writer (launch with grid `(n_kv_heads, S_NEW)`).

        Row `pid_s` of the chunk is written at `load(pos_ptr) + pid_s` (`S_NEW == 1` reduces to
        the old single-token writer exactly).
        """
        pid_h = tl.program_id(0)
        pid_s = tl.program_id(1)
        offs_d = tl.arange(0, BLOCK_D)
        mask_d = offs_d < HEAD_DIM

        in_off = (pid_h.to(tl.int64) * S_NEW + pid_s) * HEAD_DIM + offs_d
        k_val = tl.load(k_in_ptr + in_off, mask=mask_d, other=0.0).to(tl.float32)
        v_val = tl.load(v_in_ptr + in_off, mask=mask_d, other=0.0).to(tl.float32)

        k_amax = tl.maximum(tl.max(tl.abs(k_val), axis=0), 1e-6)
        v_amax = tl.maximum(tl.max(tl.abs(v_val), axis=0), 1e-6)

        ks = (k_amax / 127.0).to(tl.float16)
        vs = (v_amax / 127.0).to(tl.float16)

        k_scaled = k_val / ks.to(tl.float32)
        v_scaled = v_val / vs.to(tl.float32)
        kq = tl.floor(tl.abs(k_scaled) + 0.5) * tl.where(k_scaled < 0.0, -1.0, 1.0)
        vq = tl.floor(tl.abs(v_scaled) + 0.5) * tl.where(v_scaled < 0.0, -1.0, 1.0)

        kq_i8 = tl.minimum(tl.maximum(kq, -127.0), 127.0).to(tl.int8)
        vq_i8 = tl.minimum(tl.maximum(vq, -127.0), 127.0).to(tl.int8)

        dst_pos = tl.load(pos_ptr).to(tl.int64) + pid_s
        out_off = (pid_h.to(tl.int64) * capacity + dst_pos) * HEAD_DIM + offs_d
        scale_off = pid_h.to(tl.int64) * capacity + dst_pos

        tl.store(k_buf_ptr + out_off, kq_i8, mask=mask_d)
        tl.store(v_buf_ptr + out_off, vq_i8, mask=mask_d)
        tl.store(ks_buf_ptr + scale_off, ks)
        tl.store(vs_buf_ptr + scale_off, vs)

    @triton.jit
    def _kv_append_int4_kernel(
        k_in_ptr,    # float16 [n_kv_heads, s_new, 2 * HALF_D]
        v_in_ptr,    # float16 [n_kv_heads, s_new, 2 * HALF_D]
        k_buf_ptr,   # uint8   [n_kv_heads, capacity, HALF_D]
        v_buf_ptr,   # uint8   [n_kv_heads, capacity, HALF_D]
        ks_buf_ptr,  # float16 [n_kv_heads, capacity, 2]
        vs_buf_ptr,  # float16 [n_kv_heads, capacity, 2]
        start_pos,
        s_new,
        capacity,
        HALF_D: tl.constexpr,
        BLOCK_HD: tl.constexpr,
    ):
        """
        Fused single-launch K & V split-half INT4 quantizer + cache writer (`~50%` less VRAM &
        GDDR6 bandwidth than INT8, `~75%` less than FP16; unlocks 256K+ context on Dual T4!).
        """
        pid_h = tl.program_id(0)
        pid_s = tl.program_id(1)

        offs_hd = tl.arange(0, BLOCK_HD)
        mask_hd = offs_hd < HALF_D

        in_base = (pid_h.to(tl.int64) * s_new + pid_s) * (2 * HALF_D)
        k_lo = tl.load(k_in_ptr + in_base + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)
        k_hi = tl.load(k_in_ptr + in_base + HALF_D + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)
        v_lo = tl.load(v_in_ptr + in_base + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)
        v_hi = tl.load(v_in_ptr + in_base + HALF_D + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)

        ks_lo = (tl.maximum(tl.max(tl.abs(k_lo), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)
        ks_hi = (tl.maximum(tl.max(tl.abs(k_hi), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)
        vs_lo = (tl.maximum(tl.max(tl.abs(v_lo), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)
        vs_hi = (tl.maximum(tl.max(tl.abs(v_hi), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)

        k_lo_s = k_lo / ks_lo.to(tl.float32)
        k_hi_s = k_hi / ks_hi.to(tl.float32)
        v_lo_s = v_lo / vs_lo.to(tl.float32)
        v_hi_s = v_hi / vs_hi.to(tl.float32)

        qk_lo = tl.minimum(tl.maximum(tl.floor(tl.abs(k_lo_s) + 0.5) * tl.where(k_lo_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8
        qk_hi = tl.minimum(tl.maximum(tl.floor(tl.abs(k_hi_s) + 0.5) * tl.where(k_hi_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8
        qv_lo = tl.minimum(tl.maximum(tl.floor(tl.abs(v_lo_s) + 0.5) * tl.where(v_lo_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8
        qv_hi = tl.minimum(tl.maximum(tl.floor(tl.abs(v_hi_s) + 0.5) * tl.where(v_hi_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8

        kp_u8 = ((qk_lo & 0x0F) | ((qk_hi & 0x0F) << 4)).to(tl.uint8)
        vp_u8 = ((qv_lo & 0x0F) | ((qv_hi & 0x0F) << 4)).to(tl.uint8)

        dst_pos = start_pos + pid_s
        slot_idx = pid_h.to(tl.int64) * capacity + dst_pos
        out_off = slot_idx * HALF_D + offs_hd
        sc_off = slot_idx * 2

        tl.store(k_buf_ptr + out_off, kp_u8, mask=mask_hd)
        tl.store(v_buf_ptr + out_off, vp_u8, mask=mask_hd)
        tl.store(ks_buf_ptr + sc_off, ks_lo)
        tl.store(ks_buf_ptr + sc_off + 1, ks_hi)
        tl.store(vs_buf_ptr + sc_off, vs_lo)
        tl.store(vs_buf_ptr + sc_off + 1, vs_hi)

    @triton.jit
    def _kv_append_int4_posptr_kernel(
        k_in_ptr,
        v_in_ptr,
        k_buf_ptr,
        v_buf_ptr,
        ks_buf_ptr,
        vs_buf_ptr,
        pos_ptr,        # int32[1] GPU scalar (`start_pos`, chunk base position)
        capacity,
        HALF_D: tl.constexpr,
        BLOCK_HD: tl.constexpr,
        S_NEW: tl.constexpr = 1,
    ):
        """CUDA-Graph-capturable INT4 KV cache writer (launch with grid `(n_kv_heads, S_NEW)`).

        Row `pid_s` of the chunk is written at `load(pos_ptr) + pid_s` (`S_NEW == 1` reduces to
        the old single-token writer exactly).
        """
        pid_h = tl.program_id(0)
        pid_s = tl.program_id(1)
        offs_hd = tl.arange(0, BLOCK_HD)
        mask_hd = offs_hd < HALF_D

        in_base = (pid_h.to(tl.int64) * S_NEW + pid_s) * (2 * HALF_D)
        k_lo = tl.load(k_in_ptr + in_base + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)
        k_hi = tl.load(k_in_ptr + in_base + HALF_D + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)
        v_lo = tl.load(v_in_ptr + in_base + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)
        v_hi = tl.load(v_in_ptr + in_base + HALF_D + offs_hd, mask=mask_hd, other=0.0).to(tl.float32)

        ks_lo = (tl.maximum(tl.max(tl.abs(k_lo), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)
        ks_hi = (tl.maximum(tl.max(tl.abs(k_hi), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)
        vs_lo = (tl.maximum(tl.max(tl.abs(v_lo), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)
        vs_hi = (tl.maximum(tl.max(tl.abs(v_hi), axis=0), 1e-6) * (1.0 / 7.0)).to(tl.float16)

        k_lo_s = k_lo / ks_lo.to(tl.float32)
        k_hi_s = k_hi / ks_hi.to(tl.float32)
        v_lo_s = v_lo / vs_lo.to(tl.float32)
        v_hi_s = v_hi / vs_hi.to(tl.float32)

        qk_lo = tl.minimum(tl.maximum(tl.floor(tl.abs(k_lo_s) + 0.5) * tl.where(k_lo_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8
        qk_hi = tl.minimum(tl.maximum(tl.floor(tl.abs(k_hi_s) + 0.5) * tl.where(k_hi_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8
        qv_lo = tl.minimum(tl.maximum(tl.floor(tl.abs(v_lo_s) + 0.5) * tl.where(v_lo_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8
        qv_hi = tl.minimum(tl.maximum(tl.floor(tl.abs(v_hi_s) + 0.5) * tl.where(v_hi_s < 0.0, -1.0, 1.0), -7.0), 7.0).to(tl.int32) + 8

        kp_u8 = ((qk_lo & 0x0F) | ((qk_hi & 0x0F) << 4)).to(tl.uint8)
        vp_u8 = ((qv_lo & 0x0F) | ((qv_hi & 0x0F) << 4)).to(tl.uint8)

        dst_pos = tl.load(pos_ptr).to(tl.int64) + pid_s
        slot_idx = pid_h.to(tl.int64) * capacity + dst_pos
        out_off = slot_idx * HALF_D + offs_hd
        sc_off = slot_idx * 2

        tl.store(k_buf_ptr + out_off, kp_u8, mask=mask_hd)
        tl.store(v_buf_ptr + out_off, vp_u8, mask=mask_hd)
        tl.store(ks_buf_ptr + sc_off, ks_lo)
        tl.store(ks_buf_ptr + sc_off + 1, ks_hi)
        tl.store(vs_buf_ptr + sc_off, vs_lo)
        tl.store(vs_buf_ptr + sc_off + 1, vs_hi)

    @triton.jit
    def _int4_gqa_flash_decode_split_kernel(
        q_ptr,          # float16 [n_heads, 2 * HALF_D]
        k_ptr,          # uint8   [n_kv_heads, capacity, HALF_D]
        ks_ptr,         # float16 [n_kv_heads, capacity, 2]
        v_ptr,          # uint8   [n_kv_heads, capacity, HALF_D]
        vs_ptr,         # float16 [n_kv_heads, capacity, 2]
        part_out_ptr,   # float32 [n_heads, num_splits, 2 * HALF_D]
        part_lse_ptr,   # float32 [n_heads, num_splits]
        total_len,
        capacity,
        win_start,
        sm_scale,
        WIN_RING: tl.constexpr,
        GQA_GROUP: tl.constexpr,
        NUM_SPLITS: tl.constexpr,
        BLOCK_S: tl.constexpr,
        HALF_D: tl.constexpr,
        Q_LEN: tl.constexpr = 1,
        WINDOW_SIZE: tl.constexpr = 0,
    ):
        """
        Fused Split-KV Flash-Decoding directly over packed INT4 K and V caches (50% less GDDR6 traffic
        than INT8, 75% less than FP16, with optional zero-copy modulo ring window `WIN_RING=True`).
        """
        head_idx = tl.program_id(0)
        split_idx = tl.program_id(1)
        kv_head = (head_idx // Q_LEN) // GQA_GROUP
        # `total_len` covers the union of all query windows, beginning at
        # win_start. Each query sees only its causal prefix and own window.
        query_end = win_start + total_len - (Q_LEN - 1 - head_idx % Q_LEN)
        if WIN_RING:
            query_start = tl.maximum(win_start, query_end - WINDOW_SIZE) if WINDOW_SIZE > 0 else win_start
        else:
            query_start = 0
        visible_len = query_end - query_start
        tokens_per_split = (visible_len + NUM_SPLITS - 1) // NUM_SPLITS
        split_start = split_idx * tokens_per_split
        split_end = tl.minimum(visible_len, split_start + tokens_per_split)

        offs_hd = tl.arange(0, HALF_D)
        head_dim = 2 * HALF_D
        lse_offset = head_idx * NUM_SPLITS + split_idx
        out_offset = lse_offset * head_dim

        neg_inf = tl.zeros((), dtype=tl.float32) - float("inf")
        if split_start >= split_end:
            tl.store(part_lse_ptr + lse_offset, neg_inf)
            z_hd = tl.zeros((HALF_D,), dtype=tl.float32)
            tl.store(part_out_ptr + out_offset + offs_hd, z_hd)
            tl.store(part_out_ptr + out_offset + HALF_D + offs_hd, z_hd)
            return

        q_lo = tl.load(q_ptr + head_idx * head_dim + offs_hd).to(tl.float32) * sm_scale
        q_hi = tl.load(q_ptr + head_idx * head_dim + HALF_D + offs_hd).to(tl.float32) * sm_scale

        m_i = tl.zeros((), dtype=tl.float32) - float("inf")
        l_i = tl.zeros((), dtype=tl.float32)
        acc_lo = tl.zeros((HALF_D,), dtype=tl.float32)
        acc_hi = tl.zeros((HALF_D,), dtype=tl.float32)

        kv_h64 = kv_head.to(tl.int64)
        cap64 = capacity.to(tl.int64)
        k_base = k_ptr + kv_h64 * cap64 * HALF_D
        v_base = v_ptr + kv_h64 * cap64 * HALF_D
        ks_base = ks_ptr + kv_h64 * cap64 * 2
        vs_base = vs_ptr + kv_h64 * cap64 * 2

        for s_off in range(split_start, split_end, BLOCK_S):
            offs_s = s_off + tl.arange(0, BLOCK_S)
            mask_s = offs_s < split_end
            if WIN_RING:
                offs_s64 = ((query_start + offs_s) % capacity).to(tl.int64)
            else:
                offs_s64 = offs_s.to(tl.int64)

            kp = tl.load(
                k_base + offs_s64[:, None] * HALF_D + offs_hd[None, :],
                mask=mask_s[:, None],
                other=0x88,
            )
            ks_lo = tl.load(ks_base + offs_s64 * 2, mask=mask_s, other=0.0).to(tl.float32)
            ks_hi = tl.load(ks_base + offs_s64 * 2 + 1, mask=mask_s, other=0.0).to(tl.float32)

            k_lo = (kp & 0x0F).to(tl.float32) - 8.0
            k_hi = ((kp >> 4) & 0x0F).to(tl.float32) - 8.0

            scores = tl.sum(k_lo * q_lo[None, :], axis=1) * ks_lo + tl.sum(k_hi * q_hi[None, :], axis=1) * ks_hi
            scores = tl.where(mask_s, scores, float("-inf"))

            m_ij = tl.max(scores, axis=0)
            m_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_new)
            p = tl.exp(scores - m_new)

            vp = tl.load(
                v_base + offs_s64[:, None] * HALF_D + offs_hd[None, :],
                mask=mask_s[:, None],
                other=0x88,
            )
            vs_lo = tl.load(vs_base + offs_s64 * 2, mask=mask_s, other=0.0).to(tl.float32)
            vs_hi = tl.load(vs_base + offs_s64 * 2 + 1, mask=mask_s, other=0.0).to(tl.float32)

            v_lo = (vp & 0x0F).to(tl.float32) - 8.0
            v_hi = ((vp >> 4) & 0x0F).to(tl.float32) - 8.0

            acc_lo = acc_lo * alpha + tl.sum((p * vs_lo)[:, None] * v_lo, axis=0)
            acc_hi = acc_hi * alpha + tl.sum((p * vs_hi)[:, None] * v_hi, axis=0)
            l_i = l_i * alpha + tl.sum(p, axis=0)
            m_i = m_new

        inv_l = 1.0 / tl.maximum(l_i, 1e-20)
        tl.store(part_out_ptr + out_offset + offs_hd, acc_lo * inv_l)
        tl.store(part_out_ptr + out_offset + HALF_D + offs_hd, acc_hi * inv_l)
        tl.store(part_lse_ptr + lse_offset, m_i + tl.log(tl.maximum(l_i, 1e-20)))

    @triton.jit
    def _int4_gqa_flash_decode_posptr_kernel(
        q_ptr,
        k_ptr,
        ks_ptr,
        v_ptr,
        vs_ptr,
        part_out_ptr,
        part_lse_ptr,
        pos_ptr,        # int32[1] GPU scalar (`cur_pos`, the chunk base position)
        capacity,
        sm_scale,
        GQA_GROUP: tl.constexpr,
        NUM_SPLITS: tl.constexpr,
        BLOCK_S: tl.constexpr,
        HALF_D: tl.constexpr,
        Q_LEN: tl.constexpr = 1,
    ):
        """CUDA-Graph-capturable Split-KV INT4 Flash-Decoding reading positions from GPU `pos_ptr`.

        `Q_LEN` queries share one captured graph (`q` rows are laid out `[head, q_row]`); query row
        `r` attends the causal prefix `[0, cur_pos + 1 + r)` (`Q_LEN == 1` reduces to the old path).
        """
        head_idx = tl.program_id(0)
        split_idx = tl.program_id(1)
        q_row = head_idx % Q_LEN
        kv_head = (head_idx // Q_LEN) // GQA_GROUP

        total_len = tl.load(pos_ptr).to(tl.int32) + 1 + q_row
        tokens_per_split = (total_len + NUM_SPLITS - 1) // NUM_SPLITS
        split_start = split_idx * tokens_per_split
        split_end = tl.minimum(total_len, split_start + tokens_per_split)

        offs_hd = tl.arange(0, HALF_D)
        head_dim = 2 * HALF_D
        lse_offset = head_idx * NUM_SPLITS + split_idx
        out_offset = lse_offset * head_dim

        neg_inf = tl.zeros((), dtype=tl.float32) - float("inf")
        if split_start >= split_end:
            tl.store(part_lse_ptr + lse_offset, neg_inf)
            z_hd = tl.zeros((HALF_D,), dtype=tl.float32)
            tl.store(part_out_ptr + out_offset + offs_hd, z_hd)
            tl.store(part_out_ptr + out_offset + HALF_D + offs_hd, z_hd)
            return

        q_lo = tl.load(q_ptr + head_idx * head_dim + offs_hd).to(tl.float32) * sm_scale
        q_hi = tl.load(q_ptr + head_idx * head_dim + HALF_D + offs_hd).to(tl.float32) * sm_scale

        m_i = tl.zeros((), dtype=tl.float32) - float("inf")
        l_i = tl.zeros((), dtype=tl.float32)
        acc_lo = tl.zeros((HALF_D,), dtype=tl.float32)
        acc_hi = tl.zeros((HALF_D,), dtype=tl.float32)

        kv_h64 = kv_head.to(tl.int64)
        cap64 = capacity.to(tl.int64)
        k_base = k_ptr + kv_h64 * cap64 * HALF_D
        v_base = v_ptr + kv_h64 * cap64 * HALF_D
        ks_base = ks_ptr + kv_h64 * cap64 * 2
        vs_base = vs_ptr + kv_h64 * cap64 * 2

        for s_off in range(split_start, split_end, BLOCK_S):
            offs_s = s_off + tl.arange(0, BLOCK_S)
            mask_s = offs_s < split_end
            offs_s64 = offs_s.to(tl.int64)

            kp = tl.load(
                k_base + offs_s64[:, None] * HALF_D + offs_hd[None, :],
                mask=mask_s[:, None],
                other=0x88,
            )
            ks_lo = tl.load(ks_base + offs_s64 * 2, mask=mask_s, other=0.0).to(tl.float32)
            ks_hi = tl.load(ks_base + offs_s64 * 2 + 1, mask=mask_s, other=0.0).to(tl.float32)

            k_lo = (kp & 0x0F).to(tl.float32) - 8.0
            k_hi = ((kp >> 4) & 0x0F).to(tl.float32) - 8.0

            scores = tl.sum(k_lo * q_lo[None, :], axis=1) * ks_lo + tl.sum(k_hi * q_hi[None, :], axis=1) * ks_hi
            scores = tl.where(mask_s, scores, float("-inf"))

            m_ij = tl.max(scores, axis=0)
            m_new = tl.maximum(m_i, m_ij)
            alpha = tl.exp(m_i - m_new)
            p = tl.exp(scores - m_new)

            vp = tl.load(
                v_base + offs_s64[:, None] * HALF_D + offs_hd[None, :],
                mask=mask_s[:, None],
                other=0x88,
            )
            vs_lo = tl.load(vs_base + offs_s64 * 2, mask=mask_s, other=0.0).to(tl.float32)
            vs_hi = tl.load(vs_base + offs_s64 * 2 + 1, mask=mask_s, other=0.0).to(tl.float32)

            v_lo = (vp & 0x0F).to(tl.float32) - 8.0
            v_hi = ((vp >> 4) & 0x0F).to(tl.float32) - 8.0

            acc_lo = acc_lo * alpha + tl.sum((p * vs_lo)[:, None] * v_lo, axis=0)
            acc_hi = acc_hi * alpha + tl.sum((p * vs_hi)[:, None] * v_hi, axis=0)
            l_i = l_i * alpha + tl.sum(p, axis=0)
            m_i = m_new

        inv_l = 1.0 / tl.maximum(l_i, 1e-20)
        tl.store(part_out_ptr + out_offset + offs_hd, acc_lo * inv_l)
        tl.store(part_out_ptr + out_offset + HALF_D + offs_hd, acc_hi * inv_l)
        tl.store(part_lse_ptr + lse_offset, m_i + tl.log(tl.maximum(l_i, 1e-20)))

    @triton.jit
    def _gated_deltanet_scan_kernel(
        q_ptr,      # float32 [S, H_K, D_k] (pre-scaled by 1 / sqrt(D_k); tiled via pid_h % H_K!)
        k_ptr,      # float32 [S, H_K, D_k]
        v_ptr,      # float32 [S, H_v, D_v]
        decay_ptr,  # float32 [S, H_v]     (exp(g_t))
        beta_ptr,   # float32 [S, H_v]     (sigmoid(beta_t))
        state_ptr,  # float32 [H_v, D_v, D_k] (updated IN-PLACE)
        out_ptr,    # float32 [S, H_v, D_v]
        seq_len,
        H_K: tl.constexpr,
        H_V: tl.constexpr,
        D_K: tl.constexpr,
        D_V: tl.constexpr,
        BLOCK_K: tl.constexpr,
        BLOCK_V: tl.constexpr,
    ):
        """
        Fused Register-Resident Gated-DeltaNet Scan Kernel (for Qwen3.5 / Qwen3.6 / Qwen3.8 / Qwen3Next).
        Each threadblock owns a `[BLOCK_V, D_K]` slice of one V-head's recurrent state `[D_V, D_K]`
        entirely in GPU registers across all `seq_len` steps (zero per-token global state traffic!).
        """
        pid_h = tl.program_id(0)
        pid_v = tl.program_id(1)
        pid_kh = pid_h % H_K

        offs_v = pid_v * BLOCK_V + tl.arange(0, BLOCK_V)
        offs_k = tl.arange(0, BLOCK_K)
        mask_v = offs_v < D_V
        mask_k = offs_k < D_K
        mask_vk = mask_v[:, None] & mask_k[None, :]

        state_base = state_ptr + pid_h * (D_V * D_K)
        s_ptrs = state_base + offs_v[:, None] * D_K + offs_k[None, :]
        state = tl.load(s_ptrs, mask=mask_vk, other=0.0).to(tl.float32)

        for t in range(0, seq_len):
            hb_off = t * H_V + pid_h
            decay_t = tl.load(decay_ptr + hb_off).to(tl.float32)
            beta_t = tl.load(beta_ptr + hb_off).to(tl.float32)

            qk_off = (t * H_K + pid_kh) * D_K + offs_k
            q_t = tl.load(q_ptr + qk_off, mask=mask_k, other=0.0).to(tl.float32)
            k_t = tl.load(k_ptr + qk_off, mask=mask_k, other=0.0).to(tl.float32)

            v_off = hb_off * D_V + offs_v
            v_t = tl.load(v_ptr + v_off, mask=mask_v, other=0.0).to(tl.float32)

            state = state * decay_t
            kv_mem = tl.sum(state * k_t[None, :], axis=1)
            delta = (v_t - kv_mem) * beta_t
            state = state + delta[:, None] * k_t[None, :]
            y_t = tl.sum(state * q_t[None, :], axis=1)

            tl.store(out_ptr + v_off, y_t, mask=mask_v)

        tl.store(s_ptrs, state, mask=mask_vk)

    @triton.jit
    def _causal_conv1d_silu_step_kernel(
        qkv_in_ptr,      # float16/float32 [conv_dim]
        conv_state_ptr,  # float32 [conv_dim, K_HIST] (updated IN-PLACE)
        conv_w_ptr,      # float32 [conv_dim, CONV_K]
        conv_out_ptr,    # float32 [conv_dim]
        conv_dim,
        CONV_K: tl.constexpr,
        K_HIST: tl.constexpr,
        BLOCK_C: tl.constexpr,
    ):
        """Fused S=1 causal depthwise Conv1D + in-place state shift + SiLU in 1 launch (~2 us on T4)."""
        pid_c = tl.program_id(0)
        offs_c = pid_c * BLOCK_C + tl.arange(0, BLOCK_C)
        mask_c = offs_c < conv_dim

        x_new = tl.load(qkv_in_ptr + offs_c, mask=mask_c, other=0.0).to(tl.float32)
        acc = tl.zeros((BLOCK_C,), dtype=tl.float32)

        for j in tl.static_range(0, K_HIST):
            s_j = tl.load(conv_state_ptr + offs_c * K_HIST + j, mask=mask_c, other=0.0).to(tl.float32)
            w_j = tl.load(conv_w_ptr + offs_c * CONV_K + j, mask=mask_c, other=0.0).to(tl.float32)
            acc += s_j * w_j
            if j > 0:
                tl.store(conv_state_ptr + offs_c * K_HIST + (j - 1), s_j, mask=mask_c)

        if K_HIST > 0:
            tl.store(conv_state_ptr + offs_c * K_HIST + (K_HIST - 1), x_new, mask=mask_c)

        w_last = tl.load(conv_w_ptr + offs_c * CONV_K + K_HIST, mask=mask_c, other=0.0).to(tl.float32)
        acc += x_new * w_last
        silu = acc / (1.0 + tl.exp(-acc))
        tl.store(conv_out_ptr + offs_c, silu, mask=mask_c)

    @triton.jit
    def _gated_deltanet_decode_step_kernel(
        conv_out_ptr,   # float32 [2 * key_dim + d_inner]
        beta_raw_ptr,   # float16/float32 [H_V]
        alpha_raw_ptr,  # float16/float32 [H_V]
        ssm_dt_ptr,     # float32 [H_V]
        ssm_a_ptr,      # float32 [H_V]
        state_ptr,      # float32 [H_V, D_V, D_K] (updated IN-PLACE)
        out_ptr,        # float32 [H_V, D_V]
        key_dim,
        q_scale,
        eps,
        H_K: tl.constexpr,
        H_V: tl.constexpr,
        D_K: tl.constexpr,
        D_V: tl.constexpr,
        BLOCK_K: tl.constexpr,
        BLOCK_V: tl.constexpr,
    ):
        """
        Fused S=1 Gated-DeltaNet Decode Kernel: computes `beta` sigmoid, `alpha+dt` softplus,
        `decay` exp, `Q`/`K` L2-Norm, tiled K-head broadcast, and recurrent state update in 1 launch!
        """
        pid_h = tl.program_id(0)
        pid_v = tl.program_id(1)
        pid_kh = pid_h % H_K

        offs_v = pid_v * BLOCK_V + tl.arange(0, BLOCK_V)
        offs_k = tl.arange(0, BLOCK_K)
        mask_v = offs_v < D_V
        mask_k = offs_k < D_K
        mask_vk = mask_v[:, None] & mask_k[None, :]

        b_raw = tl.load(beta_raw_ptr + pid_h).to(tl.float32)
        beta_t = 1.0 / (1.0 + tl.exp(-b_raw))

        a_val = tl.load(alpha_raw_ptr + pid_h).to(tl.float32) + tl.load(ssm_dt_ptr + pid_h).to(tl.float32)
        sp = tl.where(a_val > 20.0, a_val, tl.log(1.0 + tl.exp(tl.maximum(a_val, -80.0))))
        decay_t = tl.exp(sp * tl.load(ssm_a_ptr + pid_h).to(tl.float32))

        q_raw = tl.load(conv_out_ptr + pid_kh * D_K + offs_k, mask=mask_k, other=0.0).to(tl.float32)
        k_raw = tl.load(conv_out_ptr + key_dim + pid_kh * D_K + offs_k, mask=mask_k, other=0.0).to(tl.float32)
        q_t = q_raw * tl.rsqrt(tl.sum(q_raw * q_raw, axis=0) + eps) * q_scale
        k_t = k_raw * tl.rsqrt(tl.sum(k_raw * k_raw, axis=0) + eps)

        v_off = pid_h * D_V + offs_v
        v_t = tl.load(conv_out_ptr + 2 * key_dim + v_off, mask=mask_v, other=0.0).to(tl.float32)

        state_base = state_ptr + pid_h * (D_V * D_K)
        s_ptrs = state_base + offs_v[:, None] * D_K + offs_k[None, :]
        state = tl.load(s_ptrs, mask=mask_vk, other=0.0).to(tl.float32)

        state = state * decay_t
        kv_mem = tl.sum(state * k_t[None, :], axis=1)
        delta = (v_t - kv_mem) * beta_t
        state = state + delta[:, None] * k_t[None, :]
        y_t = tl.sum(state * q_t[None, :], axis=1)

        tl.store(out_ptr + v_off, y_t, mask=mask_v)
        tl.store(s_ptrs, state, mask=mask_vk)

    @triton.jit
    def _causal_conv1d_silu_multirow_kernel(
        qkv_in_ptr,      # float16/float32 [S, conv_dim]
        conv_state_ptr,  # float32 [1, conv_dim, K_HIST] (updated IN-PLACE)
        conv_w_ptr,      # float32 [conv_dim, CONV_K]
        conv_out_ptr,    # float32 [S, conv_dim]
        conv_in_ptr,     # float32 [1, conv_dim, K_HIST + S] (written if SAVE_CONV_IN or CONV_K != 4)
        S,
        conv_dim,
        CONV_K: tl.constexpr,
        K_HIST: tl.constexpr,
        SAVE_CONV_IN: tl.constexpr,
        BLOCK_C: tl.constexpr,
    ):
        """
        Fused multi-row (`1 <= S <= 16`) causal depthwise Conv1D + in-place state shift + SiLU
        in 1 launch (optionally emitting `conv_in` for zero-recompute speculative verify rollback).
        """
        pid_c = tl.program_id(0)
        offs_c = pid_c * BLOCK_C + tl.arange(0, BLOCK_C)
        mask_c = offs_c < conv_dim
        offs_c64 = offs_c.to(tl.int64)
        in_stride = K_HIST + S

        if CONV_K == 4 and K_HIST == 3:
            h0 = tl.load(conv_state_ptr + offs_c64 * 3 + 0, mask=mask_c, other=0.0).to(tl.float32)
            h1 = tl.load(conv_state_ptr + offs_c64 * 3 + 1, mask=mask_c, other=0.0).to(tl.float32)
            h2 = tl.load(conv_state_ptr + offs_c64 * 3 + 2, mask=mask_c, other=0.0).to(tl.float32)

            w0 = tl.load(conv_w_ptr + offs_c64 * 4 + 0, mask=mask_c, other=0.0).to(tl.float32)
            w1 = tl.load(conv_w_ptr + offs_c64 * 4 + 1, mask=mask_c, other=0.0).to(tl.float32)
            w2 = tl.load(conv_w_ptr + offs_c64 * 4 + 2, mask=mask_c, other=0.0).to(tl.float32)
            w3 = tl.load(conv_w_ptr + offs_c64 * 4 + 3, mask=mask_c, other=0.0).to(tl.float32)

            if SAVE_CONV_IN:
                tl.store(conv_in_ptr + offs_c64 * in_stride + 0, h0, mask=mask_c)
                tl.store(conv_in_ptr + offs_c64 * in_stride + 1, h1, mask=mask_c)
                tl.store(conv_in_ptr + offs_c64 * in_stride + 2, h2, mask=mask_c)

            for t in range(0, S):
                x = tl.load(qkv_in_ptr + t * conv_dim + offs_c64, mask=mask_c, other=0.0).to(tl.float32)
                if SAVE_CONV_IN:
                    tl.store(conv_in_ptr + offs_c64 * in_stride + (3 + t), x, mask=mask_c)
                acc = h0 * w0 + h1 * w1 + h2 * w2 + x * w3
                silu = acc / (1.0 + tl.exp(-acc))
                tl.store(conv_out_ptr + t * conv_dim + offs_c64, silu, mask=mask_c)
                h0 = h1
                h1 = h2
                h2 = x

            tl.store(conv_state_ptr + offs_c64 * 3 + 0, h0, mask=mask_c)
            tl.store(conv_state_ptr + offs_c64 * 3 + 1, h1, mask=mask_c)
            tl.store(conv_state_ptr + offs_c64 * 3 + 2, h2, mask=mask_c)
        else:
            for j in tl.static_range(0, K_HIST):
                s_j = tl.load(conv_state_ptr + offs_c64 * K_HIST + j, mask=mask_c, other=0.0).to(tl.float32)
                tl.store(conv_in_ptr + offs_c64 * in_stride + j, s_j, mask=mask_c)
            for t in range(0, S):
                x = tl.load(qkv_in_ptr + t * conv_dim + offs_c64, mask=mask_c, other=0.0).to(tl.float32)
                tl.store(conv_in_ptr + offs_c64 * in_stride + (K_HIST + t), x, mask=mask_c)
                acc = tl.zeros((BLOCK_C,), dtype=tl.float32)
                for j in tl.static_range(0, CONV_K):
                    v_j = tl.load(conv_in_ptr + offs_c64 * in_stride + (t + j), mask=mask_c, other=0.0).to(tl.float32)
                    w_j = tl.load(conv_w_ptr + offs_c64 * CONV_K + j, mask=mask_c, other=0.0).to(tl.float32)
                    acc += v_j * w_j
                silu = acc / (1.0 + tl.exp(-acc))
                tl.store(conv_out_ptr + t * conv_dim + offs_c64, silu, mask=mask_c)
            for j in tl.static_range(0, K_HIST):
                s_new_j = tl.load(conv_in_ptr + offs_c64 * in_stride + (S + j), mask=mask_c, other=0.0).to(tl.float32)
                tl.store(conv_state_ptr + offs_c64 * K_HIST + j, s_new_j, mask=mask_c)

    @triton.jit
    def _gated_deltanet_multirow_verify_kernel(
        conv_out_ptr,    # float32 [S, conv_dim]
        beta_raw_ptr,    # float16/float32 [S, H_V]
        alpha_raw_ptr,   # float16/float32 [S, H_V]
        ssm_dt_ptr,      # float32 [H_V]
        ssm_a_ptr,       # float32 [H_V]
        state_ptr,       # float32 [H_V, D_V, D_K] (updated IN-PLACE)
        out_ptr,         # float32 [S, H_V, D_V]
        pre_ssm_ptr,     # float32 [H_V, D_V, D_K] (written if SAVE_INPUTS)
        q_save_ptr,      # float32 [S, H_K, D_K]   (written if SAVE_INPUTS)
        k_save_ptr,      # float32 [S, H_K, D_K]   (written if SAVE_INPUTS)
        v_save_ptr,      # float32 [S, H_V, D_V]   (written if SAVE_INPUTS)
        decay_save_ptr,  # float32 [S, H_V]        (written if SAVE_INPUTS)
        beta_save_ptr,   # float32 [S, H_V]        (written if SAVE_INPUTS)
        S,
        conv_dim,
        key_dim,
        q_scale,
        eps,
        SAVE_INPUTS: tl.constexpr,
        H_K: tl.constexpr,
        H_V: tl.constexpr,
        D_K: tl.constexpr,
        D_V: tl.constexpr,
        BLOCK_K: tl.constexpr,
        BLOCK_V: tl.constexpr,
    ):
        """
        Fused Multi-Row (`2 <= S <= 16`) Gated-DeltaNet Speculative Verify Kernel:
        Keeps `ssm_state [BLOCK_V, BLOCK_K]` in registers across all `S` tokens, fuses `beta` sigmoid,
        `alpha+dt` softplus, `decay` exp, `Q/K` L2-norm, and `H_K -> H_V` broadcast, and optionally
        snapshots `pre_ssm` + lightweight `(q_scaled, k, v, decay, beta)` for zero-recompute rollback!
        """
        pid_h = tl.program_id(0)
        pid_v = tl.program_id(1)
        pid_kh = pid_h % H_K

        offs_v = pid_v * BLOCK_V + tl.arange(0, BLOCK_V)
        offs_k = tl.arange(0, BLOCK_K)
        mask_v = offs_v < D_V
        mask_k = offs_k < D_K
        mask_vk = mask_v[:, None] & mask_k[None, :]

        dt_h = tl.load(ssm_dt_ptr + pid_h).to(tl.float32)
        a_h = tl.load(ssm_a_ptr + pid_h).to(tl.float32)

        state_off = pid_h.to(tl.int64) * (D_V * D_K) + offs_v[:, None] * D_K + offs_k[None, :]
        state = tl.load(state_ptr + state_off, mask=mask_vk, other=0.0).to(tl.float32)
        if SAVE_INPUTS:
            tl.store(pre_ssm_ptr + state_off, state, mask=mask_vk)

        for t in range(0, S):
            b_raw = tl.load(beta_raw_ptr + t * H_V + pid_h).to(tl.float32)
            beta_t = 1.0 / (1.0 + tl.exp(-b_raw))

            a_val = tl.load(alpha_raw_ptr + t * H_V + pid_h).to(tl.float32) + dt_h
            sp = tl.where(a_val > 20.0, a_val, tl.log(1.0 + tl.exp(tl.maximum(a_val, -80.0))))
            decay_t = tl.exp(sp * a_h)

            c_row = conv_out_ptr + t * conv_dim
            q_raw = tl.load(c_row + pid_kh * D_K + offs_k, mask=mask_k, other=0.0).to(tl.float32)
            k_raw = tl.load(c_row + key_dim + pid_kh * D_K + offs_k, mask=mask_k, other=0.0).to(tl.float32)
            q_t = q_raw * tl.rsqrt(tl.sum(q_raw * q_raw, axis=0) + eps) * q_scale
            k_t = k_raw * tl.rsqrt(tl.sum(k_raw * k_raw, axis=0) + eps)

            v_off = pid_h * D_V + offs_v
            v_t = tl.load(c_row + 2 * key_dim + v_off, mask=mask_v, other=0.0).to(tl.float32)

            if SAVE_INPUTS:
                tl.store(v_save_ptr + (t * H_V + pid_h) * D_V + offs_v, v_t, mask=mask_v)
                if pid_v == 0:
                    tl.store(decay_save_ptr + t * H_V + pid_h, decay_t)
                    tl.store(beta_save_ptr + t * H_V + pid_h, beta_t)
                    if pid_h < H_K:
                        qk_off = (t * H_K + pid_h) * D_K + offs_k
                        tl.store(q_save_ptr + qk_off, q_t, mask=mask_k)
                        tl.store(k_save_ptr + qk_off, k_t, mask=mask_k)

            state = state * decay_t
            kv_mem = tl.sum(state * k_t[None, :], axis=1)
            delta = (v_t - kv_mem) * beta_t
            state = state + delta[:, None] * k_t[None, :]
            y_t = tl.sum(state * q_t[None, :], axis=1)

            tl.store(out_ptr + (t * H_V + pid_h) * D_V + offs_v, y_t, mask=mask_v)

        tl.store(state_ptr + state_off, state, mask=mask_vk)


def exec_gated_deltanet_scan(
    q_scaled: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    decay: torch.Tensor,
    beta: torch.Tensor,
    state: torch.Tensor,
) -> torch.Tensor:
    """
    Execute exact Gated-DeltaNet recurrence over `S` tokens, updating `state` `[H_v, D_v, D_k]` in-place.
      q_scaled: float32 `[S, H_k, D_k]` (`H_k` divides `H_v`; tiled automatically without `.repeat`!)
      k       : float32 `[S, H_k, D_k]`
      v       : float32 `[S, H_v, D_v]`
      decay   : float32 `[S, H_v]`
      beta    : float32 `[S, H_v]`
      state   : float32 `[H_v, D_v, D_k]`
    Returns `out` of shape `float32[S, H_v, D_v]`.
    """
    S, H_k, D_k = q_scaled.shape
    H_v = v.shape[1]
    D_v = v.shape[-1]

    if HAS_TRITON and state.is_cuda and D_k <= 256:
        dev = state.device
        with cuda_device_scope(dev):
            q_c = q_scaled.to(dev).contiguous()
            k_c = k.to(dev).contiguous()
            v_c = v.to(dev).contiguous()
            dec_c = decay.to(dev).contiguous()
            bet_c = beta.to(dev).contiguous()
            out = torch.empty((S, H_v, D_v), device=dev, dtype=torch.float32)

            BLOCK_K = triton.next_power_of_2(D_k)
            BLOCK_V = 16 if BLOCK_K >= 128 else 32
            grid = (H_v, triton.cdiv(D_v, BLOCK_V))
            _gated_deltanet_scan_kernel[grid](
                q_c,
                k_c,
                v_c,
                dec_c,
                bet_c,
                state,
                out,
                S,
                H_K=H_k,
                H_V=H_v,
                D_K=D_k,
                D_V=D_v,
                BLOCK_K=BLOCK_K,
                BLOCK_V=BLOCK_V,
                num_warps=2,
            )
            return out

    # Exact FP32 PyTorch fallback (CPU or D_k > 256)
    if H_k != H_v:
        rep = H_v // H_k
        q_scaled = q_scaled.repeat(1, rep, 1)
        k = k.repeat(1, rep, 1)
    out = torch.empty((S, H_v, D_v), device=q_scaled.device, dtype=torch.float32)
    for t in range(S):
        state.mul_(decay[t].view(H_v, 1, 1))
        k_t = k[t]
        kv_mem = (state * k_t.unsqueeze(1)).sum(dim=-1)
        delta = (v[t] - kv_mem) * beta[t].unsqueeze(-1)
        state.add_(delta.unsqueeze(-1) * k_t.unsqueeze(1))
        out[t] = (state * q_scaled[t].unsqueeze(1)).sum(dim=-1)
    return out


def exec_gated_deltanet_decode_step(
    qkv_mix: torch.Tensor,
    beta_raw: torch.Tensor,
    alpha_raw: torch.Tensor,
    conv_state: torch.Tensor,
    conv_weight: torch.Tensor,
    ssm_dt: torch.Tensor,
    ssm_a: torch.Tensor,
    ssm_state: torch.Tensor,
    key_dim: int,
    num_k_heads: int,
    num_v_heads: int,
    head_k_dim: int,
    head_v_dim: int,
    q_scale: float,
    eps: float,
) -> Optional[torch.Tensor]:
    """
    Fast CUDA Triton S=1 Gated-DeltaNet path (2 kernel launches instead of 22!).
    Returns `float32[1, H_v, D_v]` on CUDA, or `None` when CPU fallback should be used.
    """
    if not (HAS_TRITON and ssm_state.is_cuda and head_k_dim <= 256):
        return None
    dev = ssm_state.device
    conv_dim = conv_weight.shape[0]
    conv_k = conv_weight.shape[-1]
    k_hist = conv_state.shape[-1]
    if conv_k != k_hist + 1:
        return None

    with cuda_device_scope(dev):
        qkv_c = qkv_mix.reshape(-1).contiguous()
        b_c = beta_raw.reshape(-1).contiguous()
        a_c = alpha_raw.reshape(-1).contiguous()
        conv_out = torch.empty((conv_dim,), device=dev, dtype=torch.float32)
        out = torch.empty((1, num_v_heads, head_v_dim), device=dev, dtype=torch.float32)

        BLOCK_C = 256
        _causal_conv1d_silu_step_kernel[(triton.cdiv(conv_dim, BLOCK_C),)](
            qkv_c,
            conv_state,
            conv_weight,
            conv_out,
            conv_dim,
            CONV_K=conv_k,
            K_HIST=k_hist,
            BLOCK_C=BLOCK_C,
            num_warps=4,
        )

        BLOCK_K = triton.next_power_of_2(head_k_dim)
        BLOCK_V = 16 if BLOCK_K >= 128 else 32
        _gated_deltanet_decode_step_kernel[(num_v_heads, triton.cdiv(head_v_dim, BLOCK_V))](
            conv_out,
            b_c,
            a_c,
            ssm_dt,
            ssm_a,
            ssm_state,
            out,
            key_dim,
            q_scale,
            eps,
            H_K=num_k_heads,
            H_V=num_v_heads,
            D_K=head_k_dim,
            D_V=head_v_dim,
            BLOCK_K=BLOCK_K,
            BLOCK_V=BLOCK_V,
            num_warps=2,
        )
        return out


def exec_gated_deltanet_multirow(
    qkv_mix: torch.Tensor,
    beta_raw: torch.Tensor,
    alpha_raw: torch.Tensor,
    conv_state: torch.Tensor,
    conv_weight: torch.Tensor,
    ssm_dt: torch.Tensor,
    ssm_a: torch.Tensor,
    ssm_state: torch.Tensor,
    key_dim: int,
    num_k_heads: int,
    num_v_heads: int,
    head_k_dim: int,
    head_v_dim: int,
    q_scale: float,
    eps: float,
    rec_state: Optional["LayerRecurrentState"] = None,
) -> Optional[torch.Tensor]:
    """
    Fast CUDA Triton Multi-Row (`2 <= S <= 16`) Gated-DeltaNet path (2 kernel launches per layer
    instead of ~20 eager PyTorch ops, saving ~768 launches per speculative verify step on 48-GDN-layer
    models while optionally recording zero-recompute rollback inputs!).
    Returns `float32[S, H_v, D_v]` on CUDA, or `None` when fallback should be used.
    """
    S = int(qkv_mix.shape[1]) if qkv_mix.dim() == 3 else int(qkv_mix.shape[0])
    if not (HAS_TRITON and ssm_state.is_cuda and head_k_dim <= 256 and 1 < S <= 16):
        return None
    dev = ssm_state.device
    conv_dim = int(conv_weight.shape[0])
    conv_k = int(conv_weight.shape[-1])
    k_hist = int(conv_state.shape[-1])
    if conv_k != k_hist + 1:
        return None

    save_inputs = bool(rec_state is not None and rec_state.verify_active)
    need_conv_in = save_inputs or not (conv_k == 4 and k_hist == 3)

    with cuda_device_scope(dev):
        qkv_c = qkv_mix.reshape(S, conv_dim).contiguous()
        b_c = beta_raw.reshape(S, num_v_heads).contiguous()
        a_c = alpha_raw.reshape(S, num_v_heads).contiguous()
        conv_out = torch.empty((S, conv_dim), device=dev, dtype=torch.float32)
        out = torch.empty((S, num_v_heads, head_v_dim), device=dev, dtype=torch.float32)

        conv_in = (
            torch.empty((1, conv_dim, k_hist + S), device=dev, dtype=torch.float32)
            if need_conv_in
            else conv_out
        )

        BLOCK_C = 256
        _causal_conv1d_silu_multirow_kernel[(triton.cdiv(conv_dim, BLOCK_C),)](
            qkv_c,
            conv_state,
            conv_weight,
            conv_out,
            conv_in,
            S,
            conv_dim,
            CONV_K=conv_k,
            K_HIST=k_hist,
            SAVE_CONV_IN=save_inputs,
            BLOCK_C=BLOCK_C,
            num_warps=4,
        )

        if save_inputs:
            pre_ssm = torch.empty_like(ssm_state)
            q_save = torch.empty((S, num_k_heads, head_k_dim), device=dev, dtype=torch.float32)
            k_save = torch.empty((S, num_k_heads, head_k_dim), device=dev, dtype=torch.float32)
            v_save = torch.empty((S, num_v_heads, head_v_dim), device=dev, dtype=torch.float32)
            decay_save = torch.empty((S, num_v_heads), device=dev, dtype=torch.float32)
            beta_save = torch.empty((S, num_v_heads), device=dev, dtype=torch.float32)
        else:
            pre_ssm = ssm_state
            q_save = out
            k_save = out
            v_save = out
            decay_save = conv_out
            beta_save = conv_out

        BLOCK_K = triton.next_power_of_2(head_k_dim)
        BLOCK_V = 16 if BLOCK_K >= 128 else 32
        _gated_deltanet_multirow_verify_kernel[(num_v_heads, triton.cdiv(head_v_dim, BLOCK_V))](
            conv_out,
            b_c,
            a_c,
            ssm_dt,
            ssm_a,
            ssm_state,
            out,
            pre_ssm,
            q_save,
            k_save,
            v_save,
            decay_save,
            beta_save,
            S,
            conv_dim,
            key_dim,
            q_scale,
            eps,
            SAVE_INPUTS=save_inputs,
            H_K=num_k_heads,
            H_V=num_v_heads,
            D_K=head_k_dim,
            D_V=head_v_dim,
            BLOCK_K=BLOCK_K,
            BLOCK_V=BLOCK_V,
            num_warps=2,
        )

        if save_inputs and rec_state is not None:
            rec_state.record_verify_inputs(
                pre_ssm_state=pre_ssm,
                conv_in=conv_in,
                q_scaled=q_save,
                k=k_save,
                v=v_save,
                decay=decay_save,
                beta=beta_save,
                hist_len=k_hist,
            )
        return out


class LayerRecurrentState:
    """
    Fixed-memory recurrent state (`conv_state` + `ssm_state`) for linear-attention / Gated-DeltaNet layers.
    Uses O(1) constant VRAM (~3.1 MiB per layer on Qwen3.8-27B) regardless of whether context is 4K or 128K!
    """

    def __init__(self, device: torch.device) -> None:
        self.device = device
        self.conv_state: Optional[torch.Tensor] = None  # float32 [1, conv_dim, conv_kernel - 1]
        self.ssm_state: Optional[torch.Tensor] = None   # float32 [num_v_heads, head_v_dim, head_k_dim]
        self.verify_active: bool = False
        # When an S>1 verify CUDA graph is captured, holds permanent references to the graph-owned
        # verify-cache buffers so `begin_recurrent_verify()` can re-point `_verify_cache` at them.
        self._graph_verify_cache: Optional[Tuple[torch.Tensor, ...]] = None
        self._verify_cache: Optional[
            Tuple[
                torch.Tensor,  # pre_ssm_state [H_v, D_v, D_k]
                torch.Tensor,  # conv_in [1, conv_dim, hist_len + S]
                torch.Tensor,  # q_scaled [S, H_k, D_k]
                torch.Tensor,  # k [S, H_k, D_k]
                torch.Tensor,  # v [S, H_v, D_v]
                torch.Tensor,  # decay [S, H_v]
                torch.Tensor,  # beta [S, H_v]
                int,           # hist_len
            ]
        ] = None

    def ensure_allocated(
        self,
        conv_dim: int,
        conv_kernel: int,
        num_v_heads: int,
        head_v_dim: int,
        head_k_dim: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        hist_len = max(1, conv_kernel - 1)
        if self.conv_state is None:
            self.conv_state = torch.zeros((1, conv_dim, hist_len), device=self.device, dtype=torch.float32)
        if self.ssm_state is None:
            self.ssm_state = torch.zeros(
                (num_v_heads, head_v_dim, head_k_dim), device=self.device, dtype=torch.float32
            )
        return self.conv_state, self.ssm_state

    def record_verify_inputs(
        self,
        pre_ssm_state: torch.Tensor,
        conv_in: torch.Tensor,
        q_scaled: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        decay: torch.Tensor,
        beta: torch.Tensor,
        hist_len: int,
    ) -> None:
        self._verify_cache = (pre_ssm_state, conv_in, q_scaled, k, v, decay, beta, hist_len)

    def commit_verify_prefix(self, n_keep: int) -> bool:
        """
        Zero-recompute speculative rollback: settle `conv_state` and `ssm_state` to exact row `n_keep`
        (`0 <= n_keep <= S`) by replaying ONLY the lightweight recurrence scan over `[:n_keep]`,
        avoiding a 64-layer trunk GEMV/MLP/Attention re-forward on draft rejection!
        """
        if self._verify_cache is None or self.conv_state is None or self.ssm_state is None:
            return False
        pre_ssm, conv_in, q_scaled, k, v, decay, beta, hist_len = self._verify_cache
        total_s = int(q_scaled.shape[0])
        if n_keep < 0 or n_keep > total_s:
            self._verify_cache = None
            return False
        with torch.inference_mode():
            if hist_len > 0 and conv_in.shape[-1] >= n_keep + hist_len:
                self.conv_state.copy_(conv_in[:, :, n_keep : n_keep + hist_len])
            if n_keep < total_s:
                self.ssm_state.copy_(pre_ssm)
                if n_keep > 0:
                    exec_gated_deltanet_scan(
                        q_scaled[:n_keep],
                        k[:n_keep],
                        v[:n_keep],
                        decay[:n_keep],
                        beta[:n_keep],
                        self.ssm_state,
                    )
        self._verify_cache = None
        self.verify_active = False
        return True

    def clear_verify_inputs(self) -> None:
        self._verify_cache = None
        self.verify_active = False

    def reset(self) -> None:
        self.clear_verify_inputs()
        with torch.inference_mode():
            if self.conv_state is not None:
                self.conv_state.zero_()
            if self.ssm_state is not None:
                self.ssm_state.zero_()

    def clone_state(self, target_device: Optional[torch.device] = None) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
        if self.conv_state is None or self.ssm_state is None:
            return None
        dev = target_device or self.device
        with torch.inference_mode():
            return (self.conv_state.to(dev).clone(), self.ssm_state.to(dev).clone())

    def restore_state(self, snap: Optional[Tuple[torch.Tensor, torch.Tensor]]) -> None:
        if snap is None:
            self.reset()
            return
        c_snap, s_snap = snap
        with torch.inference_mode():
            self.conv_state = c_snap.to(self.device).clone()
            self.ssm_state = s_snap.to(self.device).clone()

    def allocated_bytes(self) -> int:
        total = 0
        if self.conv_state is not None:
            total += self.conv_state.numel() * self.conv_state.element_size()
        if self.ssm_state is not None:
            total += self.ssm_state.numel() * self.ssm_state.element_size()
        return total


class LayerKVCache:
    """
    Single-layer KV cache residing on that layer's target GPU (`cuda:0` or `cuda:1`), supporting:
    - `int4` (~75% VRAM & bandwidth savings vs FP16, ~50% vs INT8; unlocks 256K+ context on Dual T4!),
    - `int8` (~50% VRAM savings vs FP16), or
    - `fp16` storage,
    with sequential peak-VRAM-minimizing paged growth (`page_growth=2048` / 256-token page granularity)
    and optional bounded Sliding-Window Ring Buffer (`sliding_window` for `draft-mtp`).
    """

    def __init__(
        self,
        n_kv_heads: int,
        head_dim: int,
        max_seq_len: int,
        device: torch.device,
        kv_dtype: Literal["int4", "int8", "fp16"] = "int8",
        page_growth: int = 2048,
        sliding_window: Optional[int] = None,
    ) -> None:
        if kv_dtype == "int4" and (head_dim % 2 != 0):
            raise ValueError(f"kv_dtype='int4' requires even head_dim, got {head_dim}")
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.device = device
        self.kv_dtype: Literal["int4", "int8", "fp16"] = kv_dtype
        self.page_growth = page_growth
        self.sliding_window: Optional[int] = (
            int(sliding_window) if (sliding_window is not None and int(sliding_window) > 0) else None
        )
        # +16 guard slots ensure speculative chaining + rollback never overwrites active window history!
        self._ring_cap: Optional[int] = (
            min(max_seq_len, self.sliding_window + 16) if self.sliding_window is not None else None
        )

        self.capacity = 0
        self.seq_len = 0
        self.logical_seq_len = 0
        self.k_buf: Optional[torch.Tensor] = None
        self.v_buf: Optional[torch.Tensor] = None
        self.k_scale: Optional[torch.Tensor] = None
        self.v_scale: Optional[torch.Tensor] = None
        self._part_out_scratch: Optional[torch.Tensor] = None
        self._graph_scratch_out: Optional[torch.Tensor] = None
        self._graph_scratch_lse: Optional[torch.Tensor] = None
        self._part_lse_scratch: Optional[torch.Tensor] = None

    def _ensure_capacity(self, needed_len: int) -> None:
        if self._ring_cap is not None:
            if self.capacity >= self._ring_cap:
                return
            new_cap = self._ring_cap
        else:
            if needed_len > self.max_seq_len:
                raise ValueError(
                    f"Context length {needed_len} exceeds configured max_seq_len ({self.max_seq_len}). "
                    f"Increase `max_context_len` when starting the engine."
                )
            if needed_len <= self.capacity:
                return
            new_cap = min(
                self.max_seq_len,
                max(self.page_growth, ((needed_len + self.page_growth - 1) // self.page_growth) * self.page_growth),
            )

        copy_len = min(self.seq_len, self.capacity)
        if self.kv_dtype == "int4":
            storage_dtype = torch.uint8
            storage_dim = self.head_dim // 2
        elif self.kv_dtype == "int8":
            storage_dtype = torch.int8
            storage_dim = self.head_dim
        else:
            storage_dtype = torch.float16
            storage_dim = self.head_dim

        # Sequential single-buffer growth: drops old `k_buf` BEFORE allocating `new_v` so PyTorch's
        # CUDA caching allocator immediately recycles the freed block (eliminates 2x VRAM spikes at 128K-256K!).
        new_k = torch.empty((self.n_kv_heads, new_cap, storage_dim), device=self.device, dtype=storage_dtype)
        if copy_len > 0 and self.k_buf is not None:
            new_k[:, :copy_len, :].copy_(self.k_buf[:, :copy_len, :])
        self.k_buf = new_k
        del new_k

        new_v = torch.empty((self.n_kv_heads, new_cap, storage_dim), device=self.device, dtype=storage_dtype)
        if copy_len > 0 and self.v_buf is not None:
            new_v[:, :copy_len, :].copy_(self.v_buf[:, :copy_len, :])
        self.v_buf = new_v
        del new_v

        if self.kv_dtype == "int4":
            new_ks = torch.empty((self.n_kv_heads, new_cap, 2), device=self.device, dtype=torch.float16)
            if copy_len > 0 and self.k_scale is not None:
                new_ks[:, :copy_len, :].copy_(self.k_scale[:, :copy_len, :])
            self.k_scale = new_ks
            del new_ks

            new_vs = torch.empty((self.n_kv_heads, new_cap, 2), device=self.device, dtype=torch.float16)
            if copy_len > 0 and self.v_scale is not None:
                new_vs[:, :copy_len, :].copy_(self.v_scale[:, :copy_len, :])
            self.v_scale = new_vs
            del new_vs
        elif self.kv_dtype == "int8":
            new_ks = torch.empty((self.n_kv_heads, new_cap), device=self.device, dtype=torch.float16)
            if copy_len > 0 and self.k_scale is not None:
                new_ks[:, :copy_len].copy_(self.k_scale[:, :copy_len])
            self.k_scale = new_ks
            del new_ks

            new_vs = torch.empty((self.n_kv_heads, new_cap), device=self.device, dtype=torch.float16)
            if copy_len > 0 and self.v_scale is not None:
                new_vs[:, :copy_len].copy_(self.v_scale[:, :copy_len])
            self.v_scale = new_vs
            del new_vs

        self.capacity = new_cap

    def _write_contiguous_chunk(
        self,
        k_s: torch.Tensor,
        v_s: torch.Tensor,
        dst_pos: int,
        pos_tensor: Optional[torch.Tensor] = None,
    ) -> None:
        """Write `[n_kv_heads, s_chunk, head_dim]` into physical `[dst_pos : dst_pos + s_chunk]`."""
        assert self.k_buf is not None and self.v_buf is not None
        s_chunk = int(k_s.shape[1])
        end_pos = dst_pos + s_chunk

        if self.kv_dtype == "int4":
            assert self.k_scale is not None and self.v_scale is not None
            half_d = self.head_dim // 2
            if HAS_TRITON and self.k_buf.is_cuda and half_d <= 256:
                dev = self.k_buf.device
                with cuda_device_scope(dev):
                    k_c = k_s.to(device=dev, dtype=torch.float16).contiguous()
                    v_c = v_s.to(device=dev, dtype=torch.float16).contiguous()
                    BLOCK_HD = triton.next_power_of_2(max(16, half_d))
                    if pos_tensor is not None:
                        _kv_append_int4_posptr_kernel[(self.n_kv_heads, s_chunk)](
                            k_c,
                            v_c,
                            self.k_buf,
                            self.v_buf,
                            self.k_scale,
                            self.v_scale,
                            pos_tensor,
                            self.capacity,
                            HALF_D=half_d,
                            BLOCK_HD=BLOCK_HD,
                            S_NEW=s_chunk,
                            num_warps=2,
                        )
                    else:
                        _kv_append_int4_kernel[(self.n_kv_heads, s_chunk)](
                            k_c,
                            v_c,
                            self.k_buf,
                            self.v_buf,
                            self.k_scale,
                            self.v_scale,
                            dst_pos,
                            s_chunk,
                            self.capacity,
                            HALF_D=half_d,
                            BLOCK_HD=BLOCK_HD,
                            num_warps=2,
                        )
            else:
                k_lo, k_hi = k_s[..., :half_d].float(), k_s[..., half_d:].float()
                v_lo, v_hi = v_s[..., :half_d].float(), v_s[..., half_d:].float()
                ks_lo = (k_lo.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / 7.0).to(torch.float16)
                ks_hi = (k_hi.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / 7.0).to(torch.float16)
                vs_lo = (v_lo.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / 7.0).to(torch.float16)
                vs_hi = (v_hi.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / 7.0).to(torch.float16)

                def _q4_u8(val: torch.Tensor, sc: torch.Tensor) -> torch.Tensor:
                    s_val = val / sc.float()
                    r_val = torch.floor(s_val.abs() + 0.5) * torch.where(s_val < 0.0, -1.0, 1.0)
                    return (r_val.clamp(-7.0, 7.0).to(torch.int32) + 8) & 0x0F

                kp = (_q4_u8(k_lo, ks_lo) | (_q4_u8(k_hi, ks_hi) << 4)).to(torch.uint8)
                vp = (_q4_u8(v_lo, vs_lo) | (_q4_u8(v_hi, vs_hi) << 4)).to(torch.uint8)
                ks_pair = torch.cat([ks_lo, ks_hi], dim=-1)
                vs_pair = torch.cat([vs_lo, vs_hi], dim=-1)
                if pos_tensor is not None:
                    pos_vec = pos_tensor.to(torch.long).view(1) + torch.arange(
                        s_chunk, device=self.k_buf.device, dtype=torch.long
                    )
                    self.k_buf.index_copy_(1, pos_vec, kp)
                    self.v_buf.index_copy_(1, pos_vec, vp)
                    self.k_scale.index_copy_(1, pos_vec, ks_pair)
                    self.v_scale.index_copy_(1, pos_vec, vs_pair)
                else:
                    self.k_buf[:, dst_pos:end_pos, :] = kp
                    self.v_buf[:, dst_pos:end_pos, :] = vp
                    self.k_scale[:, dst_pos:end_pos, :] = ks_pair
                    self.v_scale[:, dst_pos:end_pos, :] = vs_pair
        elif self.kv_dtype == "int8":
            assert self.k_scale is not None and self.v_scale is not None
            if HAS_TRITON and self.k_buf.is_cuda and self.head_dim <= 512:
                dev = self.k_buf.device
                with cuda_device_scope(dev):
                    k_c = k_s.to(device=dev, dtype=torch.float16).contiguous()
                    v_c = v_s.to(device=dev, dtype=torch.float16).contiguous()
                    BLOCK_D = triton.next_power_of_2(self.head_dim)
                    if pos_tensor is not None:
                        _kv_append_int8_posptr_kernel[(self.n_kv_heads, s_chunk)](
                            k_c,
                            v_c,
                            self.k_buf,
                            self.v_buf,
                            self.k_scale,
                            self.v_scale,
                            pos_tensor,
                            self.capacity,
                            HEAD_DIM=self.head_dim,
                            BLOCK_D=BLOCK_D,
                            S_NEW=s_chunk,
                            num_warps=2,
                        )
                    else:
                        _kv_append_int8_kernel[(self.n_kv_heads, s_chunk)](
                            k_c,
                            v_c,
                            self.k_buf,
                            self.v_buf,
                            self.k_scale,
                            self.v_scale,
                            dst_pos,
                            s_chunk,
                            self.capacity,
                            HEAD_DIM=self.head_dim,
                            BLOCK_D=BLOCK_D,
                            num_warps=2,
                        )
            elif pos_tensor is not None:
                k_amax = k_s.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6)
                v_amax = v_s.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6)
                ks = (k_amax / 127.0).to(torch.float16)
                vs = (v_amax / 127.0).to(torch.float16)
                kq = torch.round(k_s / ks).clamp(-127, 127).to(torch.int8)
                vq = torch.round(v_s / vs).clamp(-127, 127).to(torch.int8)
                pos_vec = pos_tensor.to(torch.long).view(1) + torch.arange(
                    s_chunk, device=self.k_buf.device, dtype=torch.long
                )
                self.k_buf.index_copy_(1, pos_vec, kq)
                self.v_buf.index_copy_(1, pos_vec, vq)
                self.k_scale.index_copy_(1, pos_vec, ks.squeeze(-1))
                self.v_scale.index_copy_(1, pos_vec, vs.squeeze(-1))
            else:
                k_amax = k_s.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6)
                v_amax = v_s.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6)
                ks = (k_amax / 127.0).to(torch.float16)
                vs = (v_amax / 127.0).to(torch.float16)
                self.k_buf[:, dst_pos:end_pos, :] = torch.round(k_s / ks).clamp(-127, 127).to(torch.int8)
                self.v_buf[:, dst_pos:end_pos, :] = torch.round(v_s / vs).clamp(-127, 127).to(torch.int8)
                self.k_scale[:, dst_pos:end_pos] = ks.squeeze(-1)
                self.v_scale[:, dst_pos:end_pos] = vs.squeeze(-1)
        else:
            if pos_tensor is not None:
                pos_vec = pos_tensor.to(torch.long).view(1) + torch.arange(
                    s_chunk, device=self.k_buf.device, dtype=torch.long
                )
                self.k_buf.index_copy_(1, pos_vec, k_s.to(torch.float16))
                self.v_buf.index_copy_(1, pos_vec, v_s.to(torch.float16))
            else:
                self.k_buf[:, dst_pos:end_pos, :] = k_s.to(torch.float16)
                self.v_buf[:, dst_pos:end_pos, :] = v_s.to(torch.float16)

    def append(
        self,
        k_new: torch.Tensor,
        v_new: torch.Tensor,
        start_pos: int,
        pos_tensor: Optional[torch.Tensor] = None,
    ) -> int:
        """
        Append `k_new`, `v_new` of shape `[1, n_kv_heads, S_new, head_dim]` at `start_pos`.
        When `pos_tensor` (`int32[1]` on GPU) is supplied, the write offsets are derived on-GPU
        from `pos_tensor` (`base + row`), making the path CUDA-Graph-capturable for any `S_new`.
        When `sliding_window` is configured, wraps writes into the bounded ring buffer (`O(W)` VRAM).
        """
        s_new = int(k_new.shape[2])
        end_pos = start_pos + s_new

        if self.sliding_window is not None:
            self._ensure_capacity(1)
            assert self.k_buf is not None and self.v_buf is not None
            cap = self.capacity
            k_s = k_new.squeeze(0)
            v_s = v_new.squeeze(0)
            w_start = start_pos
            if s_new > cap:
                keep = cap
                k_s = k_s[:, -keep:, :]
                v_s = v_s[:, -keep:, :]
                w_start = end_pos - keep
                s_write = keep
            else:
                s_write = s_new

            if w_start + s_write <= cap:
                self._write_contiguous_chunk(k_s, v_s, w_start)
            else:
                r0 = w_start % cap
                first_n = min(s_write, cap - r0)
                self._write_contiguous_chunk(k_s[:, :first_n, :], v_s[:, :first_n, :], r0)
                rem_n = s_write - first_n
                if rem_n > 0:
                    self._write_contiguous_chunk(k_s[:, first_n:, :], v_s[:, first_n:, :], 0)

            self.logical_seq_len = end_pos
            self.seq_len = min(end_pos, self.sliding_window)
            return end_pos

        if pos_tensor is None:
            self._ensure_capacity(end_pos)
        assert self.k_buf is not None and self.v_buf is not None
        self._write_contiguous_chunk(k_new.squeeze(0), v_new.squeeze(0), start_pos, pos_tensor=pos_tensor)
        self.logical_seq_len = end_pos
        self.seq_len = end_pos
        return end_pos

    def rewind_to(self, length: int) -> None:
        """Rewind logical and physical KV cache length cleanly (ring-window aware)."""
        clamped = max(0, int(length))
        if self.sliding_window is not None:
            self.logical_seq_len = min(self.logical_seq_len, clamped)
            self.seq_len = min(self.logical_seq_len, self.sliding_window)
        else:
            self.logical_seq_len = clamped
            self.seq_len = clamped

    def get_dequant_slice(self, start: int, end: int, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
        """Dequantize only `[start:end]` physical KV chunk -> `[1, n_kv_heads, end-start, head_dim]`."""
        assert self.k_buf is not None and self.v_buf is not None
        if self.kv_dtype == "int4":
            assert self.k_scale is not None and self.v_scale is not None
            kp = self.k_buf[:, start:end, :]
            vp = self.v_buf[:, start:end, :]
            ks = self.k_scale[:, start:end, :].to(dtype)
            vs = self.v_scale[:, start:end, :].to(dtype)
            k_lo = ((kp & 0x0F).to(dtype) - 8.0) * ks[..., 0:1]
            k_hi = (((kp >> 4) & 0x0F).to(dtype) - 8.0) * ks[..., 1:2]
            v_lo = ((vp & 0x0F).to(dtype) - 8.0) * vs[..., 0:1]
            v_hi = (((vp >> 4) & 0x0F).to(dtype) - 8.0) * vs[..., 1:2]
            return torch.cat([k_lo, k_hi], dim=-1).unsqueeze(0), torch.cat([v_lo, v_hi], dim=-1).unsqueeze(0)
        if self.kv_dtype == "int8":
            assert self.k_scale is not None and self.v_scale is not None
            ks = self.k_scale[:, start:end].to(dtype).unsqueeze(-1)
            vs = self.v_scale[:, start:end].to(dtype).unsqueeze(-1)
            k = self.k_buf[:, start:end, :].to(dtype) * ks
            v = self.v_buf[:, start:end, :].to(dtype) * vs
            return k.unsqueeze(0), v.unsqueeze(0)
        return self.k_buf[:, start:end, :].to(dtype).unsqueeze(0), self.v_buf[:, start:end, :].to(dtype).unsqueeze(0)

    def _get_ring_dequant_window(
        self,
        win_start: int,
        end_pos: int,
        dtype: torch.dtype,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Return chronological `[win_start : end_pos)` KV window from a ring buffer."""
        cap = self.capacity
        if end_pos <= cap:
            return self.get_dequant_slice(win_start, end_pos, dtype)
        r0 = win_start % cap
        r1 = end_pos % cap
        if r0 < r1:
            return self.get_dequant_slice(r0, r1, dtype)
        k_a, v_a = self.get_dequant_slice(r0, cap, dtype)
        if r1 == 0:
            return k_a, v_a
        k_b, v_b = self.get_dequant_slice(0, r1, dtype)
        return torch.cat([k_a, k_b], dim=2), torch.cat([v_a, v_b], dim=2)

    def attend(
        self,
        q: torch.Tensor,
        start_pos: int,
        sm_scale: float,
        softcap: float = 0.0,
        kv_chunk_size: int = 4096,
        pos_tensor: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Compute GQA causal attention for `q` `[1, n_heads, q_len, head_dim]`.
        Uses:
          1. Fused Split-KV INT4 / INT8 attention for up to 8 queries (GPU position pointers read the chunk base position)
          2. PyTorch SDPA fast path when `eff_len <= kv_chunk_size` and `softcap == 0`
          3. Online-Softmax Chunked Attention for ultra-long contexts (32K-256K+) so VRAM never spikes!
        """
        _, n_heads, q_len, head_dim = q.shape
        gqa_group = n_heads // self.n_kv_heads

        is_ring = self.sliding_window is not None
        if is_ring:
            assert self.sliding_window is not None
            end_pos = self.logical_seq_len
            win_cap = self.sliding_window + q_len - 1
            win_start = max(0, end_pos - win_cap)
            total_len = end_pos - win_start
        else:
            end_pos = self.seq_len
            win_start = 0
            total_len = self.seq_len

        # Path 1: Triton Split-KV INT4 / INT8 attention for short decode/verify batches.
        is_pow2_hd = head_dim in (32, 64, 128, 256)
        if (
            HAS_TRITON
            and q.is_cuda
            and self.kv_dtype in ("int4", "int8")
            and 1 <= q_len <= 8
            and (
                (pos_tensor is not None and not is_ring and q_len <= 8)
                or (
                    pos_tensor is None
                    and (
                        q_len == 1
                        or (start_pos + q_len == end_pos
                            and (not is_ring or self.capacity >= self.sliding_window + q_len - 1))
                    )
                )
            )
            and softcap == 0.0
            and is_pow2_hd
            and (pos_tensor is not None or total_len >= 1)
        ):
            assert self.k_buf is not None and self.v_buf is not None
            assert self.k_scale is not None and self.v_scale is not None
            dev = self.k_buf.device
            with cuda_device_scope(dev):
                if pos_tensor is not None:
                    num_splits = 32
                elif total_len < 128:
                    num_splits = 2
                elif total_len < 1024:
                    num_splits = 8
                elif total_len < 8192:
                    num_splits = 16
                else:
                    num_splits = 32
                block_s = 32 if head_dim >= 256 else 64

                rows = n_heads * q_len
                if pos_tensor is not None:
                    # Graph-reserved scratch: ONE max-size buffer per layer shared by every captured
                    # S=1..8 graph (dim-0 slicing stays contiguous for any row count). Allocated during
                    # capture setup so graph capture never allocates.
                    if self._graph_scratch_out is None or self._graph_scratch_out.shape[0] < rows:
                        self._graph_scratch_out = torch.empty((n_heads * 8, 32, head_dim), device=dev, dtype=torch.float32)
                        self._graph_scratch_lse = torch.empty((n_heads * 8, 32), device=dev, dtype=torch.float32)
                    assert self._graph_scratch_lse is not None
                    part_out = self._graph_scratch_out[:rows]
                    part_lse = self._graph_scratch_lse[:rows]
                else:
                    if (self._part_out_scratch is None
                        or self._part_out_scratch.shape != (rows, num_splits, head_dim)):
                        # Kernels index with NUM_SPLITS as the row stride: sliced
                        # 32-wide buffers are not contiguous for fewer splits.
                        self._part_out_scratch = torch.empty((rows, num_splits, head_dim), device=dev, dtype=torch.float32)
                        self._part_lse_scratch = torch.empty((rows, num_splits), device=dev, dtype=torch.float32)
                    part_out = self._part_out_scratch
                    part_lse = self._part_lse_scratch

                q_contig = q.to(device=dev, dtype=torch.float16).contiguous().view(rows, head_dim)
                grid = (rows, num_splits)
                if self.kv_dtype == "int4":
                    half_d = head_dim // 2
                    if pos_tensor is not None:
                        _int4_gqa_flash_decode_posptr_kernel[grid](
                            q_contig,
                            self.k_buf,
                            self.k_scale,
                            self.v_buf,
                            self.v_scale,
                            part_out,
                            part_lse,
                            pos_tensor,
                            self.capacity,
                            sm_scale,
                            GQA_GROUP=gqa_group,
                            NUM_SPLITS=num_splits,
                            BLOCK_S=block_s,
                            HALF_D=half_d,
                            Q_LEN=q_len,
                            num_warps=4,
                        )
                    else:
                        _int4_gqa_flash_decode_split_kernel[grid](
                            q_contig,
                            self.k_buf,
                            self.k_scale,
                            self.v_buf,
                            self.v_scale,
                            part_out,
                            part_lse,
                            total_len,
                            self.capacity,
                            win_start,
                            sm_scale,
                            WIN_RING=is_ring,
                            Q_LEN=q_len,
                            WINDOW_SIZE=self.sliding_window if is_ring else 0,
                            GQA_GROUP=gqa_group,
                            NUM_SPLITS=num_splits,
                            BLOCK_S=block_s,
                            HALF_D=half_d,
                            num_warps=4,
                        )
                else:
                    if pos_tensor is not None:
                        _int8_gqa_flash_decode_posptr_kernel[grid](
                            q_contig,
                            self.k_buf,
                            self.k_scale,
                            self.v_buf,
                            self.v_scale,
                            part_out,
                            part_lse,
                            pos_tensor,
                            self.capacity,
                            sm_scale,
                            GQA_GROUP=gqa_group,
                            NUM_SPLITS=num_splits,
                            BLOCK_S=block_s,
                            HEAD_DIM=head_dim,
                            Q_LEN=q_len,
                            num_warps=4,
                        )
                    else:
                        _int8_gqa_flash_decode_split_kernel[grid](
                            q_contig,
                            self.k_buf,
                            self.k_scale,
                            self.v_buf,
                            self.v_scale,
                            part_out,
                            part_lse,
                            total_len,
                            self.capacity,
                            win_start,
                            sm_scale,
                            WIN_RING=is_ring,
                            Q_LEN=q_len,
                            WINDOW_SIZE=self.sliding_window if is_ring else 0,
                            GQA_GROUP=gqa_group,
                            NUM_SPLITS=num_splits,
                            BLOCK_S=block_s,
                            HEAD_DIM=head_dim,
                            num_warps=4,
                        )
                out_fp16 = torch.empty((rows, head_dim), device=dev, dtype=torch.float16)
                _flash_decode_combine_kernel[(rows,)](
                    part_out,
                    part_lse,
                    out_fp16,
                    NUM_SPLITS=num_splits,
                    HEAD_DIM=head_dim,
                    num_warps=2,
                )
                return out_fp16.to(q.dtype).view(1, n_heads, q_len, head_dim)

        # Path 2: Short/Medium Context (or Sliding-Window Ring) -> Fused PyTorch SDPA
        if (total_len <= kv_chunk_size or is_ring) and softcap == 0.0:
            if is_ring:
                k_full, v_full = self._get_ring_dequant_window(win_start, end_pos, q.dtype)
            else:
                k_full, v_full = self.get_dequant_slice(0, total_len, q.dtype)

            # Prefer native GQA (`enable_gqa`, torch >= 2.5) so K/V are never expanded in memory;
            # fall back to `repeat_interleave` on unsupported backends/builds.
            sdpa_gqa = gqa_group > 1

            def _sdpa(
                _q: torch.Tensor,
                _k: torch.Tensor,
                _v: torch.Tensor,
                **_kw: Any,
            ) -> torch.Tensor:
                if sdpa_gqa:
                    try:
                        return F.scaled_dot_product_attention(_q, _k, _v, enable_gqa=True, **_kw)
                    except Exception:
                        return F.scaled_dot_product_attention(
                            _q,
                            _k.repeat_interleave(gqa_group, dim=1),
                            _v.repeat_interleave(gqa_group, dim=1),
                            **_kw,
                        )
                return F.scaled_dot_product_attention(_q, _k, _v, **_kw)

            if q_len == 1:
                return _sdpa(q, k_full, v_full, is_causal=False, scale=sm_scale)
            if start_pos == 0 and win_start == 0:
                return _sdpa(q, k_full, v_full, is_causal=True, scale=sm_scale)

            # Speculative / multi-token append (`start_pos > 0` or ring window): explicit causal + window mask
            q_pos = torch.arange(start_pos, start_pos + q_len, device=q.device).unsqueeze(1)
            k_pos = torch.arange(win_start, end_pos, device=q.device).unsqueeze(0)
            if is_ring and self.sliding_window is not None:
                attn_mask = (k_pos <= q_pos) & (k_pos > (q_pos - self.sliding_window))
            else:
                attn_mask = k_pos <= q_pos  # [q_len, total_len] bool mask (True = keep)
            return _sdpa(q, k_full, v_full, attn_mask=attn_mask, scale=sm_scale)

        # Path 3: Online-Softmax Chunked Attention for Big Contexts (32K-256K+) or Gemma Softcapping.
        # GQA group is folded into the query-row dimension (`2*qc` rows per kv-head) so K/V chunks are
        # never expanded; tensor-core fp16/bf16 matmuls with FP32 online-softmax bookkeeping.
        n_kv = self.n_kv_heads
        q_chunk_len = 512
        outputs: List[torch.Tensor] = []
        # Prefill matmuls prefer fp16 compute on CUDA tensor cores; CPU stays fp32 for accuracy.
        compute_dtype = torch.float16 if q.is_cuda and head_dim % 8 == 0 else torch.float32

        for q_off in range(0, q_len, q_chunk_len):
            q_end = min(q_len, q_off + q_chunk_len)
            qc_len = q_end - q_off
            q_sub = q[:, :, q_off:q_end, :]  # [1, H, qc_len, D]
            n_elems_g = gqa_group * qc_len
            # [1, H=nkv*g, qc, D] -> [1, nkv, g*qc, D] (row r = j*qc + s covers group head j, position s)
            q_fold = (
                q_sub.to(compute_dtype)
                .view(1, n_kv, gqa_group, qc_len, head_dim)
                .reshape(1, n_kv, n_elems_g, head_dim)
            )
            # Per folded row r: query position = start_pos + q_off + (r % qc_len)
            row_abs_pos = (torch.arange(n_elems_g, device=q.device) % qc_len) + (start_pos + q_off)
            max_visible_k = min(total_len, start_pos + q_end)

            m_i = torch.full((1, n_kv, n_elems_g, 1), float("-inf"), device=q.device, dtype=torch.float32)
            l_i = torch.zeros((1, n_kv, n_elems_g, 1), device=q.device, dtype=torch.float32)
            acc_o = torch.zeros((1, n_kv, n_elems_g, head_dim), device=q.device, dtype=torch.float32)

            for kv_off in range(0, max_visible_k, kv_chunk_size):
                kv_end = min(total_len, kv_off + kv_chunk_size)
                k_chunk, v_chunk = self.get_dequant_slice(kv_off, kv_end, compute_dtype)

                scores = torch.matmul(q_fold, k_chunk.transpose(-2, -1)).float() * sm_scale
                if softcap > 0.0:
                    scores = torch.tanh(scores / softcap) * softcap

                if kv_end > start_pos + q_off:
                    # Only the chunks overlapping the causal boundary need a mask
                    k_abs_pos = torch.arange(kv_off, kv_end, device=q.device).view(1, 1, 1, -1)
                    scores = scores.masked_fill(k_abs_pos > row_abs_pos.view(1, 1, n_elems_g, 1), float("-inf"))

                m_ij = scores.amax(dim=-1, keepdim=True)
                m_new = torch.maximum(m_i, m_ij)
                # Guard against (-inf) - (-inf) = NaN when an entire row is masked
                m_new_safe = torch.where(torch.isinf(m_new), torch.zeros_like(m_new), m_new)
                m_i_safe = torch.where(torch.isinf(m_i), torch.zeros_like(m_i), m_i)

                alpha = torch.where(torch.isinf(m_i), torch.zeros_like(m_i), torch.exp(m_i_safe - m_new_safe))
                p = torch.exp(scores - m_new_safe)

                acc_o = acc_o * alpha + torch.matmul(p.to(compute_dtype), v_chunk).float()
                l_i = l_i * alpha + p.sum(dim=-1, keepdim=True)
                m_i = m_new

            out = (acc_o / l_i.clamp(min=1e-20)).to(q.dtype)
            # Unfold: [1, nkv, g*qc, D] -> [1, nkv, g, qc, D] -> [1, H, qc, D] (head h = kv_head * g + j)
            outputs.append(out.view(1, n_kv, gqa_group, qc_len, head_dim).reshape(1, n_heads, qc_len, head_dim))

        return torch.cat(outputs, dim=2)


class EngineKVCacheManager:
    """
    Coordinates per-layer KV caches (for full-attention layers) AND fixed-size recurrent states
    (for Gated-DeltaNet / SSM hybrid layers) across 1 or 2+ GPUs, while tracking cached token IDs
    for instant Prefix Cache Reuse across multi-turn OpenAI chat conversations.
    """

    def __init__(
        self,
        n_layers: int,
        n_kv_heads: int,
        head_dim: int,
        max_seq_len: int,
        layer_devices: Sequence[torch.device],
        kv_dtype: Literal["int4", "int8", "fp16"] = "int8",
        recurrent_layers: Optional[Sequence[bool]] = None,
        tp_devices: Optional[Sequence[torch.device]] = None,
    ) -> None:
        rec_flags = list(recurrent_layers) if recurrent_layers is not None else [False] * n_layers
        self.has_recurrent = any(rec_flags)
        self.tp_devices: List[torch.device] = list(tp_devices) if tp_devices and len(tp_devices) > 1 else []
        tp_world = len(self.tp_devices)

        if tp_world > 1:
            self.layers: List[Optional[LayerKVCache]] = [None] * n_layers
            self.recurrent_states: List[Optional[LayerRecurrentState]] = [None] * n_layers
            self.tp_layers: List[Optional[List[LayerKVCache]]] = [
                None
                if rec_flags[i]
                else [
                    LayerKVCache(
                        n_kv_heads=n_kv_heads // tp_world,
                        head_dim=head_dim,
                        max_seq_len=max_seq_len,
                        device=dev,
                        kv_dtype=kv_dtype,
                    )
                    for dev in self.tp_devices
                ]
                for i in range(n_layers)
            ]
            self.tp_recurrent_states: List[Optional[List[LayerRecurrentState]]] = [
                [LayerRecurrentState(device=dev) for dev in self.tp_devices] if rec_flags[i] else None
                for i in range(n_layers)
            ]
            # expose rank-0 caches on `.layers` for backward-compatible inspection
            for i in range(n_layers):
                if self.tp_layers[i] is not None:
                    self.layers[i] = self.tp_layers[i][0]
                if self.tp_recurrent_states[i] is not None:
                    self.recurrent_states[i] = self.tp_recurrent_states[i][0]
        else:
            self.layers = [
                None
                if rec_flags[i]
                else LayerKVCache(
                    n_kv_heads=n_kv_heads,
                    head_dim=head_dim,
                    max_seq_len=max_seq_len,
                    device=layer_devices[i],
                    kv_dtype=kv_dtype,
                )
                for i in range(n_layers)
            ]
            self.recurrent_states = [
                LayerRecurrentState(device=layer_devices[i]) if rec_flags[i] else None
                for i in range(n_layers)
            ]
            self.tp_layers = [None] * n_layers
            self.tp_recurrent_states = [None] * n_layers

        self.cached_tokens: List[int] = []
        self.recurrent_seq_len: int = 0
        # Lightweight CPU-offloaded prompt checkpoint for hybrid models (0 MB GPU VRAM overhead)
        self._ckpt_len: int = 0
        self._ckpt_states: Dict[Tuple[int, int], Optional[Tuple[torch.Tensor, torch.Tensor]]] = {}

        # Private MTP (`blk.{n_layers}.nextn.*`) full-attention KV cache (supports compact `mtp_cache_start` + ring window)
        self._max_seq_len = max_seq_len
        self._kv_dtype: Literal["int4", "int8", "fp16"] = kv_dtype
        self.has_mtp: bool = False
        self.mtp_kv_window: Optional[int] = None
        self.mtp_kv_cache: Optional[LayerKVCache] = None
        self.mtp_tp_kv_caches: List[LayerKVCache] = []
        self.mtp_cache_start: int = 0
        self.mtp_exact_kv_len: int = 0
        self._mtp_ckpt_len: int = 0
        self._mtp_ckpt_h: Optional[torch.Tensor] = None

    def init_mtp_cache(
        self,
        n_kv_heads: int,
        head_dim: int,
        device: torch.device,
        tp_devices: Optional[Sequence[torch.device]] = None,
        sliding_window: Optional[int] = 4096,
    ) -> None:
        """
        Allocate private full-attention KV cache for the embedded GGUF MTP (`NextN`) draft block.
        Defaults to a `4096`-token Sliding-Window Ring Buffer (`~4 MiB` total VRAM even at 256K trunk context!).
        """
        win = int(sliding_window) if (sliding_window is not None and int(sliding_window) > 0) else None
        self.mtp_kv_window = win
        tp_list = list(tp_devices) if tp_devices and len(tp_devices) > 1 else []
        if tp_list:
            tp_world = len(tp_list)
            self.mtp_tp_kv_caches = [
                LayerKVCache(
                    n_kv_heads=n_kv_heads // tp_world,
                    head_dim=head_dim,
                    max_seq_len=self._max_seq_len,
                    device=dev,
                    kv_dtype=self._kv_dtype,
                    sliding_window=win,
                )
                for dev in tp_list
            ]
            self.mtp_kv_cache = self.mtp_tp_kv_caches[0]
        else:
            self.mtp_tp_kv_caches = []
            self.mtp_kv_cache = LayerKVCache(
                n_kv_heads=n_kv_heads,
                head_dim=head_dim,
                max_seq_len=self._max_seq_len,
                device=device,
                kv_dtype=self._kv_dtype,
                sliding_window=win,
            )
        self.has_mtp = True
        self.mtp_cache_start = 0
        self.mtp_exact_kv_len = 0

    def _iter_mtp_kv_caches(self) -> List[LayerKVCache]:
        if self.mtp_tp_kv_caches:
            return list(self.mtp_tp_kv_caches)
        if self.mtp_kv_cache is not None:
            return [self.mtp_kv_cache]
        return []

    def reset_mtp_cache(self, cache_start: int = 0) -> None:
        """Rebase private MTP KV cache to start compactly at `cache_start` (`kv_pos = abs_pos - cache_start`)."""
        self.mtp_cache_start = max(0, int(cache_start))
        self.mtp_exact_kv_len = self.mtp_cache_start
        for lkv in self._iter_mtp_kv_caches():
            lkv.rewind_to(0)

    def rollback_mtp_to(self, abs_len: int) -> None:
        """Rewind private MTP KV cache to `abs_len` (absolute token position)."""
        if not self.has_mtp:
            return
        if abs_len < self.mtp_cache_start:
            self.reset_mtp_cache(abs_len)
            return
        rel_len = abs_len - self.mtp_cache_start
        for lkv in self._iter_mtp_kv_caches():
            lkv.rewind_to(rel_len)
        self.mtp_exact_kv_len = min(self.mtp_exact_kv_len, abs_len)

    def _iter_all_kv_caches(self) -> List[LayerKVCache]:
        out: List[LayerKVCache] = []
        if self.tp_devices:
            for grp in self.tp_layers:
                if grp is not None:
                    out.extend(grp)
        else:
            for lkv in self.layers:
                if lkv is not None:
                    out.append(lkv)
        return out

    def _iter_all_recurrent_states(self) -> List[Tuple[Tuple[int, int], LayerRecurrentState]]:
        out: List[Tuple[Tuple[int, int], LayerRecurrentState]] = []
        if self.tp_devices:
            for idx, grp in enumerate(self.tp_recurrent_states):
                if grp is not None:
                    for rank, st in enumerate(grp):
                        out.append(((idx, rank), st))
        else:
            for idx, st in enumerate(self.recurrent_states):
                if st is not None:
                    out.append(((idx, 0), st))
        return out

    def ensure_all_kv_capacity(self, needed_len: int) -> None:
        """Pre-grow all KV cache pages outside CUDA Graph capture so buffer addresses stay static."""
        for lkv in self._iter_all_kv_caches():
            lkv._ensure_capacity(needed_len)
        if self.has_mtp:
            rel_need = max(1, needed_len - self.mtp_cache_start)
            for lkv in self._iter_mtp_kv_caches():
                lkv._ensure_capacity(rel_need)

    def match_prefix_length(self, prompt_tokens: Sequence[int]) -> int:
        """
        Return how many leading tokens of `prompt_tokens` are already materialized in cache.
        Always leaves at least 1 token to forward so logits for the next token are freshly produced!
        """
        if not self.cached_tokens or not prompt_tokens:
            return 0
        limit = min(len(self.cached_tokens), len(prompt_tokens) - 1)
        matched = 0
        while matched < limit and self.cached_tokens[matched] == prompt_tokens[matched]:
            matched += 1

        if not self.has_recurrent:
            return matched

        # Hybrid recurrent models: live GPU state sits at `self.recurrent_seq_len` (instant multi-turn hit!)
        if matched == self.recurrent_seq_len:
            return matched
        if self._ckpt_len > 0 and matched >= self._ckpt_len:
            return self._ckpt_len
        return 0

    def save_prompt_checkpoint(self) -> None:
        """Save CPU-offloaded recurrent state checkpoint at prompt boundary for shared-system-prompt reuse."""
        if not self.has_recurrent:
            return
        cpu_dev = torch.device("cpu")
        self._ckpt_len = self.recurrent_seq_len
        self._ckpt_states = {
            key: st.clone_state(target_device=cpu_dev)
            for key, st in self._iter_all_recurrent_states()
        }

    def begin_recurrent_verify(self) -> None:
        """Arm recurrent layers to cache lightweight scan inputs for zero-recompute rejection rollback."""
        if not self.has_recurrent:
            return
        for _key, st in self._iter_all_recurrent_states():
            st.verify_active = True
            if getattr(st, "_graph_verify_cache", None) is not None:
                # Verify CUDA-graph mode (`forward_verify_step`): the captured graph rewrites the
                # fixed verify-cache buffers on every replay; re-point at them instead of dropping them.
                st._verify_cache = st._graph_verify_cache
            else:
                st._verify_cache = None

    def commit_recurrent_verify(self, start_pos: int, n_keep: int) -> bool:
        """
        Commit `n_keep` verified rows across all recurrent layers in ~0.2 ms (no trunk re-forward!).
        Returns `True` if every recurrent layer settled via cached scan inputs.
        """
        if not self.has_recurrent:
            return True
        all_ok = True
        for _key, st in self._iter_all_recurrent_states():
            if not st.commit_verify_prefix(n_keep):
                all_ok = False
        if all_ok:
            self.recurrent_seq_len = start_pos + n_keep
        return all_ok

    def end_recurrent_verify(self) -> None:
        if not self.has_recurrent:
            return
        for _key, st in self._iter_all_recurrent_states():
            st.clear_verify_inputs()

    def snapshot_recurrent_gpu(self) -> Dict[Tuple[int, int], Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """Fast on-GPU snapshot of recurrent states prior to speculative verification."""
        if not self.has_recurrent:
            return {}
        return {
            key: st.clone_state()
            for key, st in self._iter_all_recurrent_states()
        }

    def restore_recurrent_gpu(
        self,
        snap: Dict[Tuple[int, int], Optional[Tuple[torch.Tensor, torch.Tensor]]],
        seq_len: int,
    ) -> None:
        if not self.has_recurrent:
            return
        for key, st in self._iter_all_recurrent_states():
            if key in snap:
                st.restore_state(snap[key])
        self.recurrent_seq_len = seq_len

    def rollback_to(self, length: int) -> None:
        """Rollback attention KV caches (O(1)) and synchronize hybrid recurrent states cleanly."""
        for layer_kv in self._iter_all_kv_caches():
            layer_kv.rewind_to(length)
        del self.cached_tokens[length:]
        if self.has_mtp:
            if length == 0:
                self.reset_mtp_cache(0)
            else:
                self.rollback_mtp_to(length)

        if self.has_recurrent:
            if length == 0:
                for _key, st in self._iter_all_recurrent_states():
                    st.reset()
                self.recurrent_seq_len = 0
            elif length == self.recurrent_seq_len:
                pass
            elif length == self._ckpt_len and self._ckpt_states:
                for key, st in self._iter_all_recurrent_states():
                    st.restore_state(self._ckpt_states.get(key))
                self.recurrent_seq_len = length

    def record_tokens(self, start_pos: int, new_tokens: Sequence[int]) -> None:
        del self.cached_tokens[start_pos:]
        self.cached_tokens.extend(int(t) for t in new_tokens)
        if self.has_recurrent:
            self.recurrent_seq_len = len(self.cached_tokens)

    def clear(self) -> None:
        self._ckpt_len = 0
        self._ckpt_states.clear()
        self._mtp_ckpt_len = 0
        self._mtp_ckpt_h = None
        self.rollback_to(0)

    def allocated_vram_bytes(self) -> int:
        total = 0
        for lkv in self._iter_all_kv_caches() + self._iter_mtp_kv_caches():
            if lkv.k_buf is not None:
                total += lkv.k_buf.numel() * lkv.k_buf.element_size() * 2
            if lkv.k_scale is not None:
                total += lkv.k_scale.numel() * lkv.k_scale.element_size() * 2
        for _key, st in self._iter_all_recurrent_states():
            total += st.allocated_bytes()
        return total

    def projected_max_vram_bytes(self) -> int:
        """Exact bytes the KV buffers would occupy at their configured full capacity
        (non-ring layers: `max_seq_len`; sliding-window layers: their ring cap).
        Does NOT allocate anything; recurrent-state bytes are counted as already allocated."""
        total = 0
        for lkv in self._iter_all_kv_caches() + self._iter_mtp_kv_caches():
            cap = lkv._ring_cap if lkv._ring_cap is not None else lkv.max_seq_len
            if lkv.kv_dtype == "int4":
                per_pos_per_head = lkv.head_dim + 8  # K+V int4 packed nibbles + fp16 K/V scales
            elif lkv.kv_dtype == "int8":
                per_pos_per_head = 2 * lkv.head_dim + 4  # K+V int8 + fp16 K/V scales
            else:
                per_pos_per_head = 4 * lkv.head_dim  # K+V fp16
            total += cap * lkv.n_kv_heads * per_pos_per_head
        for _key, st in self._iter_all_recurrent_states():
            total += st.allocated_bytes()
        return total


In [ ]:
%%writefile pygguf_engine/model.py
"""
Unified Multi-GPU GGUF Transformer (`0% llama.cpp`) supporting:
- Dense, Sparse Mixture-of-Experts (MoE), AND Hybrid Gated-DeltaNet + Gated Attention GGUF checkpoints
- Architectures: Llama / Llama-3, Mistral, Mixtral, Qwen2, Qwen2-MoE, Qwen3, Qwen3-MoE,
  Qwen3.5 / Qwen3.6 / Qwen3.8 (`qwen35`, `qwen35moe`, `qwen3next`), Gemma 2, Gemma 3
- Dual-Mode Multi-GPU Execution on Kaggle T4 x2 (`cuda:0` + `cuda:1`):
  1. 2-GPU Tensor Parallelism (`TP=2`, activates both T4 memory buses simultaneously at 640 GB/s!)
  2. Byte-balanced Pipeline Parallelism (`PP=2`) + Per-Stage `torch.cuda.CUDAGraph` `M=1` Decode Replay!
- Zero-VRAM Horizontal Weight Fusion (`FusedLinearPack` for QKV, Gate+Up, and DeltaNet projections)
- `logits_to_keep` slicing before `lm_head` (prevents 15GB+ vocab-logit OOMs during 64K-128K prefill!)
"""

from __future__ import annotations

import math
import time
from typing import Any, Dict, List, Literal, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .gguf_parser import GGUFFile, GGUFModelConfig
from .kv_cache import (
    EngineKVCacheManager,
    LayerKVCache,
    LayerRecurrentState,
    exec_gated_deltanet_decode_step,
    exec_gated_deltanet_multirow,
    exec_gated_deltanet_scan,
)
from .quant_kernels import (
    FusedLinearPack,
    GGUFQuantLinear,
    GGUFQuantMoEExperts,
    cuda_device_scope,
    exec_fused_act_mul_rowscale,
    exec_fused_add_rmsnorm,
    exec_fused_rmsnorm,
    exec_fused_rmsnorm_silu_gate,
    load_float_tensor,
)


class GGUFRMSNorm(nn.Module):
    def __init__(self, weight: torch.Tensor, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.dim = int(weight.shape[-1])
        self.register_buffer("weight", weight.to(torch.float32).contiguous(), persistent=False)

    def forward(self, x: torch.Tensor, out_dtype: Optional[torch.dtype] = None) -> torch.Tensor:
        return exec_fused_rmsnorm(x, self.weight, self.eps, out_dtype=out_dtype)

    def forward_with_residual(
        self,
        x: torch.Tensor,
        delta: torch.Tensor,
        out_dtype: torch.dtype = torch.float16,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        return exec_fused_add_rmsnorm(x, delta, self.weight, self.eps, out_dtype=out_dtype)


class RotaryEmbeddingCache:
    """
    Precomputed RoPE cos/sin table per GPU device supporting both NeoX (split-half) and GGUF-Llama (interleaved).

    Also implements context-extension scaling so GGUF checkpoints can run far beyond their trained
    window (`max_context_len` larger than the checkpoint's native context):
      - `linear`    — positions divided by `factor` (inv_freq / factor)
      - `ntk-aware` — base re-scaled as `theta * factor ** (dim / (dim - 2))`
      - `yarn`      — YaRN ramp (correction dims from `beta_fast`/`beta_slow` around the original
                      context length) plus the standard `mscale = 1 + 0.1 * ext_factor * log(factor)`
                      attention-factor, applied by multiplying the softmax scale (`self.attn_factor`).
    """

    def __init__(
        self,
        rope_dim: int,
        rope_theta: float,
        neox_style: bool,
        rope_scaling_type: str = "none",
        rope_scaling_factor: float = 1.0,
        rope_original_context_len: int = 0,
        rope_yarn_beta_fast: float = 32.0,
        rope_yarn_beta_slow: float = 1.0,
        rope_yarn_ext_factor: float = 1.0,
        rope_yarn_attn_factor: float = 0.0,
        rope_yarn_log_mul: float = 0.1,
        max_context_len: int = 0,
    ) -> None:
        self.rope_dim = rope_dim
        self.rope_theta = rope_theta
        self.neox_style = neox_style
        self.rope_scaling_type = self._normalize_scaling_type(rope_scaling_type)
        self.rope_scaling_factor = float(rope_scaling_factor) if rope_scaling_factor else 1.0
        if not math.isfinite(self.rope_scaling_factor) or self.rope_scaling_factor <= 0.0:
            self.rope_scaling_factor = 1.0
        self.rope_original_context_len = int(rope_original_context_len or 0)
        self.rope_yarn_beta_fast = float(rope_yarn_beta_fast) if rope_yarn_beta_fast else 32.0
        self.rope_yarn_beta_slow = float(rope_yarn_beta_slow) if rope_yarn_beta_slow else 1.0
        self.rope_yarn_ext_factor = float(rope_yarn_ext_factor) if rope_yarn_ext_factor else 1.0
        self.rope_yarn_attn_factor = float(rope_yarn_attn_factor) if rope_yarn_attn_factor else 0.0
        self.rope_yarn_log_mul = float(rope_yarn_log_mul) if rope_yarn_log_mul and rope_yarn_log_mul > 0.0 else 0.1
        self.max_context_len = int(max_context_len or 0)
        self.attn_factor = self._compute_attn_factor()
        self._tables_per_dev: Dict[torch.device, Tuple[torch.Tensor, torch.Tensor]] = {}

    @staticmethod
    def _normalize_scaling_type(raw: Optional[str]) -> str:
        stype = (raw or "none").strip().lower().replace("_", "-")
        if stype in ("", "none", "false", "0"):
            return "none"
        if stype in ("linear", "pos-interpolation", "position-interpolation"):
            return "linear"
        if stype in ("ntk", "ntk-aware", "ntkaware", "dynamic-ntk"):
            return "ntk-aware"
        if stype == "yarn":
            return "yarn"
        # Unknown scaling types must not silently lie about positions; fail loudly instead.
        raise ValueError(f"Unsupported rope scaling type: {raw!r} (expected one of: none, linear, ntk-aware, yarn)")

    def _find_correction_dim(self, num_rotations: float, base: float, max_pos: int) -> float:
        return (self.rope_dim * math.log(max_pos / (num_rotations * 2 * math.pi))) / (2 * math.log(base))

    def _yarn_original_max_pos(self) -> int:
        if self.rope_original_context_len > 0:
            return self.rope_original_context_len
        if self.max_context_len > 0:
            return self.max_context_len
        return 4096

    def _inv_freq(self, device: torch.device) -> torch.Tensor:
        idx = torch.arange(0, self.rope_dim, 2, device=device, dtype=torch.float32)
        base = float(self.rope_theta)
        factor = self.rope_scaling_factor
        stype = self.rope_scaling_type
        if stype == "linear" and factor != 1.0:
            return 1.0 / (base ** (idx / self.rope_dim)) / factor
        if stype == "ntk-aware" and factor != 1.0:
            base_ntk = base * (factor ** (self.rope_dim / max(1, self.rope_dim - 2)))
            return 1.0 / (base_ntk ** (idx / self.rope_dim))
        inv_freq_orig = 1.0 / (base ** (idx / self.rope_dim))
        if stype == "yarn" and factor != 1.0:
            original_max_pos = self._yarn_original_max_pos()
            low = math.floor(self._find_correction_dim(self.rope_yarn_beta_fast, base, original_max_pos))
            high = math.ceil(self._find_correction_dim(self.rope_yarn_beta_slow, base, original_max_pos))
            if high <= low:
                high = low + 1
            ramp_idx = torch.arange(0, self.rope_dim // 2, device=device, dtype=torch.float32)
            ramp = torch.clamp((ramp_idx - float(low)) / float(high - low), 0.0, 1.0)
            inv_freq_interp = inv_freq_orig / factor
            return inv_freq_interp * ramp + inv_freq_orig * (1.0 - ramp)
        return inv_freq_orig

    def _compute_attn_factor(self) -> float:
        if self.rope_scaling_type == "yarn" and self.rope_scaling_factor > 1.0:
            if self.rope_yarn_attn_factor > 0.0:
                return float(self.rope_yarn_attn_factor)
            ext = self.rope_yarn_ext_factor if self.rope_yarn_ext_factor > 0.0 else 1.0
            return 1.0 + self.rope_yarn_log_mul * ext * math.log(self.rope_scaling_factor)
        return 1.0

    def ensure_capacity(self, device: torch.device, needed: int) -> Tuple[torch.Tensor, torch.Tensor]:
        cached = self._tables_per_dev.get(device)
        if cached is None or cached[0].shape[0] < needed:
            new_cap = max(8192, ((needed + 4095) // 4096) * 4096)
            with cuda_device_scope(device):
                inv_freq = self._inv_freq(device)
                positions = torch.arange(0, new_cap, device=device, dtype=torch.float32)
                freqs = torch.outer(positions, inv_freq)
                cached = (torch.cos(freqs).contiguous(), torch.sin(freqs).contiguous())
            self._tables_per_dev[device] = cached
        return cached

    def _get_cos_sin(
        self,
        device: torch.device,
        start_pos: int,
        seq_len: int,
        pos_tensor: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        if pos_tensor is not None:
            # CUDA-graph-safe positions: `pos_tensor` (`int32[1]`) holds the chunk base position;
            # row `i` of a multi-token chunk sits at `base + i` (S=1 reduces to the old path exactly).
            cached = self._tables_per_dev.get(device)
            if cached is None:
                cached = self.ensure_capacity(device, self.max_context_len if self.max_context_len > 0 else 8192)
            base = pos_tensor.to(torch.long).view(1)
            if seq_len == 1:
                full_idx = base
            else:
                full_idx = base + torch.arange(seq_len, device=base.device, dtype=torch.long)
            cos = cached[0].index_select(0, full_idx).view(1, seq_len, 1, -1)
            sin = cached[1].index_select(0, full_idx).view(1, seq_len, 1, -1)
            return cos, sin

        needed = start_pos + seq_len
        cached = self.ensure_capacity(device, needed)
        cos = cached[0][start_pos:needed].view(1, seq_len, 1, -1)
        sin = cached[1][start_pos:needed].view(1, seq_len, 1, -1)
        return cos, sin

    def apply(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        start_pos: int,
        pos_tensor: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        q: `[B, S, n_heads, head_dim]`
        k: `[B, S, n_kv_heads, head_dim]`
        """
        seq_len = q.shape[1]
        cos, sin = self._get_cos_sin(q.device, start_pos, seq_len, pos_tensor=pos_tensor)
        return self._rotate(q, cos, sin), self._rotate(k, cos, sin)

    def _rotate(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        x_rot = x[..., : self.rope_dim].float()
        x_pass = x[..., self.rope_dim :]

        if self.neox_style:
            half = self.rope_dim // 2
            x1 = x_rot[..., :half]
            x2 = x_rot[..., half:]
            rotated = torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(orig_dtype)
        else:
            x_even = x_rot[..., 0::2]
            x_odd = x_rot[..., 1::2]
            out_even = x_even * cos - x_odd * sin
            out_odd = x_even * sin + x_odd * cos
            rotated = torch.stack([out_even, out_odd], dim=-1).flatten(-2).to(orig_dtype)

        if x_pass.numel() > 0:
            return torch.cat([rotated, x_pass], dim=-1)
        return rotated


class GGUFAttention(nn.Module):
    def __init__(
        self,
        config: GGUFModelConfig,
        rope: RotaryEmbeddingCache,
        n_heads: int,
        n_kv_heads: int,
        qkv_pack: FusedLinearPack,
        o_proj: GGUFQuantLinear,
        has_q_gate: bool,
        is_pre_fused_qkv: bool,
        q_norm: Optional[GGUFRMSNorm],
        k_norm: Optional[GGUFRMSNorm],
    ) -> None:
        super().__init__()
        self.config = config
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = config.head_dim
        self.rope = rope
        self.qkv_pack = qkv_pack
        self.o_proj = o_proj
        self.has_q_gate = has_q_gate
        self.is_pre_fused_qkv = is_pre_fused_qkv
        self.q_norm = q_norm
        self.k_norm = k_norm

        if config.query_pre_attn_scalar > 0:
            self.sm_scale = 1.0 / math.sqrt(config.query_pre_attn_scalar)
        else:
            self.sm_scale = 1.0 / math.sqrt(self.head_dim)

    @classmethod
    def from_gguf(
        cls,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        rope: RotaryEmbeddingCache,
        device: torch.device,
    ) -> "GGUFAttention":
        prefix = f"blk.{layer_idx}"
        if f"{prefix}.attn_q.weight" in gguf_file.tensors:
            q_proj = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_q.weight", f"{prefix}.attn_q.bias", device=device
            )
            k_proj = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_k.weight", f"{prefix}.attn_k.bias", device=device
            )
            v_proj = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_v.weight", f"{prefix}.attn_v.bias", device=device
            )
            has_q_gate = q_proj.out_features == 2 * config.n_heads * config.head_dim
            qkv_pack = FusedLinearPack([q_proj, k_proj, v_proj])
            is_pre_fused = False
            del q_proj, k_proj, v_proj
        else:
            qkv_single = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_qkv.weight", f"{prefix}.attn_qkv.bias", device=device
            )
            has_q_gate = False
            qkv_pack = FusedLinearPack([qkv_single])
            is_pre_fused = True
            del qkv_single

        o_proj = GGUFQuantLinear.from_gguf(
            gguf_file, f"{prefix}.attn_output.weight", f"{prefix}.attn_output.bias", device=device
        )

        q_norm_name = f"{prefix}.attn_q_norm.weight"
        k_norm_name = f"{prefix}.attn_k_norm.weight"
        q_norm = (
            GGUFRMSNorm(load_float_tensor(gguf_file, q_norm_name, device=device), config.rms_norm_eps)
            if q_norm_name in gguf_file.tensors
            else None
        )
        k_norm = (
            GGUFRMSNorm(load_float_tensor(gguf_file, k_norm_name, device=device), config.rms_norm_eps)
            if k_norm_name in gguf_file.tensors
            else None
        )
        return cls(
            config=config,
            rope=rope,
            n_heads=config.n_heads,
            n_kv_heads=config.n_kv_heads,
            qkv_pack=qkv_pack,
            o_proj=o_proj,
            has_q_gate=has_q_gate,
            is_pre_fused_qkv=is_pre_fused,
            q_norm=q_norm,
            k_norm=k_norm,
        )

    @classmethod
    def from_gguf_tp_shards(
        cls,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        rope: RotaryEmbeddingCache,
        tp_devices: Sequence[torch.device],
    ) -> List["GGUFAttention"]:
        tp = len(tp_devices)
        # Repack full tensors on host; only final TP shards reside on GPUs.
        load_dev = torch.device("cpu")
        prefix = f"blk.{layer_idx}"
        h_per = config.n_heads // tp
        kv_per = config.n_kv_heads // tp
        hd = config.head_dim

        if f"{prefix}.attn_q.weight" in gguf_file.tensors:
            q_full = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_q.weight", f"{prefix}.attn_q.bias", device=load_dev
            )
            k_full = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_k.weight", f"{prefix}.attn_k.bias", device=load_dev
            )
            v_full = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_v.weight", f"{prefix}.attn_v.bias", device=load_dev
            )
            has_q_gate = q_full.out_features == 2 * config.n_heads * hd
            q_rows_per = h_per * (2 * hd if has_q_gate else hd)
            kv_rows_per = kv_per * hd
            q_shards = [q_full.slice_output_rows(slice(r * q_rows_per, (r + 1) * q_rows_per), tp_devices[r]) for r in range(tp)]
            k_shards = [k_full.slice_output_rows(slice(r * kv_rows_per, (r + 1) * kv_rows_per), tp_devices[r]) for r in range(tp)]
            v_shards = [v_full.slice_output_rows(slice(r * kv_rows_per, (r + 1) * kv_rows_per), tp_devices[r]) for r in range(tp)]
            del q_full, k_full, v_full
        else:
            qkv_full = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.attn_qkv.weight", f"{prefix}.attn_qkv.bias", device=load_dev
            )
            has_q_gate = False
            q_total = config.n_heads * hd
            kv_total = config.n_kv_heads * hd
            q_rows_per = h_per * hd
            kv_rows_per = kv_per * hd
            q_shards = [qkv_full.slice_output_rows(slice(r * q_rows_per, (r + 1) * q_rows_per), tp_devices[r]) for r in range(tp)]
            k_shards = [
                qkv_full.slice_output_rows(slice(q_total + r * kv_rows_per, q_total + (r + 1) * kv_rows_per), tp_devices[r])
                for r in range(tp)
            ]
            v_shards = [
                qkv_full.slice_output_rows(
                    slice(q_total + kv_total + r * kv_rows_per, q_total + kv_total + (r + 1) * kv_rows_per),
                    tp_devices[r],
                )
                for r in range(tp)
            ]
            del qkv_full

        o_full = GGUFQuantLinear.from_gguf(
            gguf_file, f"{prefix}.attn_output.weight", f"{prefix}.attn_output.bias", device=load_dev
        )
        o_blk_cols = h_per * hd
        o_shards = [
            o_full.slice_input_blocks([r], o_blk_cols, tp_devices[r], keep_bias=(r == 0))
            for r in range(tp)
        ]
        del o_full

        q_norm_name = f"{prefix}.attn_q_norm.weight"
        k_norm_name = f"{prefix}.attn_k_norm.weight"
        q_nw = load_float_tensor(gguf_file, q_norm_name, device=torch.device("cpu")) if q_norm_name in gguf_file.tensors else None
        k_nw = load_float_tensor(gguf_file, k_norm_name, device=torch.device("cpu")) if k_norm_name in gguf_file.tensors else None

        shards: List[GGUFAttention] = []
        for r, dev in enumerate(tp_devices):
            with cuda_device_scope(dev):
                pack = FusedLinearPack([q_shards[r], k_shards[r], v_shards[r]])
                qn = GGUFRMSNorm(q_nw.to(dev), config.rms_norm_eps) if q_nw is not None else None
                kn = GGUFRMSNorm(k_nw.to(dev), config.rms_norm_eps) if k_nw is not None else None
                shards.append(
                    cls(
                        config=config,
                        rope=rope,
                        n_heads=h_per,
                        n_kv_heads=kv_per,
                        qkv_pack=pack,
                        o_proj=o_shards[r],
                        has_q_gate=has_q_gate,
                        is_pre_fused_qkv=False,
                        q_norm=qn,
                        k_norm=kn,
                    )
                )
        return shards

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: LayerKVCache,
        start_pos: int,
        pos_tensor: Optional[torch.Tensor] = None,
        kv_pos: Optional[int] = None,
    ) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        attn_gate: Optional[torch.Tensor] = None

        if self.is_pre_fused_qkv:
            q_dim = self.n_heads * self.head_dim
            kv_dim = self.n_kv_heads * self.head_dim
            qkv_raw = self.qkv_pack.forward_single_concat(x)
            q_raw, k_raw, v_raw = torch.split(qkv_raw, [q_dim, kv_dim, kv_dim], dim=-1)
            q = q_raw.view(bsz, seq_len, self.n_heads, self.head_dim)
            k = k_raw.view(bsz, seq_len, self.n_kv_heads, self.head_dim)
            v = v_raw.view(bsz, seq_len, self.n_kv_heads, self.head_dim)
        else:
            q_raw, k_raw, v_raw = self.qkv_pack(x)
            if self.has_q_gate:
                q_and_gate = q_raw.view(bsz, seq_len, self.n_heads, self.head_dim * 2)
                q, attn_gate = q_and_gate.chunk(2, dim=-1)
            else:
                q = q_raw.view(bsz, seq_len, self.n_heads, self.head_dim)
            k = k_raw.view(bsz, seq_len, self.n_kv_heads, self.head_dim)
            v = v_raw.view(bsz, seq_len, self.n_kv_heads, self.head_dim)

        if self.q_norm is not None:
            q = self.q_norm(q)
        if self.k_norm is not None:
            k = self.k_norm(k)

        q, k = self.rope.apply(q, k, start_pos, pos_tensor=pos_tensor)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        eff_kv_pos = start_pos if kv_pos is None else kv_pos
        kv_cache.append(k, v, eff_kv_pos, pos_tensor=pos_tensor)
        # YaRN folds its `mscale` attention factor into the softmax scale (equivalent to scaling `q`).
        attn_out = kv_cache.attend(
            q,
            start_pos=eff_kv_pos,
            sm_scale=self.sm_scale * getattr(self.rope, "attn_factor", 1.0),
            softcap=self.config.attn_logit_softcapping,
            pos_tensor=pos_tensor,
        )
        attn_out = attn_out.transpose(1, 2).contiguous().view(bsz, seq_len, self.n_heads * self.head_dim)
        if attn_gate is not None:
            gate_sig = torch.sigmoid(attn_gate.float()).to(attn_out.dtype)
            attn_out = attn_out * gate_sig.reshape(bsz, seq_len, self.n_heads * self.head_dim)
        return self.o_proj(attn_out)


class GGUFGatedDeltaNet(nn.Module):
    """
    Gated-DeltaNet Linear-Attention / SSM block for Qwen3.5 / Qwen3.6 / Qwen3.8 (`qwen35`, `qwen35moe`, `qwen3next`).
    Fuses `[wqkv | wqkv_gate | ssm_beta | ssm_alpha]` + 2-launch Triton S=1 decode step + fused Gated-RMSNorm!
    """

    def __init__(
        self,
        eps: float,
        proj_pack: FusedLinearPack,
        ssm_out: GGUFQuantLinear,
        conv_weight: torch.Tensor,
        ssm_dt: torch.Tensor,
        ssm_a: torch.Tensor,
        ssm_norm_weight: torch.Tensor,
        d_inner: int,
        num_v_heads: int,
        head_v_dim: int,
        conv_dim: int,
        key_dim: int,
        head_k_dim: int,
        num_k_heads: int,
    ) -> None:
        super().__init__()
        self.eps = eps
        self.proj_pack = proj_pack
        self.ssm_out = ssm_out
        self.d_inner = d_inner
        self.num_v_heads = num_v_heads
        self.head_v_dim = head_v_dim
        self.conv_dim = conv_dim
        self.key_dim = key_dim
        self.head_k_dim = head_k_dim
        self.num_k_heads = num_k_heads
        self.conv_kernel = int(conv_weight.shape[-1])
        self.q_scale = 1.0 / math.sqrt(head_k_dim)

        self.register_buffer("conv_weight", conv_weight.reshape(conv_dim, 1, self.conv_kernel).contiguous(), persistent=False)
        self.register_buffer("ssm_dt", ssm_dt.view(-1).contiguous(), persistent=False)
        self.register_buffer("ssm_a", ssm_a.view(-1).contiguous(), persistent=False)
        self.register_buffer("ssm_norm_weight", ssm_norm_weight.to(torch.float32).view(-1).contiguous(), persistent=False)

    @classmethod
    def from_gguf(
        cls,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        device: torch.device,
    ) -> "GGUFGatedDeltaNet":
        prefix = f"blk.{layer_idx}"
        wqkv = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.attn_qkv.weight", device=device)
        wqkv_gate = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.attn_gate.weight", device=device)
        ssm_beta = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ssm_beta.weight", device=device)
        ssm_alpha = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ssm_alpha.weight", device=device)
        ssm_out = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ssm_out.weight", device=device)

        d_inner = wqkv_gate.out_features
        num_v_heads = ssm_beta.out_features
        head_v_dim = d_inner // num_v_heads
        conv_dim = wqkv.out_features
        key_dim = (conv_dim - d_inner) // 2
        head_k_dim = config.ssm_state_size if config.ssm_state_size > 0 else head_v_dim
        num_k_heads = key_dim // head_k_dim

        proj_pack = FusedLinearPack([wqkv, wqkv_gate, ssm_beta, ssm_alpha])
        del wqkv, wqkv_gate, ssm_beta, ssm_alpha

        conv_w = load_float_tensor(gguf_file, f"{prefix}.ssm_conv1d.weight", device=device, dtype=torch.float32)
        ssm_dt = load_float_tensor(gguf_file, f"{prefix}.ssm_dt.bias", device=device, dtype=torch.float32)
        ssm_a = load_float_tensor(gguf_file, f"{prefix}.ssm_a", device=device, dtype=torch.float32)
        norm_w = load_float_tensor(gguf_file, f"{prefix}.ssm_norm.weight", device=device, dtype=torch.float32)

        return cls(
            eps=config.rms_norm_eps,
            proj_pack=proj_pack,
            ssm_out=ssm_out,
            conv_weight=conv_w,
            ssm_dt=ssm_dt,
            ssm_a=ssm_a,
            ssm_norm_weight=norm_w,
            d_inner=d_inner,
            num_v_heads=num_v_heads,
            head_v_dim=head_v_dim,
            conv_dim=conv_dim,
            key_dim=key_dim,
            head_k_dim=head_k_dim,
            num_k_heads=num_k_heads,
        )

    @classmethod
    def from_gguf_tp_shards(
        cls,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        tp_devices: Sequence[torch.device],
    ) -> List["GGUFGatedDeltaNet"]:
        """
        Exact Tiled-Head Tensor Parallelism for Gated-DeltaNet (`V` head `vh` pairs with `K` head `vh % num_k_heads`).
        Rank `r` owns `k_heads = [r*(Hk/tp) .. (r+1)*(Hk/tp) - 1]` and all `rep` V-head tiles `tile*Hk + k_heads`,
        so zero cross-GPU communication is needed until the final `ssm_out` all-reduce!
        """
        tp = len(tp_devices)
        # Repack full tensors on host; only final TP shards reside on GPUs.
        load_dev = torch.device("cpu")
        prefix = f"blk.{layer_idx}"

        wqkv_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.attn_qkv.weight", device=load_dev)
        wgate_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.attn_gate.weight", device=load_dev)
        wbeta_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ssm_beta.weight", device=load_dev)
        walpha_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ssm_alpha.weight", device=load_dev)
        wout_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ssm_out.weight", device=load_dev)

        d_inner = wgate_full.out_features
        num_v_heads = wbeta_full.out_features
        head_v_dim = d_inner // num_v_heads
        conv_dim = wqkv_full.out_features
        key_dim = (conv_dim - d_inner) // 2
        head_k_dim = config.ssm_state_size if config.ssm_state_size > 0 else head_v_dim
        num_k_heads = key_dim // head_k_dim
        rep = num_v_heads // num_k_heads

        hk_per = num_k_heads // tp
        hv_per = num_v_heads // tp
        key_dim_per = hk_per * head_k_dim
        d_inner_per = hv_per * head_v_dim
        conv_dim_per = 2 * key_dim_per + d_inner_per

        conv_w_cpu = load_float_tensor(gguf_file, f"{prefix}.ssm_conv1d.weight", device=torch.device("cpu"), dtype=torch.float32)
        ssm_dt_cpu = load_float_tensor(gguf_file, f"{prefix}.ssm_dt.bias", device=torch.device("cpu"), dtype=torch.float32).view(-1)
        ssm_a_cpu = load_float_tensor(gguf_file, f"{prefix}.ssm_a", device=torch.device("cpu"), dtype=torch.float32).view(-1)
        norm_w_cpu = load_float_tensor(gguf_file, f"{prefix}.ssm_norm.weight", device=torch.device("cpu"), dtype=torch.float32)

        shards: List[GGUFGatedDeltaNet] = []
        for r, dev in enumerate(tp_devices):
            k_heads_r = list(range(r * hk_per, (r + 1) * hk_per))
            # Tiled V-head indices preserving `[tile_0_k_heads, tile_1_k_heads, ...]` order!
            v_heads_r = [tile * num_k_heads + kh for tile in range(rep) for kh in k_heads_r]

            q_rows = [kh * head_k_dim + d for kh in k_heads_r for d in range(head_k_dim)]
            k_rows = [key_dim + kh * head_k_dim + d for kh in k_heads_r for d in range(head_k_dim)]
            v_rows_in_qkv = [2 * key_dim + vh * head_v_dim + d for vh in v_heads_r for d in range(head_v_dim)]
            qkv_row_idx = q_rows + k_rows + v_rows_in_qkv
            gate_row_idx = [vh * head_v_dim + d for vh in v_heads_r for d in range(head_v_dim)]

            wqkv_r = wqkv_full.slice_output_rows(qkv_row_idx, dev)
            wgate_r = wgate_full.slice_output_rows(gate_row_idx, dev)
            wbeta_r = wbeta_full.slice_output_rows(v_heads_r, dev)
            walpha_r = walpha_full.slice_output_rows(v_heads_r, dev)
            wout_r = wout_full.slice_input_blocks(v_heads_r, head_v_dim, dev, keep_bias=(r == 0))

            qkv_idx_cpu = torch.tensor(qkv_row_idx, dtype=torch.long)
            vh_idx_cpu = torch.tensor(v_heads_r, dtype=torch.long)

            with cuda_device_scope(dev):
                pack_r = FusedLinearPack([wqkv_r, wgate_r, wbeta_r, walpha_r])
                shards.append(
                    cls(
                        eps=config.rms_norm_eps,
                        proj_pack=pack_r,
                        ssm_out=wout_r,
                        conv_weight=conv_w_cpu.index_select(0, qkv_idx_cpu).to(dev),
                        ssm_dt=ssm_dt_cpu.index_select(0, vh_idx_cpu).to(dev),
                        ssm_a=ssm_a_cpu.index_select(0, vh_idx_cpu).to(dev),
                        ssm_norm_weight=norm_w_cpu.to(dev),
                        d_inner=d_inner_per,
                        num_v_heads=hv_per,
                        head_v_dim=head_v_dim,
                        conv_dim=conv_dim_per,
                        key_dim=key_dim_per,
                        head_k_dim=head_k_dim,
                        num_k_heads=hk_per,
                    )
                )

        del wqkv_full, wgate_full, wbeta_full, walpha_full, wout_full
        return shards

    def forward(self, x: torch.Tensor, rec_state: LayerRecurrentState) -> torch.Tensor:
        bsz, seq_len, _ = x.shape
        if bsz != 1:
            raise ValueError(f"GGUFGatedDeltaNet expects batch_size=1, got {bsz}")

        conv_state, ssm_state = rec_state.ensure_allocated(
            self.conv_dim, self.conv_kernel, self.num_v_heads, self.head_v_dim, self.head_k_dim
        )

        qkv_mix, z, beta_raw, alpha_raw = self.proj_pack(x)

        # Ultra-fast S=1 CUDA Triton Decode Path (2 launches instead of 22!)
        if seq_len == 1:
            y_fast = exec_gated_deltanet_decode_step(
                qkv_mix=qkv_mix,
                beta_raw=beta_raw,
                alpha_raw=alpha_raw,
                conv_state=conv_state,
                conv_weight=self.conv_weight,
                ssm_dt=self.ssm_dt,
                ssm_a=self.ssm_a,
                ssm_state=ssm_state,
                key_dim=self.key_dim,
                num_k_heads=self.num_k_heads,
                num_v_heads=self.num_v_heads,
                head_k_dim=self.head_k_dim,
                head_v_dim=self.head_v_dim,
                q_scale=self.q_scale,
                eps=self.eps,
            )
            if y_fast is not None:
                z_heads = z.view(1, self.num_v_heads, self.head_v_dim)
                y_gated = exec_fused_rmsnorm_silu_gate(y_fast, z_heads, self.ssm_norm_weight, self.eps)
                return self.ssm_out(y_gated.reshape(1, 1, self.d_inner))
        elif 1 < seq_len <= 16:
            # Ultra-fast Multi-Row (`2 <= S <= 16`) Speculative Verify Path (2 launches + zero-recompute rollback!)
            y_multi = exec_gated_deltanet_multirow(
                qkv_mix=qkv_mix,
                beta_raw=beta_raw,
                alpha_raw=alpha_raw,
                conv_state=conv_state,
                conv_weight=self.conv_weight,
                ssm_dt=self.ssm_dt,
                ssm_a=self.ssm_a,
                ssm_state=ssm_state,
                key_dim=self.key_dim,
                num_k_heads=self.num_k_heads,
                num_v_heads=self.num_v_heads,
                head_k_dim=self.head_k_dim,
                head_v_dim=self.head_v_dim,
                q_scale=self.q_scale,
                eps=self.eps,
                rec_state=rec_state,
            )
            if y_multi is not None:
                z_heads = z.view(seq_len, self.num_v_heads, self.head_v_dim)
                y_gated = exec_fused_rmsnorm_silu_gate(y_multi, z_heads, self.ssm_norm_weight, self.eps).to(x.dtype)
                return self.ssm_out(y_gated.reshape(1, seq_len, self.d_inner))

        beta = torch.sigmoid(beta_raw.float()).squeeze(0)
        alpha = alpha_raw.float().squeeze(0)
        g = F.softplus(alpha + self.ssm_dt) * self.ssm_a
        decay = torch.exp(g)

        qkv_t = qkv_mix.float().transpose(1, 2)
        conv_in = torch.cat([conv_state, qkv_t], dim=-1)
        if self.conv_kernel > 1:
            conv_state.copy_(conv_in[:, :, -(self.conv_kernel - 1) :])
        conv_out = F.silu(F.conv1d(conv_in, self.conv_weight, groups=self.conv_dim))
        conv_out = conv_out.squeeze(0).transpose(0, 1).contiguous()

        q_raw, k_raw, v_raw = torch.split(conv_out, [self.key_dim, self.key_dim, self.d_inner], dim=-1)
        q = q_raw.view(seq_len, self.num_k_heads, self.head_k_dim)
        k = k_raw.view(seq_len, self.num_k_heads, self.head_k_dim)
        v = v_raw.view(seq_len, self.num_v_heads, self.head_v_dim)

        q = q * torch.rsqrt(q.pow(2).sum(dim=-1, keepdim=True) + self.eps)
        k = k * torch.rsqrt(k.pow(2).sum(dim=-1, keepdim=True) + self.eps)
        q_scaled = q * self.q_scale

        if rec_state.verify_active and 1 < seq_len <= 16:
            rec_state.record_verify_inputs(
                pre_ssm_state=ssm_state.clone(),
                conv_in=conv_in,
                q_scaled=q_scaled,
                k=k,
                v=v,
                decay=decay,
                beta=beta,
                hist_len=max(1, self.conv_kernel - 1),
            )

        # `exec_gated_deltanet_scan` tiles `num_k_heads -> num_v_heads` natively without `.repeat()`!
        y = exec_gated_deltanet_scan(q_scaled, k, v, decay, beta, ssm_state)

        z_heads = z.view(seq_len, self.num_v_heads, self.head_v_dim)
        y_gated = exec_fused_rmsnorm_silu_gate(y, z_heads, self.ssm_norm_weight, self.eps).to(x.dtype)
        return self.ssm_out(y_gated.reshape(1, seq_len, self.d_inner))


class GGUFDenseMLP(nn.Module):
    def __init__(
        self,
        gate_up_pack: FusedLinearPack,
        down_proj: GGUFQuantLinear,
        use_gelu: bool,
    ) -> None:
        super().__init__()
        self.gate_up_pack = gate_up_pack
        self.down_proj = down_proj
        self.use_gelu = use_gelu

    @classmethod
    def from_gguf(
        cls,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        device: torch.device,
        prefix_override: Optional[str] = None,
    ) -> "GGUFDenseMLP":
        prefix = prefix_override or f"blk.{layer_idx}"
        use_gelu = config.arch.startswith("gemma")
        gate_proj = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_gate.weight", device=device)
        up_proj = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_up.weight", device=device)
        down_proj = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_down.weight", device=device)
        pack = FusedLinearPack([gate_proj, up_proj])
        del gate_proj, up_proj
        return cls(pack, down_proj, use_gelu=use_gelu)

    @classmethod
    def from_gguf_tp_shards(
        cls,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        tp_devices: Sequence[torch.device],
    ) -> List["GGUFDenseMLP"]:
        tp = len(tp_devices)
        # Repack full tensors on host; only final TP shards reside on GPUs.
        load_dev = torch.device("cpu")
        prefix = f"blk.{layer_idx}"
        use_gelu = config.arch.startswith("gemma")

        gate_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_gate.weight", device=load_dev)
        up_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_up.weight", device=load_dev)
        down_full = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_down.weight", device=load_dev)

        inter_per = gate_full.out_features // tp
        shards: List[GGUFDenseMLP] = []
        for r, dev in enumerate(tp_devices):
            row_sl = slice(r * inter_per, (r + 1) * inter_per)
            g_r = gate_full.slice_output_rows(row_sl, dev)
            u_r = up_full.slice_output_rows(row_sl, dev)
            d_r = down_full.slice_input_blocks([r], inter_per, dev, keep_bias=(r == 0))
            with cuda_device_scope(dev):
                shards.append(cls(FusedLinearPack([g_r, u_r]), d_r, use_gelu=use_gelu))

        del gate_full, up_full, down_full
        return shards

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.gate_up_pack.is_single_contiguous:
            gu = self.gate_up_pack.forward_single_concat(x)
            down_in, row_scale = exec_fused_act_mul_rowscale(gu, None, use_gelu=self.use_gelu)
        else:
            gate, up = self.gate_up_pack(x)
            down_in, row_scale = exec_fused_act_mul_rowscale(gate, up, use_gelu=self.use_gelu)
        return self.down_proj(down_in.to(x.dtype)).float() * row_scale


class GGUFSparseMoEMLP(nn.Module):
    """
    Sparse Mixture-of-Experts FFN block supporting:
    - Qwen3-MoE (`Qwen3-30B-A3B`), Qwen3.5-MoE (`Qwen3.5-35B-A3B`), Qwen2-MoE, and Mixtral!
    - Only reads active `top_k` expert tensors per token from GPU VRAM.
    """

    def __init__(
        self,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        device: torch.device,
    ) -> None:
        super().__init__()
        self.config = config
        self.n_experts = config.n_experts
        self.top_k = max(1, config.n_experts_used)
        self.norm_topk_prob = config.norm_topk_prob
        self.use_gelu = config.arch.startswith("gemma")

        prefix = f"blk.{layer_idx}"
        router_w = load_float_tensor(gguf_file, f"{prefix}.ffn_gate_inp.weight", device=device, dtype=torch.float32)
        self.register_buffer("router_weight", router_w, persistent=False)

        self.gate_exps = GGUFQuantMoEExperts.from_gguf(gguf_file, f"{prefix}.ffn_gate_exps.weight", device=device)
        self.up_exps = GGUFQuantMoEExperts.from_gguf(gguf_file, f"{prefix}.ffn_up_exps.weight", device=device)
        self.down_exps = GGUFQuantMoEExperts.from_gguf(gguf_file, f"{prefix}.ffn_down_exps.weight", device=device)

        shexp_gate_name = f"{prefix}.ffn_gate_shexp.weight"
        if shexp_gate_name in gguf_file.tensors:
            sh_g = GGUFQuantLinear.from_gguf(gguf_file, shexp_gate_name, device=device)
            sh_u = GGUFQuantLinear.from_gguf(gguf_file, f"{prefix}.ffn_up_shexp.weight", device=device)
            self.shared_gate_up: Optional[FusedLinearPack] = FusedLinearPack([sh_g, sh_u])
            del sh_g, sh_u
            self.shared_down: Optional[GGUFQuantLinear] = GGUFQuantLinear.from_gguf(
                gguf_file, f"{prefix}.ffn_down_shexp.weight", device=device
            )
            shexp_inp_name = f"{prefix}.ffn_gate_inp_shexp.weight"
            if shexp_inp_name in gguf_file.tensors:
                sh_inp = load_float_tensor(gguf_file, shexp_inp_name, device=device, dtype=torch.float32).view(1, -1)
                self.register_buffer("shared_inp_gate", sh_inp, persistent=False)
            else:
                self.shared_inp_gate = None
        else:
            self.shared_gate_up = None
            self.shared_down = None
            self.shared_inp_gate = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_shape = x.shape
        x_2d = x.reshape(-1, orig_shape[-1])  # [M, H]

        router_logits = F.linear(x_2d.float(), self.router_weight)  # [M, E]
        probs = F.softmax(router_logits, dim=-1, dtype=torch.float32)
        topk_weights, topk_indices = torch.topk(probs, k=self.top_k, dim=-1)  # [M, K]

        if self.norm_topk_prob:
            topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True).clamp(min=1e-20)

        out_2d = torch.zeros_like(x_2d, dtype=torch.float32)

        for exp_id in topk_indices.unique().tolist():
            exp_mask = topk_indices == exp_id
            tok_idx, slot_idx = torch.where(exp_mask)
            x_e = x_2d.index_select(0, tok_idx)

            g_e = self.gate_exps.forward_expert(exp_id, x_e)
            u_e = self.up_exps.forward_expert(exp_id, x_e)
            down_in_e, row_scale_e = exec_fused_act_mul_rowscale(g_e, u_e, use_gelu=self.use_gelu)
            d_e = self.down_exps.forward_expert(exp_id, down_in_e.to(x_e.dtype)).float() * row_scale_e

            w_e = topk_weights[tok_idx, slot_idx].unsqueeze(-1)
            out_2d.index_add_(0, tok_idx, d_e * w_e)

        if self.shared_gate_up is not None and self.shared_down is not None:
            if self.shared_gate_up.is_single_contiguous:
                sh_gu = self.shared_gate_up.forward_single_concat(x_2d)
                sh_in, sh_row_scale = exec_fused_act_mul_rowscale(sh_gu, None, use_gelu=self.use_gelu)
            else:
                sh_g, sh_u = self.shared_gate_up(x_2d)
                sh_in, sh_row_scale = exec_fused_act_mul_rowscale(sh_g, sh_u, use_gelu=self.use_gelu)
            sh_out = self.shared_down(sh_in.to(x_2d.dtype)).float() * sh_row_scale
            if self.shared_inp_gate is not None:
                sh_scale = torch.sigmoid(F.linear(x_2d.float(), self.shared_inp_gate))
                sh_out = sh_out * sh_scale
            out_2d = out_2d + sh_out

        return out_2d.reshape(orig_shape)


def _resolve_block_norm_keys(gguf_file: GGUFFile, layer_idx: int) -> Tuple[Optional[str], str, Optional[str]]:
    prefix = f"blk.{layer_idx}"
    ffn_norm_key = f"{prefix}.ffn_norm.weight"
    post_attn_key = (
        f"{prefix}.post_attention_norm.weight"
        if f"{prefix}.post_attention_norm.weight" in gguf_file.tensors
        else f"{prefix}.attn_post_norm.weight"
    )
    if ffn_norm_key in gguf_file.tensors:
        attn_post = post_attn_key if post_attn_key in gguf_file.tensors else None
        eff_ffn_norm = ffn_norm_key
    else:
        attn_post = None
        eff_ffn_norm = post_attn_key

    post_ffn_cand = (
        f"{prefix}.post_ffw_norm.weight"
        if f"{prefix}.post_ffw_norm.weight" in gguf_file.tensors
        else f"{prefix}.ffn_post_norm.weight"
    )
    ffn_post = post_ffn_cand if post_ffn_cand in gguf_file.tensors else None
    return attn_post, eff_ffn_norm, ffn_post


class GGUFTransformerBlock(nn.Module):
    def __init__(
        self,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        rope: RotaryEmbeddingCache,
        device: torch.device,
    ) -> None:
        super().__init__()
        self.device = device
        prefix = f"blk.{layer_idx}"

        self.attn_norm = GGUFRMSNorm(
            load_float_tensor(gguf_file, f"{prefix}.attn_norm.weight", device=device),
            config.rms_norm_eps,
        )

        if f"{prefix}.ssm_out.weight" in gguf_file.tensors:
            self.is_recurrent = True
            self.attn: nn.Module = GGUFGatedDeltaNet.from_gguf(gguf_file, config, layer_idx, device=device)
        else:
            self.is_recurrent = False
            self.attn = GGUFAttention.from_gguf(gguf_file, config, layer_idx, rope, device=device)

        attn_post_key, eff_ffn_key, ffn_post_key = _resolve_block_norm_keys(gguf_file, layer_idx)
        self.attn_post_norm = (
            GGUFRMSNorm(load_float_tensor(gguf_file, attn_post_key, device=device), config.rms_norm_eps)
            if attn_post_key is not None
            else None
        )
        self.ffn_norm = GGUFRMSNorm(
            load_float_tensor(gguf_file, eff_ffn_key, device=device),
            config.rms_norm_eps,
        )

        if f"{prefix}.ffn_gate_exps.weight" in gguf_file.tensors:
            self.mlp: nn.Module = GGUFSparseMoEMLP(gguf_file, config, layer_idx, device=device)
        else:
            self.mlp = GGUFDenseMLP.from_gguf(gguf_file, config, layer_idx, device=device)

        self.ffn_post_norm = (
            GGUFRMSNorm(load_float_tensor(gguf_file, ffn_post_key, device=device), config.rms_norm_eps)
            if ffn_post_key is not None
            else None
        )

    def forward(
        self,
        x: torch.Tensor,
        kv_cache: Optional[LayerKVCache],
        start_pos: int,
        rec_state: Optional[LayerRecurrentState] = None,
        compute_dtype: torch.dtype = torch.float16,
        pos_tensor: Optional[torch.Tensor] = None,
        kv_pos: Optional[int] = None,
    ) -> torch.Tensor:
        if x.device != self.device:
            x = x.to(self.device, non_blocking=True)

        with cuda_device_scope(self.device):
            normed_attn = self.attn_norm(x, out_dtype=compute_dtype)
            if self.is_recurrent:
                assert rec_state is not None
                attn_delta = self.attn(normed_attn, rec_state).float()
            else:
                assert kv_cache is not None
                attn_delta = self.attn(
                    normed_attn, kv_cache, start_pos, pos_tensor=pos_tensor, kv_pos=kv_pos
                ).float()

            if self.attn_post_norm is not None:
                attn_delta = self.attn_post_norm(attn_delta, out_dtype=torch.float32)
            h, normed_ffn = self.ffn_norm.forward_with_residual(x, attn_delta, out_dtype=compute_dtype)

            ffn_delta = self.mlp(normed_ffn).float()
            if self.ffn_post_norm is not None:
                ffn_delta = self.ffn_post_norm(ffn_delta, out_dtype=torch.float32)
            return h + ffn_delta


class TPTransformerBlock(nn.Module):
    """
    2-GPU (or N-GPU) Tensor-Parallel Transformer Block:
    Enqueues Attention / Gated-DeltaNet and SwiGLU MLP shards across all TP GPUs asynchronously
    so BOTH Tesla T4 GDDR6 memory buses (`2 x 320 = 640 GB/s`) stream weights simultaneously!
    """

    def __init__(
        self,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        rope: RotaryEmbeddingCache,
        tp_devices: Sequence[torch.device],
    ) -> None:
        super().__init__()
        self.tp_devices = list(tp_devices)
        self.primary_device = self.tp_devices[0]
        prefix = f"blk.{layer_idx}"

        attn_nw_cpu = load_float_tensor(gguf_file, f"{prefix}.attn_norm.weight", device=torch.device("cpu"))
        self.attn_norm = GGUFRMSNorm(attn_nw_cpu.to(self.primary_device), config.rms_norm_eps)

        if f"{prefix}.ssm_out.weight" in gguf_file.tensors:
            self.is_recurrent = True
            self.attn_shards = nn.ModuleList(
                GGUFGatedDeltaNet.from_gguf_tp_shards(gguf_file, config, layer_idx, self.tp_devices)
            )
        else:
            self.is_recurrent = False
            self.attn_shards = nn.ModuleList(
                GGUFAttention.from_gguf_tp_shards(gguf_file, config, layer_idx, rope, self.tp_devices)
            )

        attn_post_key, eff_ffn_key, ffn_post_key = _resolve_block_norm_keys(gguf_file, layer_idx)
        self.attn_post_norm = (
            GGUFRMSNorm(
                load_float_tensor(gguf_file, attn_post_key, device=self.primary_device),
                config.rms_norm_eps,
            )
            if attn_post_key is not None
            else None
        )
        self.ffn_norm = GGUFRMSNorm(
            load_float_tensor(gguf_file, eff_ffn_key, device=self.primary_device),
            config.rms_norm_eps,
        )
        self.mlp_shards = nn.ModuleList(
            GGUFDenseMLP.from_gguf_tp_shards(gguf_file, config, layer_idx, self.tp_devices)
        )
        self.ffn_post_norm = (
            GGUFRMSNorm(
                load_float_tensor(gguf_file, ffn_post_key, device=self.primary_device),
                config.rms_norm_eps,
            )
            if ffn_post_key is not None
            else None
        )

    def forward_tp(
        self,
        x: torch.Tensor,
        tp_kv_caches: Optional[List[LayerKVCache]],
        start_pos: int,
        tp_rec_states: Optional[List[LayerRecurrentState]] = None,
        compute_dtype: torch.dtype = torch.float16,
        kv_pos: Optional[int] = None,
    ) -> torch.Tensor:
        dev0 = self.primary_device
        if x.device != dev0:
            x = x.to(dev0, non_blocking=True)

        with cuda_device_scope(dev0):
            normed_0 = self.attn_norm(x, out_dtype=compute_dtype)

        # Broadcast tiny FP16 activation (`~10 KB` at S=1) to peer GPUs, then enqueue ALL shards asynchronously!
        normed_inputs = [normed_0] + [normed_0.to(dev, non_blocking=True) for dev in self.tp_devices[1:]]

        attn_outs: List[torch.Tensor] = []
        for r, dev in enumerate(self.tp_devices):
            with cuda_device_scope(dev):
                if self.is_recurrent:
                    assert tp_rec_states is not None
                    attn_outs.append(self.attn_shards[r](normed_inputs[r], tp_rec_states[r]))
                else:
                    assert tp_kv_caches is not None
                    attn_outs.append(self.attn_shards[r](normed_inputs[r], tp_kv_caches[r], start_pos, kv_pos=kv_pos))

        with cuda_device_scope(dev0):
            attn_delta = attn_outs[0].float()
            for part in attn_outs[1:]:
                attn_delta = attn_delta + part.to(dev0, non_blocking=True).float()
            if self.attn_post_norm is not None:
                attn_delta = self.attn_post_norm(attn_delta, out_dtype=torch.float32)
            h, normed_ffn_0 = self.ffn_norm.forward_with_residual(x, attn_delta, out_dtype=compute_dtype)

        ffn_inputs = [normed_ffn_0] + [normed_ffn_0.to(dev, non_blocking=True) for dev in self.tp_devices[1:]]
        mlp_outs: List[torch.Tensor] = []
        for r, dev in enumerate(self.tp_devices):
            with cuda_device_scope(dev):
                mlp_outs.append(self.mlp_shards[r](ffn_inputs[r]))

        with cuda_device_scope(dev0):
            ffn_delta = mlp_outs[0].float()
            for part in mlp_outs[1:]:
                ffn_delta = ffn_delta + part.to(dev0, non_blocking=True).float()
            if self.ffn_post_norm is not None:
                ffn_delta = self.ffn_post_norm(ffn_delta, out_dtype=torch.float32)
            return h + ffn_delta


def _has_complete_mtp_block(gguf_file: GGUFFile, config: GGUFModelConfig) -> bool:
    """Return `True` iff GGUF contains a complete `blk.{n_layers}.nextn.*` MTP draft block."""
    if config.nextn_predict_layers < 1:
        return False
    p = f"blk.{config.n_layers}"
    t = gguf_file.tensors
    has_nextn = (
        f"{p}.nextn.eh_proj.weight" in t
        and f"{p}.nextn.enorm.weight" in t
        and f"{p}.nextn.hnorm.weight" in t
    )
    has_attn = f"{p}.attn_norm.weight" in t and (
        f"{p}.attn_qkv.weight" in t or f"{p}.attn_q.weight" in t
    )
    has_ffn = f"{p}.ffn_gate.weight" in t or f"{p}.ffn_gate_exps.weight" in t
    return bool(has_nextn and has_attn and has_ffn)


def _can_tp_block(gguf_file: GGUFFile, config: GGUFModelConfig, lid: int, tp_world: int) -> bool:
    p = f"blk.{lid}"
    if f"{p}.ffn_gate_exps.weight" in gguf_file.tensors:
        return False
    inter = int(gguf_file.tensors[f"{p}.ffn_gate.weight"].shape[0])
    if inter % tp_world != 0 or (inter // tp_world) % 32 != 0:
        return False
    if f"{p}.ssm_out.weight" in gguf_file.tensors:
        d_inner = int(gguf_file.tensors[f"{p}.attn_gate.weight"].shape[0])
        num_v_heads = int(gguf_file.tensors[f"{p}.ssm_beta.weight"].shape[0])
        conv_dim = int(gguf_file.tensors[f"{p}.attn_qkv.weight"].shape[0])
        key_dim = (conv_dim - d_inner) // 2
        head_v_dim = d_inner // num_v_heads
        head_k_dim = config.ssm_state_size if config.ssm_state_size > 0 else head_v_dim
        num_k_heads = key_dim // head_k_dim
        if num_k_heads % tp_world != 0 or num_v_heads % num_k_heads != 0 or head_v_dim % 32 != 0:
            return False
    return True


def _can_use_tensor_parallel(
    gguf_file: GGUFFile,
    config: GGUFModelConfig,
    tp_world: int,
) -> bool:
    """Verify exact group-aligned divisibility across all transformer blocks before enabling TP."""
    if tp_world < 2:
        return False
    if config.n_heads % tp_world != 0 or config.n_kv_heads % tp_world != 0:
        return False
    if ((config.n_heads // tp_world) * config.head_dim) % 32 != 0:
        return False

    for lid in range(config.n_layers):
        if not _can_tp_block(gguf_file, config, lid, tp_world):
            return False
    return True


class GGUFMTPDraftHead(nn.Module):
    """
    Native embedded GGUF Multi-Token Prediction (`blk.{n_layers}.nextn.*` / `llama.cpp` `graph_mtp`) head:
      `[enorm(embed(tok)), hnorm(h_prev)] -> eh_proj -> Full-Attention Decoder Block -> shared_head_norm -> lm_head`
    Supports both single-GPU/`PP` (`GGUFTransformerBlock`, dense or MoE) and `TP=2` (`TPTransformerBlock`)!
    """

    def __init__(
        self,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        layer_idx: int,
        rope: RotaryEmbeddingCache,
        device: torch.device,
        tp_devices: Optional[Sequence[torch.device]] = None,
        compute_dtype: torch.dtype = torch.float16,
    ) -> None:
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.compute_dtype = compute_dtype
        p = f"blk.{layer_idx}"

        tp_list = list(tp_devices) if tp_devices and len(tp_devices) > 1 and _can_tp_block(gguf_file, config, layer_idx, len(tp_devices)) else []
        self.is_tensor_parallel = bool(tp_list)
        self.tp_devices = tp_list
        self.device = tp_list[0] if tp_list else device

        self.enorm = GGUFRMSNorm(
            load_float_tensor(gguf_file, f"{p}.nextn.enorm.weight", device=self.device),
            config.rms_norm_eps,
        )
        self.hnorm = GGUFRMSNorm(
            load_float_tensor(gguf_file, f"{p}.nextn.hnorm.weight", device=self.device),
            config.rms_norm_eps,
        )

        tp_w = len(self.tp_devices)
        eh_rows = int(gguf_file.tensors[f"{p}.nextn.eh_proj.weight"].shape[0])
        stage_eh = self.is_tensor_parallel and eh_rows % tp_w == 0 and (eh_rows // tp_w) % 32 == 0
        eh_full = GGUFQuantLinear.from_gguf(
            gguf_file, f"{p}.nextn.eh_proj.weight",
            device=torch.device("cpu") if stage_eh else self.device,
        )
        if self.is_tensor_parallel and eh_full.out_features % tp_w == 0 and (eh_full.out_features // tp_w) % 32 == 0:
            chunk = eh_full.out_features // tp_w
            self.eh_proj_shards = nn.ModuleList(
                [eh_full.slice_output_rows(slice(r * chunk, (r + 1) * chunk), self.tp_devices[r]) for r in range(tp_w)]
            )
            self.eh_proj: Optional[GGUFQuantLinear] = None
            del eh_full
        else:
            self.eh_proj_shards = nn.ModuleList()
            self.eh_proj = eh_full

        self.own_embed: Optional[GGUFQuantLinear] = (
            GGUFQuantLinear.from_gguf(gguf_file, f"{p}.nextn.embed_tokens.weight", device=self.device)
            if f"{p}.nextn.embed_tokens.weight" in gguf_file.tensors
            else None
        )

        if self.is_tensor_parallel:
            self.block: nn.Module = TPTransformerBlock(
                gguf_file=gguf_file,
                config=config,
                layer_idx=layer_idx,
                rope=rope,
                tp_devices=self.tp_devices,
            )
        else:
            self.block = GGUFTransformerBlock(
                gguf_file=gguf_file,
                config=config,
                layer_idx=layer_idx,
                rope=rope,
                device=self.device,
            )

        self.shared_head_norm: Optional[GGUFRMSNorm] = (
            GGUFRMSNorm(
                load_float_tensor(gguf_file, f"{p}.nextn.shared_head_norm.weight", device=self.device),
                config.rms_norm_eps,
            )
            if f"{p}.nextn.shared_head_norm.weight" in gguf_file.tensors
            else None
        )
        self.own_lm_head: Optional[GGUFQuantLinear] = (
            GGUFQuantLinear.from_gguf(gguf_file, f"{p}.nextn.shared_head_head.weight", device=self.device)
            if f"{p}.nextn.shared_head_head.weight" in gguf_file.tensors
            else None
        )

    def _project_eh(self, cat_eh: torch.Tensor) -> torch.Tensor:
        if len(self.eh_proj_shards) > 0:
            eh_inputs = [cat_eh] + [cat_eh.to(dev, non_blocking=True) for dev in self.tp_devices[1:]]
            parts: List[torch.Tensor] = []
            for r, dev in enumerate(self.tp_devices):
                with cuda_device_scope(dev):
                    parts.append(self.eh_proj_shards[r](eh_inputs[r]))
            with cuda_device_scope(self.device):
                return torch.cat([pt.to(self.device, non_blocking=True) for pt in parts], dim=-1).float()
        assert self.eh_proj is not None
        with cuda_device_scope(self.device):
            return self.eh_proj(cat_eh).float()

    def _forward_block(
        self,
        token_ids: torch.Tensor,
        h_prev: torch.Tensor,
        kv_manager: EngineKVCacheManager,
        start_pos: int,
        trunk_token_embd: GGUFQuantLinear,
    ) -> torch.Tensor:
        if self.own_embed is not None:
            with cuda_device_scope(self.device):
                emb = self.own_embed.embed_lookup(token_ids.to(self.device), dtype=torch.float32)
        else:
            emb_dev = trunk_token_embd.scales.device
            with cuda_device_scope(emb_dev):
                emb = trunk_token_embd.embed_lookup(token_ids.to(emb_dev), dtype=torch.float32)
            if emb.device != self.device:
                emb = emb.to(self.device, non_blocking=True)

        with cuda_device_scope(self.device):
            if self.config.embed_scale != 1.0:
                emb = emb * self.config.embed_scale
            e_norm = self.enorm(emb, out_dtype=self.compute_dtype)
            h_norm = self.hnorm(h_prev.to(self.device, non_blocking=True), out_dtype=self.compute_dtype)
            cat_eh = torch.cat([e_norm, h_norm], dim=-1)

        x = self._project_eh(cat_eh)
        eff_kv_pos = start_pos - kv_manager.mtp_cache_start

        if self.is_tensor_parallel:
            return self.block.forward_tp(  # type: ignore[operator]
                x,
                kv_manager.mtp_tp_kv_caches,
                start_pos,
                None,
                self.compute_dtype,
                kv_pos=eff_kv_pos,
            )
        assert kv_manager.mtp_kv_cache is not None
        return self.block(
            x,
            kv_manager.mtp_kv_cache,
            start_pos,
            None,
            self.compute_dtype,
            kv_pos=eff_kv_pos,
        )

    @torch.inference_mode()
    def prefill_catch_up(
        self,
        prompt_ids: torch.Tensor,
        trunk_h_norm: torch.Tensor,
        kv_manager: EngineKVCacheManager,
        start_pos: int,
        trunk_token_embd: GGUFQuantLinear,
        prev_h: Optional[torch.Tensor] = None,
    ) -> None:
        """
        Populate MTP full-attention KV cache over `prompt_ids` `[1, S]` at `[start_pos .. start_pos + S)`
        using `[prev_h (or 0), trunk_h_norm[:-1]]`, skipping `lm_head` so prefill overhead is `< 1.5%`!
        When `kv_manager.mtp_kv_window` (`W`, e.g. `4096`) is set and `S > W`, automatically rebases
        `mtp_cache_start` and forwards ONLY the trailing `W` tokens (`O(W)` time & VRAM even on 256K prompts!).
        """
        seq_len = int(prompt_ids.shape[1])
        if seq_len <= 0:
            return
        win = kv_manager.mtp_kv_window
        with cuda_device_scope(self.device):
            if win is not None and seq_len > win:
                skip = seq_len - win
                new_abs_start = start_pos + skip
                kv_manager.reset_mtp_cache(new_abs_start)
                prompt_ids = prompt_ids[:, skip:]
                h_in = trunk_h_norm[:, skip - 1 : seq_len - 1, :].to(self.device, non_blocking=True)
                start_pos = new_abs_start
                seq_len = win
            else:
                th = trunk_h_norm.to(self.device, non_blocking=True)
                if prev_h is not None:
                    first_h = prev_h.to(device=self.device, dtype=th.dtype, non_blocking=True).view(1, 1, -1)
                else:
                    first_h = torch.zeros((1, 1, self.config.hidden_size), device=self.device, dtype=th.dtype)
                h_in = torch.cat([first_h, th[:, :-1, :]], dim=1) if seq_len > 1 else first_h

        prefill_chunk = min(2048, win) if (win is not None and win > 0) else 2048
        for off in range(0, seq_len, prefill_chunk):
            end = min(seq_len, off + prefill_chunk)
            _ = self._forward_block(
                prompt_ids[:, off:end],
                h_in[:, off:end, :],
                kv_manager,
                start_pos + off,
                trunk_token_embd,
            )
        kv_manager.mtp_exact_kv_len = start_pos + seq_len

    @torch.inference_mode()
    def forward_step(
        self,
        token_ids: torch.Tensor,
        h_prev: torch.Tensor,
        kv_manager: EngineKVCacheManager,
        start_pos: int,
        trunk_model: "GGUFTransformerModel",
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Fused catch-up + draft step: runs MTP block over `S >= 1` rows (`[catchup_rows..., seed_row]`),
        applies `shared_head_norm` + `lm_head` ONLY on the final row, and returns `(logits_f32[1,1,V], h_nextn[1,1,H])`!
        """
        block_out = self._forward_block(
            token_ids,
            h_prev,
            kv_manager,
            start_pos,
            trunk_model.token_embd,
        )
        last_raw = block_out[:, -1:, :]
        if self.shared_head_norm is not None:
            with cuda_device_scope(self.device):
                h_nextn = self.shared_head_norm(last_raw.to(self.device), out_dtype=self.compute_dtype)
        else:
            norm_dev = trunk_model.first_device if trunk_model.is_tensor_parallel else trunk_model.last_device
            with cuda_device_scope(norm_dev):
                h_nextn = trunk_model.output_norm(last_raw.to(norm_dev), out_dtype=self.compute_dtype)

        if self.own_lm_head is not None:
            with cuda_device_scope(self.device):
                logits_f32 = self.own_lm_head(h_nextn.to(self.device)).float()
                if self.config.final_logit_softcapping > 0.0:
                    cap = self.config.final_logit_softcapping
                    logits_f32 = torch.tanh(logits_f32 / cap) * cap
            return logits_f32, h_nextn
        return trunk_model.project_normed_hidden(h_nextn), h_nextn


def resolve_device_list(devices: Optional[Sequence[str]] = None) -> List[torch.device]:
    if devices is not None and len(devices) > 0:
        return [torch.device(d) for d in devices]
    if torch.cuda.is_available():
        return [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
    return [torch.device("cpu")]


def compute_layer_device_map(
    n_layers: int,
    devices: Optional[Sequence[str]] = None,
    gguf_file: Optional[GGUFFile] = None,
    reserve_mtp_on_last: bool = False,
) -> List[torch.device]:
    """
    Distribute transformer blocks across all available GPUs (e.g., `cuda:0` and `cuda:1` on Kaggle T4 x2)
    accounting for exact per-block GGUF tensor sizes + `token_embd.weight` (GPU 0) + `output.weight` / MTP (last GPU),
    so free VRAM headroom is balanced evenly across GPUs!
    """
    dev_list = resolve_device_list(devices)
    if len(dev_list) == 1 or n_layers <= 1:
        return [dev_list[0]] * n_layers

    n_devs = len(dev_list)
    free_bytes: List[float] = []
    for d in dev_list:
        if d.type == "cuda":
            free_b, _ = torch.cuda.mem_get_info(d)
            free_bytes.append(max(float(free_b), 1.0))
        else:
            free_bytes.append(16.0 * (1024**3))

    if gguf_file is not None:
        soa_factor = 1.12
        embd_b = float(gguf_file.tensors["token_embd.weight"].n_bytes) * soa_factor if "token_embd.weight" in gguf_file.tensors else 0.0
        head_b = float(gguf_file.tensors["output.weight"].n_bytes) * soa_factor if "output.weight" in gguf_file.tensors else 0.0
        mtp_b = 0.0

        layer_bytes = [0.0] * n_layers
        for t_name, t_info in gguf_file.tensors.items():
            if t_name.startswith("blk."):
                parts = t_name.split(".", 2)
                if len(parts) >= 2 and parts[1].isdigit():
                    lid = int(parts[1])
                    if 0 <= lid < n_layers:
                        layer_bytes[lid] += float(t_info.n_bytes) * soa_factor
                    elif reserve_mtp_on_last and lid == n_layers:
                        mtp_b += float(t_info.n_bytes) * soa_factor

        avail = list(free_bytes)
        avail[0] = max(1.0, avail[0] - embd_b)
        avail[-1] = max(1.0, avail[-1] - head_b - mtp_b)

        total_avail = sum(avail)
        total_layer_b = max(sum(layer_bytes), 1.0)
        targets = [total_layer_b * (a / total_avail) for a in avail]

        mapping: List[torch.device] = []
        dev_idx = 0
        acc_b = 0.0
        for lid in range(n_layers):
            remaining_layers = n_layers - lid
            remaining_devs = n_devs - 1 - dev_idx
            if (
                dev_idx < n_devs - 1
                and len(mapping) > 0
                and (acc_b + 0.5 * layer_bytes[lid] > targets[dev_idx])
                and remaining_layers >= remaining_devs
            ):
                dev_idx += 1
                acc_b = 0.0
            if remaining_layers == remaining_devs and dev_idx < n_devs - 1:
                dev_idx += 1
                acc_b = 0.0
            mapping.append(dev_list[dev_idx])
            acc_b += layer_bytes[lid]
        return mapping

    free_bytes[0] *= 0.94
    total_w = sum(free_bytes)
    counts = [max(1, round(n_layers * (w / total_w))) for w in free_bytes]
    while sum(counts) > n_layers:
        counts[counts.index(max(counts))] -= 1
    while sum(counts) < n_layers:
        counts[counts.index(min(counts))] += 1

    mapping = []
    for dev, cnt in zip(dev_list, counts):
        mapping.extend([dev] * cnt)
    return mapping


class GGUFTransformerModel(nn.Module):
    """
    End-to-end GGUF Causal LM supporting both:
    - Multi-GPU Tensor Parallelism (`TP=2`, streaming weights from all GPUs in parallel!), and
    - Pipeline Parallelism (`PP`) + Per-Stage `torch.cuda.CUDAGraph` single-token decode replay!
    """

    def __init__(
        self,
        gguf_file: GGUFFile,
        config: GGUFModelConfig,
        devices: Optional[Sequence[str]] = None,
        compute_dtype: torch.dtype = torch.float16,
        parallel_mode: Literal["auto", "tp", "pp"] = "auto",
        enable_cuda_graph: bool = True,
        load_mtp: bool = True,
    ) -> None:
        super().__init__()
        self.config = config
        self.compute_dtype = compute_dtype
        dev_list = resolve_device_list(devices)

        want_tp = False
        if parallel_mode == "tp" and len(dev_list) >= 2:
            want_tp = _can_use_tensor_parallel(gguf_file, config, len(dev_list))
        elif parallel_mode == "auto" and len(dev_list) >= 2 and all(d.type == "cuda" for d in dev_list) and not config.is_moe:
            want_tp = _can_use_tensor_parallel(gguf_file, config, len(dev_list))

        self.is_tensor_parallel = want_tp
        self.tp_devices: List[torch.device] = list(dev_list) if want_tp else []
        self.enable_cuda_graph = bool(enable_cuda_graph and not self.is_tensor_parallel and not config.is_moe)
        has_mtp_in_gguf = bool(load_mtp and _has_complete_mtp_block(gguf_file, config))

        self.rope = RotaryEmbeddingCache(
            rope_dim=config.rope_dim,
            rope_theta=config.rope_theta,
            neox_style=config.rope_neox,
            rope_scaling_type=config.rope_scaling_type,
            rope_scaling_factor=config.rope_scaling_factor,
            rope_original_context_len=config.rope_original_context_len,
            rope_yarn_beta_fast=config.rope_yarn_beta_fast,
            rope_yarn_beta_slow=config.rope_yarn_beta_slow,
            rope_yarn_ext_factor=config.rope_yarn_ext_factor,
            rope_yarn_attn_factor=config.rope_yarn_attn_factor,
            rope_yarn_log_mul=config.rope_yarn_log_mul,
            max_context_len=config.max_context_len,
        )

        load_start = time.monotonic()
        def report_load(message: str) -> None:
            print(f"[PyGGUF] Loading {message} ({time.monotonic() - load_start:.1f}s elapsed)", flush=True)

        if self.is_tensor_parallel:
            self.first_device = self.tp_devices[0]
            self.last_device = self.tp_devices[0]
            self.layer_devices = [self.first_device] * config.n_layers

            report_load("token embeddings")
            self.token_embd = GGUFQuantLinear.from_gguf(gguf_file, "token_embd.weight", device=self.first_device)
            def load_tp_block(i: int) -> TPTransformerBlock:
                report_load(f"layer {i + 1}/{config.n_layers} (TP)")
                return TPTransformerBlock(
                        gguf_file=gguf_file,
                        config=config,
                        layer_idx=i,
                        rope=self.rope,
                        tp_devices=self.tp_devices,
                    )
            self.layers = nn.ModuleList([load_tp_block(i) for i in range(config.n_layers)])
            self.recurrent_layer_flags: List[bool] = [bool(blk.is_recurrent) for blk in self.layers]
            self.has_recurrent_layers: bool = any(self.recurrent_layer_flags)

            self.output_norm = GGUFRMSNorm(
                load_float_tensor(gguf_file, "output_norm.weight", device=self.first_device),
                config.rms_norm_eps,
            )

            # Shard vocab projection across all TP GPUs so 248K-vocab `lm_head` also runs at 2x bandwidth!
            tp_w = len(self.tp_devices)
            report_load("output head")
            if "output.weight" in gguf_file.tensors:
                head_full = GGUFQuantLinear.from_gguf(gguf_file, "output.weight", device=torch.device("cpu"))
                self.tied_embeddings = False
            else:
                head_full = self.token_embd
                self.tied_embeddings = True

            vocab = head_full.out_features
            v_chunk = (vocab + tp_w - 1) // tp_w
            self.lm_head_shards = nn.ModuleList(
                [
                    head_full.slice_output_rows(slice(r * v_chunk, min(vocab, (r + 1) * v_chunk)), self.tp_devices[r])
                    for r in range(tp_w)
                ]
            )
            self.lm_head: Optional[GGUFQuantLinear] = self.lm_head_shards[0]
            if not self.tied_embeddings:
                del head_full
        else:
            self.layer_devices = compute_layer_device_map(
                config.n_layers,
                devices,
                gguf_file=gguf_file,
                reserve_mtp_on_last=has_mtp_in_gguf,
            )
            self.first_device = self.layer_devices[0]
            self.last_device = self.layer_devices[-1]

            report_load("token embeddings")
            self.token_embd = GGUFQuantLinear.from_gguf(gguf_file, "token_embd.weight", device=self.first_device)
            def load_pp_block(i: int) -> GGUFTransformerBlock:
                report_load(f"layer {i + 1}/{config.n_layers} ({self.layer_devices[i]})")
                return GGUFTransformerBlock(
                        gguf_file=gguf_file,
                        config=config,
                        layer_idx=i,
                        rope=self.rope,
                        device=self.layer_devices[i],
                    )
            self.layers = nn.ModuleList([load_pp_block(i) for i in range(config.n_layers)])
            self.recurrent_layer_flags = [bool(blk.is_recurrent) for blk in self.layers]
            self.has_recurrent_layers = any(self.recurrent_layer_flags)

            self.output_norm = GGUFRMSNorm(
                load_float_tensor(gguf_file, "output_norm.weight", device=self.last_device),
                config.rms_norm_eps,
            )

            self.lm_head_shards = nn.ModuleList()
            report_load("output head")
            if "output.weight" in gguf_file.tensors:
                self.lm_head = GGUFQuantLinear.from_gguf(gguf_file, "output.weight", device=self.last_device)
                self.tied_embeddings = False
            else:
                self.lm_head = None
                self.tied_embeddings = True

        if has_mtp_in_gguf:
            report_load("MTP draft head")
        self.mtp_head: Optional[GGUFMTPDraftHead] = (
            GGUFMTPDraftHead(
                gguf_file=gguf_file,
                config=config,
                layer_idx=config.n_layers,
                rope=self.rope,
                device=self.last_device,
                tp_devices=self.tp_devices if self.is_tensor_parallel else None,
                compute_dtype=compute_dtype,
            )
            if has_mtp_in_gguf
            else None
        )

        # Per-GPU-stage CUDA Graph state for M=1 decode (on PP / 1-GPU CUDA runs)
        self._decode_graphs_ready = False
        self._decode_graph_caps: Tuple[int, ...] = ()
        self._stage_ranges: List[Tuple[torch.device, int, int]] = []
        self._stage_graphs: List[Any] = []
        self._stage_in_bufs: List[torch.Tensor] = []
        self._stage_out_bufs: List[torch.Tensor] = []
        self._stage_pos_bufs: Dict[torch.device, torch.Tensor] = {}
        # Per-S CUDA Graph state for S=2..8 verify steps (speculative decode). Buffers are created
        # once per S and shared by every capture; they must survive M=1 re-captures, so they are
        # never recreated — hence dedicated pos/token bufs (the M=1 capture recreates its own).
        self._verify_graphs_failed = False
        self._verify_token_bufs: Dict[int, torch.Tensor] = {}
        self._verify_pos_bufs: Dict[torch.device, torch.Tensor] = {}
        self._verify_stage_data: Dict[
            int, Tuple[List[Tuple[torch.device, int, int]], List[Any], List[torch.Tensor], List[torch.Tensor]]
        ] = {}
        self._verify_head_data: Dict[
            int, Tuple[Any, torch.Tensor, torch.Tensor, torch.Tensor]
        ] = {}
        self._verify_graph_caps: Dict[int, Tuple[int, ...]] = {}
        self._verify_graphs_ready: Dict[int, bool] = {}
        self._graph_token_buf: Optional[torch.Tensor] = None
        self._graph_logits_buf: Optional[torch.Tensor] = None
        self._graph_normed_buf: Optional[torch.Tensor] = None
        self._head_graph: Any = None
        self._head_in_buf: Optional[torch.Tensor] = None

    def _compute_pp_stages(self) -> List[Tuple[torch.device, int, int]]:
        stages: List[Tuple[torch.device, int, int]] = []
        if not self.layer_devices:
            return stages
        cur_dev = self.layer_devices[0]
        s_start = 0
        for idx in range(1, len(self.layer_devices)):
            if self.layer_devices[idx] != cur_dev:
                stages.append((cur_dev, s_start, idx))
                cur_dev = self.layer_devices[idx]
                s_start = idx
        stages.append((cur_dev, s_start, len(self.layer_devices)))
        return stages

    def _capture_decode_graphs(self, kv_manager: EngineKVCacheManager) -> None:
        """
        Capture per-GPU `torch.cuda.CUDAGraph`s for `M=1` decode.
        Each stage graph runs strictly on a single CUDA device so non-P2P Dual-T4 hosts never trigger
        forbidden cross-device stream syncs during graph capture!
        """
        stages = self._compute_pp_stages()
        self._stage_ranges = stages
        self._stage_graphs = []
        self._stage_in_bufs = []
        self._stage_out_bufs = []
        self._stage_pos_bufs = {}

        # Snapshot live recurrent & KV state so warmup/capture leaves model state 100% untouched!
        rec_snap = kv_manager.snapshot_recurrent_gpu() if kv_manager.has_recurrent else {}
        saved_lens = [lkv.seq_len for lkv in kv_manager._iter_all_kv_caches()]

        # Ensure RoPE & recurrent buffers exist on every stage GPU before capture
        unique_devs = {dev for dev, _, _ in stages} | {self.first_device, self.last_device}
        for dev in unique_devs:
            self.rope.ensure_capacity(dev, self.config.max_context_len)
            with cuda_device_scope(dev):
                self._stage_pos_bufs[dev] = torch.zeros((1,), device=dev, dtype=torch.int32)

        for idx, blk in enumerate(self.layers):
            if blk.is_recurrent:
                st = kv_manager.recurrent_states[idx]
                assert st is not None
                dn: GGUFGatedDeltaNet = blk.attn  # type: ignore[assignment]
                st.ensure_allocated(dn.conv_dim, dn.conv_kernel, dn.num_v_heads, dn.head_v_dim, dn.head_k_dim)
            else:
                lkv = kv_manager.layers[idx]
                assert lkv is not None
                n_h = blk.attn.n_heads  # type: ignore[attr-defined]
                with cuda_device_scope(lkv.device):
                    if lkv._part_out_scratch is None:
                        lkv._part_out_scratch = torch.empty((n_h, 32, lkv.head_dim), device=lkv.device, dtype=torch.float32)
                        lkv._part_lse_scratch = torch.empty((n_h, 32), device=lkv.device, dtype=torch.float32)
                    if lkv._graph_scratch_out is None:
                        hd = lkv.head_dim
                        lkv._graph_scratch_out = torch.empty((n_h * 8, 32, hd), device=lkv.device, dtype=torch.float32)
                        lkv._graph_scratch_lse = torch.empty((n_h * 8, 32), device=lkv.device, dtype=torch.float32)

        with cuda_device_scope(self.first_device):
            self._graph_token_buf = torch.zeros((1, 1), device=self.first_device, dtype=torch.long)

        for stage_idx, (dev, l0, l1) in enumerate(stages):
            pos_buf = self._stage_pos_bufs[dev]
            with cuda_device_scope(dev):
                in_buf = torch.zeros((1, 1, self.config.hidden_size), device=dev, dtype=torch.float32)

                def _run_stage(
                    _s_idx: int = stage_idx,
                    _l0: int = l0,
                    _l1: int = l1,
                    _in_b: torch.Tensor = in_buf,
                    _pos_b: torch.Tensor = pos_buf,
                ) -> torch.Tensor:
                    if _s_idx == 0:
                        assert self._graph_token_buf is not None
                        h_cur = self.token_embd.embed_lookup(self._graph_token_buf, dtype=torch.float32)
                        if self.config.embed_scale != 1.0:
                            h_cur = h_cur * self.config.embed_scale
                    else:
                        h_cur = _in_b
                    for b_i in range(_l0, _l1):
                        h_cur = self.layers[b_i](
                            h_cur,
                            kv_manager.layers[b_i],
                            0,
                            kv_manager.recurrent_states[b_i],
                            self.compute_dtype,
                            pos_tensor=_pos_b,
                        )
                    return h_cur

                # Warmup on side stream before capture
                s_warm = torch.cuda.Stream(device=dev)
                s_warm.wait_stream(torch.cuda.current_stream(dev))
                with torch.cuda.stream(s_warm):
                    _ = _run_stage()
                torch.cuda.current_stream(dev).wait_stream(s_warm)

                g = torch.cuda.CUDAGraph()
                with torch.cuda.graph(g, stream=s_warm):
                    out_buf = _run_stage()

                self._stage_in_bufs.append(in_buf)
                self._stage_out_bufs.append(out_buf)
                self._stage_graphs.append(g)

        head_dev = self.last_device if self.lm_head is not None else self.first_device
        with cuda_device_scope(head_dev):
            head_in = torch.zeros((1, 1, self.config.hidden_size), device=head_dev, dtype=torch.float32)

            def _run_head(_hin: torch.Tensor = head_in) -> Tuple[torch.Tensor, torch.Tensor]:
                if self.last_device == head_dev:
                    normed = self.output_norm(_hin, out_dtype=self.compute_dtype)
                else:
                    normed = _hin.to(self.compute_dtype)
                proj = self.lm_head(normed) if self.lm_head is not None else self.token_embd(normed)
                lf = proj.float()
                if self.config.final_logit_softcapping > 0.0:
                    cap = self.config.final_logit_softcapping
                    lf = torch.tanh(lf / cap) * cap
                return lf, normed

            sh_warm = torch.cuda.Stream(device=head_dev)
            sh_warm.wait_stream(torch.cuda.current_stream(head_dev))
            with torch.cuda.stream(sh_warm):
                _ = _run_head()
            torch.cuda.current_stream(head_dev).wait_stream(sh_warm)

            gh = torch.cuda.CUDAGraph()
            with torch.cuda.graph(gh, stream=sh_warm):
                logits_buf, normed_buf = _run_head()

            self._head_in_buf = head_in
            self._head_graph = gh
            self._graph_logits_buf = logits_buf
            self._graph_normed_buf = normed_buf

        # Restore live KV lengths & recurrent states untouched
        for lkv, slen in zip(kv_manager._iter_all_kv_caches(), saved_lens):
            lkv.rewind_to(slen)
        if kv_manager.has_recurrent:
            kv_manager.restore_recurrent_gpu(rec_snap, kv_manager.recurrent_seq_len)

        self._decode_graph_caps = tuple(lkv.capacity for lkv in kv_manager._iter_all_kv_caches())
        self._decode_graphs_ready = True

    @torch.inference_mode()
    def forward_decode_step(
        self,
        token_id: int,
        kv_manager: EngineKVCacheManager,
        start_pos: int,
        return_normed_hidden: bool = False,
    ) -> Any:
        """Fast single-token (`M=1`) decode using `torch.cuda.CUDAGraph` replay when supported."""
        if (
            not self.enable_cuda_graph
            or self.first_device.type != "cuda"
            or (self.last_device != self.first_device and self.lm_head is None)
        ):
            step_inp = torch.tensor([[token_id]], device=self.first_device, dtype=torch.long)
            return self.forward(
                step_inp,
                kv_manager,
                start_pos=start_pos,
                logits_to_keep=1,
                return_normed_hidden=return_normed_hidden,
            )

        # Ensure KV cache pages won't reallocate mid-step; re-capture only when page capacity grows (every 2048 toks)
        kv_manager.ensure_all_kv_capacity(start_pos + 1)
        cur_caps = tuple(lkv.capacity for lkv in kv_manager._iter_all_kv_caches())
        if not self._decode_graphs_ready or cur_caps != self._decode_graph_caps:
            try:
                self._capture_decode_graphs(kv_manager)
            except Exception:
                self.enable_cuda_graph = False
                step_inp = torch.tensor([[token_id]], device=self.first_device, dtype=torch.long)
                return self.forward(
                    step_inp,
                    kv_manager,
                    start_pos=start_pos,
                    logits_to_keep=1,
                    return_normed_hidden=return_normed_hidden,
                )

        assert self._graph_token_buf is not None and self._head_in_buf is not None and self._graph_logits_buf is not None
        with cuda_device_scope(self.first_device):
            self._graph_token_buf.fill_(token_id)
        for dev, pbuf in self._stage_pos_bufs.items():
            with cuda_device_scope(dev):
                pbuf.fill_(start_pos)

        for s_idx, (dev, _, _) in enumerate(self._stage_ranges):
            if s_idx > 0:
                self._stage_in_bufs[s_idx].copy_(self._stage_out_bufs[s_idx - 1])
            with cuda_device_scope(dev):
                self._stage_graphs[s_idx].replay()

        for lkv in kv_manager._iter_all_kv_caches():
            lkv.rewind_to(start_pos + 1)

        head_dev = self._head_in_buf.device
        self._head_in_buf.copy_(self._stage_out_bufs[-1])
        with cuda_device_scope(head_dev):
            self._head_graph.replay()
        if return_normed_hidden:
            assert self._graph_normed_buf is not None
            return self._graph_logits_buf, self._graph_normed_buf
        return self._graph_logits_buf

    def _verify_graph_eligibility(self, kv_manager: EngineKVCacheManager) -> Tuple[bool, str]:
        """
        Verify CUDA graphs (`S=2..8`) need every attention layer on the fused posptr Split-KV path:
        `int4`/`int8` KV, power-of-two head_dim, no softcap, no sliding-window ring (ring writes would
        bake python `start_pos` into the capture). Recurrent (GatedDeltaNet) layers are graph-safe.
        """
        if self.is_tensor_parallel or self.tp_devices:
            return False, "tensor-parallel"
        if getattr(self.config, "is_moe", False):
            return False, "moe"
        if not self.enable_cuda_graph:
            return False, "cuda-graph-disabled"
        if self.first_device.type != "cuda":
            return False, "non-cuda"
        if self.last_device != self.first_device and self.lm_head is None:
            return False, "sharded-head"
        if float(getattr(self.config, "attn_logit_softcapping", 0.0)) != 0.0:
            return False, "attn-softcap"
        hd = int(getattr(self.config, "head_dim", 0))
        if hd not in (32, 64, 128, 256):
            return False, "head-dim"
        for idx, blk in enumerate(self.layers):
            if getattr(blk, "is_recurrent", False):
                continue
            lkv = kv_manager.layers[idx]
            if lkv is None:
                return False, f"layer-{idx}-missing-kv"
            if lkv.kv_dtype not in ("int4", "int8"):
                return False, f"layer-{idx}-kv-dtype-{lkv.kv_dtype}"
            if lkv.sliding_window is not None:
                return False, f"layer-{idx}-sliding-window"
        return True, "ok"

    def _capture_verify_graphs(self, kv_manager: EngineKVCacheManager, s_len: int) -> None:
        """
        Capture per-GPU `torch.cuda.CUDAGraph`s for a whole `S=s_len` verify forward (all layers,
        logits head included). Mirrors `_capture_decode_graphs` but for multi-row speculative
        verification: positions flow through `pos_tensor` (`int32[1]` base position) everywhere,
        recurrent layers record their zero-recompute rollback buffers into graph-owned memory.
        """
        ok, reason = self._verify_graph_eligibility(kv_manager)
        if not ok:
            raise RuntimeError(f"verify graphs not eligible: {reason}")
        stages = self._compute_pp_stages()
        # Snapshot live recurrent & KV state so warmup/capture leaves model state 100% untouched!
        rec_snap = kv_manager.snapshot_recurrent_gpu() if kv_manager.has_recurrent else {}
        saved_lens = [lkv.seq_len for lkv in kv_manager._iter_all_kv_caches()]

        unique_devs = {dev for dev, _, _ in stages} | {self.first_device, self.last_device}
        for dev in unique_devs:
            self.rope.ensure_capacity(dev, self.config.max_context_len)
            with cuda_device_scope(dev):
                if dev not in self._verify_pos_bufs:
                    self._verify_pos_bufs[dev] = torch.zeros((1,), device=dev, dtype=torch.int32)

        for idx, blk in enumerate(self.layers):
            if blk.is_recurrent:
                st = kv_manager.recurrent_states[idx]
                assert st is not None
                dn: GGUFGatedDeltaNet = blk.attn  # type: ignore[assignment]
                st.ensure_allocated(dn.conv_dim, dn.conv_kernel, dn.num_v_heads, dn.head_v_dim, dn.head_k_dim)
            else:
                lkv = kv_manager.layers[idx]
                assert lkv is not None
                n_h = blk.attn.n_heads  # type: ignore[attr-defined]
                with cuda_device_scope(lkv.device):
                    if lkv._graph_scratch_out is None:
                        hd = lkv.head_dim
                        lkv._graph_scratch_out = torch.empty((n_h * 8, 32, hd), device=lkv.device, dtype=torch.float32)
                        lkv._graph_scratch_lse = torch.empty((n_h * 8, 32), device=lkv.device, dtype=torch.float32)

        # Arm recurrent layers so the capture records the SAVE path (rollback buffers land in the
        # graph-private pool and stay valid across every replay).
        for _key, st in kv_manager._iter_all_recurrent_states():
            st.verify_active = True
            st._verify_cache = None

        hidden = self.config.hidden_size
        with cuda_device_scope(self.first_device):
            if s_len not in self._verify_token_bufs:
                self._verify_token_bufs[s_len] = torch.zeros((1, s_len), device=self.first_device, dtype=torch.long)

        stage_graphs: List[Any] = []
        stage_ins: List[torch.Tensor] = []
        stage_outs: List[torch.Tensor] = []
        for stage_idx, (dev, l0, l1) in enumerate(stages):
            pos_buf = self._verify_pos_bufs[dev]
            with cuda_device_scope(dev):
                in_buf = torch.zeros((1, s_len, hidden), device=dev, dtype=torch.float32)

                def _run_stage(
                    _s_idx: int = stage_idx,
                    _l0: int = l0,
                    _l1: int = l1,
                    _in_b: torch.Tensor = in_buf,
                    _pos_b: torch.Tensor = pos_buf,
                    _sl: int = s_len,
                ) -> torch.Tensor:
                    if _s_idx == 0:
                        assert self._verify_token_bufs[_sl] is not None
                        h_cur = self.token_embd.embed_lookup(self._verify_token_bufs[_sl], dtype=torch.float32)
                        if self.config.embed_scale != 1.0:
                            h_cur = h_cur * self.config.embed_scale
                    else:
                        h_cur = _in_b
                    for b_i in range(_l0, _l1):
                        h_cur = self.layers[b_i](
                            h_cur,
                            kv_manager.layers[b_i],
                            0,
                            kv_manager.recurrent_states[b_i],
                            self.compute_dtype,
                            pos_tensor=_pos_b,
                        )
                    return h_cur

                # Warmup on side stream before capture
                s_warm = torch.cuda.Stream(device=dev)
                s_warm.wait_stream(torch.cuda.current_stream(dev))
                with torch.cuda.stream(s_warm):
                    _ = _run_stage()
                torch.cuda.current_stream(dev).wait_stream(s_warm)

                g = torch.cuda.CUDAGraph()
                with torch.cuda.graph(g, stream=s_warm):
                    out_buf = _run_stage()

                stage_graphs.append(g)
                stage_ins.append(in_buf)
                stage_outs.append(out_buf)

        head_dev = self.last_device if self.lm_head is not None else self.first_device
        with cuda_device_scope(head_dev):
            head_in = torch.zeros((1, s_len, hidden), device=head_dev, dtype=torch.float32)

            def _run_head(_hin: torch.Tensor = head_in) -> Tuple[torch.Tensor, torch.Tensor]:
                if self.last_device == head_dev:
                    normed = self.output_norm(_hin, out_dtype=self.compute_dtype)
                else:
                    normed = _hin.to(self.compute_dtype)
                proj = self.lm_head(normed) if self.lm_head is not None else self.token_embd(normed)
                lf = proj.float()
                if self.config.final_logit_softcapping > 0.0:
                    cap = self.config.final_logit_softcapping
                    lf = torch.tanh(lf / cap) * cap
                return lf, normed

            sh_warm = torch.cuda.Stream(device=head_dev)
            sh_warm.wait_stream(torch.cuda.current_stream(head_dev))
            with torch.cuda.stream(sh_warm):
                _ = _run_head()
            torch.cuda.current_stream(head_dev).wait_stream(sh_warm)

            gh = torch.cuda.CUDAGraph()
            with torch.cuda.graph(gh, stream=sh_warm):
                logits_buf, normed_buf = _run_head()

        # Persist the graph-owned rollback buffers for replay-time zero-recompute commits.
        for _key, st in kv_manager._iter_all_recurrent_states():
            st._graph_verify_cache = st._verify_cache
            st._verify_cache = None
            st.verify_active = False

        # Restore live KV lengths & recurrent states untouched
        for lkv, slen in zip(kv_manager._iter_all_kv_caches(), saved_lens):
            lkv.rewind_to(slen)
        if kv_manager.has_recurrent:
            kv_manager.restore_recurrent_gpu(rec_snap, kv_manager.recurrent_seq_len)

        self._verify_stage_data[s_len] = (stages, stage_graphs, stage_ins, stage_outs)
        self._verify_head_data[s_len] = (gh, head_in, logits_buf, normed_buf)
        self._verify_graph_caps[s_len] = tuple(lkv.capacity for lkv in kv_manager._iter_all_kv_caches())
        self._verify_graphs_ready[s_len] = True

    @torch.inference_mode()
    def forward_verify_step(
        self,
        verify_ids: Any,
        kv_manager: EngineKVCacheManager,
        start_pos: int,
        return_normed_hidden: bool = False,
    ) -> Any:
        """
        Multi-row (`S=2..8`) speculative verification forward. Returns `[1, S, vocab]` logits (every
        position, for acceptance checking + the bonus token), plus all rows' post-`output_norm` hidden
        states when `return_normed_hidden=True`.

        Replays per-stage `torch.cuda.CUDAGraph`s when the model is eligible (all attention layers on
        the fused posptr Split-KV path, recurrent layers armed); falls back to the eager `forward()`
        otherwise, so behavior is never silently degraded — just faster or identical to today.
        """
        if isinstance(verify_ids, torch.Tensor):
            verify_list = [int(t) for t in verify_ids.view(-1).tolist()]
        else:
            verify_list = [int(t) for t in verify_ids]
        s_len = len(verify_list)
        step_inp = torch.tensor([verify_list], dtype=torch.long, device=self.first_device)

        def _eager() -> Any:
            return self.forward(
                step_inp,
                kv_manager,
                start_pos=start_pos,
                logits_to_keep=s_len,
                return_normed_hidden=return_normed_hidden,
            )

        if (
            self._verify_graphs_failed
            or s_len < 2
            or s_len > 8
            or not self._verify_graph_eligibility(kv_manager)[0]
        ):
            return _eager()

        # Ensure KV cache pages won't reallocate mid-step; re-capture only when page capacity grows
        kv_manager.ensure_all_kv_capacity(start_pos + s_len)
        cur_caps = tuple(lkv.capacity for lkv in kv_manager._iter_all_kv_caches())
        if not self._verify_graphs_ready.get(s_len, False) or self._verify_graph_caps.get(s_len) != cur_caps:
            try:
                self._capture_verify_graphs(kv_manager, s_len)
            except Exception:
                self._verify_graphs_failed = True
                return _eager()

        stages, graphs, ins, outs = self._verify_stage_data[s_len]
        gh, head_in, logits_buf, normed_buf = self._verify_head_data[s_len]

        with cuda_device_scope(self.first_device):
            self._verify_token_bufs[s_len].copy_(step_inp)
        for dev, pbuf in self._verify_pos_bufs.items():
            with cuda_device_scope(dev):
                pbuf.fill_(start_pos)

        for s_idx, (dev, _, _) in enumerate(stages):
            if s_idx > 0:
                ins[s_idx].copy_(outs[s_idx - 1])
            with cuda_device_scope(dev):
                graphs[s_idx].replay()

        for lkv in kv_manager._iter_all_kv_caches():
            lkv.rewind_to(start_pos + s_len)

        head_in.copy_(outs[-1])
        with cuda_device_scope(head_in.device):
            gh.replay()
        if return_normed_hidden:
            return logits_buf, normed_buf
        return logits_buf

    def _norm_trunk_hidden(self, raw_h: torch.Tensor) -> torch.Tensor:
        target_dev = self.first_device if self.is_tensor_parallel else self.last_device
        with cuda_device_scope(target_dev):
            return self.output_norm(raw_h.to(target_dev), out_dtype=self.compute_dtype)

    def project_normed_hidden(self, h_norm: torch.Tensor) -> torch.Tensor:
        """Project post-`output_norm` hidden states `[1, S, H]` through `lm_head` (TP-sharded or single-GPU)."""
        if self.is_tensor_parallel:
            h0 = h_norm.to(self.first_device, non_blocking=True)
            h_inputs = [h0] + [h0.to(dev, non_blocking=True) for dev in self.tp_devices[1:]]
            logit_parts: List[torch.Tensor] = []
            for r, dev in enumerate(self.tp_devices):
                with cuda_device_scope(dev):
                    logit_parts.append(self.lm_head_shards[r](h_inputs[r]))
            with cuda_device_scope(self.first_device):
                logits = torch.cat([p.to(self.first_device, non_blocking=True) for p in logit_parts], dim=-1)
        else:
            if self.lm_head is not None:
                with cuda_device_scope(self.last_device):
                    logits = self.lm_head(h_norm.to(self.last_device))
            else:
                with cuda_device_scope(self.first_device):
                    logits = self.token_embd(h_norm.to(self.first_device))

        with cuda_device_scope(logits.device):
            logits_f32 = logits.float()
            if self.config.final_logit_softcapping > 0.0:
                cap = self.config.final_logit_softcapping
                logits_f32 = torch.tanh(logits_f32 / cap) * cap
            return logits_f32

    @torch.inference_mode()
    def forward(
        self,
        input_ids: torch.Tensor,
        kv_manager: EngineKVCacheManager,
        start_pos: int = 0,
        logits_to_keep: int = 1,
        return_normed_hidden: bool = False,
    ) -> Any:
        """
        Forward `input_ids` `[1, S]` starting at `start_pos`.
        Slices hidden states to `logits_to_keep` BEFORE projecting to vocab size!
        When `return_normed_hidden=True`, also returns ALL rows' post-`output_norm` hidden states (`t_h_nextn`)
        without computing extra `lm_head` vocab rows!
        """
        with cuda_device_scope(self.first_device):
            ids = input_ids.to(self.first_device)
            h = self.token_embd.embed_lookup(ids, dtype=torch.float32)
            if self.config.embed_scale != 1.0:
                h = h * self.config.embed_scale

        seq_len = h.shape[1]
        prefill_chunk = 2048
        if seq_len > prefill_chunk and logits_to_keep == 1:
            normed_chunks: List[torch.Tensor] = []
            last_h: Optional[torch.Tensor] = None
            for offset in range(0, seq_len, prefill_chunk):
                end = min(seq_len, offset + prefill_chunk)
                sub_h = h[:, offset:end, :]
                for idx, block in enumerate(self.layers):
                    if self.is_tensor_parallel:
                        sub_h = block.forward_tp(  # type: ignore[operator]
                            sub_h,
                            kv_manager.tp_layers[idx],
                            start_pos + offset,
                            kv_manager.tp_recurrent_states[idx],
                            self.compute_dtype,
                        )
                    else:
                        sub_h = block(
                            sub_h,
                            kv_manager.layers[idx],
                            start_pos + offset,
                            kv_manager.recurrent_states[idx],
                            self.compute_dtype,
                        )
                if return_normed_hidden:
                    normed_chunks.append(self._norm_trunk_hidden(sub_h))
                last_h = sub_h[:, -1:, :]
            assert last_h is not None
            if return_normed_hidden:
                h_norm_all = torch.cat(normed_chunks, dim=1)
                logits_f32 = self.project_normed_hidden(h_norm_all[:, -1:, :])
                return logits_f32, h_norm_all
            return self.project_normed_hidden(self._norm_trunk_hidden(last_h))

        for idx, block in enumerate(self.layers):
            if self.is_tensor_parallel:
                h = block.forward_tp(  # type: ignore[operator]
                    h,
                    kv_manager.tp_layers[idx],
                    start_pos,
                    kv_manager.tp_recurrent_states[idx],
                    self.compute_dtype,
                )
            else:
                h = block(
                    h,
                    kv_manager.layers[idx],
                    start_pos,
                    kv_manager.recurrent_states[idx],
                    self.compute_dtype,
                )

        if return_normed_hidden:
            h_norm_all = self._norm_trunk_hidden(h)
            h_for_logits = (
                h_norm_all[:, -logits_to_keep:, :]
                if (logits_to_keep > 0 and h_norm_all.shape[1] > logits_to_keep)
                else h_norm_all
            )
            return self.project_normed_hidden(h_for_logits), h_norm_all

        if logits_to_keep > 0 and h.shape[1] > logits_to_keep:
            h = h[:, -logits_to_keep:, :]
        return self.project_normed_hidden(self._norm_trunk_hidden(h))


In [ ]:
%%writefile pygguf_engine/engine.py
"""
High-speed Python GGUF Inference Engine (`PyGGUFEngine`) featuring:
- Zero-copy GGUF loading onto single or multi-GPU setups (Kaggle Dual T4 32GB VRAM)
- Quantized INT8 KV Cache (`kv_dtype="int8"`) for 32K-128K+ context windows
- Automatic Multi-Turn Prefix KV Cache Reuse (`cache_prompt=True`)
- Zero-VRAM N-Gram Self-Speculative Decoding (`speculative_ngram=True`)
- GPU-accelerated Top-K / Top-P / Min-P / Repetition-Penalty sampling & streaming
"""

from __future__ import annotations

import asyncio
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, AsyncIterator, Dict, Iterator, List, Literal, Optional, Sequence, Tuple, Union

import torch

from .gguf_parser import GGUFFile, GGUFModelConfig, GGUFTokenizer
from .kv_cache import EngineKVCacheManager
from .model import GGUFTransformerModel
from .quant_kernels import cuda_device_scope


@dataclass
class SamplingConfig:
    max_tokens: int = 512
    temperature: float = 0.7
    top_p: float = 0.95
    top_k: int = 40
    min_p: float = 0.05
    repetition_penalty: float = 1.0
    stop_strings: List[str] = field(default_factory=list)
    stop_token_ids: List[int] = field(default_factory=list)
    cache_prompt: bool = True
    seed: Optional[int] = None
    # Live per-request speculative decoding overrides (`None` = use engine defaults)
    use_mtp: Optional[bool] = None
    use_ngram: Optional[bool] = None
    mtp_draft_tokens: Optional[int] = None
    mtp_adaptive: Optional[bool] = None
    mtp_confidence_threshold: Optional[float] = None


@dataclass
class GenerationResult:
    text: str
    prompt_tokens: int
    cached_prefix_tokens: int
    completion_tokens: int
    finish_reason: Literal["stop", "length", "tool_calls"]
    prefill_time_s: float
    decode_time_s: float
    speculative_accepted: int = 0
    speculative_drafted: int = 0
    mtp_accepted: int = 0
    mtp_drafted: int = 0
    reasoning_content: str = ""
    tool_calls: List[Dict[str, Any]] = field(default_factory=list)

    @property
    def prompt_tok_per_s(self) -> float:
        evaluated = max(1, self.prompt_tokens - self.cached_prefix_tokens)
        return evaluated / max(self.prefill_time_s, 1e-6)

    @property
    def decode_tok_per_s(self) -> float:
        return self.completion_tokens / max(self.decode_time_s, 1e-6)


@dataclass
class StreamChunk:
    delta_text: str
    token_id: int
    finish_reason: Optional[Literal["stop", "length"]] = None
    stats: Optional[GenerationResult] = None


def _build_filtered_probs_gpu(
    logits_1d: torch.Tensor,
    cfg: SamplingConfig,
    recent_tokens: Optional[Sequence[int]] = None,
) -> Tuple[torch.Tensor, Optional[torch.Tensor], torch.Tensor]:
    """
    Apply repetition penalty, temperature, top-k, min-p, and top-p purely on GPU without intermediate CPU syncs.
    Returns `(probs_1d, cand_indices_or_None, penalized_logits_1d)`.
    """
    logits = logits_1d.float()
    if cfg.repetition_penalty != 1.0 and recent_tokens:
        logits = logits.clone()
        uniq_ids = torch.tensor(list(set(recent_tokens[-256:])), device=logits.device, dtype=torch.long)
        score = logits.index_select(0, uniq_ids)
        score = torch.where(score < 0, score * cfg.repetition_penalty, score / cfg.repetition_penalty)
        logits.scatter_(0, uniq_ids, score)

    if cfg.temperature <= 1e-5:
        return logits, None, logits

    scaled = logits * (1.0 / cfg.temperature)

    if cfg.top_k > 0 and cfg.top_k < scaled.numel():
        cand_logits, cand_indices = torch.topk(scaled, cfg.top_k)
    else:
        cand_logits = scaled
        cand_indices = None

    probs = torch.softmax(cand_logits, dim=-1)

    if cfg.min_p > 0.0:
        p_max = probs.max()
        probs = torch.where(probs < cfg.min_p * p_max, torch.zeros_like(probs), probs)

    if 0.0 < cfg.top_p < 1.0:
        if cand_indices is not None:
            cum_probs = torch.cumsum(probs, dim=-1)
            probs = probs.masked_fill((cum_probs - probs) > cfg.top_p, 0.0)
        else:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            sorted_probs = sorted_probs.masked_fill((cum_probs - sorted_probs) > cfg.top_p, 0.0)
            probs = torch.zeros_like(probs).scatter_(0, sorted_idx, sorted_probs)

    probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)
    prob_sum = probs.sum()
    fallback = torch.zeros_like(probs)
    if cand_indices is not None:
        fallback[0] = 1.0
    else:
        fallback.scatter_(0, logits.argmax().unsqueeze(0), 1.0)
    probs = torch.where(prob_sum > 0.0, probs / prob_sum.clamp(min=1e-20), fallback)
    return probs, cand_indices, logits


def sample_token_gpu(
    logits_1d: torch.Tensor,
    cfg: SamplingConfig,
    recent_tokens: Optional[Sequence[int]] = None,
    generator: Optional[torch.Generator] = None,
) -> int:
    """Zero-extra-sync GPU token sampler (single 8-byte `.item()` transfer per token!)."""
    with cuda_device_scope(logits_1d.device):
        probs, cand_indices, logits = _build_filtered_probs_gpu(logits_1d, cfg, recent_tokens)
        if cfg.temperature <= 1e-5:
            return int(torch.argmax(logits).item())

        sampled_local = torch.multinomial(probs, num_samples=1, generator=generator).squeeze(-1)
        if cand_indices is not None:
            return int(cand_indices.gather(0, sampled_local).item())
        return int(sampled_local.item())


def verify_or_sample_draft_token_gpu(
    logits_1d: torch.Tensor,
    draft_token: int,
    cfg: SamplingConfig,
    recent_tokens: Optional[Sequence[int]] = None,
    generator: Optional[torch.Generator] = None,
) -> Tuple[int, bool]:
    """
    Exact lossless speculative verification for deterministic N-gram draft `q(draft_token) = 1`
    at ANY temperature (`T = 0.0 .. 2.0`):
    - Accepts `draft_token` with probability `P_target(draft_token)`.
    - On rejection, samples strictly from `P_res(x != draft_token) = P_target(x) / (1 - P_target(draft_token))`,
      preserving 100% mathematical equivalence to standard autoregressive sampling!
    """
    with cuda_device_scope(logits_1d.device):
        probs, cand_indices, logits = _build_filtered_probs_gpu(logits_1d, cfg, recent_tokens)
        if cfg.temperature <= 1e-5:
            pred = int(torch.argmax(logits).item())
            return pred, (pred == draft_token)

        if cand_indices is not None:
            match_mask = cand_indices == draft_token
            p_draft = (probs * match_mask.to(probs.dtype)).sum()
        else:
            match_mask = None
            p_draft = probs[draft_token]

        u = torch.rand((), device=logits.device, dtype=torch.float32, generator=generator)
        if bool((u < p_draft).item()):
            return draft_token, True

        if cand_indices is not None:
            assert match_mask is not None
            res_probs = probs.masked_fill(match_mask, 0.0)
        else:
            res_probs = probs.clone()
            res_probs[draft_token] = 0.0

        res_sum = res_probs.sum()
        if float(res_sum.item()) <= 1e-20:
            return draft_token, True

        res_probs = res_probs / res_sum
        sampled_local = torch.multinomial(res_probs, num_samples=1, generator=generator).squeeze(-1)
        if cand_indices is not None:
            return int(cand_indices.gather(0, sampled_local).item()), False
        return int(sampled_local.item()), False


def _argmax_with_confidence(logits_1d: torch.Tensor) -> Tuple[int, float]:
    """
    Return `(top_token_id, softmax_probability_of_top_token)` in a SINGLE 16-byte GPU-to-CPU sync
    so confidence-gated adaptive `draft-mtp` (`k = 1..3`) adds zero extra PCIe round-trips!
    """
    with cuda_device_scope(logits_1d.device):
        row = logits_1d.float()
        max_val, top_idx = torch.max(row, dim=-1)
        p_top = torch.exp(max_val - torch.logsumexp(row, dim=-1))
        packed = torch.stack([top_idx.to(torch.float64), p_top.to(torch.float64)]).tolist()
        return int(packed[0]), float(packed[1])


def find_ngram_draft(
    tokens: Sequence[int],
    max_draft: int = 5,
    ngram_min: int = 2,
    ngram_max: int = 4,
) -> List[int]:
    """
    Zero-VRAM Self-Speculative N-Gram Drafter:
    Scans context history from newest to oldest for an `n`-gram matching the current suffix
    and returns up to `max_draft` continuation tokens.
    """
    total = len(tokens)
    if total < ngram_min * 2 or max_draft <= 0:
        return []

    search_limit = max(0, total - 4096)
    for n in range(min(ngram_max, total // 2), ngram_min - 1, -1):
        suffix = tokens[total - n :]
        last_tok = suffix[-1]
        # Reverse scan for fast recency match (avoids per-position slice allocations)
        for pos in range(total - n - 1, search_limit + n - 2, -1):
            if tokens[pos] != last_tok:
                continue
            ok = True
            for off in range(1, n):
                if tokens[pos - off] != suffix[n - 1 - off]:
                    ok = False
                    break
            if not ok:
                continue
            end = min(total - n, pos + 1 + max_draft)
            draft = list(tokens[pos + 1 : end])
            if draft:
                return draft
    return []


class PyGGUFEngine:
    """
    Pure-Python + PyTorch + OpenAI Triton GGUF Engine (`0% llama.cpp`).
    """

    def __init__(
        self,
        gguf_path: Union[str, Path],
        *,
        hf_tokenizer: Optional[str] = None,
        chat_template: Optional[str] = None,
        chat_template_mode: Literal["auto", "gguf", "qwen35_xml", "qwen3_json"] = "auto",
        default_enable_thinking: bool = True,
        max_context_len: Optional[int] = None,
        kv_dtype: Literal["int4", "int8", "fp16"] = "int8",
        devices: Optional[Sequence[str]] = None,
        parallel_mode: Literal["auto", "tp", "pp"] = "auto",
        enable_cuda_graph: bool = True,
        speculative_mtp: bool = True,
        mtp_draft_tokens: Optional[int] = None,
        mtp_adaptive: bool = True,
        mtp_max_draft_tokens: int = 3,
        mtp_confidence_threshold: float = 0.55,
        mtp_kv_window: Optional[int] = 4096,
        speculative_ngram: bool = True,
        speculative_max_draft: int = 5,
        model_alias: Optional[str] = None,
        rope_scaling_type: Optional[Literal["none", "linear", "ntk-aware", "yarn"]] = None,
        rope_scaling_factor: Optional[float] = None,
        rope_scaling_original_context_length: Optional[int] = None,
        rope_yarn_beta_fast: Optional[float] = None,
        rope_yarn_beta_slow: Optional[float] = None,
        rope_yarn_ext_factor: Optional[float] = None,
        rope_yarn_attn_factor: Optional[float] = None,
        rope_yarn_log_mul: Optional[float] = None,
    ) -> None:
        t0 = time.perf_counter()
        self.gguf_path = Path(gguf_path)
        self.default_enable_thinking = default_enable_thinking
        self.speculative_mtp = bool(speculative_mtp)
        if mtp_draft_tokens is not None:
            # Explicit integer `mtp_draft_tokens` pins exact depth `k` unless overridden per request
            self.mtp_draft_tokens = max(1, min(8, int(mtp_draft_tokens)))
            self.mtp_adaptive = False
        else:
            self.mtp_draft_tokens = 1
            self.mtp_adaptive = bool(mtp_adaptive)
        self.mtp_max_draft_tokens = max(1, min(8, int(mtp_max_draft_tokens)))
        self.mtp_confidence_threshold = float(mtp_confidence_threshold)
        self.mtp_kv_window: Optional[int] = (
            int(mtp_kv_window) if (mtp_kv_window is not None and int(mtp_kv_window) > 0) else None
        )
        print(f"[PyGGUF] Opening GGUF checkpoint: {self.gguf_path}")

        with GGUFFile(self.gguf_path) as gf:
            self.config: GGUFModelConfig = gf.build_model_config()
            if max_context_len is not None:
                self.config.max_context_len = max_context_len
            # Optional RoPE context-extension overrides (otherwise the GGUF's own
            # `{arch}.rope.scaling.*` metadata, if any, is used verbatim).
            if rope_scaling_type is not None:
                self.config.rope_scaling_type = rope_scaling_type
            if rope_scaling_factor is not None:
                self.config.rope_scaling_factor = float(rope_scaling_factor)
            if rope_scaling_original_context_length is not None:
                self.config.rope_original_context_len = int(rope_scaling_original_context_length)
            if rope_yarn_beta_fast is not None:
                self.config.rope_yarn_beta_fast = float(rope_yarn_beta_fast)
            if rope_yarn_beta_slow is not None:
                self.config.rope_yarn_beta_slow = float(rope_yarn_beta_slow)
            if rope_yarn_ext_factor is not None:
                self.config.rope_yarn_ext_factor = float(rope_yarn_ext_factor)
            if rope_yarn_attn_factor is not None:
                self.config.rope_yarn_attn_factor = float(rope_yarn_attn_factor)
            if rope_yarn_log_mul is not None:
                self.config.rope_yarn_log_mul = float(rope_yarn_log_mul)

            self.model_id = model_alias or self.config.name or self.gguf_path.stem
            rope_note = ""
            if getattr(self.config, "rope_scaling_type", "none") != "none" and float(getattr(self.config, "rope_scaling_factor", 1.0) or 1.0) != 1.0:
                rope_note = f" | RoPE={self.config.rope_scaling_type} x{self.config.rope_scaling_factor:g}"
                if self.config.rope_scaling_type == "yarn" and self.config.rope_original_context_len:
                    rope_note += f" (orig {self.config.rope_original_context_len})"
            print(
                f"[PyGGUF] Architecture={self.config.arch} | Layers={self.config.n_layers} | "
                f"Hidden={self.config.hidden_size} | Heads={self.config.n_heads}/{self.config.n_kv_heads} | "
                f"MoE={self.config.is_moe} ({self.config.n_experts}x top-{self.config.n_experts_used}) | "
                f"MaxCtx={self.config.max_context_len} | KV={kv_dtype}{rope_note}"
            )

            self.tokenizer = GGUFTokenizer(
                gf,
                self.config,
                hf_tokenizer=hf_tokenizer,
                chat_template=chat_template,
                chat_template_mode=chat_template_mode,
            )
            self.model = GGUFTransformerModel(
                gf,
                self.config,
                devices=devices,
                compute_dtype=torch.float16,
                parallel_mode=parallel_mode,
                enable_cuda_graph=enable_cuda_graph,
                load_mtp=self.speculative_mtp,
            )
            self.model.eval()

        self.kv_dtype: Literal["int4", "int8", "fp16"] = kv_dtype
        self.kv_manager = EngineKVCacheManager(
            n_layers=self.config.n_layers,
            n_kv_heads=self.config.n_kv_heads,
            head_dim=self.config.head_dim,
            max_seq_len=self.config.max_context_len,
            layer_devices=self.model.layer_devices,
            kv_dtype=kv_dtype,
            recurrent_layers=self.model.recurrent_layer_flags,
            tp_devices=self.model.tp_devices if self.model.is_tensor_parallel else None,
        )
        if self.model.mtp_head is not None:
            self.kv_manager.init_mtp_cache(
                n_kv_heads=self.config.n_kv_heads,
                head_dim=self.config.head_dim,
                device=self.model.mtp_head.device,
                tp_devices=self.model.mtp_head.tp_devices if self.model.mtp_head.is_tensor_parallel else None,
                sliding_window=self.mtp_kv_window,
            )
        self._mtp_boundary_h: Dict[int, torch.Tensor] = {}

        self.speculative_ngram = speculative_ngram
        self.speculative_max_draft = speculative_max_draft
        self._async_lock = asyncio.Lock()
        # Dedicated persistent GPU worker thread so CUDA driver contexts & Triton module handles stay hot!
        self._gpu_executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="pygguf-gpu")

        load_s = time.perf_counter() - t0
        active_devs = self.model.tp_devices if self.model.is_tensor_parallel else self.model.layer_devices
        dev_summary = [str(d) for d in active_devs] if self.model.is_tensor_parallel else sorted({str(d) for d in active_devs})
        par_label = f"TP={len(self.model.tp_devices)}" if self.model.is_tensor_parallel else (
            "PP+CUDAGraph" if self.model.enable_cuda_graph and any(d.type == "cuda" for d in active_devs) else "PP"
        )
        n_rec = sum(self.model.recurrent_layer_flags)
        n_attn = self.config.n_layers - n_rec
        if self.model.mtp_head is not None:
            k_desc = (
                f"adaptive k=1..{max(self.mtp_draft_tokens, self.mtp_max_draft_tokens)}"
                if self.mtp_adaptive
                else f"k={self.mtp_draft_tokens}"
            )
            win_desc = f", win={self.mtp_kv_window}" if self.mtp_kv_window else ""
            mtp_status = f"enabled draft-mtp blk.{self.config.n_layers} ({k_desc}{win_desc})"
        elif self.config.nextn_predict_layers > 0:
            mtp_status = f"disabled {self.config.nextn_predict_layers} MTP (0 GB VRAM)"
        else:
            mtp_status = ""
        hybrid_parts = []
        if n_rec > 0:
            hybrid_parts.append(f"{n_rec} GatedDeltaNet + {n_attn} FullAttn")
        if mtp_status:
            hybrid_parts.append(mtp_status)
        hybrid_note = f" | Hybrid=({', '.join(hybrid_parts)})" if hybrid_parts else ""
        print(f"[PyGGUF] Loaded '{self.model_id}' ({par_label}) across {dev_summary} in {load_s:.2f}s{hybrid_note}.")

        if any(d.type == "cuda" for d in active_devs):
            # Run warmup directly on the persistent GPU worker thread so its CUDA driver contexts are primed!
            self._gpu_executor.submit(self._warmup_cuda_kernels).result()

    @torch.inference_mode()
    def _warmup_cuda_kernels(self) -> None:
        """Pre-compile Triton PTX kernels (including M=2 multi-row GEMV, MTP head, and M=1 CUDA Graphs)."""
        t_w = time.perf_counter()
        dummy_prefill = torch.zeros((1, 12), dtype=torch.long)
        _, h_warm = self.model(dummy_prefill, self.kv_manager, start_pos=0, logits_to_keep=1, return_normed_hidden=True)
        # Warm up M=2 weight-reusing multi-row GEMV verify specialization!
        dummy_verify = torch.zeros((1, 2), dtype=torch.long)
        self.model(dummy_verify, self.kv_manager, start_pos=12, logits_to_keep=2)
        if self.model.mtp_head is not None:
            self.model.mtp_head.prefill_catch_up(
                dummy_prefill, h_warm, self.kv_manager, start_pos=0, trunk_token_embd=self.model.token_embd
            )
            self.model.mtp_head.forward_step(
                torch.zeros((1, 1), dtype=torch.long),
                h_warm[:, -1:, :],
                self.kv_manager,
                start_pos=12,
                trunk_model=self.model,
            )
        self.model.forward_decode_step(0, self.kv_manager, start_pos=14)
        self.kv_manager.clear()
        self._mtp_boundary_h.clear()
        torch.cuda.synchronize()
        cg_note = " + CUDAGraph" if self.model._decode_graphs_ready else ""
        mtp_note = " + draft-mtp" if self.model.mtp_head is not None else ""
        print(f"[PyGGUF] Compiled & warmed up Triton GPU kernels{cg_note}{mtp_note} in {time.perf_counter() - t_w:.2f}s.")

    def vram_report(self) -> Dict[str, Any]:
        gpus = []
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                alloc = torch.cuda.memory_allocated(i) / (1024**3)
                total = torch.cuda.get_device_properties(i).total_memory / (1024**3)
                gpus.append(
                    {
                        "gpu": i,
                        "name": torch.cuda.get_device_name(i),
                        "allocated_gib": round(alloc, 2),
                        "total_gib": round(total, 2),
                    }
                )
        return {
            "model_id": self.model_id,
            "arch": self.config.arch,
            "is_moe": self.config.is_moe,
            "parallel_mode": f"tp={len(self.model.tp_devices)}" if self.model.is_tensor_parallel else "pp",
            "cuda_graph": bool(self.model._decode_graphs_ready),
            "mtp_loaded": self.model.mtp_head is not None,
            "speculative_mtp_default": bool(self.speculative_mtp and self.model.mtp_head is not None),
            "mtp_draft_tokens_default": self.mtp_draft_tokens,
            "mtp_adaptive_default": self.mtp_adaptive,
            "mtp_max_draft_tokens": self.mtp_max_draft_tokens,
            "mtp_confidence_threshold": self.mtp_confidence_threshold,
            "mtp_kv_window": self.mtp_kv_window,
            "speculative_ngram_default": self.speculative_ngram,
            "max_context_len": self.config.max_context_len,
            "rope_scaling_type": getattr(self.config, "rope_scaling_type", "none"),
            "rope_scaling_factor": getattr(self.config, "rope_scaling_factor", 1.0),
            "rope_original_context_len": getattr(self.config, "rope_original_context_len", 0),
            "kv_dtype": self.kv_dtype,
            "cached_prefix_tokens": len(self.kv_manager.cached_tokens),
            "kv_cache_mib": round(self.kv_manager.allocated_vram_bytes() / (1024**2), 2),
            "kv_cache_max_mib": round(self.kv_manager.projected_max_vram_bytes() / (1024**2), 2),
            "gpus": gpus,
        }

    @torch.inference_mode()
    def stream_tokens(
        self,
        prompt_or_ids: Union[str, Sequence[int]],
        sampling: Optional[SamplingConfig] = None,
    ) -> Iterator[StreamChunk]:
        """
        Core streaming token generator with Prefix KV Cache Reuse + N-Gram Self-Speculative Decoding.
        """
        cfg = sampling or SamplingConfig()
        prompt_ids = (
            self.tokenizer.encode(prompt_or_ids, add_special_tokens=True)
            if isinstance(prompt_or_ids, str)
            else list(prompt_or_ids)
        )
        if not prompt_ids:
            prompt_ids = [self.config.bos_token_id or 0]

        max_new = max(1, min(cfg.max_tokens, self.config.max_context_len - len(prompt_ids)))
        stop_ids = set(self.config.eos_token_ids) | set(cfg.stop_token_ids)

        rng: Optional[torch.Generator] = None
        if cfg.seed is not None:
            rng = torch.Generator(device=self.model.last_device if self.model.lm_head else self.model.first_device)
            rng.manual_seed(cfg.seed)

        # Resolve per-request speculative decoding overrides (`SamplingConfig.use_mtp`, `use_ngram`, `mtp_draft_tokens`, `mtp_adaptive`)
        eff_use_mtp = (
            (self.speculative_mtp if cfg.use_mtp is None else bool(cfg.use_mtp))
            and (self.model.mtp_head is not None)
            and (cfg.temperature <= 1.2)
        )
        if cfg.mtp_draft_tokens is None:
            eff_mtp_adaptive = bool(self.mtp_adaptive if cfg.mtp_adaptive is None else cfg.mtp_adaptive)
            base_mtp_k = (
                max(self.mtp_draft_tokens, self.mtp_max_draft_tokens)
                if eff_mtp_adaptive
                else self.mtp_draft_tokens
            )
            eff_mtp_k = max(1, min(8, int(base_mtp_k)))
        else:
            eff_mtp_adaptive = bool(cfg.mtp_adaptive) if cfg.mtp_adaptive is not None else False
            eff_mtp_k = max(1, min(8, int(cfg.mtp_draft_tokens)))
        eff_mtp_conf_thresh = float(
            self.mtp_confidence_threshold
            if cfg.mtp_confidence_threshold is None
            else cfg.mtp_confidence_threshold
        )
        eff_use_ngram = (
            (self.speculative_ngram if cfg.use_ngram is None else bool(cfg.use_ngram))
            and (self.speculative_max_draft > 0)
            and (cfg.temperature <= 1.2)
        )

        # 1. Prefix KV Cache Reuse
        if cfg.cache_prompt:
            prefix_hit = self.kv_manager.match_prefix_length(prompt_ids)
            self.kv_manager.rollback_to(prefix_hit)
        else:
            prefix_hit = 0
            self.kv_manager.clear()
            self._mtp_boundary_h.clear()

        # 2. Prompt Prefill (only evaluates uncached tail `prompt_ids[prefix_hit:]`!)
        t_prefill_start = time.perf_counter()
        uncached_ids = prompt_ids[prefix_hit:]
        inp = torch.tensor([uncached_ids], dtype=torch.long)
        prefill_h_norm: Optional[torch.Tensor] = None
        if eff_use_mtp:
            logits, prefill_h_norm = self.model(
                inp, self.kv_manager, start_pos=prefix_hit, logits_to_keep=1, return_normed_hidden=True
            )
            assert self.model.mtp_head is not None and prefill_h_norm is not None
            if prefix_hit == 0:
                self.kv_manager.reset_mtp_cache(0)
                self.model.mtp_head.prefill_catch_up(
                    inp,
                    prefill_h_norm,
                    self.kv_manager,
                    start_pos=0,
                    trunk_token_embd=self.model.token_embd,
                    prev_h=None,
                )
            elif self.kv_manager.mtp_exact_kv_len == prefix_hit and prefix_hit in self._mtp_boundary_h:
                self.model.mtp_head.prefill_catch_up(
                    inp,
                    prefill_h_norm,
                    self.kv_manager,
                    start_pos=prefix_hit,
                    trunk_token_embd=self.model.token_embd,
                    prev_h=self._mtp_boundary_h[prefix_hit],
                )
            else:
                # Seamless rebase when a previous turn had MTP disabled via live override!
                self.kv_manager.reset_mtp_cache(prefix_hit)
                self.model.mtp_head.prefill_catch_up(
                    inp,
                    prefill_h_norm,
                    self.kv_manager,
                    start_pos=prefix_hit,
                    trunk_token_embd=self.model.token_embd,
                    prev_h=None,
                )
            prompt_end_h = prefill_h_norm[:, -1, :].detach().clone()
            self._mtp_boundary_h = {len(prompt_ids): prompt_end_h}
        else:
            logits = self.model(inp, self.kv_manager, start_pos=prefix_hit, logits_to_keep=1)

        self.kv_manager.record_tokens(prefix_hit, uncached_ids)
        self.kv_manager.save_prompt_checkpoint()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        prefill_time_s = time.perf_counter() - t_prefill_start

        # Sample first token from last prompt position
        t_decode_start = time.perf_counter()
        all_tokens: List[int] = list(prompt_ids)
        generated_ids: List[int] = []
        decoded_so_far = ""
        spec_accepted = 0
        spec_drafted = 0
        mtp_accepted = 0
        mtp_drafted = 0
        finish_reason: Literal["stop", "length"] = "length"

        first_tok = sample_token_gpu(logits[0, -1, :], cfg, all_tokens, rng)
        if first_tok in stop_ids:
            finish_reason = "stop"
            res = GenerationResult(
                text="",
                prompt_tokens=len(prompt_ids),
                cached_prefix_tokens=prefix_hit,
                completion_tokens=0,
                finish_reason=finish_reason,
                prefill_time_s=prefill_time_s,
                decode_time_s=time.perf_counter() - t_decode_start,
            )
            yield StreamChunk(delta_text="", token_id=first_tok, finish_reason=finish_reason, stats=res)
            return

        all_tokens.append(first_tok)
        generated_ids.append(first_tok)
        new_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        delta = new_text[len(decoded_so_far) :]
        decoded_so_far = new_text

        # Check string stop sequences on first token
        stopped_by_str = False
        for s_stop in cfg.stop_strings:
            if s_stop and s_stop in decoded_so_far:
                idx = decoded_so_far.index(s_stop)
                decoded_so_far = decoded_so_far[:idx]
                stopped_by_str = True
                break

        if stopped_by_str or len(generated_ids) >= max_new:
            finish_reason = "stop" if stopped_by_str else "length"
            res = GenerationResult(
                text=decoded_so_far,
                prompt_tokens=len(prompt_ids),
                cached_prefix_tokens=prefix_hit,
                completion_tokens=len(generated_ids),
                finish_reason=finish_reason,
                prefill_time_s=prefill_time_s,
                decode_time_s=time.perf_counter() - t_decode_start,
            )
            yield StreamChunk(delta_text=decoded_so_far, token_id=first_tok, finish_reason=finish_reason, stats=res)
            return

        yield StreamChunk(delta_text=delta, token_id=first_tok)

        # 3. Autoregressive Decode Loop (Invariant: `len(all_tokens) == kv_len + 1`)
        # Hybrid Speculative Decoding: Zero-cost N-Gram suffix match first -> Native GGUF `draft-mtp` head second!
        eff_ngram_min = 3 if self.kv_manager.has_recurrent else 2
        mtp_pending_start_pos = len(prompt_ids)
        mtp_pending_tokens: List[int] = [first_tok] if eff_use_mtp else []
        mtp_pending_h: List[torch.Tensor] = (
            [prefill_h_norm[:, -1:, :].detach().clone()] if (eff_use_mtp and prefill_h_norm is not None) else []
        )
        prompt_boundary_entry = dict(self._mtp_boundary_h)

        while len(generated_ids) < max_new:
            kv_pos = len(all_tokens) - 1
            cur_tok = all_tokens[-1]
            remaining = max_new - len(generated_ids)

            draft: List[int] = []
            draft_source: Literal["none", "ngram", "mtp"] = "none"

            if remaining > 1 and eff_use_ngram:
                draft = find_ngram_draft(
                    all_tokens,
                    max_draft=min(self.speculative_max_draft, remaining - 1),
                    ngram_min=eff_ngram_min,
                )
                if draft:
                    draft_source = "ngram"

            if not draft and remaining > 1 and eff_use_mtp and mtp_pending_tokens:
                assert self.model.mtp_head is not None
                max_k = min(eff_mtp_k, remaining - 1)
                self.kv_manager.rollback_mtp_to(self.kv_manager.mtp_exact_kv_len)
                tok_batch = torch.tensor([mtp_pending_tokens], dtype=torch.long)
                h_batch = torch.cat(mtp_pending_h, dim=1)
                d_logits, h_chain = self.model.mtp_head.forward_step(
                    tok_batch,
                    h_batch,
                    self.kv_manager,
                    start_pos=mtp_pending_start_pos,
                    trunk_model=self.model,
                )
                self.kv_manager.mtp_exact_kv_len = mtp_pending_start_pos + len(mtp_pending_tokens)
                mtp_pending_start_pos = self.kv_manager.mtp_exact_kv_len
                mtp_pending_tokens.clear()
                mtp_pending_h.clear()

                if eff_mtp_adaptive and max_k > 1:
                    d0, conf0 = _argmax_with_confidence(d_logits[0, 0])
                    draft = [d0]
                    draft_source = "mtp"
                    cum_conf = conf0
                    if d0 not in stop_ids:
                        for step_i in range(1, max_k):
                            if cum_conf < eff_mtp_conf_thresh:
                                break
                            tok_i = torch.tensor([[draft[-1]]], dtype=torch.long)
                            d_logits_i, h_chain = self.model.mtp_head.forward_step(
                                tok_i,
                                h_chain,
                                self.kv_manager,
                                start_pos=kv_pos + step_i,
                                trunk_model=self.model,
                            )
                            di, confi = _argmax_with_confidence(d_logits_i[0, 0])
                            draft.append(di)
                            cum_conf *= confi
                            if di in stop_ids:
                                break
                else:
                    d0 = int(torch.argmax(d_logits[0, 0]).item())
                    draft = [d0]
                    draft_source = "mtp"
                    if max_k > 1 and d0 not in stop_ids:
                        for step_i in range(1, max_k):
                            tok_i = torch.tensor([[draft[-1]]], dtype=torch.long)
                            d_logits_i, h_chain = self.model.mtp_head.forward_step(
                                tok_i,
                                h_chain,
                                self.kv_manager,
                                start_pos=kv_pos + step_i,
                                trunk_model=self.model,
                            )
                            di = int(torch.argmax(d_logits_i[0, 0]).item())
                            draft.append(di)
                            if di in stop_ids:
                                break

            if not draft:
                if eff_use_mtp:
                    step_logits, step_h_norm = self.model.forward_decode_step(
                        cur_tok, self.kv_manager, start_pos=kv_pos, return_normed_hidden=True
                    )
                else:
                    step_logits = self.model.forward_decode_step(cur_tok, self.kv_manager, start_pos=kv_pos)
                    step_h_norm = None
                self.kv_manager.record_tokens(kv_pos, [cur_tok])
                cand_tokens = [sample_token_gpu(step_logits[0, 0, :], cfg, all_tokens, rng)]
                if eff_use_mtp and step_h_norm is not None:
                    h_cur = step_h_norm[:, 0:1, :].detach().clone()
                    mtp_pending_tokens.append(cand_tokens[0])
                    mtp_pending_h.append(h_cur)
                    self._mtp_boundary_h = {**prompt_boundary_entry, kv_pos + 1: h_cur.squeeze(1)}
            else:
                # Single batched weight-reusing GEMV verification pass over `[cur_tok] + draft`!
                spec_drafted += len(draft)
                if draft_source == "mtp":
                    mtp_drafted += len(draft)
                verify_seq = [cur_tok] + draft
                v_len = len(verify_seq)
                if self.kv_manager.has_recurrent:
                    self.kv_manager.begin_recurrent_verify()
                # Replays a captured whole-step CUDA graph on eligible models (int4/int8 KV, non-MoE,
                # single-node PP/1-GPU); falls back to the eager forward otherwise.
                step_logits_h = self.model.forward_verify_step(
                    verify_seq,
                    self.kv_manager,
                    start_pos=kv_pos,
                    return_normed_hidden=eff_use_mtp,
                )
                if eff_use_mtp:
                    step_logits, step_h_norm = step_logits_h
                else:
                    step_logits = step_logits_h
                    step_h_norm = None

                cand_tokens = []
                accepted_in_step = 0
                if cfg.temperature <= 1e-5 and cfg.repetition_penalty == 1.0:
                    # Fast single-sync greedy batch check across all D+1 positions!
                    greedy_preds = step_logits[0].argmax(dim=-1).tolist()
                    for j, d_tok in enumerate(draft):
                        pred_tok = int(greedy_preds[j])
                        cand_tokens.append(pred_tok)
                        if pred_tok != d_tok or pred_tok in stop_ids:
                            break
                        accepted_in_step += 1
                    else:
                        cand_tokens.append(int(greedy_preds[len(draft)]))
                else:
                    for j, d_tok in enumerate(draft):
                        pred_tok, ok = verify_or_sample_draft_token_gpu(
                            step_logits[0, j, :], d_tok, cfg, all_tokens + cand_tokens, rng
                        )
                        cand_tokens.append(pred_tok)
                        if not ok or pred_tok in stop_ids:
                            break
                        accepted_in_step += 1
                    else:
                        bonus_tok = sample_token_gpu(step_logits[0, len(draft), :], cfg, all_tokens + cand_tokens, rng)
                        cand_tokens.append(bonus_tok)

                spec_accepted += accepted_in_step
                if draft_source == "mtp":
                    mtp_accepted += accepted_in_step

                # Zero-recompute commit of `cur_tok` + `accepted_in_step` verified draft tokens!
                valid_kv_new = verify_seq[: 1 + accepted_in_step]
                n_keep = len(valid_kv_new)
                if self.kv_manager.has_recurrent:
                    if accepted_in_step < len(draft):
                        self.kv_manager.commit_recurrent_verify(kv_pos, n_keep)
                    else:
                        self.kv_manager.end_recurrent_verify()
                self.kv_manager.rollback_to(kv_pos + n_keep)
                self.kv_manager.record_tokens(kv_pos, valid_kv_new)

                if eff_use_mtp and step_h_norm is not None:
                    for j, c_tok in enumerate(cand_tokens):
                        mtp_pending_tokens.append(c_tok)
                        mtp_pending_h.append(step_h_norm[:, j : j + 1, :].detach().clone())
                    self._mtp_boundary_h = {
                        **prompt_boundary_entry,
                        kv_pos + n_keep: step_h_norm[:, n_keep - 1, :].detach().clone(),
                    }

            # Emit newly produced tokens one by one, checking stop conditions cleanly
            hit_end = False
            for idx_c, next_tok in enumerate(cand_tokens):
                if next_tok in stop_ids:
                    finish_reason = "stop"
                    hit_end = True
                    # Roll back any extra speculative KV entries beyond the tokens actually kept
                    self.kv_manager.rollback_to(min(len(self.kv_manager.cached_tokens), len(all_tokens)))
                    break

                all_tokens.append(next_tok)
                generated_ids.append(next_tok)
                # O(1) sliding-window incremental detokenization (preserves multi-byte UTF-8 & spaces)
                if len(generated_ids) <= 24:
                    new_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
                    piece = new_text[len(decoded_so_far) :]
                    decoded_so_far = new_text
                else:
                    prev_tail = self.tokenizer.decode(generated_ids[-16:-1], skip_special_tokens=True)
                    curr_tail = self.tokenizer.decode(generated_ids[-16:], skip_special_tokens=True)
                    piece = curr_tail[len(prev_tail) :] if curr_tail.startswith(prev_tail) else self.tokenizer.decode([next_tok], skip_special_tokens=True)
                    decoded_so_far += piece

                for s_stop in cfg.stop_strings:
                    if s_stop and s_stop in decoded_so_far:
                        cut = decoded_so_far.index(s_stop)
                        piece = piece[: max(0, len(piece) - (len(decoded_so_far) - cut))]
                        decoded_so_far = decoded_so_far[:cut]
                        finish_reason = "stop"
                        hit_end = True
                        break

                if len(generated_ids) >= max_new and not hit_end:
                    finish_reason = "length"
                    hit_end = True

                if hit_end:
                    if torch.cuda.is_available():
                        torch.cuda.synchronize()
                    res = GenerationResult(
                        text=decoded_so_far,
                        prompt_tokens=len(prompt_ids),
                        cached_prefix_tokens=prefix_hit,
                        completion_tokens=len(generated_ids),
                        finish_reason=finish_reason,
                        prefill_time_s=prefill_time_s,
                        decode_time_s=time.perf_counter() - t_decode_start,
                        speculative_accepted=spec_accepted,
                        speculative_drafted=spec_drafted,
                        mtp_accepted=mtp_accepted,
                        mtp_drafted=mtp_drafted,
                    )
                    yield StreamChunk(delta_text=piece, token_id=next_tok, finish_reason=finish_reason, stats=res)
                    return

                yield StreamChunk(delta_text=piece, token_id=next_tok)

            if hit_end:
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                res = GenerationResult(
                    text=decoded_so_far,
                    prompt_tokens=len(prompt_ids),
                    cached_prefix_tokens=prefix_hit,
                    completion_tokens=len(generated_ids),
                    finish_reason=finish_reason,
                    prefill_time_s=prefill_time_s,
                    decode_time_s=time.perf_counter() - t_decode_start,
                    speculative_accepted=spec_accepted,
                    speculative_drafted=spec_drafted,
                    mtp_accepted=mtp_accepted,
                    mtp_drafted=mtp_drafted,
                )
                yield StreamChunk(delta_text="", token_id=all_tokens[-1], finish_reason=finish_reason, stats=res)
                return

    def generate(
        self,
        prompt_or_ids: Union[str, Sequence[int]],
        sampling: Optional[SamplingConfig] = None,
    ) -> GenerationResult:
        last_stats: Optional[GenerationResult] = None
        pieces: List[str] = []
        for chunk in self.stream_tokens(prompt_or_ids, sampling):
            if chunk.delta_text:
                pieces.append(chunk.delta_text)
            if chunk.stats is not None:
                last_stats = chunk.stats
        assert last_stats is not None
        return last_stats

    def chat(
        self,
        messages: Sequence[Dict[str, Any]],
        sampling: Optional[SamplingConfig] = None,
        enable_thinking: Optional[bool] = None,
        tools: Optional[Sequence[Dict[str, Any]]] = None,
        tool_choice: Any = None,
    ) -> GenerationResult:
        from .agentic_protocol import ReasoningAndToolStreamParser

        formatted = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
            tools=tools,
            tool_choice=tool_choice,
            default_enable_thinking=self.default_enable_thinking,
        )
        raw_res = self.generate(formatted, sampling)
        parser = ReasoningAndToolStreamParser(
            prompt_str=formatted,
            tools=tools,
            parse_tool_calls=(tool_choice != "none"),
            emit_reasoning=True,
        )
        parser.feed(raw_res.text)
        parser.finalize()
        raw_res.text = parser.full_content
        raw_res.reasoning_content = parser.full_reasoning
        if parser.has_tool_calls:
            raw_res.tool_calls = [tc.to_openai_dict() for tc in parser.parsed_tool_calls]
            raw_res.finish_reason = "tool_calls"
        return raw_res

    async def astream_tokens(
        self,
        prompt_or_ids: Union[str, Sequence[int]],
        sampling: Optional[SamplingConfig] = None,
    ) -> AsyncIterator[StreamChunk]:
        """Thread-offloaded async token generator so FastAPI's event loop stays 100% responsive."""
        async with self._async_lock:
            loop = asyncio.get_running_loop()
            queue: asyncio.Queue[Union[StreamChunk, BaseException, None]] = asyncio.Queue()

            def _produce() -> None:
                try:
                    for ch in self.stream_tokens(prompt_or_ids, sampling):
                        loop.call_soon_threadsafe(queue.put_nowait, ch)
                except BaseException as exc:
                    loop.call_soon_threadsafe(queue.put_nowait, exc)
                finally:
                    loop.call_soon_threadsafe(queue.put_nowait, None)

            fut = loop.run_in_executor(self._gpu_executor, _produce)
            try:
                while True:
                    item = await queue.get()
                    if item is None:
                        break
                    if isinstance(item, BaseException):
                        raise item
                    yield item
            finally:
                await fut


In [ ]:
%%writefile pygguf_engine/openai_server.py
"""
OpenAI-Compatible FastAPI Server for `PyGGUFEngine` + Built-in Cloudflare HTTPS Tunnel
for instant Kaggle Notebook external access (`https://xxxx.trycloudflare.com/v1`).
"""

from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import threading
import time
import urllib.request
import uuid
from typing import Any, AsyncIterator, Dict, List, Literal, Optional, Sequence, Union

from fastapi import Depends, FastAPI, Header, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, ConfigDict, Field

from .agentic_protocol import ReasoningAndToolStreamParser, resolve_enable_thinking
from .engine import PyGGUFEngine, SamplingConfig


# ==============================================================================
# 1. OpenAI-Compatible Request / Response Schemas (Full Tools + Reasoning)
# ==============================================================================

class ChatMessageInput(BaseModel):
    model_config = ConfigDict(extra="allow")

    role: str
    content: Union[str, List[Any], None] = None
    name: Optional[str] = None
    tool_calls: Optional[List[Dict[str, Any]]] = None
    tool_call_id: Optional[str] = None
    reasoning_content: Optional[str] = None

    def normalized_text(self) -> str:
        if self.content is None:
            return ""
        if isinstance(self.content, str):
            return self.content
        parts: List[str] = []
        for item in self.content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item.get("text", "")))
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join(parts)

    def to_template_dict(self) -> Dict[str, Any]:
        d: Dict[str, Any] = {
            "role": self.role,
            "content": self.normalized_text(),
        }
        if self.name is not None:
            d["name"] = self.name
        if self.tool_calls:
            d["tool_calls"] = self.tool_calls
        if self.tool_call_id is not None:
            d["tool_call_id"] = self.tool_call_id
        if self.reasoning_content is not None:
            d["reasoning_content"] = self.reasoning_content
        return d


class StreamOptions(BaseModel):
    model_config = ConfigDict(extra="allow")

    include_usage: bool = False


class ChatCompletionRequest(BaseModel):
    model_config = ConfigDict(extra="allow")

    model: Optional[str] = None
    messages: List[ChatMessageInput]
    tools: Optional[List[Dict[str, Any]]] = None
    tool_choice: Optional[Union[str, Dict[str, Any]]] = None
    parallel_tool_calls: Optional[bool] = None
    max_tokens: Optional[int] = None
    max_completion_tokens: Optional[int] = None
    temperature: float = 0.7
    top_p: float = 0.95
    top_k: int = 40
    min_p: float = 0.05
    repetition_penalty: float = 1.0
    stop: Optional[Union[str, List[str]]] = None
    stream: bool = False
    stream_options: Optional[StreamOptions] = None
    seed: Optional[int] = None
    cache_prompt: bool = True
    enable_thinking: Optional[bool] = None
    reasoning_effort: Optional[str] = None
    thinking: Optional[Dict[str, Any]] = None
    include_reasoning: Optional[bool] = None
    chat_template_kwargs: Optional[Dict[str, Any]] = None
    # Per-request speculative decoding controls (`extra_body={"speculative_mtp": False}` etc.)
    speculative: Optional[bool] = None
    speculative_mtp: Optional[bool] = None
    speculative_ngram: Optional[bool] = None
    mtp_draft_tokens: Optional[int] = None
    mtp_adaptive: Optional[bool] = None
    mtp_confidence_threshold: Optional[float] = None


class CompletionRequest(BaseModel):
    model_config = ConfigDict(extra="allow")

    model: Optional[str] = None
    prompt: Union[str, List[int]]
    max_tokens: int = 512
    temperature: float = 0.7
    top_p: float = 0.95
    top_k: int = 40
    min_p: float = 0.05
    repetition_penalty: float = 1.0
    stop: Optional[Union[str, List[str]]] = None
    stream: bool = False
    seed: Optional[int] = None
    cache_prompt: bool = True
    speculative: Optional[bool] = None
    speculative_mtp: Optional[bool] = None
    speculative_ngram: Optional[bool] = None
    mtp_draft_tokens: Optional[int] = None
    mtp_adaptive: Optional[bool] = None
    mtp_confidence_threshold: Optional[float] = None


# ==============================================================================
# 2. FastAPI Application Factory
# ==============================================================================

def _normalize_stops(stop: Optional[Union[str, List[str]]]) -> List[str]:
    if stop is None:
        return []
    if isinstance(stop, str):
        return [stop] if stop else []
    return [s for s in stop if s]


def create_openai_app(
    engine: PyGGUFEngine,
    api_key: Optional[str] = None,
    include_reasoning: bool = True,
) -> FastAPI:
    """Create a FastAPI app exposing `/v1/models`, `/v1/chat/completions`, `/v1/completions`, and `/health`."""
    app = FastAPI(
        title="PyGGUF OpenAI Server",
        version="1.0.0",
        description="100% Python + PyTorch + OpenAI Triton GGUF Serving Engine (0% llama.cpp)",
    )
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )

    created_ts = int(time.time())

    async def verify_auth(authorization: Optional[str] = Header(default=None)) -> None:
        if not api_key:
            return
        if not authorization or not authorization.startswith("Bearer "):
            raise HTTPException(
                status_code=status.HTTP_401_UNAUTHORIZED,
                detail="Missing Bearer API key in Authorization header.",
            )
        token = authorization[len("Bearer ") :].strip()
        if token != api_key:
            raise HTTPException(
                status_code=status.HTTP_401_UNAUTHORIZED,
                detail="Invalid API key.",
            )

    @app.get("/health")
    @app.get("/v1/health")
    async def health() -> Dict[str, Any]:
        return {"status": "ok", **engine.vram_report()}

    @app.get("/v1/models", dependencies=[Depends(verify_auth)])
    async def list_models() -> Dict[str, Any]:
        return {
            "object": "list",
            "data": [
                {
                    "id": engine.model_id,
                    "object": "model",
                    "created": created_ts,
                    "owned_by": "pygguf-engine",
                    "meta": {
                        "arch": engine.config.arch,
                        "max_context_len": engine.config.max_context_len,
                        "kv_dtype": engine.kv_dtype,
                        "is_moe": engine.config.is_moe,
                        "mtp_loaded": engine.model.mtp_head is not None,
                        "speculative_mtp_default": bool(engine.speculative_mtp and engine.model.mtp_head is not None),
                        "mtp_draft_tokens_default": engine.mtp_draft_tokens,
                        "speculative_ngram_default": engine.speculative_ngram,
                    },
                }
            ],
        }

    @app.post("/v1/chat/completions", dependencies=[Depends(verify_auth)])
    async def chat_completions(req: ChatCompletionRequest) -> Any:
        norm_messages = [m.to_template_dict() for m in req.messages]
        eff_thinking = resolve_enable_thinking(
            messages=norm_messages,
            explicit_enable_thinking=req.enable_thinking,
            chat_template_kwargs=req.chat_template_kwargs,
            reasoning_effort=req.reasoning_effort,
            thinking_param=req.thinking,
            default_enable_thinking=engine.default_enable_thinking,
        )
        emit_reasoning = req.include_reasoning if req.include_reasoning is not None else include_reasoning
        parse_tools = req.tool_choice != "none"

        prompt_str = engine.tokenizer.apply_chat_template(
            norm_messages,
            add_generation_prompt=True,
            enable_thinking=eff_thinking,
            tools=req.tools,
            tool_choice=req.tool_choice,
            default_enable_thinking=engine.default_enable_thinking,
        )

        max_toks = req.max_completion_tokens or req.max_tokens or 1024
        use_mtp = req.speculative_mtp if req.speculative_mtp is not None else req.speculative
        use_ngram = req.speculative_ngram if req.speculative_ngram is not None else req.speculative
        sampling = SamplingConfig(
            max_tokens=max_toks,
            temperature=req.temperature,
            top_p=req.top_p,
            top_k=req.top_k,
            min_p=req.min_p,
            repetition_penalty=req.repetition_penalty,
            stop_strings=_normalize_stops(req.stop),
            cache_prompt=req.cache_prompt,
            seed=req.seed,
            use_mtp=use_mtp,
            use_ngram=use_ngram,
            mtp_draft_tokens=req.mtp_draft_tokens,
            mtp_adaptive=req.mtp_adaptive,
            mtp_confidence_threshold=req.mtp_confidence_threshold,
        )

        req_id = f"chatcmpl-{uuid.uuid4().hex[:24]}"
        model_name = req.model or engine.model_id
        now = int(time.time())

        if not req.stream:
            parser = ReasoningAndToolStreamParser(
                prompt_str=prompt_str,
                tools=req.tools,
                parse_tool_calls=parse_tools,
                emit_reasoning=emit_reasoning,
            )
            stats = None
            async for ch in engine.astream_tokens(prompt_str, sampling):
                if ch.delta_text:
                    parser.feed(ch.delta_text)
                if ch.stats is not None:
                    stats = ch.stats
            parser.finalize()
            assert stats is not None

            clean_content = parser.full_content
            msg_payload: Dict[str, Any] = {
                "role": "assistant",
                "content": (None if (parser.has_tool_calls and not clean_content.strip()) else clean_content),
            }
            if emit_reasoning and parser.full_reasoning:
                msg_payload["reasoning_content"] = parser.full_reasoning
            if parser.has_tool_calls:
                msg_payload["tool_calls"] = [tc.to_openai_dict() for tc in parser.parsed_tool_calls]
                finish_reason: str = "tool_calls"
            else:
                finish_reason = stats.finish_reason

            return {
                "id": req_id,
                "object": "chat.completion",
                "created": now,
                "model": model_name,
                "choices": [
                    {
                        "index": 0,
                        "message": msg_payload,
                        "finish_reason": finish_reason,
                    }
                ],
                "usage": {
                    "prompt_tokens": stats.prompt_tokens,
                    "completion_tokens": stats.completion_tokens,
                    "total_tokens": stats.prompt_tokens + stats.completion_tokens,
                    "prompt_tokens_details": {"cached_tokens": stats.cached_prefix_tokens},
                },
                "timings": {
                    "prompt_per_second": round(stats.prompt_tok_per_s, 2),
                    "predicted_per_second": round(stats.decode_tok_per_s, 2),
                    "prefill_ms": round(stats.prefill_time_s * 1000, 2),
                    "decode_ms": round(stats.decode_time_s * 1000, 2),
                    "speculative_accepted": stats.speculative_accepted,
                    "speculative_drafted": stats.speculative_drafted,
                    "mtp_accepted": stats.mtp_accepted,
                    "mtp_drafted": stats.mtp_drafted,
                },
            }

        async def _sse_generator() -> AsyncIterator[str]:
            parser = ReasoningAndToolStreamParser(
                prompt_str=prompt_str,
                tools=req.tools,
                parse_tool_calls=parse_tools,
                emit_reasoning=emit_reasoning,
            )
            emitted_tool_idx = 0

            # Initial role chunk expected by OpenAI SDK / OpenCode / Cline / Continue
            role_chunk = {
                "id": req_id,
                "object": "chat.completion.chunk",
                "created": now,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"role": "assistant"}, "finish_reason": None}],
            }
            yield f"data: {json.dumps(role_chunk)}\n\n"

            def _format_emission(em: Any) -> List[str]:
                nonlocal emitted_tool_idx
                sse_Payloads: List[str] = []
                if em.reasoning_delta:
                    r_chunk = {
                        "id": req_id,
                        "object": "chat.completion.chunk",
                        "created": now,
                        "model": model_name,
                        "choices": [
                            {
                                "index": 0,
                                "delta": {"reasoning_content": em.reasoning_delta},
                                "finish_reason": None,
                            }
                        ],
                    }
                    sse_Payloads.append(f"data: {json.dumps(r_chunk, ensure_ascii=False)}\n\n")
                if em.content_delta:
                    c_chunk = {
                        "id": req_id,
                        "object": "chat.completion.chunk",
                        "created": now,
                        "model": model_name,
                        "choices": [
                            {
                                "index": 0,
                                "delta": {"content": em.content_delta},
                                "finish_reason": None,
                            }
                        ],
                    }
                    sse_Payloads.append(f"data: {json.dumps(c_chunk, ensure_ascii=False)}\n\n")
                if em.tool_calls:
                    tc_deltas = []
                    for tc in em.tool_calls:
                        tc_deltas.append(tc.to_openai_stream_delta(emitted_tool_idx))
                        emitted_tool_idx += 1
                    t_chunk = {
                        "id": req_id,
                        "object": "chat.completion.chunk",
                        "created": now,
                        "model": model_name,
                        "choices": [
                            {
                                "index": 0,
                                "delta": {"tool_calls": tc_deltas},
                                "finish_reason": None,
                            }
                        ],
                    }
                    sse_Payloads.append(f"data: {json.dumps(t_chunk, ensure_ascii=False)}\n\n")
                return sse_Payloads

            final_stats = None
            raw_finish_reason: str = "stop"
            try:
                async for ch in engine.astream_tokens(prompt_str, sampling):
                    if ch.delta_text:
                        for sse_str in _format_emission(parser.feed(ch.delta_text)):
                            yield sse_str
                    if ch.finish_reason is not None:
                        final_stats = ch.stats
                        raw_finish_reason = ch.finish_reason

                # Flush any remaining buffered content / auto-healed tool call at stream end
                for sse_str in _format_emission(parser.finalize()):
                    yield sse_str

                eff_finish = "tool_calls" if parser.has_tool_calls else raw_finish_reason
                end_chunk = {
                    "id": req_id,
                    "object": "chat.completion.chunk",
                    "created": now,
                    "model": model_name,
                    "choices": [{"index": 0, "delta": {}, "finish_reason": eff_finish}],
                }
                yield f"data: {json.dumps(end_chunk)}\n\n"
            except Exception as exc:
                import traceback
                traceback.print_exc()
                err_chunk = {
                    "id": req_id,
                    "object": "chat.completion.chunk",
                    "created": now,
                    "model": model_name,
                    "choices": [
                        {
                            "index": 0,
                            "delta": {"content": f"\n[PyGGUF Server Error: {type(exc).__name__}: {exc}]"},
                            "finish_reason": "stop",
                        }
                    ],
                }
                yield f"data: {json.dumps(err_chunk)}\n\n"
                yield "data: [DONE]\n\n"
                return

            if req.stream_options and req.stream_options.include_usage and final_stats is not None:
                usage_chunk = {
                    "id": req_id,
                    "object": "chat.completion.chunk",
                    "created": now,
                    "model": model_name,
                    "choices": [],
                        "usage": {
                            "prompt_tokens": final_stats.prompt_tokens,
                            "completion_tokens": final_stats.completion_tokens,
                            "total_tokens": final_stats.prompt_tokens + final_stats.completion_tokens,
                        },
                        "timings": {
                            "prompt_per_second": round(final_stats.prompt_tok_per_s, 2),
                            "predicted_per_second": round(final_stats.decode_tok_per_s, 2),
                            "prefill_ms": round(final_stats.prefill_time_s * 1000, 2),
                            "decode_ms": round(final_stats.decode_time_s * 1000, 2),
                            "speculative_accepted": final_stats.speculative_accepted,
                            "speculative_drafted": final_stats.speculative_drafted,
                            "mtp_accepted": final_stats.mtp_accepted,
                            "mtp_drafted": final_stats.mtp_drafted,
                        },
                    }
                yield f"data: {json.dumps(usage_chunk)}\n\n"

            yield "data: [DONE]\n\n"

        return StreamingResponse(_sse_generator(), media_type="text/event-stream")

    @app.post("/v1/completions", dependencies=[Depends(verify_auth)])
    async def completions(req: CompletionRequest) -> Any:
        use_mtp = req.speculative_mtp if req.speculative_mtp is not None else req.speculative
        use_ngram = req.speculative_ngram if req.speculative_ngram is not None else req.speculative
        sampling = SamplingConfig(
            max_tokens=req.max_tokens,
            temperature=req.temperature,
            top_p=req.top_p,
            top_k=req.top_k,
            min_p=req.min_p,
            repetition_penalty=req.repetition_penalty,
            stop_strings=_normalize_stops(req.stop),
            cache_prompt=req.cache_prompt,
            seed=req.seed,
            use_mtp=use_mtp,
            use_ngram=use_ngram,
            mtp_draft_tokens=req.mtp_draft_tokens,
            mtp_adaptive=req.mtp_adaptive,
            mtp_confidence_threshold=req.mtp_confidence_threshold,
        )
        req_id = f"cmpl-{uuid.uuid4().hex[:24]}"
        model_name = req.model or engine.model_id
        now = int(time.time())

        if not req.stream:
            stats = None
            async for ch in engine.astream_tokens(req.prompt, sampling):
                if ch.stats is not None:
                    stats = ch.stats
            assert stats is not None
            return {
                "id": req_id,
                "object": "text_completion",
                "created": now,
                "model": model_name,
                "choices": [{"index": 0, "text": stats.text, "finish_reason": stats.finish_reason}],
                "usage": {
                    "prompt_tokens": stats.prompt_tokens,
                    "completion_tokens": stats.completion_tokens,
                    "total_tokens": stats.prompt_tokens + stats.completion_tokens,
                },
            }

        async def _sse_cmpl() -> AsyncIterator[str]:
            async for ch in engine.astream_tokens(req.prompt, sampling):
                if ch.delta_text or ch.finish_reason is not None:
                    payload = {
                        "id": req_id,
                        "object": "text_completion",
                        "created": now,
                        "model": model_name,
                        "choices": [{"index": 0, "text": ch.delta_text, "finish_reason": ch.finish_reason}],
                    }
                    yield f"data: {json.dumps(payload)}\n\n"
            yield "data: [DONE]\n\n"

        return StreamingResponse(_sse_cmpl(), media_type="text/event-stream")

    return app


# ==============================================================================
# 3. Cloudflare Quick Tunnel Helper (Free Public HTTPS URL from Kaggle!)
# ==============================================================================

class CloudflareTunnel:
    """Spawns a free `trycloudflare.com` HTTPS tunnel pointing to the local OpenAI server."""

    def __init__(self, port: int = 8000, bin_path: str = "/tmp/cloudflared") -> None:
        self.port = port
        self.bin_path = bin_path
        self.public_url: Optional[str] = None
        self._proc: Optional[subprocess.Popen[str]] = None

    def start(self, timeout: float = 30.0) -> str:
        bin_file = shutil.which("cloudflared") or self.bin_path
        if not os.path.exists(bin_file):
            print("[Tunnel] Downloading cloudflared static Linux x64 binary...")
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                bin_file,
            )
            os.chmod(bin_file, 0o755)

        self._proc = subprocess.Popen(
            [bin_file, "tunnel", "--url", f"http://127.0.0.1:{self.port}", "--no-autoupdate"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
        found_event = threading.Event()

        def _reader() -> None:
            assert self._proc is not None and self._proc.stdout is not None
            for line in self._proc.stdout:
                m = url_pattern.search(line)
                if m and self.public_url is None:
                    self.public_url = m.group(0)
                    found_event.set()

        threading.Thread(target=_reader, daemon=True).start()
        if not found_event.wait(timeout=timeout) or not self.public_url:
            self.stop()
            raise TimeoutError("Timed out waiting for Cloudflare Quick Tunnel URL.")
        return self.public_url

    def stop(self) -> None:
        if self._proc is not None and self._proc.poll() is None:
            self._proc.terminate()
            try:
                self._proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self._proc.kill()
        self._proc = None


In [ ]:
%%writefile kaggle_serve.py
"""
kaggle_serve.py
===============
One-call Kaggle Notebook & CLI launcher for `pygguf_engine` (0% llama.cpp).

Features:
- Fast 1-2 Gbps GGUF download directly into `/kaggle/tmp` (~1.1 TB scratch disk)
- Loads Dense, Sparse MoE, & Hybrid Gated-DeltaNet GGUFs (`Qwen3.8-27B-UD-Q4_K_M`, `Qwen3.5`, `Qwen3-30B-A3B`, `Llama-3`, `Gemma-3`)
  onto Single T4 or Dual T4 (`cuda:0` + `cuda:1`, 30-32 GB VRAM)
- Starts FastAPI OpenAI-compatible API (`/v1/chat/completions`, `/v1/completions`, `/v1/models`)
- Optional free Cloudflare HTTPS Tunnel (`https://xxxx.trycloudflare.com/v1`) + API Key

Usage inside a Kaggle Notebook cell:
------------------------------------
    from kaggle_serve import download_gguf_kaggle, launch_kaggle_server

    gguf_path = download_gguf_kaggle(
        repo_id="bartowski/Qwen_Qwen3-8B-GGUF",
        filename="Qwen_Qwen3-8B-Q4_K_M.gguf",
    )

    handle = launch_kaggle_server(
        gguf_path=gguf_path,
        hf_tokenizer="Qwen/Qwen3-8B",   # Optional (uses embedded GGUF tokenizer if omitted!)
        max_context_len=65536,          # 64K context with INT8 KV cache
        kv_dtype="int8",                # "int4" unlocks 128K-256K+ context
        # Either the GGUF's own {arch}.rope.scaling.* metadata is used verbatim, or override here:
        # rope_scaling_type="yarn", rope_scaling_factor=4.0, rope_scaling_original_context_length=16384,
        enable_tunnel=True,             # Exposes public https://....trycloudflare.com/v1
        api_key="kaggle-secret-key",
        background=True,                # Keeps notebook cell non-blocking!
    )
"""

from __future__ import annotations

import argparse
import gc
import os
import subprocess
import sys
import threading
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional, Sequence, Union

import torch
import uvicorn

from pygguf_engine import CloudflareTunnel, PyGGUFEngine, create_openai_app


def download_gguf_kaggle(
    repo_id: str,
    filename: str,
    dest_dir: Union[str, Path] = "/kaggle/tmp/gguf_models",
    hf_token: Optional[str] = None,
) -> Path:
    """
    Download a `.gguf` file into `/kaggle/tmp` (avoiding `/kaggle/working`'s 20GB cap)
    using `hf_transfer` to saturate Kaggle's 1-2 Gbps Google Cloud network link.
    """
    dest = Path(dest_dir) if Path("/kaggle").exists() else Path("./models")
    dest.mkdir(parents=True, exist_ok=True)
    target = dest / filename
    if target.exists() and target.stat().st_size > 0:
        print(f"[Kaggle] Using cached GGUF file: {target}")
        return target

    try:
        import huggingface_hub  # noqa: F401
    except ImportError:
        subprocess.check_call(["pip", "install", "-q", "huggingface_hub[hf_transfer]"])

    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    from huggingface_hub import hf_hub_download

    print(f"[Kaggle] Downloading {repo_id}/{filename} -> {dest} ...")
    t0 = time.time()
    out_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=str(dest),
        token=hf_token or os.environ.get("HF_TOKEN"),
    )
    dt = max(time.time() - t0, 1e-3)
    gib = Path(out_path).stat().st_size / (1024**3)
    print(f"[Kaggle] Downloaded {gib:.2f} GiB in {dt:.1f}s ({gib * 8 / dt:.2f} Gbps)")
    return Path(out_path)


@dataclass
class ServerHandle:
    engine: Optional[PyGGUFEngine]
    local_base_url: str
    public_base_url: Optional[str]
    api_key: Optional[str]
    uvicorn_server: Optional[uvicorn.Server]
    tunnel: Optional[CloudflareTunnel] = None
    thread: Optional[threading.Thread] = None

    def stop(self, timeout: float = 30.0) -> None:
        """Stop a background server and release owned model references.

        Active requests must finish before the server exits. On timeout, keep the
        model alive and raise rather than start a second model on top of it.
        Other user-held references to the engine can still keep VRAM allocated.
        """
        if timeout < 0:
            raise ValueError("timeout must be non-negative")
        server = self.uvicorn_server
        if server is not None:
            server.should_exit = True
        if self.tunnel is not None:
            self.tunnel.stop()
            self.tunnel = None
        thread = self.thread
        if thread is not None and thread.is_alive():
            if thread is threading.current_thread():
                raise RuntimeError("Cannot wait for the server from its own thread")
            thread.join(timeout=timeout)
            if thread.is_alive():
                raise RuntimeError(
                    f"Server is still shutting down after {timeout:g}s; finish or cancel active requests "
                    "and call stop() again before launching another model."
                )
        self.thread = None
        self.uvicorn_server = None
        self.engine = None
        # Uvicorn's config retains the app and its closure over the engine.
        # Drop local references as well before collecting cyclic objects.
        del server, thread
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def launch_kaggle_server(
    gguf_path: Union[str, Path],
    *,
    hf_tokenizer: Optional[str] = None,
    chat_template: Optional[str] = None,
    chat_template_mode: Literal["auto", "gguf", "qwen35_xml", "qwen3_json"] = "auto",
    enable_thinking: bool = True,
    include_reasoning: bool = True,
    model_alias: Optional[str] = None,
    max_context_len: int = 32768,
    kv_dtype: Literal["int4", "int8", "fp16"] = "int8",
    devices: Optional[Sequence[str]] = None,
    parallel_mode: Literal["auto", "tp", "pp"] = "auto",
    cpu_threads: Optional[int] = None,
    enable_cuda_graph: bool = True,
    speculative_mtp: bool = True,
    mtp_draft_tokens: Optional[int] = None,
    mtp_adaptive: bool = True,
    mtp_max_draft_tokens: int = 3,
    mtp_confidence_threshold: float = 0.55,
    mtp_kv_window: Optional[int] = 4096,
    speculative_ngram: bool = True,
    rope_scaling_type: Optional[Literal["none", "linear", "ntk-aware", "yarn"]] = None,
    rope_scaling_factor: Optional[float] = None,
    rope_scaling_original_context_length: Optional[int] = None,
    rope_yarn_beta_fast: Optional[float] = None,
    rope_yarn_beta_slow: Optional[float] = None,
    rope_yarn_ext_factor: Optional[float] = None,
    rope_yarn_attn_factor: Optional[float] = None,
    rope_yarn_log_mul: Optional[float] = None,
    host: str = "0.0.0.0",
    port: int = 8000,
    api_key: Optional[str] = None,
    enable_tunnel: bool = True,
    background: bool = True,
) -> ServerHandle:
    """Load the GGUF model into `PyGGUFEngine`, launch the OpenAI server, and optionally open Cloudflare Tunnel."""
    if cpu_threads is not None:
        if isinstance(cpu_threads, bool) or not isinstance(cpu_threads, int) or cpu_threads < 1:
            raise ValueError("cpu_threads must be a positive integer or None")
        # PyTorch intra-op setting is process-wide; set it before model construction.
        torch.set_num_threads(cpu_threads)
        print(f"[PyGGUF] PyTorch CPU intra-op threads: {torch.get_num_threads()}")

    # Reclaim any orphaned GPU VRAM from earlier notebook cell exceptions/re-runs before loading weights
    for attr in ("last_traceback", "last_value", "last_type"):
        if hasattr(sys, attr):
            setattr(sys, attr, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    engine = PyGGUFEngine(
        gguf_path=gguf_path,
        hf_tokenizer=hf_tokenizer,
        chat_template=chat_template,
        chat_template_mode=chat_template_mode,
        default_enable_thinking=enable_thinking,
        max_context_len=max_context_len,
        kv_dtype=kv_dtype,
        devices=devices,
        parallel_mode=parallel_mode,
        enable_cuda_graph=enable_cuda_graph,
        speculative_mtp=speculative_mtp,
        mtp_draft_tokens=mtp_draft_tokens,
        mtp_adaptive=mtp_adaptive,
        mtp_max_draft_tokens=mtp_max_draft_tokens,
        mtp_confidence_threshold=mtp_confidence_threshold,
        mtp_kv_window=mtp_kv_window,
        speculative_ngram=speculative_ngram,
        model_alias=model_alias,
        rope_scaling_type=rope_scaling_type,
        rope_scaling_factor=rope_scaling_factor,
        rope_scaling_original_context_length=rope_scaling_original_context_length,
        rope_yarn_beta_fast=rope_yarn_beta_fast,
        rope_yarn_beta_slow=rope_yarn_beta_slow,
        rope_yarn_ext_factor=rope_yarn_ext_factor,
        rope_yarn_attn_factor=rope_yarn_attn_factor,
        rope_yarn_log_mul=rope_yarn_log_mul,
    )
    app = create_openai_app(engine, api_key=api_key, include_reasoning=include_reasoning)
    uv_cfg = uvicorn.Config(app, host=host, port=port, log_level="warning")
    server = uvicorn.Server(uv_cfg)

    local_v1 = f"http://127.0.0.1:{port}/v1"
    tunnel: Optional[CloudflareTunnel] = None
    public_v1: Optional[str] = None

    if not background:
        if enable_tunnel:
            tunnel = CloudflareTunnel(port=port)
            pub_root = tunnel.start()
            public_v1 = f"{pub_root}/v1"
            _print_banner(engine.model_id, local_v1, public_v1, api_key)
        else:
            _print_banner(engine.model_id, local_v1, None, api_key)
        server.run()
        return ServerHandle(engine, local_v1, public_v1, api_key, server, tunnel, None)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    # Wait until local FastAPI server responds on /health
    deadline = time.time() + 30.0
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=1.0) as resp:
                if resp.status == 200:
                    break
        except Exception:
            time.sleep(0.2)

    if enable_tunnel:
        tunnel = CloudflareTunnel(port=port)
        pub_root = tunnel.start()
        public_v1 = f"{pub_root}/v1"

    _print_banner(engine.model_id, local_v1, public_v1, api_key)
    return ServerHandle(engine, local_v1, public_v1, api_key, server, tunnel, thread)


def _print_banner(
    model_id: str,
    local_v1: str,
    public_v1: Optional[str],
    api_key: Optional[str],
) -> None:
    shown_key = api_key or "not-required"
    target_url = public_v1 or local_v1
    print("\n" + "=" * 72)
    print(" PyGGUF OpenAI Server is LIVE (100% Python/PyTorch/Triton, 0% llama.cpp)")
    print("=" * 72)
    print(f"  Model ID        : {model_id}")
    print(f"  Local Base URL  : {local_v1}")
    if public_v1:
        print(f"  Public HTTPS URL: {public_v1}")
    print(f"  API Key         : {shown_key}")
    print("-" * 72)
    print("  Python OpenAI SDK snippet:")
    print("    from openai import OpenAI")
    print(f'    client = OpenAI(base_url="{target_url}", api_key="{shown_key}")')
    print(
        f'    resp = client.chat.completions.create(model="{model_id}", '
        'messages=[{"role": "user", "content": "Hello!"}], stream=True)'
    )
    print("=" * 72 + "\n")


def main() -> None:
    parser = argparse.ArgumentParser(description="Serve GGUF models with PyGGUF OpenAI API (no llama.cpp)")
    parser.add_argument("--gguf", type=str, default=None, help="Local path to .gguf file")
    parser.add_argument("--repo", type=str, default=None, help="HuggingFace repo_id to download GGUF from")
    parser.add_argument("--file", type=str, default=None, help="GGUF filename inside --repo")
    parser.add_argument("--tokenizer", type=str, default=None, help="Optional HuggingFace base tokenizer repo")
    parser.add_argument("--alias", type=str, default=None, help="Served OpenAI model ID")
    parser.add_argument("--ctx", type=int, default=32768, help="Max context window (e.g. 32768, 131072, 262144)")
    parser.add_argument("--kv-dtype", choices=["int4", "int8", "fp16"], default="int8", help="KV cache precision (int4 unlocks 256K+ ctx!)")
    parser.add_argument("--port", type=int, default=8000, help="HTTP server port")
    parser.add_argument("--api-key", type=str, default=os.environ.get("OPENAI_API_KEY"), help="Optional Bearer API key")
    parser.add_argument("--tunnel", action="store_true", help="Expose public https://*.trycloudflare.com/v1 URL")
    parser.add_argument("--no-speculative", action="store_true", help="Disable BOTH N-gram and draft-mtp speculative decoding")
    parser.add_argument("--no-mtp", action="store_true", help="Disable embedded GGUF draft-mtp head (0 GB MTP VRAM loaded)")
    parser.add_argument("--mtp-draft-tokens", type=int, default=None, help="Pin exact MTP draft tokens per step (1..8; omit for adaptive k=1..3)")
    parser.add_argument("--no-adaptive-mtp", action="store_true", help="Disable confidence-gated adaptive MTP depth (k=1..3)")
    parser.add_argument("--mtp-max-draft-tokens", type=int, default=3, help="Max adaptive MTP draft depth when confident (default: 3)")
    parser.add_argument("--mtp-conf-threshold", type=float, default=0.55, help="Softmax confidence threshold for adaptive MTP chaining")
    parser.add_argument("--mtp-kv-window", type=int, default=4096, help="Sliding-window ring KV size for MTP head (0 = unbounded)")
    parser.add_argument("--no-ngram", action="store_true", help="Disable N-gram self-speculative decoding")
    parser.add_argument("--chat-template", type=str, default=None, help="Optional path to custom .jinja template file")
    parser.add_argument(
        "--chat-template-mode",
        choices=["auto", "gguf", "qwen35_xml", "qwen3_json"],
        default="auto",
        help="Agentic Jinja template mode (default: auto)",
    )
    parser.add_argument("--no-thinking", action="store_true", help="Disable <think> reasoning by default")
    parser.add_argument("--hide-reasoning", action="store_true", help="Omit reasoning_content from API responses")
    parser.add_argument(
        "--parallel-mode",
        choices=["auto", "tp", "pp"],
        default="auto",
        help="Multi-GPU execution mode: auto (TP=2 for Dense/Hybrid, PP for MoE), tp, or pp",
    )
    parser.add_argument("--cpu-threads", type=int, default=None, help="PyTorch CPU intra-op threads (process-wide; default: unchanged)")
    parser.add_argument("--no-cuda-graph", action="store_true", help="Disable CUDAGraph capture for M=1 decode")
    parser.add_argument("--rope-scaling-type", choices=["none", "linear", "ntk-aware", "yarn"], default=None,
                        help="Override RoPE scaling from GGUF metadata (extend context beyond trained length)")
    parser.add_argument("--rope-scaling-factor", type=float, default=None, help="RoPE scaling factor, e.g. 2.0/4.0/8.0 for YaRN")
    parser.add_argument("--rope-scaling-orig-ctx", type=int, default=None, help="Original (pre-extension) context length for YaRN correction dims, e.g. 32768")
    parser.add_argument("--rope-yarn-beta-fast", type=float, default=None, help="YaRN beta_fast (default: 32.0)")
    parser.add_argument("--rope-yarn-beta-slow", type=float, default=None, help="YaRN beta_slow (default: 1.0)")
    parser.add_argument("--rope-yarn-ext-factor", type=float, default=None, help="YaRN ext_factor for the mscale attention factor (default: 1.0)")
    parser.add_argument("--rope-yarn-attn-factor", type=float, default=None, help="Override YaRN attention (mscale) factor explicitly (default: auto = 1 + log_mul * ext_factor * log(factor))")
    parser.add_argument("--rope-yarn-log-mul", type=float, default=None, help="YaRN mscale log-multiplier (default: 0.1)")
    args = parser.parse_args()

    if args.gguf:
        gguf_path = Path(args.gguf)
    elif args.repo and args.file:
        gguf_path = download_gguf_kaggle(args.repo, args.file)
    else:
        parser.error("Provide either `--gguf /path/to/model.gguf` OR `--repo <hf_repo> --file <name.gguf>`.")

    launch_kaggle_server(
        gguf_path=gguf_path,
        hf_tokenizer=args.tokenizer,
        chat_template=args.chat_template,
        chat_template_mode=args.chat_template_mode,
        enable_thinking=not args.no_thinking,
        include_reasoning=not args.hide_reasoning,
        model_alias=args.alias,
        max_context_len=args.ctx,
        kv_dtype=args.kv_dtype,
        parallel_mode=args.parallel_mode,
        cpu_threads=args.cpu_threads,
        enable_cuda_graph=not args.no_cuda_graph,
        speculative_mtp=(not args.no_mtp) and (not args.no_speculative),
        mtp_draft_tokens=args.mtp_draft_tokens,
        mtp_adaptive=not args.no_adaptive_mtp,
        mtp_max_draft_tokens=args.mtp_max_draft_tokens,
        mtp_confidence_threshold=args.mtp_conf_threshold,
        mtp_kv_window=args.mtp_kv_window,
        speculative_ngram=(not args.no_ngram) and (not args.no_speculative),
        rope_scaling_type=args.rope_scaling_type,
        rope_scaling_factor=args.rope_scaling_factor,
        rope_scaling_original_context_length=args.rope_scaling_orig_ctx,
        rope_yarn_beta_fast=args.rope_yarn_beta_fast,
        rope_yarn_beta_slow=args.rope_yarn_beta_slow,
        rope_yarn_ext_factor=args.rope_yarn_ext_factor,
        rope_yarn_attn_factor=args.rope_yarn_attn_factor,
        rope_yarn_log_mul=args.rope_yarn_log_mul,
        port=args.port,
        api_key=args.api_key,
        enable_tunnel=args.tunnel,
        background=False,
    )


if __name__ == "__main__":
    main()


## 2. Download Any Hybrid (`Qwen3.8-27B` / `Qwen3.5`), Dense, or Sparse MoE `.gguf` into `/kaggle/tmp`
Uses `/kaggle/tmp` (~1.1 TB disk) + `hf_transfer` (1–2 Gbps) so `/kaggle/working`'s 20 GB limit is never touched.

In [ ]:
import sys
from pathlib import Path

# Hot-reload pygguf_engine & kaggle_serve if re-running inside an active Kaggle session
for _mod in [m for m in list(sys.modules) if m == "kaggle_serve" or m.startswith("pygguf_engine")]:
    sys.modules.pop(_mod, None)

from kaggle_serve import download_gguf_kaggle, launch_kaggle_server

# --- Preset A: Dense GGUF (Qwen3-8B-Q4_K_M with TP=2 across Kaggle Dual T4 GPUs) ---
REPO_ID = "Qwen/Qwen3-8B-GGUF"
FILENAME = "Qwen3-8B-Q4_K_M.gguf"
TOKENIZER_REPO = None  # None = uses fast Rust tokenizer + hardened Agentic Jinja2 template!

# --- Preset B: Hybrid Gated-DeltaNet + Gated Attention (27B Unsloth Dynamic Q4_K_M with TP=2 on Dual T4!) ---
# REPO_ID = "unsloth/Qwen3.8-27B-GGUF"          # (or point gguf_path directly to /kaggle/tmp/...)
# FILENAME = "Qwen3.8-27B-UD-Q4_K_M.gguf"

# --- Preset C: Sparse MoE (30B-35B quality, only 3B active weights read per token -> highest tok/s!) ---
# REPO_ID = "Qwen/Qwen3-30B-A3B-GGUF"
# FILENAME = "Qwen3-30B-A3B-Q4_K_M.gguf"

gguf_path = download_gguf_kaggle(repo_id=REPO_ID, filename=FILENAME)


## 3. Launch Turbo 256K-Ready OpenAI Server + Adaptive `draft-mtp` + Cloudflare HTTPS Tunnel
- **`kv_dtype="int4"` (Default for Long Context)**: Cuts KV cache VRAM & Flash-Decode bandwidth by **~75% vs FP16 / ~50% vs INT8**, unlocking **128K–256K+ context** on Dual T4 (`"int8"` and `"fp16"` also supported).
- **`speculative_mtp=True` + `mtp_adaptive=True` (`k = 1..3`) + `mtp_kv_window=4096`**: Loads embedded GGUF `blk.{n_layers}.nextn.*` with a bounded `4096`-token ring KV cache (`~4 MiB` VRAM even at 256K!) and chains up to `3` draft tokens when GPU softmax confidence `>= 0.55`. Set **`speculative_mtp=False`** to skip loading `blk.{n_layers}` (**saves ~0.5 GB VRAM**), or toggle live per request via `extra_body={"speculative_mtp": False}`!
- **RoPE context extension**: checkpoints trained for a shorter window can be pushed to `max_context_len` with `rope_scaling_type="yarn", rope_scaling_factor=4.0, rope_scaling_original_context_length=32768` (YaRN mscale attention factor applied automatically; `linear` and `ntk-aware` also supported). GGUF `{arch}.rope.scaling.*` metadata is used automatically when present. Extension never re-trains weights — quality is checkpoint-dependent.
- **`parallel_mode="auto"`**: Automatically enables **2-GPU Tensor Parallelism (`TP=2`, 640 GB/s combined memory bandwidth)** on Dual T4 for Dense & Hybrid (`Qwen3.8-27B`) models, or **Pipeline Parallelism (`PP`)** for Sparse MoE models.
- **`enable_thinking=True`**: Extracts `<think>...</think>` into `reasoning_content` (**zero `<think></think>` tags in `content`!**). Set **`enable_thinking=False`** for **3x–5x faster end-to-end replies** when chain-of-thought isn't needed.

In [ ]:
import gc
import torch

# Cleanly stop any previous server_handle if re-running this cell in the same Kaggle session
if "server_handle" in globals() and server_handle is not None:
    # Wait for active requests to finish; do not launch a second model on timeout.
    server_handle.stop(timeout=60.0)
    server_handle = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

API_KEY = "kaggle-pygguf-key"   # Set to None to disable Bearer auth
ENABLE_PUBLIC_TUNNEL = True      # Exposes https://xxxx.trycloudflare.com/v1 for OpenCode / Claude Code / Cline / Cursor

server_handle = launch_kaggle_server(
    gguf_path=gguf_path,
    hf_tokenizer=TOKENIZER_REPO,
    chat_template=None,          # Optional path to a custom .jinja file (None = built-in hardened template)
    chat_template_mode="auto",   # "auto" | "qwen35_xml" | "qwen3_json" | "gguf"
    enable_thinking=True,        # Set False for 3x-5x faster turn completion without <think> CoT!
    include_reasoning=True,      # Set False to completely omit reasoning_content from JSON/SSE responses
    cpu_threads=4,              # PyTorch CPU intra-op threads; benchmark 2, 4 and 8
    parallel_mode="auto",        # "auto" = TP=2 (640 GB/s) on Dual-T4 Dense/Hybrid, PP on MoE | "tp" | "pp"
    # --- Dual-T4 measured (temp 0.6, 128 new tokens): keep speculation ON! ---
    # ~9.1 tok/s with MTP k=2 (75-77% acceptance, ~2.3 tokens per verify step) vs ~5.5 tok/s spec-off
    # (PP stage CUDA graphs replay only M=1 steps, and T4 kernels are the bottleneck there, not dispatch).
    # Try extra_body={"mtp_draft_tokens": 3} (see "5b. MTP depth sweep"). llama.cpp: ~20 tok/s (TP).
    enable_cuda_graph=True,      # Captures M=1 CUDA Graphs automatically on PP / single-GPU runs
    speculative_mtp=True,        # Native GGUF `blk.{L}.nextn.*` draft-mtp head (set False to save ~0.5 GB VRAM!)
    mtp_draft_tokens=None,       # None = Confidence-Gated Adaptive k=1..3 (or pass 1..8 to pin fixed k)
    mtp_adaptive=True,           # Chains k=2..3 MTP tokens when cumulative softmax confidence >= 0.55!
    mtp_kv_window=4096,          # Bounded 4096-token Ring KV for MTP (~4 MiB VRAM even at 256K trunk context!)
    speculative_ngram=True,      # Hybrid zero-VRAM N-gram suffix drafter + draft-mtp across ALL temperatures!
    max_context_len=131072,      # 128K context out of the box (set 262144 for full 256K — ~2.1 GiB KV in int4!)
    kv_dtype="int4",             # "int4" (~75% VRAM & BW savings, 256K-ready!) | "int8" | "fp16"
    # RoPE context extension (only needed when the checkpoint's TRAINED ctx is shorter than max_context_len):
    # GGUF {arch}.rope.scaling.* metadata is honored automatically (llama.cpp-compatible keys), otherwise override:
    # rope_scaling_type="yarn", rope_scaling_factor=4.0, rope_scaling_original_context_length=32768,
    port=8000,
    api_key=API_KEY,
    enable_tunnel=ENABLE_PUBLIC_TUNNEL,
    background=True,
)


## 4. Test Streaming Speed, Clean `reasoning_content`, & Native Multi-Turn Tool Calling

In [ ]:
import json
import time
from openai import OpenAI

client = OpenAI(
    base_url=server_handle.local_base_url,  # Use server_handle.public_base_url in external harnesses!
    api_key=API_KEY,
)

print("Models:", [m.id for m in client.models.list().data])
print("VRAM, Parallelism & MTP Report:", server_handle.engine.vram_report())

# --- Test A: Streaming Chat (notice zero <think></think> in delta.content!) ---
print("\n=== 1. Streaming Chat (TP=2 + Native draft-mtp) ===")
t0 = time.perf_counter()
stream = client.chat.completions.create(
    model=server_handle.engine.model_id,
    messages=[
        {"role": "developer", "content": "You are a concise AI systems assistant."},
        {"role": "user", "content": "Give 2 short bullet points on why TP=2 + draft-mtp accelerates Dual T4 GPUs."},
    ],
    temperature=0.6,
    max_tokens=256,
    stream=True,
    stream_options={"include_usage": True},
    # Optional live per-request overrides:
    # extra_body={"speculative_mtp": False}                  # disable MTP for this call only
    # extra_body={"speculative_mtp": True, "mtp_draft_tokens": 2}
)
printed_think_header = False
printed_ans_header = False
comp_toks = 0
for chunk in stream:
    if getattr(chunk, "usage", None) is not None:
        comp_toks = chunk.usage.completion_tokens
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    rc = getattr(delta, "reasoning_content", None)
    if rc:
        if not printed_think_header:
            print("[Reasoning]: ", end="", flush=True)
            printed_think_header = True
        print(rc, end="", flush=True)
    if delta.content:
        if not printed_ans_header:
            print("\n\n[Answer]:\n", end="", flush=True)
            printed_ans_header = True
        print(delta.content, end="", flush=True)
elapsed = time.perf_counter() - t0
print(f"\n(Generated {comp_toks} tokens in {elapsed:.2f}s -> {comp_toks / max(elapsed, 1e-6):.1f} e2e tok/s)")

# --- Test B: Native OpenAI Multi-Turn Tool Calling ---
print("\n=== 2. Native OpenAI Tool Calling ===")
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_gpu_cluster_status",
            "description": "Query live GPU VRAM and temperature metrics for a cluster node.",
            "parameters": {
                "type": "object",
                "properties": {
                    "node_name": {"type": "string", "description": "Cluster node identifier"},
                    "include_temps": {"type": "boolean", "description": "Include Celsius temperatures"},
                },
                "required": ["node_name"],
            },
        },
    }
]

tool_msgs = [{"role": "user", "content": "Check GPU cluster status on node kaggle-t4x2 including temperatures."}]
r1 = client.chat.completions.create(
    model=server_handle.engine.model_id,
    messages=tool_msgs,
    tools=tools,
    temperature=0.0,
    max_tokens=384,
)
msg1 = r1.choices[0].message
print("Finish Reason:", r1.choices[0].finish_reason, "| Timings (incl. MTP):", r1.model_extra.get("timings"))
print("Parsed Tool Calls:", msg1.tool_calls)

if msg1.tool_calls:
    tc = msg1.tool_calls[0]
    args = json.loads(tc.function.arguments)
    tool_msgs.append(msg1.model_dump(exclude_none=True))
    tool_msgs.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps({"node": args.get("node_name"), "gpus": "2x Tesla T4", "free_vram_gib": 8.4, "temp_c": 58}),
        }
    )
    r2 = client.chat.completions.create(
        model=server_handle.engine.model_id,
        messages=tool_msgs,
        tools=tools,
        temperature=0.2,
        max_tokens=256,
    )
    print("Final Tool-Grounded Answer:", r2.choices[0].message.content)


## 5. Throughput A/B Benchmark (single cell, all configs)

Runs the same prompt through every speculative/parallelism-related knob and prints `timings` (server-side per-request stats), so you get one JSON blob + one table to paste back.

- **Cold cache**: `cache_prompt=False` on every call so prefix-cache hits never skew prefill.
- **Fixed `seed=42`** for all sampling-based cases (temperature > 0) so acceptance rates are comparable.
- **`stream=False`**: the sync response carries the full `timings` dict (`predicted_per_second`, `prefill_ms`, `decode_ms`, `speculative_accepted/drafted`, `mtp_accepted/drafted`).
- Cases: `1` = engine-as-launched default, `2` = speculation OFF (M=1 CUDA-graph replay), `3` = n-gram only, `4` = MTP only, `5` = greedy spec OFF, `6` = greedy engine default, `7` = spec OFF + thinking disabled.
- Greedy (`temperature=0.0`) exercises the fast single-sync batched verify path; temperature 0.6 exercises `verify_or_sample_draft_token_gpu`.
- Measured result (dual T4, Qwen3.8-27B UD-Q4_K_M): **speculation ON wins decisively** — MTP-only 9.14 pred tok/s > engine default 8.69 > greedy spec-on 7.92 > spec-off ~5.5. n-gram is acceptance-neutral at temperature 0.6 (7/26) but free (zero VRAM).
- Expected runtime ~15-25 min. Set `MAX_TOKENS = 256` for direct comparison with the 52.44s baseline from the streaming test.
- **Paste the printed per-case JSON lines + final table back to the assistant for analysis.**


In [ ]:
import json
import time
import urllib.request

BASE_URL = getattr(server_handle, "local_base_url", "http://127.0.0.1:8000/v1/").rstrip("/")
MODEL_ID = server_handle.engine.model_id
AUTH = {"Content-Type": "application/json"}
_ak = None
try:
    _ak = API_KEY  # type: ignore[name-defined]
except NameError:
    pass
if _ak:
    AUTH["Authorization"] = f"Bearer {_ak}"

MAX_TOKENS = 128   # 256 to match the streaming-test baseline exactly
PROMPT_TEXT = "Give 2 short bullet points on why speculative decoding accelerates LLM inference."

def _post(payload, timeout=7200):
    req = urllib.request.Request(f"{BASE_URL}/chat/completions",
                                 data=json.dumps(payload).encode("utf-8"),
                                 headers=AUTH, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def run_case(name, **overrides):
    payload = dict(
        model=MODEL_ID,
        messages=[{"role": "user", "content": PROMPT_TEXT}],
        max_tokens=MAX_TOKENS,
        temperature=0.6,
        cache_prompt=False,
        seed=42,
        stream=False,
    )
    payload.update(overrides)
    t0 = time.perf_counter()
    resp = _post(payload)
    wall = time.perf_counter() - t0
    u = resp.get("usage", {}) or {}
    tim = resp.get("timings", {}) or {}
    ct = int(u.get("completion_tokens", 0))
    row = {
        "case": name,
        "wall_s": round(wall, 2),
        "completion_tokens": ct,
        "e2e_tok_per_s": round(ct / max(wall, 1e-6), 2),
        "predicted_per_second": tim.get("predicted_per_second"),
        "prefill_ms": tim.get("prefill_ms"),
        "decode_ms": tim.get("decode_ms"),
        "cached_prefix_tokens": (u.get("prompt_tokens_details") or {}).get("cached_tokens"),
        "spec_accepted": tim.get("speculative_accepted"),
        "spec_drafted": tim.get("speculative_drafted"),
        "mtp_accepted": tim.get("mtp_accepted"),
        "mtp_drafted": tim.get("mtp_drafted"),
    }
    print("RESULT " + json.dumps(row))
    return row

results = []
results.append(run_case("1_engine_default(temp0.6)"))
results.append(run_case("2_spec_off_pp_graphs(temp0.6)", speculative_mtp=False, speculative_ngram=False))
results.append(run_case("3_ngram_only(temp0.6)", speculative_mtp=False, speculative_ngram=True))
results.append(run_case("4_mtp_only(temp0.6)", speculative_mtp=True, speculative_ngram=False))
results.append(run_case("5_greedy_spec_off", temperature=0.0, speculative_mtp=False, speculative_ngram=False))
results.append(run_case("6_greedy_engine_default", temperature=0.0))
results.append(run_case("7_spec_off_no_thinking", speculative_mtp=False, speculative_ngram=False, enable_thinking=False))

print()
print(f"{'case':34s} {'wall_s':>7s} {'toks':>5s} {'e2e_tok/s':>9s} {'pred_tok/s':>10s} {'mtp_acc/dft':>11s}")
for r in results:
    mtp = f"{r['mtp_accepted']}/{r['mtp_drafted']}" if r['mtp_accepted'] is not None else "-"
    print(f"{r['case']:34s} {r['wall_s']:7.2f} {r['completion_tokens']:5d} {r['e2e_tok_per_s']:9.2f} {str(r['predicted_per_second']):>10s} {mtp:>11s}")
print()
print("Paste the RESULT lines and this table back to the assistant.")


## 5b. MTP depth sweep (per-request, no relaunch)

Same methodology as the A/B cell above: cold prefix cache (`cache_prompt=False`), fixed `seed=42`, `stream=False` for server-side `timings`. The engine launch config is printed first so results are self-describing (useful when A/B-ing `parallel_mode='tp'` vs `'pp'` after a relaunch too).

- `8_mtp_k1` — single draft token per step
- `9_mtp_k3` — three draft tokens per step (acceptance ~0.75 ⇒ expect ~3.x tokens per verify step if per-step cost stays near weight-bound)
- `10_mtp_adaptive_0.55` — adaptive chaining, engine default confidence threshold 0.55 (max depth 3)
- `11_mtp_adaptive_0.30` — adaptive chaining with a looser threshold (chains deeper on marginal confidence)

To also A/B **TP vs PP with speculation on**: relaunch with `parallel_mode="tp"` and re-run this cell (verify steps are weight-read-bound, so TP should win by the bandwidth delta).

**Paste the RESULT lines + table back to the assistant.**


In [ ]:
import json
import time
import urllib.request

BASE_URL = getattr(server_handle, "local_base_url", "http://127.0.0.1:8000/v1/").rstrip("/")
MODEL_ID = server_handle.engine.model_id
AUTH = {"Content-Type": "application/json"}
_ak = None
try:
    _ak = API_KEY  # type: ignore[name-defined]
except NameError:
    pass
if _ak:
    AUTH["Authorization"] = f"Bearer {_ak}"

MAX_TOKENS = 128
PROMPT_TEXT = "Give 2 short bullet points on why speculative decoding accelerates LLM inference."

try:
    _vr = server_handle.engine.vram_report()
    keep = {k: _vr[k] for k in ("parallel_mode", "cuda_graph", "mtp_draft_tokens_default",
                                "mtp_adaptive_default", "mtp_max_draft_tokens",
                                "mtp_confidence_threshold", "speculative_ngram_default", "kv_dtype") if k in _vr}
    print("ENGINE_CONFIG " + json.dumps(keep))
except Exception as exc:  # never block the benchmark on telemetry
    print("ENGINE_CONFIG unavailable:", exc)

def _post(payload, timeout=7200):
    req = urllib.request.Request(f"{BASE_URL}/chat/completions",
                                 data=json.dumps(payload).encode("utf-8"),
                                 headers=AUTH, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def run_case(name, **overrides):
    payload = dict(
        model=MODEL_ID,
        messages=[{"role": "user", "content": PROMPT_TEXT}],
        max_tokens=MAX_TOKENS,
        temperature=0.6,
        cache_prompt=False,
        seed=42,
        stream=False,
    )
    payload.update(overrides)
    t0 = time.perf_counter()
    resp = _post(payload)
    wall = time.perf_counter() - t0
    u = resp.get("usage", {}) or {}
    tim = resp.get("timings", {}) or {}
    ct = int(u.get("completion_tokens", 0))
    row = {
        "case": name,
        "wall_s": round(wall, 2),
        "completion_tokens": ct,
        "e2e_tok_per_s": round(ct / max(wall, 1e-6), 2),
        "predicted_per_second": tim.get("predicted_per_second"),
        "prefill_ms": tim.get("prefill_ms"),
        "decode_ms": tim.get("decode_ms"),
        "spec_accepted": tim.get("speculative_accepted"),
        "spec_drafted": tim.get("speculative_drafted"),
        "mtp_accepted": tim.get("mtp_accepted"),
        "mtp_drafted": tim.get("mtp_drafted"),
    }
    print("RESULT " + json.dumps(row))
    return row

results = []
results.append(run_case("8_mtp_k1", mtp_draft_tokens=1))
results.append(run_case("9_mtp_k3", mtp_draft_tokens=3))
results.append(run_case("10_mtp_adaptive_0.55", mtp_adaptive=True))
results.append(run_case("11_mtp_adaptive_0.30", mtp_adaptive=True, mtp_confidence_threshold=0.30))

print()
print(f"{'case':28s} {'wall_s':>7s} {'toks':>5s} {'e2e_tok/s':>9s} {'pred_tok/s':>10s} {'mtp_acc/dft':>11s}")
for r in results:
    mtp = f"{r['mtp_accepted']}/{r['mtp_drafted']}" if r['mtp_accepted'] is not None else "-"
    print(f"{r['case']:28s} {r['wall_s']:7.2f} {r['completion_tokens']:5d} {r['e2e_tok_per_s']:9.2f} {str(r['predicted_per_second']):>10s} {mtp:>11s}")
print()
print("Paste the ENGINE_CONFIG line, RESULT lines and this table back to the assistant.")


In [ ]:
# Optional: Keep cell alive for hours if using external harnesses (OpenCode / Claude Code / Cline / Continue / Cursor)
# import time
# while True:
#     time.sleep(60)
